# Rebuild our quantum circuit submission

Google Quantum AI's [Verifiable Quantum Circuit Compilation competition](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation)
asks us to make three quantum calculations cheaper while preserving their answers.
**invsqrt** computes inverse square-root arithmetic. **dqi** decodes an error-correcting code.
**dfthcsos** prepares a step in a chemistry simulation.

A higher ratio means a lower cost than Google's reference. Our recorded local ratios are
**13.16× / 14.23× / 1.0×** (geometric mean about **5.72**); the same submission received
**5.68935 (about 5.69) on the public leaderboard**. These are historical results, not
a claim that this run reproduces the hidden evaluation's score.

This notebook rebuilds submission-003 from source, compares every output byte by SHA-256,
and runs fresh checks using the competition verifiers. The final file is `submission.zip`.
No earlier notebook output or verifier receipt is used.

## Run this notebook

Attach only the competition data. The default path is
`/kaggle/input/google-verifiable-quantum-circuit-compilation`.
For a local run, set **VQCC_DATA** to the extracted competition package root (the directory
containing `kickmix`, `verifiers`, and `sample_submission`). This is the only data-root setting.
The packaged native library requires **CPython 3.13 on Linux**. NumPy and SciPy must be
available; this notebook installs nothing and makes no network request. It prints their
versions and fails if the source, native library, verifier, generated files or archive differ.

The generator source is embedded verbatim from [c8bbfbde18bb27f4f89aac90f4b62437877b60f4](https://github.com/clduab11/quantum-hackathon/tree/c8bbfbde18bb27f4f89aac90f4b62437877b60f4),
a commit on `main`. Every embedded file has its own SHA-256 below. This avoids a runtime
clone and works with internet disabled. The notebook builder extracts these files with
`git show`; its source remains beside this notebook in the repository.

Verification defaults to **full**: 64 invsqrt shots for each of 3,133,452 inputs,
102,400 dqi shots, and 64 dfthcsos shots per address. Budget roughly 30 minutes for
verification plus the build; actual times are printed. If a confirmed kernel time limit
prevents full verification, set `VQCC_VERIFY_MODE=sample` before execution: it uses
1 invsqrt shot per input, 1,536 dqi shots and 1 dfthcsos shot per address. A sample PASS
is only a smoke check, not full verification or a submission clearance. The public salt
is fixed once; no seed or salt search occurs. Four verifier workers bound CPU use.

In [1]:
import hashlib
from pathlib import Path, PurePosixPath


def check_hash(path, expected):
    path = Path(path)
    with path.open('rb') as stream:
        actual = hashlib.file_digest(stream, 'sha256').hexdigest()
    print(f'{path.name}: SHA-256 {actual}', flush=True)
    if actual != expected:
        raise ValueError(f'SHA-256 mismatch for {path.name}: {actual} != {expected}')
    return actual


def check_files(root, expected):
    root = Path(root)
    actual = {p.relative_to(root).as_posix() for p in root.rglob('*') if p.is_file()}
    if actual != set(expected):
        raise ValueError(f'File set mismatch: missing={sorted(set(expected)-actual)}, '
                         f'extra={sorted(actual-set(expected))}')
    for name, digest in sorted(expected.items()):
        path = root / name
        if path.is_symlink() or not path.resolve().is_relative_to(root.resolve()):
            raise ValueError(f'Unsafe output path: {name}')
        print(name, end=': ')
        check_hash(path, digest)


def restore_sources(root, sources, hashes):
    if set(sources) != set(hashes):
        raise ValueError('Source inventory mismatch')
    for name, source in sources.items():
        path = PurePosixPath(name)
        if path.is_absolute() or '..' in path.parts or '\\' in name:
            raise ValueError(f'Unsafe source path: {name}')
        if hashlib.sha256(source.encode('utf-8')).hexdigest() != hashes[name]:
            raise ValueError(f'SHA-256 mismatch for vendored source: {name}')
    root = Path(root)
    root.mkdir(parents=True, exist_ok=False)
    for name, source in sorted(sources.items()):
        path = root / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(source.encode('utf-8'))
        print(f'{name}: SHA-256 {hashes[name]}')

In [2]:
CONFIG = {'commit': 'c8bbfbde18bb27f4f89aac90f4b62437877b60f4', 'repository': 'https://github.com/clduab11/quantum-hackathon', 'manifest': 'ledger/manifests/candidates/pra228-dqi-invsqrt-572.json', 'manifest_sha256': '5d76c29f18a43c54c4402385e3c37cd07c18316c8a6e327d4479d12eddd6a8b9', 'files': {'dfthcsos/angle_k_0.kmb': '7bfb3983accc6ff97d6ab41a88deae924f1d1c97cf30e60879d0bff5688a8e25', 'dfthcsos/angle_k_1.kmb': 'a04b6d0d57ce851b123af265066aaab2ab33353ea049d26acd534727466d72b6', 'dfthcsos/angle_k_2.kmb': '5c1b8987bc3fa1a4b8607e470f8c8f1392c22e7a7dbcb228f0ce9ec986e18058', 'dfthcsos/angle_k_3.kmb': '16ac67cdcaf3e2c6abdd98c4138ff1ae8a56cfc4ae15622bcdb3d3a3ba1e22cf', 'dfthcsos/angle_k_4.kmb': 'd21cdd4c7e94196654aab15ebcabfe10ce8d343151a5e8dc3672e76684b6628d', 'dfthcsos/angle_k_5.kmb': '63b8d4cba36955e6098321f3b2ef3e34401dd276fa629b2e490d043b1743880b', 'dfthcsos/angle_k_6.kmb': '5bf740428e661d035fda860142578f421969e35e0df2e4e024fbf01a7340b378', 'dfthcsos/angle_k_7.kmb': '606009582111a22bd549b795c8d115ac8bbeb124bc8da24a80380eca6d34163d', 'dfthcsos/angle_k_8.kmb': '57c7cc51f5922fff6e95d76e1f723d4c2d029a241da643b95684d6322157a8fe', 'dfthcsos/angle_k_9.kmb': '22cbe2828988ff954fc07fd971a25e6bc04a02c302bf2cc642e4c3535daaad65', 'dfthcsos/submission.npz': '14d5cc25eb3bf82a9ccb87c811db28d127bef6d28697dd190dc1e875de3bfe69', 'dqi.kmb': '529b075009f7758abeda83a63407f0c1feeb51dd70631ae4335dfd28721b7e08', 'invsqrt.kmb': '75e8a0e2341bb3e8f8ded1ec1d53e24437410c69e887da9daf7d49bfcc28c2dd'}, 'zip_sha256': 'b6709a4692802091f169332630ef54f0c0d9ffe3a6bcd26179e278c2189975ba', 'zip_bytes': 91399451, 'kickmix_sha256': 'c66b3216c4029e0cdfd09054306aa0feadf53ab2f50c1164e37f752874e5005e', 'verifiers': {'dfthcsos': '0775be7a528922367caebe9e4d051d6199811f1aefeaa4b793eaaa4621b8ec9d', 'dqi': 'f14274bc0943fb9769ddb01131be6063191a15009fe1caf1d4bdd9fd25194e65', 'invsqrt': 'ddef520dec6b68d1cf0a6f328ddd36d79eb84570b9192cf3f7acaf073f06733a'}}
SOURCES = {'LICENSE': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2026 Praxen / Chris Dukes\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'ledger/__init__.py': '', 'ledger/archive.py': '"""Submission zips: safe extraction and deterministic builds.\n\n`safe_extract` applies the archive checks from the PRA-171 inventory (§4) before anything reads\nthe files: no duplicate paths, no absolute or `..` paths, no backslashes, no links or special\nfiles.\n\n`build_zip` writes a byte-reproducible zip (sorted entries, fixed timestamps and permissions), so\nG5 can compare the SHA-256 of a rebuilt zip with the one in the manifest. Every candidate zip is\nbuilt with it.\n"""\n\nfrom __future__ import annotations\n\nimport shutil\nimport stat\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nFIXED_DATE_TIME = (1980, 1, 1, 0, 0, 0)\nFILE_MODE = 0o644\n\n\nclass UnsafeArchive(ValueError):\n    """The zip contains a path or entry type that is not allowed."""\n\n\ndef check_members(archive: zipfile.ZipFile) -> None:\n    members = archive.infolist()\n    # Compare normalised paths: "a/b", "./a/b", "a//b" and "a/b/" all extract to the same place.\n    names = [PurePosixPath(m.filename).as_posix() for m in members]\n    if len(set(names)) != len(names):\n        raise UnsafeArchive("zip has duplicate paths")\n    for member in members:\n        path = PurePosixPath(member.filename)\n        if path.is_absolute() or ".." in path.parts or "\\\\" in member.filename:\n            raise UnsafeArchive(f"zip has an unsafe path: {member.filename!r}")\n        mode = member.external_attr >> 16\n        if stat.S_IFMT(mode) not in (0, stat.S_IFREG, stat.S_IFDIR):\n            raise UnsafeArchive(f"zip contains a link or special file: {member.filename!r}")\n\n\ndef safe_extract(zip_path: Path, dest: Path) -> Path:\n    """Check every member, then extract into `dest` (which must not exist yet)."""\n    dest = Path(dest)\n    dest.mkdir(parents=True, exist_ok=False)\n    with zipfile.ZipFile(zip_path) as archive:\n        check_members(archive)\n        archive.extractall(dest)\n    return dest\n\n\ndef build_zip(src_dir: Path, dest_zip: Path) -> Path:\n    """Zip every regular file under `src_dir` deterministically."""\n    src_dir = Path(src_dir)\n    files = sorted(p for p in src_dir.rglob("*") if p.is_file())\n    for path in files:\n        if path.is_symlink():\n            raise UnsafeArchive(f"refusing to zip a symlink: {path}")\n    dest_zip = Path(dest_zip)\n    dest_zip.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(dest_zip, "w", compresslevel=6) as archive:\n        for path in files:\n            info = zipfile.ZipInfo(path.relative_to(src_dir).as_posix(), FIXED_DATE_TIME)\n            info.compress_type = zipfile.ZIP_DEFLATED\n            info.external_attr = (stat.S_IFREG | FILE_MODE) << 16\n            info.create_system = 3  # Unix, so the mode bits above are honoured\n            # Streamed, so the 886 MB dqi circuit is never held in memory.\n            large = path.stat().st_size >= zipfile.ZIP64_LIMIT\n            with path.open("rb") as src, archive.open(info, "w", force_zip64=large) as dst:\n                shutil.copyfileobj(src, dst, 1 << 20)\n    return dest_zip\n', 'notices/10-invsqrt.txt': '- The invsqrt generator under tasks/invsqrt/ is a port, not independent clean-room authorship. The module map is `tasks/invsqrt/PROVENANCE.md`. The code parent is "Quantum Circuits | Signed-Digit Memory" v3 by Artem1410 (anazemcev / ViberTeam), https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory?scriptVersionId=361502374 (v3), Apache License 2.0 as declared in that version\'s own source (SPDX header in cell 5, licence text and NOTICE in cell 20; PRA-218). Source snapshot SHA-256 79f6373302cb916a88f65a1b142024d4d2d0b14e41eb58bef3152ebd45ffd0ca, equal to the public version-pinned download. That notebook retains upstream notices for Sumit Chongder, Rudra Bhaskar, Alex Chilton, Kozykorpesh, and Kirill Tymchishin. Those notices are kept in tasks/invsqrt/signed_digit.py. Sumit run 361357149 is v1 (public version metadata, PRA-218); Dmitrii Gluzdov v6 is credited second-hand. Their originals are not loaded. Whether a later Sumit v2 exists upstream stays with PRA-205.\n- The 90-qubit Gray curvature schedule in tasks/invsqrt/schedule90.py is a port of Kozykorpesh, "Quantum AI LB score", V3 run 361677445, https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445 , Apache License 2.0, as that method is used by the attached Apache-2.0 notebooks. The pruned shift is the Dmitrii Gluzdov V6 boundary rule that schedule names (run 361592338). His notebook is not an input. The two Davio reload orders are method parameters from anazemcev, "Quantum Circuits | Source & Exact Compilation" v4, Apache License 2.0, attachment SHA-256 4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb.\n- The affine Toffoli/CSWAP rewrite in tasks/invsqrt/affine.py is a port of Kozykorpesh, "Quantum AI complete-domain affine compilation" v2, Apache License 2.0, attachment SHA-256 81befe1e69ee286271fa2eb8acf4bb760a727853bf966945318d918482b1ab24, and the measurement-uncompute rule from anazemcev v4. The peephole in tasks/invsqrt/peephole.py is a port of that compilation\'s involution pass.\n- Rotation calibration in tasks/invsqrt/calibrate.py is our scipy linprog search. The slack-maximising drop is credited to LightningV08 and to Mizero Lucky Gall. Mizero v1 has no notebook-wide licence (PRA-194 public-notebooks; PRA-199 item 2) and is reference-only: it is not loaded and it is not a shipping parent. Alex Chilton v8 is the same: credited where the signed-digit parent retains his notice, not loaded, not a licence parent. No calibration file is an input.\n- Coefficient triples are produced by tasks/invsqrt/fit_coefficients.py from the mathematical target (Lawson minimax, then an integer search whose start is the rounded Lawson coefficient). Provenance of that table is `regenerated; dataset-adapted calibration; licence unresolved`. The calibration is adapted from the dataset\'s `code/domain_tables.py` (SHA-256 59cf4d98899433dcc49d9ce512293be20d33da5334211c20e35323555994a4fb); the output equals its `code/coefficients.json` (SHA-256 98bd211f22b158bf05a4d2d934ab966e5e97e54d8307b82e2ca58910c6830763) 256 of 256 rows. Source: kozykappa/quantum-ai-dataset, version 1, datasetVersionId 21136269, https://www.kaggle.com/datasets/kozykappa/quantum-ai-dataset?datasetVersionId=21136269 . Kaggle licence label: "Other (specified in description)". Rights for both files remain unresolved (D-PRA220-01). The Kaggle coefficient dataset version 21136269 and `code/coefficients.json` are not read and are not the optimiser start. Toolbox `fusion/*.npz` and `selected/invsqrt.kmb` are not inputs.\n- `tasks/invsqrt/reference.py` is the competition package\'s reference Newton-Raphson circuit script (`verifiers/invsqrt/nr_reference_circuit.py`), Apache-2.0, Copyright Google Quantum AI authors of that package. Praxen changed the command line (`--out` writes `invsqrt.kmb`). It rebuilds the reference submission (submission-001) for G5 under `configs/gates/reference.yaml` and is not a parent of the Praxen candidate. https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/data\n', 'notices/20-dqi.txt': '- `tasks/dqi/reference_circuit.py` is the competition package\'s reference Reed–Solomon decoder (`verifiers/dqi/dqi_reference_circuit.py`), Apache-2.0, Copyright Google Quantum AI authors of that package. https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/data\n- `tasks/dqi/generate.py` is a port of a Berlekamp–Massey + Forney decoder, not an independent clean-room work. The module map is `tasks/dqi/PROVENANCE.md`. Transitive code parent: Gu, Kemeny, Yelin, and Lukin, repository `andigu/uncompute`, commit `64b5a13aa249f1360846657392e6c66817b26d0f`, Apache-2.0 (https://github.com/andigu/uncompute). Those four names are the `pyproject.toml` authors. That repository has no NOTICE file and states no copyright year; none is added here. Its `LICENSE` is copied verbatim to `tasks/dqi/bm_forney/LICENSE` (SHA-256 `cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30`). Used upstream paths: `LICENSE`; `uncompute/classical.py`; `uncompute/gadgets/linear.py`; `tasks/dqi-reed-solomon/contract.json` (field modulus changed from 4331 to 4179); `tasks/dqi-reed-solomon/circuits/discovered/constants.json` keys `bm_windows` and `psi_windows` only; `tasks/dqi-reed-solomon/circuits/discovered/circuit.py`; `tasks/dqi-reed-solomon/circuits/discovered/gen_constants.py` (formula section). Not used: `search.c`, `README.md`, `profile.pdf`, `circuit.ops.xz`, `liveness.json`, and the paper text and figures. Bibliographic citation only: Gu, Kemeny, Yelin, and Lukin, arXiv:2610.22367. Immediate code parent of the kickmix schedule: the identical `src/gen_dqi_v6` embed in Artem1410 (anazemcev / ViberTeam), "Quantum Circuits | Source & Exact Compilation" v4, Apache-2.0, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation (attachment SHA-256 `4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb`) and in Kozykorpesh (kozykappa), "Quantum AI — Complete-Domain Affine Compilation" v2, Apache-2.0, https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation (attachment SHA-256 `81befe1e69ee286271fa2eb8acf4bb760a727853bf966945318d918482b1ab24`). That embed attributes the memory schedule to Dmitrii Gluzdov\'s V6 schedule; his notebook and `femoco_public_words.npz` are not inputs. The shipped borrow route is the notebooks\' affine loan. Multiplication tables are regenerated for modulus 4179. Zenodo 10.5281/zenodo.17301475 is not an input. No k256.dev circuit or numeric target is an input.\n- anazemcev v4 cell-45: Retained opening notice from cell 45 of "Quantum Circuits | Source & Exact Compilation" v4, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation , attachment SHA-256 4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb: "Quantum Circuits resource labs (2026). Original experiments, explanations and added implementations: anazemcev / ViberTeam. Released under the Apache License, Version 2.0, subject to upstream notices." The directly copied src/gen_dqi_v6 embed supplies tasks/dqi/bm_forney/; tasks/dqi/generate.py adapts its driver and streamed sweep. The existing Gu notice and module map above remain applicable. The following credits are inherited through that embed, not claims that Praxen directly copied those earlier notebooks. Unrelated chemistry, numerical dataset and compiled-input claims in the original NOTICE do not describe inputs to this DQI port.\n- Gluzdov field/adapter: Cell-45 attribution (cell 45 of the hash-pinned anazemcev v4 notebook above), retained for tasks/dqi/bm_forney/gu_formula_derivation.py, port_runtime.py and gu_port.py: "Dmitrii Gluzdov: competition-field formula derivation, Kickmix adapter, record-reading phase sweep and readable source-built portfolio. Apache-2.0. Adapted source V3, run 361529794: https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361529794 . The separately scored V1, run 361509836, returned 6.12847 for submission 57567302. V1 and V3 contain identical embedded generator modules. https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361509836 ." Version identity and scores in this passage are the immediate parent\'s historical attribution, not newly verified measurements. Our field tables are regenerated for modulus 4179; port_runtime.py\'s import and field checks are modified as recorded in the module map.\n- Kozykorpesh symbolic lowering: Inherited through anazemcev v4 cell 45: "Kozykorpesh V3/run361677445, Apache2.0: https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445 ." The applicable DQI credit is "symbolic DQI lowering, and the source context for the portable adapters used in V4." It maps to tasks/dqi/bm_forney/symbolic_runtime.py and its use by tasks/dqi/generate.py. This is the transitive lowering lineage; the identical immediate embed in Kozykorpesh\'s Complete-Domain Affine Compilation v2 remains credited above. The original passage\'s Gray-curvature, signed-Givens, difference-bank and scratch-pebbling credits concern other tasks and are not DQI input claims.\n- Gluzdov V6: Inherited through anazemcev v4 cell 45: "Dmitrii Gluzdov V6/run361592338, Apache2.0: https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361592338 . Compact inverse-norm schedules, template-word loans and streamed sweep, building on the credited Gu decoder and earlier competition-field port." This schedule lineage maps to tasks/dqi/bm_forney/round2_gadgets.py, borrow.py and gu_port.py, and the streamed-sweep integration in tasks/dqi/generate.py. The directly used source is the immediate notebook embed; Gluzdov\'s notebook and FeMoCo numerical dataset are not inputs. Our borrow.py is the modified affine-only extract described in the module map.\n', 'notices/30-dfthcsos.txt': '- PRA-232 module provenance map: `tasks/dfthcsos/qrom.py` is a Praxen reimplementation of truncated algebraic-normal-form (ANF) loading, banked low-address differences, measurement-based ANF erasure and selective ancestor pebbling. Method-only notebook parents (read from the SHA-256-checked source attachments on PRA-194; never executed): kozykappa, "Quantum AI — Complete-Domain Affine Compilation", v2, https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation, Apache-2.0, source SHA-256 81befe1e69ee286271fa2eb8acf4bb760a727853bf966945318d918482b1ab24 (six traversal wires, `gauge_qrom.py`); anazemcev / ViberTeam, "Quantum Circuits — Source & Exact Compilation", v4, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation, Apache-2.0, SHA-256 4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb (`banked_lab.py`, exact ANF phase repair); magmacot, "Quantum Circuits — clean workspace reuse", v2, https://www.kaggle.com/code/magmacot/quantum-circuits-clean-workspace-reuse, Apache-2.0, SHA-256 f3f393b5c4958f7288f69f68cd7762a62bfcc697d94d7d19607e67ff7ed20d9a (pebbling at prefix degree 4). The banked method\'s upstream attribution includes Kozykorpesh / kozykappa, https://www.kaggle.com/code/kozykappa/quantum-ai-compact-arithmetic-banked-qrom?scriptVersionId=361433617, and Sumit Chongder, https://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149, read through Kirill Tymchishin\'s QROM + Tower42 v10; retained as credited by these parents, not independently fetched or executed.\n- PRA-232 precision experiment: ahmedberatozer, "Quantum AI — DF-THC Group Compression", v1, https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression, Apache-2.0 notebook code, source SHA-256 b221781e9bf8c1c8da8c5251c6556b294ed07def5e57f5a1ea038addbf7a57df, motivates testing 10-bit angles. `qrom.quantize_words` uses nearest-even rescaling of Google\'s baseline integer words, not that notebook\'s trained factors or quantization optimizer. No claim is made that its physical result transfers to Google\'s baseline.\n- `tasks/dfthcsos/circuit.py`, `experiment.py` and PRA-232 tests are Praxen orchestration and verification code. The existing `generate.py` supplies Google\'s attributed batch-XOR schedule and pinned baseline input. `circuit_checks.py` calls Google\'s unchanged verifier math/resource model; it reads the circuit register width and reports static cost estimates. The only numerical input is Google\'s Apache-2.0 `sample_submission/dfthcsos/submission.npz`, SHA-256 14d5cc25eb3bf82a9ccb87c811db28d127bef6d28697dd190dc1e875de3bfe69. No `femoco_public_words.npz`, `dfthc_group_32_60_30.npz`, toolbox selected circuits, chemistry directory or k256 artifact was loaded. Signed-gauge factor changes and factor refitting are not implemented here.\n- `tasks/dfthcsos/generate.py` ports the package\'s ordinary-QROM ANGLE builder (`verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py`), Apache-2.0, same authors and URL. The factorization bytes are the package\'s `sample_submission/dfthcsos/submission.npz` and are not committed.\n- PRA-198 uses Google\'s Apache-2.0 starter kit v4, https://www.kaggle.com/datasets/sohier/verifiable-circuit-compilation-starter-kit/versions/4 (package author Guang Hao Low), as separately declared code and input parents. Original downloaded files and notices are retained unchanged; exact baseline, integral, fixed-shift, encoder and verifier pins are in `ledger/manifests/inputs/pra198-google-v4.json`. Outputs are labelled "own-generated, derived from Google baseline".\n- D(w)-penalized structure refit and norm-ranked group pruning: Ahmed Berat Ozer, https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression, v1 described in PRA-194 and v2 source inspected on 2026-10-25, Apache-2.0 (`src/dfthc_train.py`). PRA-198 reimplements these methods with analytic NumPy gradients and SciPy CPU L-BFGS-B. No public factor data, words, derived target, warm start or checkpoint is used.\n- Effective-norm / Gram-gauge and banked / six-wire QROM method context: Kozykappa, Complete-Domain Affine Compilation v2, https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation; anazemcev, Source & Exact Compilation v4, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation; Apache-2.0 per PRA-194 / PRA-199. PRA-198 ports no Gram-gauge or banked-QROM code; its cost model is ordinary QROM. The module provenance map is `tasks/dfthcsos/FACTOR_PIPELINE.md`.\n', 'tasks/__init__.py': '"""Competition task packages."""\n', 'tasks/dfthcsos/generate.py': '# SPDX-License-Identifier: Apache-2.0\n# ANGLE circuit builder ported from the competition package\n# verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py\n# (Apache-2.0). Praxen wrapper: --out writes dfthcsos/angle_k_*.kmb and\n# copies the packaged reference factorization.\n#\n# The public 1.44× factor (Signed-Digit Memory v3, NPZ SHA-256\n# c5b4165e7b1dcacf1d287746642414a9ba6086b4279e5ede2bb73e21e6aebd98)\n# is not used. That notebook is Apache-2.0, but it copies the factor\n# from an earlier archive and states that the starting chemistry factors\n# originate with k256.dev, whose generator and licence are not available.\n# R10 forbids shipping that file. This generator rebuilds the package\n# reference (score target 1.0), not the 1.44× lineage.\n\n"""Write the reference dfthcsos submission from the packaged factorization.\n\nThe factorization bytes are the competition package\'s sample ``submission.npz``\n(Apache-2.0), checked by SHA-256 and copied. The ANGLE circuits are rebuilt\nhere from its ``angle_words`` with the packaged ordinary-QROM construction\n(``b_rot`` 12, ``J`` 8). ``.kmb`` files are never committed.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport os\nimport shutil\nfrom collections.abc import Sequence\nfrom math import ceil\nfrom operator import index\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport numpy.typing as npt\n\nDEFAULT_DATA = Path(\n    os.environ.get(\n        "VQCC_DATA",\n        "/home/cld-main/work/vqcc-data/google-verifiable-quantum-circuit-compilation",\n    )\n)\nREFERENCE_NPZ_SHA256 = "14d5cc25eb3bf82a9ccb87c811db28d127bef6d28697dd190dc1e875de3bfe69"\nREFERENCE_B_ROT = 12\nREFERENCE_BATCH = 8\n# Packaged sample circuits. A match means this builder reproduced them.\nREFERENCE_ANGLE_SHA256 = {\n    0: "7bfb3983accc6ff97d6ab41a88deae924f1d1c97cf30e60879d0bff5688a8e25",\n    1: "a04b6d0d57ce851b123af265066aaab2ab33353ea049d26acd534727466d72b6",\n    2: "5c1b8987bc3fa1a4b8607e470f8c8f1392c22e7a7dbcb228f0ce9ec986e18058",\n    3: "16ac67cdcaf3e2c6abdd98c4138ff1ae8a56cfc4ae15622bcdb3d3a3ba1e22cf",\n    4: "d21cdd4c7e94196654aab15ebcabfe10ce8d343151a5e8dc3672e76684b6628d",\n    5: "63b8d4cba36955e6098321f3b2ef3e34401dd276fa629b2e490d043b1743880b",\n    6: "5bf740428e661d035fda860142578f421969e35e0df2e4e024fbf01a7340b378",\n    7: "606009582111a22bd549b795c8d115ac8bbeb124bc8da24a80380eca6d34163d",\n    8: "57c7cc51f5922fff6e95d76e1f723d4c2d029a241da643b95684d6322157a8fe",\n    9: "22cbe2828988ff954fc07fd971a25e6bc04a02c302bf2cc642e4c3535daaad65",\n}\n\nSELECTION_REGISTER = "selection"\n\n\ndef word_register(j: int) -> str:\n    return f"word_{j}"\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1 << 20), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef batch_deltas(angle_words: npt.ArrayLike, batch_size: int) -> npt.NDArray[np.int64]:\n    """``(K, N+R*B, J)`` tables. ``delta[0] = batch[0]`` and later batches are XORs."""\n    words = np.asarray(angle_words)\n    if words.ndim != 2 or not np.issubdtype(words.dtype, np.integer):\n        raise ValueError("angle_words must be a 2-D integer array (N + R*B, N - 1)")\n    if np.any(words < 0):\n        raise ValueError("angle_words must be non-negative")\n    batch_size = index(batch_size)\n    if batch_size < 1:\n        raise ValueError("batch_size must be a positive integer")\n    num_vectors, num_angles = words.shape\n    num_batches = max(1, ceil(num_angles / batch_size))\n    padded = np.zeros((num_vectors, num_batches * batch_size), dtype=np.int64)\n    padded[:, :num_angles] = words\n    batches = padded.reshape(num_vectors, num_batches, batch_size).transpose(1, 0, 2)\n    deltas = batches.copy()\n    deltas[1:] ^= batches[:-1]\n    return deltas\n\n\ndef build_angle_k_circuit(delta: npt.ArrayLike, brot: int):\n    """Ordinary QROM for one ``(A, J)`` XOR-difference table."""\n    import kickmix as km\n\n    delta = np.asarray(delta, dtype=np.int64)\n    brot = index(brot)\n    if not 1 <= brot <= 63:\n        raise ValueError("brot must be an integer between 1 and 63")\n    if delta.ndim != 2 or delta.shape[0] < 2:\n        raise ValueError("delta must have shape (A, J) with A >= 2 addresses")\n    if np.any(delta < 0) or np.any(delta >= np.int64(1) << brot):\n        raise ValueError(f"words must lie in [0, 2**{brot})")\n    num_rows, batch_size = delta.shape\n    num_bits = (num_rows - 1).bit_length()\n    builder = km.CircuitBuilder()\n    selection = builder.create_quantum_register(num_bits, SELECTION_REGISTER)\n    words = [builder.create_quantum_register(brot, word_register(j)) for j in range(batch_size)]\n    ancillas = builder.alloc_qubits(num_bits - 1) if num_bits > 1 else []\n    bits = [selection[i] for i in reversed(range(num_bits))]\n\n    def load(row: int, control: Any) -> None:\n        for j in range(batch_size):\n            for bit in reversed(range(brot)):\n                if (int(delta[row, j]) >> bit) & 1:\n                    builder.cx(control, words[j][bit])\n\n    def node(control: Any, level: int, lo: int, hi: int) -> None:\n        if lo >= num_rows:\n            return\n        if hi <= num_rows and (hi - lo == 1 or np.all(delta[lo:hi] == delta[lo, 0])):\n            load(lo, control)\n            return\n        mid = (lo + hi) // 2\n        if num_rows <= mid:\n            node(control, level + 1, lo, mid)\n            return\n        bit, ancilla = bits[level], ancillas[level - 1]\n        builder.x(bit)\n        builder.init_and(control, bit, ancilla)\n        builder.x(bit)\n        node(ancilla, level + 1, lo, mid)\n        builder.cx(control, ancilla)\n        node(ancilla, level + 1, mid, hi)\n        builder.del_and(control, bit, ancilla)\n\n    half = 1 << (num_bits - 1)\n    builder.x(bits[0])\n    node(bits[0], 1, 0, half)\n    builder.x(bits[0])\n    node(bits[0], 1, half, 2 * half)\n    return builder.finish_circuit()\n\n\ndef build_angle_k_family(angle_words: npt.ArrayLike, brot: int, batch_size: int):\n    return tuple(build_angle_k_circuit(delta, brot) for delta in batch_deltas(angle_words, batch_size))\n\n\ndef load_angle_words(path: Path) -> npt.NDArray[np.int64]:\n    loaded: Any = np.load(path, allow_pickle=False)\n    with loaded as archive:\n        if "angle_words" not in archive.files:\n            raise KeyError(f"{path} has no angle_words entry")\n        words = np.array(archive["angle_words"])\n    if words.ndim != 2 or not np.issubdtype(words.dtype, np.integer):\n        raise ValueError(f"{path}: angle_words must be a 2-D integer array")\n    return words.astype(np.int64)\n\n\ndef reference_npz(data: Path) -> Path:\n    path = data / "sample_submission" / "dfthcsos" / "submission.npz"\n    if not path.is_file():\n        raise FileNotFoundError(f"packaged reference factorization is missing: {path}")\n    digest = sha256_file(path)\n    if digest != REFERENCE_NPZ_SHA256:\n        raise ValueError(f"reference submission.npz SHA-256 is {digest}, expected {REFERENCE_NPZ_SHA256}")\n    return path\n\n\ndef write_family(circuits: Sequence[Any], directory: Path) -> list[Path]:\n    directory.mkdir(parents=True, exist_ok=True)\n    paths = []\n    for k, circuit in enumerate(circuits):\n        path = directory / f"angle_k_{k}.kmb"\n        circuit.to_file(path, format="kmb")\n        paths.append(path)\n    return paths\n\n\ndef generate(out: Path, data: Path = DEFAULT_DATA) -> dict[str, str]:\n    """Write ``<out>/dfthcsos/`` and return relative path → SHA-256."""\n    source = reference_npz(data)\n    destination = out / "dfthcsos"\n    destination.mkdir(parents=True, exist_ok=True)\n    npz_path = destination / "submission.npz"\n    shutil.copyfile(source, npz_path)\n    circuits = build_angle_k_family(load_angle_words(npz_path), REFERENCE_B_ROT, REFERENCE_BATCH)\n    files = {"dfthcsos/submission.npz": sha256_file(npz_path)}\n    for k, path in enumerate(write_family(circuits, destination)):\n        digest = sha256_file(path)\n        files[f"dfthcsos/{path.name}"] = digest\n        expected = REFERENCE_ANGLE_SHA256.get(k)\n        mark = "match" if digest == expected else "DIFFERS"\n        print(f"{path.name}: sha256 {digest} ({mark})")\n    print(f"submission.npz: sha256 {files[\'dfthcsos/submission.npz\']}")\n    return files\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--out", type=Path, required=True, help="Submission root. Receives dfthcsos/.")\n    parser.add_argument("--data", type=Path, default=DEFAULT_DATA, help="Competition package root (VQCC_DATA).")\n    args = parser.parse_args()\n    generate(args.out, args.data)\n\n\nif __name__ == "__main__":\n    main()\n', 'tasks/dqi/PROVENANCE.md': '# DQI decoder provenance\n\nThis generator is a **port**. It is not an independent clean-room authorship.\nThe PRA-219 re-cut preserves the PRA-197 measurements and candidate manifest below\nas historical evidence; their recorded commits, gate results and three-task zip\nare not verification of this new head. The candidate is not queued or approved\nfor upload. Current R10 G5 requirements and candidate review gates still apply.\nThe reference configuration uses `reference_circuit.py --out DIR`; its CLI is\nretained from main so submission-001 does not switch to the BM+Forney decoder.\n\nNothing in this map is a shipment. The three-salt verifier has been run\n(D-PRA197-10). A ledger candidate still needs a clean-worktree G5 rebuild\nfrom a commit on GitHub, the PRA-203 re-check, and the PRA-204 sign-off\n(D-PRA202-05).\n\nZenodo record 17301475 (DOI 10.5281/zenodo.17301475) is **not** an input.\nNo CC-BY-4.0 attribution record is required for it on this generator.\n\n## Transitive code parent\n\n| Field | Value |\n| --- | --- |\n| Work | `andigu/uncompute` |\n| URL | https://github.com/andigu/uncompute |\n| Commit | `64b5a13aa249f1360846657392e6c66817b26d0f` (`tasks/dqi/bm_forney/UPSTREAM_COMMIT.txt`) |\n| Licence | Apache-2.0. Verbatim `LICENSE` copied to `tasks/dqi/bm_forney/LICENSE`, SHA-256 `cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30` |\n| Authors | Andi Gu, Alek H. Kemeny, Susanne F. Yelin, Mikhail D. Lukin (`pyproject.toml` `authors`) |\n| NOTICE | The repository has no NOTICE file. No copyright year is stated. None is added. |\n| Paper | Bibliographic citation only: Gu, Kemeny, Yelin, and Lukin, arXiv:2610.22367. No paper prose. No figures (`profile.pdf` is not copied). |\n\n### Used upstream paths\n\n| Upstream path | What was taken | Upstream SHA-256 |\n| --- | --- | --- |\n| `LICENSE` | verbatim | `cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30` |\n| `uncompute/classical.py` | verbatim → `gu_classical.py` | `c05c9e7a84a69e3ad0590b972642dcb30c2c29ca2613a821c82d09d53556ba62` |\n| `uncompute/gadgets/linear.py` | verbatim → `gu_linear.py` | `d0b89d494ae63ec28e8bac6f33b344c11aaaab378d0cec81683b1a5743ed25d3` |\n| `tasks/dqi-reed-solomon/contract.json` | same object, `gf_poly` 4331 → 4179 | `c2a257b333527bf8dabde3a3d2ec0464f0a3439efac9f1e8d2811659bc3bcf42` |\n| `tasks/dqi-reed-solomon/circuits/discovered/constants.json` | keys `bm_windows` and `psi_windows` only | `b5e93915f5636cf084456e2ede7cad3f195ddc869c4a4c9942aa98b8a58cc12e` |\n| `tasks/dqi-reed-solomon/circuits/discovered/circuit.py` | reached here through the notebook embed below | `d5f7dd55915e3426ab87ec75da5b76a37123cb996cd750f46ecd74be17a5ec30` |\n| `tasks/dqi-reed-solomon/circuits/discovered/gen_constants.py` | formula section only, through the notebook embed | `7e211d060edd242c54f5a34447edde4ae08eb8fda9111b37b1886db126abcee3` |\n\nNot used from that commit: `search.c`, `README.md`, `profile.pdf`, `circuit.ops.xz`, `liveness.json`, `meta.json`, `grader.py`, `reference.py`, and every other task directory.\n\n`bm_windows` and `psi_windows` in `windows.json` match those two objects in the upstream `constants.json`. The multiplication tables (`KT12`, `KT6`, `KT3`, `CUBE3Q`, `PAIR3`, `LIN`) are not copied. `gu_formula_derivation.py` regenerates them for modulus 4179 on every run.\n\n## Immediate code parent\n\nThe kickmix schedule is the `src/gen_dqi_v6` tree embedded by `%%writefile` cells.\nThose cells are byte-identical in both notebooks:\n\n| Notebook | Version | URL | Attachment SHA-256 |\n| --- | --- | --- | --- |\n| Artem1410 (anazemcev / ViberTeam), "Quantum Circuits \\| Source & Exact Compilation" | 4 | https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation | `4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb` |\n| Kozykorpesh (kozykappa), "Quantum AI — Complete-Domain Affine Compilation" | 2 | https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation | `81befe1e69ee286271fa2eb8acf4bb760a727853bf966945318d918482b1ab24` |\n\nBoth state Apache-2.0. PRA-194 read the sources. The anazemcev v4 markdown is the base of that embed. Kozykorpesh v2 carries the same decoder bytes. The shipped loan is that embed\'s affine route (`DQI_BORROW=affine`).\n\nDmitrii Gluzdov\'s V6 memory schedule is the schedule those notebooks name. His notebook is not vendored. `femoco_public_words.npz` is not an input (D-PRA202-03).\n\n## Module map\n\n"Copied material" is the notebook `%%writefile` body. "Trailing newline" means the committed file is that body plus one final newline. Hashes are of the committed bytes.\n\n| Module | Immediate source | Transitive source | Licence | Copied material | Modifications |\n| --- | --- | --- | --- | --- | --- |\n| `bm_forney/gu_classical.py` | notebook `ref/gu_classical.py` | `uncompute/classical.py` | Apache-2.0 | verbatim upstream, trailing newline | none. SHA-256 `c05c9e7a84a69e3ad0590b972642dcb30c2c29ca2613a821c82d09d53556ba62` |\n| `bm_forney/gu_linear.py` | notebook `ref/gu_linear.py` | `uncompute/gadgets/linear.py` | Apache-2.0 | verbatim upstream, trailing newline | none. SHA-256 `d0b89d494ae63ec28e8bac6f33b344c11aaaab378d0cec81683b1a5743ed25d3` |\n| `bm_forney/gu_port.py` | notebook `gu_port.py` | `discovered/circuit.py` | Apache-2.0 | notebook body, trailing newline | none relative to the notebook. The notebook already changed imports, the constants path, `final_bb_l`, the empty fixed-mask Z loop, and the borrow hook. SHA-256 `93a753e54e0ae28e0ca37a168f0c4e8a740f6e1ab82ff7106b1fdd24bb487250`. Its header mentions `REPORT.md` and `ref/gu_README.md`; those files are not in this tree. |\n| `bm_forney/gu_formula_derivation.py` | notebook `gu_formula_derivation.py` | `discovered/gen_constants.py` formula section | Apache-2.0 | notebook body, trailing newline | none relative to the notebook. The window census (`search.c` and the Psi search) is not in this file. SHA-256 `d55e238aa5c7ccb26b0d5cea8fdb911ecbb4ae54782df3e50950140164204dbb` |\n| `bm_forney/round2_gadgets.py` | notebook `round2_gadgets.py` | Gu algebra, schedule text in the notebook | Apache-2.0 | notebook body, trailing newline | none. SHA-256 `76dd907a2f19f16c38c49f6a631909cdb732d17fc7809d3797bb7c08e67c5645`. The sentence "developed in this workspace (OpenAI Codex, 2026-10-23)" is the notebook\'s own notice. It is not a Praxen authorship claim. |\n| `bm_forney/symbolic_runtime.py` | notebook `symbolic_runtime.py` | kickmix adapter in the same embed; schedule credited to Gluzdov V6 by the notebook | Apache-2.0 | notebook body, identical including newline | none. SHA-256 `f25047e0138bfe75cf177afd6d50e7694d047ed8cacf26ac83b7eaa6224058a4` |\n| `bm_forney/port_runtime.py` | notebook `port_runtime.py` | Gu `Builder` / `Journal` behaviour, reimplemented against kickmix in the notebook | Apache-2.0 | notebook adapter | **Modified.** Import paths point at `gu_classical`, `gu_linear`, and `contract.json`. `load_contract` stops unless `kickmix.GF2Field(12)` has modulus 4179 and primitive element 2. SHA-256 `cbfc4562cd110bce7f3d214ec179e02ec00e1165446504be28c0c3e44fbdf293` |\n| `bm_forney/borrow.py` | `borrow_template` in notebook `run_gu_port.py` | same embed | Apache-2.0 | affine branch of that function | **Modified extract.** Names were expanded. The notebook trie route is not copied. The loan stays an approximate support claim. SHA-256 `8fd2f83cc831d9d60b6a13d75e0a36dd5ed9033dfe9d8b5ec0ee899780cdfe39` |\n| `bm_forney/contract.json` | notebook `ref/gu_contract.json` | Gu `contract.json` | Apache-2.0 | Gu contract object | **Modified data.** `gf_poly` is 4179, not 4331. Every other parsed field matches the Gu file. SHA-256 `b5fa02899021142b7f265f1e18f366b0d6f55944023b3cf249f976c655d2097e` |\n| `bm_forney/windows.json` | the two window keys of notebook `ref/gu_constants.json` | Gu `constants.json` | Apache-2.0 | `bm_windows`, `psi_windows` | **Extract, not a rewrite.** Formula tables are omitted so they cannot be loaded. SHA-256 `a5dff27decbf564d6b810ece4dd345099eac5e4ba3d8260dcbf263f22dc9da99` |\n| `bm_forney/LICENSE` | notebook `ref/GU_LICENSE` | Gu `LICENSE` | Apache-2.0 | verbatim | none |\n| `bm_forney/UPSTREAM_COMMIT.txt` | notebook `ref/GU_COMMIT.txt` | the commit above | Apache-2.0 | the 40-hex commit and a newline | none |\n| `generate.py` | replaces notebook `run_gu_port.py` / `dqi_driver.py` | the modules above | Apache-2.0 | driver written for the `--out` contract | Sets `DQI_BORROW=affine`, regenerates tables, merges `windows.json`, and patches the streamed sweep so the symbolic builder forgets those wires. Does not read a `.kmb` or a `.npz`. |\n| `algebra.py`, `gates.py` | none | competition field and the G1/G2 hook contract | Apache-2.0 | new | Checks and gate hooks. Not decoder parents. |\n| `reference_circuit.py` | competition package `verifiers/dqi/dqi_reference_circuit.py` | Google Quantum AI, Apache-2.0 | Apache-2.0 | the 1.0 reference decoder | Kept beside the port. It is not the shipped BM generator. |\n\n`run_gu_port.py` itself is not committed. Only its affine `borrow_template` was extracted.\nThe notebook `dqi_driver.py` is not committed.\n\n## Forbidden parents (not inputs)\n\nThese names do not appear as loads in `tasks/dqi` Python or JSON:\n\n- `selected/dqi.kmb`\n- `source/decoder_input.kmb`\n- toolbox `dqi/*.npz`\n- `femoco_public_words.npz`\n- `dfthc_group_32_60_30.npz`\n\n## Two local builds, same bytes (not a candidate)\n\nBoth runs used `uv run python tasks/dqi/generate.py --out DIR` with `OMP_NUM_THREADS=1`. Neither read a `.kmb` or a `.npz`. The second run was after commit `c94611c` (the provenance commit). The circuit bytes matched.\n\n| Run | Output (not committed) | Size | SHA-256 | Q | Static Toffoli-like | Build |\n| --- | --- | --- | --- | --- | --- | --- |\n| 1 | `/tmp/pra197/out/dqi.kmb` | 823,297,489 | `529b075009f7758abeda83a63407f0c1feeb51dd70631ae4335dfd28721b7e08` | 1789 | 445229 | 156.5 s |\n| 2 | `/tmp/pra197/out-g5b/dqi.kmb` | 823,297,489 | `529b075009f7758abeda83a63407f0c1feeb51dd70631ae4335dfd28721b7e08` | 1789 | 445229 | 143.2 s |\n| 3 | `/tmp/pra197/out/dqi.kmb` (rebuilt 2026-10-24, same path) | 823,297,489 | `529b075009f7758abeda83a63407f0c1feeb51dd70631ae4335dfd28721b7e08` | 1789 | 445229 | 143.2 s |\n\nOperations in each log: 93,898,574. Field: modulus 4179, primitive element 2.\n\nPRA-194 records that same byte size for the anazemcev v4 `dqi.kmb` output. This tree did not read that file. Equal size is not a claim that the two files are the public artifact.\n\n## Three public salts (not a ledger G3)\n\n`/tmp/pra197/three-salt.jsonl` is the full verifier on the run-3 file. 102,400 shots, 4 workers. The circuit SHA-256 is the hash above. Value errors are 0 on every salt. Executed Toffolis are 445229.0, equal to the static count.\n\n| Salt | Phase errors | epsilon+ | R | Ratio vs reference dqi |\n| --- | ---: | ---: | ---: | ---: |\n| `PRA171_LOCAL_PUBLIC_FIXED_SALT_0001` | 3 | 0.000983909102617945 | 1.2055843238398525 | 14.22623531711937 |\n| `PRA171_LOCAL_PUBLIC_FIXED_SALT_0002` | 4 | 0.001016763883917165 | 1.209464049815962 | 14.18060031481362 |\n| `PRA171_LOCAL_PUBLIC_FIXED_SALT_0003` | 8 | 0.0011344950381555927 | 1.2230139975925918 | 14.02349140675107 |\n\nReference dqi cost used for the ratio: 24439465632206.816. Salt 0001 is the ledger G3 salt. Salts 0002 and 0003 are the diagnostic labels. They were not used to choose a circuit. This file is not a ledger candidate.\n\n## Clean worktree rebuild of the dqi file\n\n`ledger.benchmark_health.rebuild_in_clean_worktree` checked out commit `e1842978a78a485a1ae8c7fb9803c27a08dbbd2d` into a fresh worktree and ran `tasks/dqi/generate.py`. The output `dqi.kmb` matched SHA-256 `529b075009f7758abeda83a63407f0c1feeb51dd70631ae4335dfd28721b7e08`. Diffs: none. This is the dqi half of the G5 check. It is not a ledger G5: there is no submission zip, invsqrt and dfthcsos were not rebuilt, and this commit is not on GitHub. `git push` cannot authenticate here. The configured helper path does not exist, SSH public-key auth is denied, and the GitHub connection cannot publish an existing commit.\n\n## Ledger G0–G4 (not a shipment)\n\nCandidate `pra197-dqi-bm-forney`. Salt `PRA171_LOCAL_PUBLIC_FIXED_SALT_0001` only. Zip SHA-256 `b6709a4692802091f169332630ef54f0c0d9ffe3a6bcd26179e278c2189975ba`, 91,399,451 bytes, built with `ledger.archive.build_zip`. The zip is not committed.\n\n`dqi.kmb` in that zip is the file above. `invsqrt.kmb` is the current `tasks/invsqrt/generate.py` output, SHA-256 `75e8a0e2341bb3e8f8ded1ec1d53e24437410c69e887da9daf7d49bfcc28c2dd`. The dfthcsos files match `REFERENCE_ANGLE_SHA256` and `REFERENCE_NPZ_SHA256`. They were copied from the competition sample after that hash check. This run did not rebuild them.\n\nG0, G1, G2, G3, G4 and G5 PASS. G3 reports dqi value errors 0, phase errors 3, ratio 14.22623531711937, and a three-task geometric mean 5.721389477411995. G2 dqi used 1,536 shots and is not parity. G4 beats archive best `pra200-invsqrt-selected` (2.3612294080657024). G5 checked out `0b5858532fbef2ff3ce0be0b9e6e2c6d2cb8a28e` and rebuilt invsqrt, dqi and dfthcsos. All three file sets matched. The runner\'s enqueue step was not called. The candidate is not queued and is not shipped. PRA-203 and PRA-204 remain open. Decision rows: D-PRA197-12 and D-PRA197-13.\n', 'tasks/dqi/bm_forney/LICENSE': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'tasks/dqi/bm_forney/borrow.py': '# SPDX-License-Identifier: Apache-2.0\n# Modified extract of borrow_template from src/gen_dqi_v6/run_gu_port.py\n# in the Apache-2.0 notebooks anazemcev v4 and kozykappa v2 (same bytes).\n# This file keeps the affine route only. The notebook trie route is not copied.\n# The loan is an approximate support claim. The official verifier is the test.\n# No copyright year is added. See tasks/dqi/PROVENANCE.md.\n\n"""Route two high template words into a 24-qubit scratch loan."""\n\nfrom __future__ import annotations\n\nimport os\n\n\ndef borrow_template(builder, emitter, alloc, bm, d_width):\n    """Loan template words 35 and 36, then return a restore closure.\n\n    ``DQI_BORROW=affine`` selects one differing pivot and a six-AND ladder\n    per modulo-37 class. The other routes stay available for measurement\n    and are not the shipped default.\n    """\n    import gu_port as port\n\n    bank = bm.final_bb_l\n\n    def swaps(value):\n        slot = (value + 2) % 37\n        first = (slot + 35) % 37\n        second = (slot + 36) % 37\n        pairs = []\n        if first != 35:\n            pairs.append((35, first))\n        if second == 35:\n            second = first\n        if second != 36:\n            pairs.append((36, second))\n        return pairs\n\n    def route(reverse=False):\n        with builder.phase("borrow/unroute" if reverse else "borrow/route"):\n            mode = os.environ.get("DQI_BORROW")\n            if mode != "affine":\n                raise RuntimeError(f"shipped borrow route is affine, got {mode!r}")\n            for slot in range(37):\n                pairs = swaps(slot - 2)\n                if not pairs:\n                    continue\n                values = [v & 255 for v in range(-1, 69) if (v + 2) % 37 == slot]\n                if len(values) not in (1, 2):\n                    raise RuntimeError(f"modulo-37 class {slot} has {len(values)} values")\n                if len(values) == 2:\n                    left, right = values\n                    diff = left ^ right\n                    pivot = (diff & -diff).bit_length() - 1\n                    changed = [i for i in range(8) if i != pivot and (diff >> i) & 1]\n                    for i in changed:\n                        emitter.cx(d_width[pivot], d_width[i])\n                    bits = [d_width[i] for i in range(8) if i != pivot]\n                    target = 0\n                    bit_index = 0\n                    for i in range(8):\n                        if i == pivot:\n                            continue\n                        bit = (left >> i) & 1\n                        if i in changed:\n                            bit ^= (left >> pivot) & 1\n                        target |= bit << bit_index\n                        bit_index += 1\n                else:\n                    changed = []\n                    bits = d_width\n                    target = values[0]\n                with port.eq_test(emitter, bits, target) as control:\n                    ordered = reversed(pairs) if reverse else pairs\n                    for left_word, right_word in ordered:\n                        emitter.swaps(control, port.word_pairs([bank[left_word]], [bank[right_word]]))\n                for i in reversed(changed):\n                    emitter.cx(d_width[pivot], d_width[i])\n\n    route()\n    loan = list(bank[35]) + list(bank[36])\n    alloc.close("borrowed_template", loan)\n\n    def restore():\n        alloc.reopen("borrowed_template", loan)\n        route(reverse=True)\n\n    return restore\n', 'tasks/dqi/bm_forney/contract.json': '{\n  "_provenance": "Task-defining derived constants shared by grader.py and every circuit. grader.py re-derives all of them from its inline definitions and asserts equality with this file at import. c_stream_hex = counter-mode SHA-256 over c_tag (block i = SHA-256(tag || u32le(i))), truncated to 6143 bytes = the 49140-bit fixup vector c (bit k = bit k%8 of byte k/8; the top 4 bits of the last byte are unused); weight_cum_2p16 = the exact DQI weight law |w_k|^2 of Jordan et al. (2025), arXiv:2408.08292, at this instance (m=4095, l=35, p=2016/4096), quantized to a /2^16 grid by largest-remainder rounding (entry k = P(weight <= k) * 2^16).",\n  "gf_poly": 4179,\n  "m": 4095,\n  "n_synd": 70,\n  "b": 12,\n  "w_max": 35,\n  "c_tag": "kickmix/rs-decode-4095/fixup-c/v1",\n  "c_stream_hex": "0ab02275411d99401427df155f38df3d98dc8cd245edf6c545c44fbdfa580586048e10430b242a866358f792f5d4bf309aa3b5df457ec1be714f477bb3674d87bfe6f1e5be30e94aebbc685caf06703c256d6fde56cf1a71a44b26dd3a7f4d39f6ee5a39ed36d49a8e7c160a82ce95c221a9fbc6191b503d5c06976bb53878ace4318e6992c5f4932f6671be9cb705bf20019f44b90345f3b313d959710822c65345a2e2ec8bc270038e5582285d76fe1f53be1cccfe381d130665f33d1269c7aa3dd34878ea8d00369625db16895818731b0694f61f915d05c3f9b62b3008ce551248aa11fb39a8ea189aadb70f573e5f0ef48b8bb73acd01aad27897b0d5b18284284238aca81eed501a557a884a7091224220db6414a2b7c794c187ef6a883e7b375c5b0b0b1497d28aa0f35cfce019afeba54fbb98dc521625e100829b51b15495d8fdf1cfe2175d92668f20a62456ce4ef718b26360f2e83f44de3d452b8a131aac6dca7ef463c96510ca780ff5f76dd3e45894651052e7fae51261b65be4d020647db7f45b233d3bbb4d5941e0b7c303ceb613f3c2ed9b11127a1858f12416d6c957bc5cc58e7f1cb655790578fc64db0720c70190f95ed1c662b81b5e29405e9fa3d8e3ed252853155b5b487dd1c3c8b72771712cb8980b4844211d343db065d60d8e31040e0ebd116bd0a434b38eeede887faad8f4f2a616040b6d605648772342a892f8c571728216671979de87050e3d6a76888b24c3111f257fdb13e5ee92fcc2d851abe85ee3bd80c57404144c2a6fe4049a4a9e8ec74a4642cc8c8eb760b801f14692371add6298127eda4508e4ceb9cfd363d1b55bee8157b116f12c938d5b3f37e147c57f065155e7a4c290fbb4f616acabf0a8ade65294356083304276d56d35790d8bd9b7103f70276ff89d187c0fff73a8fe23cdab99d92f4a176725d43ee8043ce2bb852ec84f834419c4a2ef968aab729a62f09ca081e5041af453a53d7ab399c41cff68e56d30e04bdd4aca3eae09ea16ff96fc3f00fe200aee77e275a3b05f2d6c623b746950fd294fc60bb42826113566cab8358d1a435bdd025cb1905a301f7dd71957c9359f73feb93af6325b84a6628b5097d1ef7d6a8c05949ec66b7a682fcc69ce76c3d736d2e1db88c89c012af8565fd7cfbdcc1f9cb26c73d6af71a4020d6b5edab892539c62e98d81bd7ba114f960e45cc78ca2946d77cec0d232a8bf397e99d9e112c5a576887f718850303d3aba9d958b55b913c9d310a20ff7e946688e5f8f9f01230dc26900594e93581a93a08a2d294d7c9349dc211ef66b729810e3bac545d94aec168270882dbd225bce7af21585fd6d572ccb3ded7903f0211145fb0132d56c07a4834fdbe997c99ed61d9bf391754c75726fcf4896af7d205cabebafee5d8d34a9f7d091ce5557a7c984cbd73076dbc76526668476a9b55c610bb68614be2662ae4e9d4cd823c6fabd7ccf627e672e56dff4555987168ba7b87be4b69cfcd78f6e3a6a2d6faf054b2c5787bbcc92688bdbe59ba6f0925b9d38922df8398022f27dac378b1e19b616eb65215350707341357c5cc4f006b3e12b073a03a1bd8edd06af7e31562effd6ba689130ea72f39106d075d09312625897741a7a82f00a0f5f0b91618ed22da815ad6a3e446e7621b9889c96eedc287785751985ee5622ecd80bfe2324893ba8656c290b3faa2b7c8726515f5e605809a7ae4b6e04fb3fd6ef0165eb01b0b0e6d8dc38b3bb7e7f18a6c983611e8ee601ce55a8897aa244a68d3ddcc1241a28d039874c1c352fbd99e505eb00d5e490d2b033fd4e8472a94a8ef947766e5881e1bba7513e899785b82148119f06676350d4ddb8f35905c53b0e05781d5ea6aa819c5152e4b9d3e4662dfbb5295467af2246af9b3743229f5f4a13edaae3860a99309a7fbd7a5c91d9b373f0cc597e30b9194cb5b3d491c27c6cb227f71939d0d836c588830e39e3e6b8436bcc9af6c76f07513b37cdf63ef37e1c1045a5d333a0c19c3960d02e9cbc2c0b1c196bab8c8ebdb7267dc1daa55e4820a3f351c0aa0e97850d9c8bc887faf4805cff561e93b4c1a25bdb8dd100180360855298b7f7e64bdcfc158a5fef4bdd811832c8e6266accde6f332df161e7ddb2d1627546c966da622d0a6a1a045251dda529b4e10fdd14f9f3fc94ea45ea3b29a56564ec9d30fd8c0f736b5ab7d7a1a45175347d391328170d559dc0da5c6d661a4513bfab803fa1eed3d2c658575ac5de52e6ae74a1addfdcb2d598e96826fd625147491b9bfa2ba270d531600ce2791e32e7fdf4fd7ec1152ee6596894c6e917c6f94e1e79094a82bdac1d5ae4b10c7062256cbc754c2fdc6546080c7242eeca80418cb10fb089f721e9e3552b2f9d0686533d9113d338883f246ac8e5e8028108797952b041ed934c65ec9d541c0d4766682e260e735627b2463d899c1bffe99c9337cf959e2947d01a91cd2d88951029cbc4726991190d1fae04117573905c8619b0afaca7163ef533e9c3bf8c279602488cdc0636c114e28697799b0242cbb310c3f557b0d9e6428ebe3d635d201d5ee183ece5400056943ba890e490a219f4e33bed5809bfea2c74138bfcb3aec67fca982aca9890de419c6054cb0c95f0504c01000fd05e86efab211b4fc7a722712c4f483d72b440065ed3869230623924e3254b2ebb7ffa29f4beea6bba9dc32f3572f78b749fba1540009a0b657aae24fea3d77241ce786ac68618c44c84a19b5f650c1217be19568c28ce5a95316de929f113b876c480456e73eaf94d1c5e1abe81d1f70ade42b87df32f0f7dc56520e7cca6679efe320c0ec1340e9a3f249871f8c7bc36c2e57920e5a2fe96917adf1c9c56809a733e7179246429092cac1a9ca5ffe668fb23a69579de104d3e08e86ce1b0f7bed3abf800608fd02853221cdcafab1ff8dd6eb9aa6577af77df9b3e774a62094f23a6cda60b64a54d1577c1fe0526087b86b3aac84d67db4a937bac13469a330e1fc28974cd2d60c07221de0980b1d46a7102317a381f951c8f0029ab2199b0f5a3fbbb5947cd25c291bff0dd37b635628d2882ff3ed73ad8bdeff3c9444ff97ebe21bb543cc5c3f7f1a0608c5ae28b6a5b591f43895cc088a078ed465f0dd8b289219ee1d1d4df353087b2e96d18c400ce7a8ac5d1e30e3920c5aba753d67e198bd51c57fb90f20ccd518f44a3162213e50f67f712e16d5fab5cbe32f585a817b10cb3e68f53c71c04d9c5a400be6701178f1681a9dc073511118eb668819290697a180380eaae3b68af08fddefdfe233f8cdda37eb90ae6d625b0119d54722ba4ccf62d2b44b272aae1d98ab954130e59e7240cc72dd9566459a2af5dca2ad2663dc4b6b8a48dc68b33ac575ffbd46da82f1bc31223c7e280853a6a2c6667152a3875a6d77f41a54a1f9f6e518f50e6a4e1ae06e81e77fd5461fe87516818a9a67a388e6b8ef41b03d521fb1034a59ce4ada4037844b815f00fe1560aa615052a02c2ec5d288bb441b905aa8fd78c1978632a4288d010b4d2357ec48a2535b827ebab108f2eba026a136043874363916cbd300a02b7624163bd7d2dc4688de111a94d5bce2c0f9b332d3b0f647ca94359d80c7785709942fb553165eb24e09489ba8fcbcb8992b89ab24f715dab250fdb51ccfcf0c2f5aded499a106c45fabd22542baa0c722e78c3e8196136a8ff93c46a1bda8222b6119a90c61fce032b382c52b6f308160d6190c85dabd93b633291969ddcb30b13e3bbf401dcd656944523181086bf5cede9ce66e756c3190df3dcdf268fb9b07dc642d2497625c7f87625780a537318fb5f719d12c384d3baa14094de38c440784b0036ef5e4ec873486ccb35baeab9185f9f5a53227e2e61540c8df2db7428a74aab497e16f4ec9c6ff63083339ad26d2e674b33d7f737f3a391302fa8fa367e8b035d78bc9db74145c5e6d338c8706bf0f70f7f39402d6c29d69760985f4ab54c30b1df29c6d26d3d7ecfd02749718c1decc5ecd7f8b5b007d3a533568217aba921f1bb56bf3c0ec82f1f4ae28acab500338de9a704893b5a33965a295a37f11649a90ce06b37ca0086a7f600e92a3d92a0462726bef2b5cf16818d28ff44cbc6742c4d8c5db383d3e35767c0849797bc2d1d54b3d733128bcf61a2c1f81534abce1170909592c2a4318ab53dcdf6b2c722cf992502ee3d08d460540359df9120a36e0414cbd9e144ad4c62c51f8b5873c3d9c6d6c1ee2e7e58bc40a9eabed857f9c53e5c5174237f5ba8dbe14e6219fb2f85b6364f4c81c295852a54401b57ee9354589283509e880decb0e6aa301a1def95506622fc0d37b6b91ebf5483cf4ea53ec2d452a37b440825beb96070c9d5b9c01485ffa779ba8b06c308b5e50185e335cb4016c4ea86c77878b08fdda99c0bc7376c3845b8c563ea2baff72fe7bf4e5bf2a7522dc4f9527baeeaff5c77dc5b7faa9a731e15d261e264b5ed30c278553b620c43e75addc70145f0c08d5db037d14057e143e5ae55b3fd973f818b008c45b063ee7f0c4f28554af92d6881f1efbc612e6af829525b2dd0f1fe2dc8ab2e4c31deb697699e88ea4fb4d43eea3519bf91730d96a1b554e484d8d95b58be290897ff641b71fd2d786d5ce4062e78b28545b30874b9eb30d9d8b4189919a29ecbc470c907b21ad5964034bbbbcc608d665b028b75627d00c411477fa3fbc78c34727cad31de8db8789d547fbe4a9431c9e03bb61f9e2b993cf127d0bb8402d2da817e9ae35e86134a57d5ce7061f2628b4db18f5d61c4ca8e5955fa126419b45f108dd81969e197473ec06ec736b48b9dd8bb838149042cfd9d9fabfd6b75282936f2ba24ee84b85751a0b1f8892b7a29f0e4c4d35f4d3d6efad602f8794954811d6d59145b2cd0d3d9ee5a2bd88c82664a98aa6a137a6d5046c7c6b23ab544478771b14b8e147e6ecc9a3146b438fffd9c86c708eaf915afb0b05f7b33e666780a4345ee7782624cd3db51ae1dce6e079662ff726a2d54527407eb4b983b9b5592a765a74f5f35cd608b97dc534dec3cbefc1430182a568386ff09755a5034aa2a3b6ea291a0483651f75e8b92bbc3260097ab5121708df9369315e86cb1f1a0a666252263294569115fd5169f034c9ce46897d736d6616cc11d407866bdb0af5c826c534763e810b70b0cae816603fa99ff3148ae17ad2122048cd2384950842d8d4608b32aacc304d5ab4b93f038db83fc45fbc2205d33a780717fc75aaf463f3793952d9a01cc9bb5c7755e4817b2d81d6f0a7a069c2e8d034f26e75ff8f36fa35e82613048e899c022b3b5abe3e2228f41edb25dc7235090644bbcff1287861e6cb545f5903839132f9e397768fc5e3e9160bb188a10c79f511e201e12de36e1b237a47d46bd41293f61e53fa0706ca447d2883ccdea8bc2f8b53eda7d61523bc576ecb381b1139dd286eaaa79fcd22a911554391573776c3a8fbae6ebac203bd993787145b13540582a3e2150de032895e61f3544e1a32ba84bbf95e240d38fae0080528fb1ce9dfdba196e3939b5045c9d6e2ca3a4867e4facc9ef2d620330a235550a489692818542786e256264f3aeb72e565ae46e83f73354573122d7ebf9fe345bdb3d67069e2664c4cdc128bdc43809b1660239411809d4f1cad42a99821bdb111c6c4a6ba41b0145db04198f034803f56c3deee1f3a4334bd9ec37db9bea24c66f342be2512d07a959d1c00a7d36f4c44963df8b22b6b788fca5fccef9871d555af82b8b27a06497a60bf24052cfdbdca7d18731c4cc5c561ede0102fc3679699111c8def6ea7df0701983c3a3722b3fdd6438de336cf7421e2029e40ecbd63c8a2c2479607df55d68b8fcf62941fb3c0a3bc3c21e86fb4e6b4c058b5a5282607640000c2adffb79de8f25d3a25dabebc26b9fe5be6943cd854be634a4e07556a68dc7c999312d4e0f77eded93dc454d44028fe10345e778444bda4c7af6593b650ff63909740c66a34329453ee46eb5014b6ee339ae734eec6c065218f0f22eb5eb4fd2836b7175b9bffe4a85c23dad53e6ab3625ef316e34c31718f3171d38328a143bc28af58184054690e4f16e279a1e5de044c1ac8a1ff5518c13242ddbcbcb97e30c580356bd466f4156a98c66f9a24976765aa396043f3f76e2184e963892e5bb4288c181a48d50aca0da0e99642692948934fdd30d892a3ed225438253abce27472fe1e62c3f2e9bcd01dc34ada72f0ba0ee50210881e25a522ba2b40e5b55695f0e9a0e094957d33e326df57c3fa48ee6931253939b782b015dac6e591fbda9b78d5b83e08d3ac8cc638e0a802762292ce095e51b16ecceee95b5323ea8fc2254a74b5c737ec217301ea14cc44703dec852603056d6dab998e6c34f8f3276216bf10a2e114c56f11d471c49858a95c272dddfcd5c86ae19e4f516b5f7a0a0ab7f7d414de68c0167ea71bbfb1afcec51614a1c0e2c51a4196c6ed44834776d14eda80fc0b446c8c7870990a0c6d1840f3d224d23151bf7fd36bf7e859c6c4c75b1f83bd01171514a438971ffbe5691a9d5b26a121270243eb6183ee60ac4c0b386ad3ce4b5a6d8f2cdc7c61de41d278aa9c473470a54aa27a80d78e9e22e51225f7b47dc3ad97017cc571350a622a9f895f66690773f8770f724f5cd86af9ded84d26972c80b88afd8e02684ac49c41b7d8875daa700f75b56909eb0e4d7c07ce7fbc92d5686aaaab86d53e3b5e5ff626c9b15052f919071c095b044cc3c3ddf194abcc128c07012584c735058b93544f4f736059c37ef3b1c1203c82c33396523b5b3a48b193e6eaff66cf2c88ab8506cfcc90d1e84011d78aa201b63f75ab435eb3326a81edb1531f95dbc19defdce72a66db0530b290fd42d194abd3961af6af54e9ee4ce9ccf5f37f08ab3bb6a9187e4da74675ee1ed963be6dbafb355a106e1656073e433302ca784d59cc10285000cfda8466c27247c1fc4f35bd5daaf2727c6c8e15b10713ac46528be2bba8bb0c4c9bf57109750cef993e2691d12c15d226923027d6f38b735caf4842d2dd8328519591248e5e5e23a533975ffc71276678415c397bdf9b99aee7d853c66fb6ceb3fab015b05e5c51bc8adb2064c629ae9acdead63edde2f1f35f95f40dce4bbe537121356477ef6135da2039e08a1af7db68903ca2bcb14bba9963c5d9df629a0d80dd13162016ecec5ca8fb0794315e2f4293fb3e652e6ca659a13cf45286983da01110f5731577bd14ffb461ace093b5170b405bfed241a957950cc1f5fe3e3914293c86c7e9e786fd464637b4b5980cb99b2fd06e43a535dd665df1c903e9a059ae9e71562cf1ee5fa018b7b8f341a2cd19580d79f915f3fa1ae22ced811a628535f60b823969a3e9b99a926adb1f3696035876d07f149aee05d5b73a714854aa6a1d26ccc926eabd32fbe985e881ba461280c98eee75e87b9af58b447ef6b01da8b6703f832963ecb4ac70fc43098274fbc58e417bdcc17056eb2359cb2d81e439d658a81ba959147d385cd207b10c7d73cd59494d2e4c33f22e6e573d1ea6ec390f7f9c218df196536b5c71fbda6807f8847c6fb5a88ced5e53cac46ac1a9dce71328ce65d41e50d084cc585c431699be3af7587ff6ae1a0aa76da0c70cc0dc4aa7cf3bdf767cc40a23e38e4e2a307214bd3025fbb84876f9d764c08bce44776088ae9b02a502bddb463cfbb8b2eec454e0c1e5159ab5b7a872ec35cce3461ccebc03a666fbcea8d3afce5decc4f61375bee4125eaaf2386655ae2554f7323b1dc22b200da6d6cdb77b64bff6676aa9c45386050d8f6a7025e5b7e765661a06f6eee5e57a330601196a6476a76bca9c2563a4e8738fc183a44b541cc1c42d64856007af7fa2f71cae4504078d187ba875b01f3826e15e98c48946e5b68812268860126a31fc6661f8608d61f5bc44b556f5fd9223aed2f61d55c31c3569716bbc473711812563e566163a93be589a35e469711e1f3b356e5357def24ba087e77b16adf5df3054251731c318a97dd1494ac9b81dbed2050295f336515f4ef346e0ed172f519824a95319f3f6a017ddf495b014e24f37420c7b7125ea9d6766bdc2dafacc2d7aa8363894dc73c5d36008f563a692622c35a55fd100c1e59580fa75589d4fb77f27088b4257b0cb33a400d1be219b69f230b24b3397caaf5ead6d74de2493e5259ed4b1696ea8a9c47c9a1196e8b107282d30ee31527f5b5c7c8e225641b8d78228727e915080279b56c2687cc4b87734c17560911323eeb6dc0c79fa914166968f54099075349fd578dbde760b4642004f9276e5c0fd2a2e4e6f0107c463784c7d42f085232994bd70e924767bc8ccba72c7f4d19ebf8c7839f72911f5354f22b2a305da91ef5fb5c623eb1d55566b4e4d781e87a4df56eea7bd89d0d8c4e3e1ca647baa1c106751cdb66ddb72da2a69e9bb4b366515f3a6db3fa4c9d2bf097e3b5f14caadbf64b2bf119b1b951cd1379299e3af7823a8a4e03c12ef861ad48b13839867a8c3bdb7119cc79b0f0521533f52475dfab5d10c430cb1b57ee6ece34c3b085fbbdb4ae080c7f63a3596cc3088cad73e5fd2fa90d02311540ed63b07c1c3a0be14f88fbf5f5280d9c5ca02c0408f1a2be67837f1af3eb134e57fc8b1b30ac1be1",\n  "c_popcount": 24492,\n  "c_stream_sha256": "8fcd44e5a419ea29d84bb443d75f10dc9ac85dde2085b7851d92cd198eb3785a",\n  "weight_cum_2p16": [\n    0,\n    0,\n    0,\n    0,\n    0,\n    0,\n    0,\n    0,\n    0,\n    0,\n    0,\n    0,\n    0,\n    0,\n    0,\n    0,\n    0,\n    0,\n    0,\n    1,\n    4,\n    13,\n    41,\n    121,\n    333,\n    843,\n    1963,\n    4199,\n    8230,\n    14737,\n    24027,\n    35550,\n    47605,\n    57694,\n    63723,\n    65536\n  ]\n}\n', 'tasks/dqi/bm_forney/gu_classical.py': '"""uncompute.classical — classical math shared by circuits and their checks.\n\nNothing here touches a Builder: these are the pure-Python models and\ntable-builders whose OUTPUTS enter emitted circuits (lookup tables, linear\nlayers, test vectors, reference values): GF(2^n) scalar arithmetic and\nGF(2) linear-map helpers. One exp/log-table field (``GF2n``) serves every\nfinite-field circuit. ``poly`` is always the full reduction polynomial\nincluding the x^n term (e.g. 0x13 for GF(16)).\n"""\n\n\n# ---- GF(2^n) scalar arithmetic ---------------------------------------------\n\n\ndef _clmul_mod(a, b, n, poly):\n    """Return a * b in GF(2^n) mod ``poly`` (LSB-first schoolbook + reduce)."""\n    prod = 0\n    for i in range(n):\n        if (a >> i) & 1:\n            prod ^= b << i\n    for bit in range(2 * n - 2, n - 1, -1):\n        if (prod >> bit) & 1:\n            prod ^= poly << (bit - n)\n    return prod & ((1 << n) - 1)\n\n\nclass GF2n:\n    """GF(2^n) mod ``poly``, by exp/log tables over a primitive element.\n\n    ``gen`` is the smallest primitive element, so ``gen == 2`` (the element\n    x) whenever x is primitive. ``exp[k] = gen^k`` for k in [0, 2*order):\n    the table repeats once, so ``exp[log[a] + log[b]]`` needs no modulo.\n    ``log[a]`` is the exponent k with gen^k = a, for nonzero a. ``log[0]`` is 0, a\n    placeholder: callers test for zero first. Hot loops may index ``exp``\n    and ``log`` directly. The constructor raises ValueError when ``poly``\n    is not irreducible of degree n. Tables cost O(2^n), so n stays small.\n    """\n\n    __slots__ = ("exp", "gen", "log", "n", "order")\n\n    def __init__(self, n, poly):\n        self.n = n\n        self.order = order = (1 << n) - 1  # multiplicative group order\n        for gen in range(1, 1 << n) if poly >> n == 1 else ():\n            exp, x = [1], gen\n            while x != 1 and len(exp) < order:\n                exp.append(x)\n                x = _clmul_mod(x, gen, n, poly)\n            if x == 1 and len(exp) == order:\n                break\n        else:\n            raise ValueError(f"poly {poly:#x} is not irreducible of degree {n}")\n        self.gen = gen\n        self.exp = exp + exp\n        self.log = [0] * (order + 1)\n        for k, v in enumerate(exp):\n            self.log[v] = k\n\n    def mul(self, a, b):\n        """Return a * b."""\n        if a == 0 or b == 0:\n            return 0\n        return self.exp[self.log[a] + self.log[b]]\n\n    def pow(self, a, e):\n        """Return a^e for any integer e. 0^0 is 1; 0^e is 0 otherwise."""\n        if a == 0:\n            return int(e == 0)\n        return self.exp[self.log[a] * e % self.order]\n\n    def inv(self, a):\n        """Return a^-1. The inverse is totalized: 0 maps to 0."""\n        return self.exp[self.order - self.log[a]] if a else 0\n\n    def sqrt(self, a):\n        """Return the unique square root, a^(2^(n-1))."""\n        return self.exp[(self.log[a] << (self.n - 1)) % self.order] if a else 0\n\n    def trace(self, a, k=None):\n        """Return a + a^2 + a^4 + ... + a^(2^(k-1)).\n\n        ``k=None`` means k = n: the absolute trace, 0 or 1 for every a.\n        For a in the subfield GF(2^k), k dividing n, the same sum is the\n        subfield trace to GF(2).\n        """\n        t = 0\n        for _ in range(self.n if k is None else k):\n            t ^= a\n            a = self.mul(a, a)\n        return t\n\n\n# ---- GF(2) linear-map helpers ----------------------------------------------\n\n\ndef parity(x):\n    """GF(2) parity of an int\'s bits."""\n    return x.bit_count() & 1\n\n\ndef f2_rows_of(fn, n):\n    """Return the rows of the GF(2)-linear map ``fn`` on n-bit ints.\n\n    Row i is a packed int: bit j is set when input bit j feeds output\n    bit i (LSB-first, the layout ``gadgets.linear_xor`` takes). The call\n    checks linearity over all 2^n inputs and raises AssertionError on a\n    nonlinear ``fn``, which would otherwise yield a silently wrong\n    matrix.\n    """\n    cols = [fn(1 << j) for j in range(n)]\n    for v in range(1 << n):\n        image = 0\n        for j in range(n):\n            if (v >> j) & 1:\n                image ^= cols[j]\n        assert fn(v) == image, "map is not GF(2)-linear"\n    return [sum((((cols[j] >> i) & 1) << j) for j in range(n)) for i in range(n)]\n\n\ndef f2_matinv(rows, n):\n    """Inverse of the n x n GF(2) matrix ``rows`` (LSB-first packed ints)\n    by partial-pivot Gauss-Jordan; asserts invertibility.\n    """\n    A = list(rows)\n    I = [1 << i for i in range(n)]\n    for col in range(n):\n        piv = None\n        for r in range(col, n):\n            if (A[r] >> col) & 1:\n                piv = r\n                break\n        assert piv is not None, "singular matrix"\n        A[col], A[piv] = A[piv], A[col]\n        I[col], I[piv] = I[piv], I[col]\n        for r in range(n):\n            if r != col and ((A[r] >> col) & 1):\n                A[r] ^= A[col]\n                I[r] ^= I[col]\n    return I\n\n\ndef f2_transpose(rows, n):\n    """Transpose of the n x n GF(2) matrix ``rows`` (LSB-first packed\n    ints).\n    """\n    return [sum(((rows[r] >> c) & 1) << r for r in range(n)) for c in range(n)]\n', 'tasks/dqi/bm_forney/gu_formula_derivation.py': '# SPDX-License-Identifier: Apache-2.0\n# Gu et al., deterministic field formula derivation, adapted imports.\n#!/usr/bin/env python3\n"""Regenerate constants.json for the dqi-reed-solomon discovered circuit.\n\nmain() reruns every search and writes constants.json.  The searches, in\nfile order:\n\n  Formulas     the multiplication formulas and tower maps (KT12, KT6,\n               KT3, CUBE3Q, PAIR3, LIN).  Deterministic; seconds.\n  BM windows   the per-round coefficient windows (IMAXV, KMSV, KMMV).\n               search.c samples the full Berlekamp-Massey recursion over\n               403.2M weight-law draws (56 fixed seeds, 7.2M draws each)\n               and counts, per (round, slot), how often the slot changes\n               state.  Each window is the smallest one whose clipped\n               slots all have mass under 4e-5 per draw.  About 2.7\n               CPU-hours; the seeds run on a thread pool (3.5 min on 56\n               cores).\n  Psi windows  the cofactor windows of the Psi route: per CR op, the\n               minimal cyclic interval covering the union support where\n               the op acts, over 400k law draws (16 fixed seeds) plus\n               the w = 0 anchor and every w = 1 position.  BM runs under\n               the freshly derived BM windows.\n\nThe models section holds the bit-exact classical models of the circuit\nthat the Psi window search runs: the fused-Q BM recursion and the Psi\nroute (stages 5-7).  The script derives the instance from contract.json\nand does not import circuit.py, so it runs without constants.json.\n\nFormula derivation.  The field is GF(2^12), poly 4331, x = 2\nprimitive.  The tower is GF(2^12) = GF(64)(t) = GF(8)(s)(t), read top\ndown: A = x + x^64, t = x / A, nu = x^65 / A^2, u = x^65,\nA\' = u + u^8, s = u / A\', nu\' = u^9 / A\'^2, v = u^9.  Tower\ncoordinates are E12[i + 3j + 6k] = v^i s^j t^k.  The norm descends\nboth quadratic levels (N = x xbar in GF(64), then M = N Nbar in GF(8)),\nso the only nonlinear inverse left is M -> M^3 in GF(8), a 3-product\nquadratic form; M^-1 = (M^3)^2 is a free squaring.  A full inversion\ncosts N (15) + M (6) + M^3 (3) + Nbar M^-1 (12) + xbar N^-1 (30) = 66\nToffoli, and every stage maps 0 to 0.  Rejected towers: GF(64)^2 alone\nneeds N^31 (five 1-bits, so a generic GF(64) product on top of two\nquadratic forms); GF(16)^3 pays a cubic norm; plain Itoh-Tsujii needs\nat least four 42-Toffoli products.\n\n  KT12    GF(2^12) = GF(4)[x]: a 6-term product over GF(4) by CRT at\n          the four GF(4) points, infinity, and the irreducible\n          quadratics x^2+x+w, x^2+x+w^2, x^2+wx+1.  14 GF(4) products\n          x 3 = 42; the greedy prune removes nothing.  GF(8)-CRT and\n          Chudnovsky-style curves over GF(4) tie at 42; Karatsuba\n          variants give 45-51.\n  KT6     GF(64) = GF(4)[u], 3 terms, CRT at four points + infinity:\n          15 products, the best known bilinear complexity of GF(2^6).\n  KT3     the 3-term formula M3TERMS reduced modulo v\'s minimal\n          polynomial: 6 products.\n  CUBE3Q  exhaustive search over product subsets: 3 is the minimum.\n  PAIR3   elimination rank decomposition of Tr3(r s^2): rank 3.\n  LIN     the tower\'s linear maps (pure linear algebra on the\n          elements above).\n\nRUN: uv run python gen_constants.py   (writes constants.json)\n"""\n\nimport collections\nimport ctypes\nimport json\nimport os\nimport pathlib\nimport random\nimport sys\nimport time\nfrom multiprocessing import Pool\nfrom multiprocessing.pool import ThreadPool\n\nimport numpy as np\n\nfrom port_runtime import GF2n, f2_matinv, f2_transpose, parity\nfrom port_runtime import load_contract\n\nHERE = pathlib.Path(__file__).resolve().parent\nPATH = HERE / "constants.json"\n\n# The instance (contract.json), derived as circuit.py derives it.\nCONTRACT = load_contract(__file__)\nM = CONTRACT["m"]  # code length = nonzero field points\nN = CONTRACT["n_synd"]  # syndrome symbols = BM rounds\nB = CONTRACT["b"]  # bits per symbol\nW = CONTRACT["w_max"]  # unique-decoding radius n/2 = max locator degree\nLAW_CUM = CONTRACT["weight_cum_2p16"]  # the grader\'s 2^16 cumulative weight law\nNH = W // 2 + 1  # words per Euclid half E\', O\' (and per cofactor)\nFIELD = GF2n(B, CONTRACT["gf_poly"])\nEXP, LOG = FIELD.exp, FIELD.log\ngmul, ginv, gsqrt, trace = FIELD.mul, FIELD.inv, FIELD.sqrt, FIELD.trace\nDWB = (N - 2).bit_length() + 1  # the signed d counter dW: two\'s complement\nDMAX = 1 << (DWB - 1)  # two\'s-complement range of dW\n\n\ndef bitlist(m, n):\n    return [i for i in range(n) if (m >> i) & 1]\n\n\ndef bitmask(n, pred):\n    return sum(1 << i for i in range(n) if pred(i))\n\n\nDWMASK = (1 << DWB) - 1  # the dW register (two\'s complement, DWB bits)\nMMASK = 31  # m = (dW + 2) >> 2 lives in 5 bits (dW[2:7])\n# The cores this process may use (its CPU affinity set, not the whole node).\nNCPU = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()\n\n# #########################################################################\n# Formulas\n# #########################################################################\n# Every table is deterministic; the module docstring gives the derivation.\n# search_kt keeps the CRT formula only if certify_kt accepts it.\n\n# ---------------------------------------------------------------------------\n# small F2 helpers\n# ---------------------------------------------------------------------------\n\n\ndef rows_apply(rows, x):\n    """bit r of the result = parity(rows[r] & x)."""\n    v = 0\n    for r in range(len(rows)):\n        if parity(rows[r] & x):\n            v |= 1 << r\n    return v\n\n\ndef compose_rows(A, B, nb):\n    """Rows of A o B (apply B, then A); B has nb rows."""\n    out = []\n    for r in range(len(A)):\n        m = 0\n        for i in bitlist(A[r], nb):\n            m ^= B[i]\n        out.append(m)\n    return out\n\n\n# ---------------------------------------------------------------------------\n# GF(2^b) arithmetic\n# ---------------------------------------------------------------------------\n\n\ndef reduce_raw(raw):\n    """Reduce a raw F2[x] product (degree <= 2B - 2) mod the field\n    polynomial: x^k = EXP[k], since gamma = x.\n    """\n    v = 0\n    for k in bitlist(raw, 2 * B - 1):\n        v ^= EXP[k]\n    return v\n\n\n# ---------------------------------------------------------------------------\n# polynomial-multiplication formulas (Karatsuba fallback, GF(8) terms)\n# ---------------------------------------------------------------------------\n\nM3TERMS = [\n    (1, 1, 0b00111),\n    (2, 2, 0b01110),\n    (4, 4, 0b11100),\n    (3, 3, 0b00010),\n    (6, 6, 0b01000),\n    (5, 5, 0b00100),\n]\n\n\ndef kara(n):\n    """Recursive symmetric Karatsuba formula for n-term F2 polynomial\n    multiplication, [(Amask, Bmask, raw-product mask)].\n    """\n    if n == 1:\n        return [(1, 1, 1)]\n    if n == 3:\n        return list(M3TERMS)\n    n0 = n // 2\n    n1 = n - n0\n    m = max(n0, n1)\n    out = []\n    for A, B, C in kara(n0):\n        out.append((A, B, C ^ (C << n0)))\n    for A, B, C in kara(n1):\n        out.append((A << n0, B << n0, (C << n0) ^ (C << (2 * n0))))\n    for A, B, C in kara(m):\n        Am = 0\n        Bm = 0\n        for i in range(m):\n            if (A >> i) & 1:\n                if i < n0:\n                    Am ^= 1 << i\n                if i < n1:\n                    Am ^= 1 << (n0 + i)\n            if (B >> i) & 1:\n                if i < n0:\n                    Bm ^= 1 << i\n                if i < n1:\n                    Bm ^= 1 << (n0 + i)\n        out.append((Am, Bm, C << n0))\n    return out\n\n\n# ---------------------------------------------------------------------------\n# GF(4)-tower CRT bilinear formulas (deterministic; KT / KT12 / KT6)\n# ---------------------------------------------------------------------------\n\n\nclass F4:\n    """GF(4) = {0, 1, w, w+1} inside fld, index-encoded (bit 0 = 1, bit 1 =\n    w), so index addition is XOR.\n    """\n\n    def __init__(self, fld):\n        gmul = fld.mul\n        w3 = fld.exp[fld.order // 3]\n        assert gmul(w3, w3) == (w3 ^ 1) and gmul(w3, w3 ^ 1) == 1\n        self.w3 = w3\n        self.E = [0, 1, w3, w3 ^ 1]\n        self.I = {self.E[i]: i for i in range(4)}\n        self.gmul = gmul\n\n    def mul(self, a, bq):\n        return self.I[self.gmul(self.E[a], self.E[bq])]\n\n\ndef f4_irreducible_quadratics(f4):\n    """Monic irreducible x^2 + a x + b over GF(4), (a, b) in index order."""\n    out = []\n    for a in range(4):\n        for bq in range(4):\n            if all((f4.mul(r, r) ^ f4.mul(a, r) ^ bq) != 0 for r in range(4)):\n                out.append((a, bq))\n    assert len(out) == 6\n    return out\n\n\ndef f4_lams(f4, k, nquad):\n    """Return the CRT evaluation functionals for a k-term product over GF(4):\n    evaluation at 0, 1, w, w^2, at infinity, then for each of the first\n    nquad irreducible quadratics q the two reduction coordinates\n    (x^i mod q = c0_i + c1_i x) and their sum (Karatsuba over GF(4)).\n    Modulus degree = 4 + 1 + 2 nquad; the product has 2k - 1 coefficients.\n    """\n    assert 4 + 1 + 2 * nquad == 2 * k - 1, (k, nquad)\n    lams = []\n    for pt in range(4):\n        lam, p = [], 1\n        for _ in range(k):\n            lam.append(p)\n            p = f4.mul(p, pt)\n        lams.append(lam)\n    lams.append([0] * (k - 1) + [1])\n    for a, bq in f4_irreducible_quadratics(f4)[:nquad]:\n        L0, L1 = [], []\n        c0, c1 = 1, 0\n        for _ in range(k):\n            L0.append(c0)\n            L1.append(c1)\n            c0, c1 = f4.mul(c1, bq), c0 ^ f4.mul(c1, a)\n        lams.extend([L0, L1, [p ^ q for p, q in zip(L0, L1)]])\n    return lams\n\n\ndef f4_digits(f4, fld, coords, gens, k):\n    """GF(4)-digit extraction for an n = 2k dimensional coordinate system:\n    gens[i] (i < k) are field elements whose GF(4)-span {gens[i], gens[i]\n    w} is the space; coords(elt) -> n-bit coordinate vector.  Returns\n    DIG[bq] = the k GF(4)-digit indices of coordinate basis vector bq.\n    """\n    n = 2 * k\n    colsE = []\n    for i in range(k):\n        for j in range(2):\n            e = gens[i]\n            if j:\n                e = fld.mul(e, f4.w3)\n            colsE.append(coords(e))\n    EXTR = f2_matinv(f2_transpose(colsE, n), n)\n    DIG = []\n    for bq in range(n):\n        c = rows_apply(EXTR, 1 << bq)\n        DIG.append([(c >> (2 * i)) & 3 for i in range(k)])\n    return DIG\n\n\ndef solve_sym(n, prodlist, mulcoord):\n    """Reconstruction solve for symmetric operand masks: target masks T[t]\n    per product such that a*b = XOR_t parity(a&m_t) parity(b&m_t) T[t] on all\n    n x n basis pairs (verified), or None.\n    """\n    piv = {}\n    for i in range(n):\n        for j in range(n):\n            a = 0\n            for t, mk in enumerate(prodlist):\n                if ((mk >> i) & 1) and ((mk >> j) & 1):\n                    a |= 1 << t\n            r = mulcoord(i, j)\n            while a:\n                cc = a.bit_length() - 1\n                if cc in piv:\n                    pa, pr = piv[cc]\n                    a ^= pa\n                    r ^= pr\n                else:\n                    piv[cc] = (a, r)\n                    a = 0\n                    r = 0\n            if r:\n                return None\n    T = [0] * len(prodlist)\n    for cc in sorted(piv):\n        a, r = piv[cc]\n        acc = r\n        aa = a & ~(1 << cc)\n        while aa:\n            c2 = aa.bit_length() - 1\n            aa ^= 1 << c2\n            acc ^= T[c2]\n        T[cc] = acc\n    for i in range(n):\n        for j in range(n):\n            acc = 0\n            for t, mk in enumerate(prodlist):\n                if ((mk >> i) & 1) and ((mk >> j) & 1):\n                    acc ^= T[t]\n            if acc != mulcoord(i, j):\n                return None\n    return T\n\n\ndef crt_f4_formula(n, DIG, k, lams, f4mul, mulcoord):\n    """GF(4)-CRT bilinear formula on n = 2k coordinate bits: products =\n    3 F2 parities per lambda, reconstruction solved by\n    Gaussian elimination over all n x n basis pairs, deterministic greedy\n    prune in index order.  Returns [(prodmask, targetmask)] (symmetric\n    operand masks) or None.\n    """\n    prods = []\n    for lam in lams:\n        m0 = 0\n        m1 = 0\n        for bq in range(n):\n            U = 0\n            for i in range(k):\n                U ^= f4mul(lam[i], DIG[bq][i])\n            if U & 1:\n                m0 |= 1 << bq\n            if U & 2:\n                m1 |= 1 << bq\n        prods.extend([m0, m1, m0 ^ m1])\n    assert len(prods) == 3 * len(lams)\n\n    T = solve_sym(n, prods, mulcoord)\n    if T is None:\n        return None\n    keep = list(range(len(prods)))\n    for t in range(len(prods)):\n        if t not in keep:\n            continue\n        trial = [prods[u] for u in keep if u != t]\n        if solve_sym(n, trial, mulcoord) is not None:\n            keep = [u for u in keep if u != t]\n    plist = [prods[u] for u in keep]\n    T = solve_sym(n, plist, mulcoord)\n    if T is None:\n        return None\n    out = []\n    for t, mk in enumerate(plist):\n        if T[t] and mk:\n            out.append((mk, T[t]))\n    return out\n\n\ndef build_f4_crt_formula(fld):\n    """GF(2^b) multiplication over the poly basis via GF(2^b) = GF(4)[x]:\n    b/2 terms over GF(4), moduli = 4 points + infinity + (b/2 - 3)\n    irreducible quadratics (b = 12: 42 products).\n    """\n    b = fld.n\n    k = b // 2\n    try:\n        f4 = F4(fld)\n        theta = 2\n        gens = [fld.pow(theta, i) for i in range(k)]\n        DIG = f4_digits(f4, fld, lambda e: e, gens, k)\n        for v in range(1 << b):  # exhaustive digit round-trip\n            dg = [0] * k\n            for bq in bitlist(v, b):\n                dg = [x ^ y for x, y in zip(dg, DIG[bq])]\n            w = 0\n            for i in range(k):\n                w ^= fld.mul(gens[i], f4.E[dg[i]])\n            assert w == v\n        lams = f4_lams(f4, k, k - 3)\n        return crt_f4_formula(b, DIG, k, lams, f4.mul, lambda i, j: fld.mul(1 << i, 1 << j))\n    except Exception as ex:  # noqa: BLE001  (best-effort alternative: any failure falls back to kara(b))\n        print("f4-crt formula construction failed:", repr(ex))\n        return None\n\n\ndef search_kt(fld):\n    """Return the GF(2^b) multiplication formula, constants.json shape\n    [[Ab, Bb, Tb], ...].  Deterministic: the Karatsuba formula kara(b)\n    (b = 12: 54 products), replaced by the GF(4)-tower CRT formula when\n    that is smaller.\n    """\n    b = fld.n\n    RAW = kara(b)\n    # basis-exhaustive check of RAW (complete for a bilinear form)\n    for i in range(b):\n        for j in range(b):\n            pr = 0\n            for Am, Bm, Cm in RAW:\n                if parity((1 << i) & Am) and parity((1 << j) & Bm):\n                    pr ^= Cm\n            assert reduce_raw(pr) == fld.mul(1 << i, 1 << j)\n    KT = []\n    for Am, Bm, Cm in RAW:\n        tm = reduce_raw(Cm)\n        if tm:\n            KT.append([bitlist(Am, b), bitlist(Bm, b), bitlist(tm, b)])\n    F4KT = build_f4_crt_formula(fld)\n    if F4KT is not None:\n        KT_NEW = [[bitlist(mk, b), bitlist(mk, b), bitlist(tm, b)] for (mk, tm) in F4KT]\n        if certify_kt(KT_NEW, fld) and len(KT_NEW) < len(KT):\n            KT = KT_NEW\n    return KT\n\n\n# ---------------------------------------------------------------------------\n# The towers\n# ---------------------------------------------------------------------------\n\n\nclass Tower:\n    """The (2, 2, 3) tower of the module docstring.  GF(2^12) = GF(64)(t),\n    GF(64) = GF(8)(s), GF(8) = F2[v]; E_list = [v^i s^j t^k] indexed\n    i + 3j + 6k; the GF(64) coordinates are E6 = [v^i s^j] (i + 3j), the\n    GF(8) coordinates E3 = [v^i].\n    """\n\n    def __init__(self, fld):\n        self.fld = fld\n        b = fld.n\n        assert b == 12\n        h = b // 2\n        EXP, gmul, ginv, gpow = fld.exp, fld.mul, fld.inv, fld.pow\n        Ael = EXP[1 << h] ^ 2  # x^(2^h) + x = Tr(x)\n        t_el = gmul(2, ginv(Ael))  # t = x / A\n        u_el = EXP[(1 << h) + 1]  # u = x^(2^h + 1) = N(x)\n        nu_el = gmul(u_el, ginv(gmul(Ael, Ael)))  # nu = u / A^2\n        assert gpow(u_el, 1 << h) == u_el and u_el != 0\n        assert gpow(Ael, 1 << h) == Ael and Ael != 0\n        assert gpow(nu_el, 1 << h) == nu_el\n        assert gmul(t_el, t_el) ^ t_el == nu_el\n        self.Ael, self.t_el, self.u_el, self.nu_el = Ael, t_el, u_el, nu_el\n        self.h = h\n        # sub-level GF(64) = GF(8)(s)\n        Ap = gpow(u_el, 8) ^ u_el  # Tr_{6/3}(u)\n        s_el = gmul(u_el, ginv(Ap))\n        v_el = gpow(u_el, 9)  # N_{6/3}(u)\n        nup_el = gmul(v_el, ginv(gmul(Ap, Ap)))\n        assert gpow(Ap, 8) == Ap and Ap != 0\n        assert gpow(v_el, 8) == v_el and gpow(v_el, 7) == 1 and v_el != 1\n        assert gpow(nup_el, 8) == nup_el\n        assert gmul(s_el, s_el) ^ s_el == nup_el\n        self.Ap, self.s_el, self.v_el, self.nup_el = (Ap, s_el, v_el, nup_el)\n        E3 = [gpow(v_el, i) for i in range(3)]\n        E6 = E3 + [gmul(e, s_el) for e in E3]\n        E_list = E6 + [gmul(e, t_el) for e in E6]\n        self.E_sub = E6\n        self.E3 = E3\n        self.E_list = E_list\n        self.MINVR = f2_matinv(f2_transpose(E_list, b), b)\n        for v in range(1 << b):\n            c = self.coords_full(v)\n            w = 0\n            for u in bitlist(c, b):\n                w ^= E_list[u]\n            assert w == v\n        # projections poly -> (a0, a1) and the conjugate\'s constant part\n        self.A0ROWS = self.MINVR[0:h]\n        self.A1ROWS = self.MINVR[h:b]\n        self.A01ROWS = [self.A0ROWS[r] ^ self.A1ROWS[r] for r in range(h)]\n        # squaring and multiply-by-nu on subfield coords\n        self.SQR = f2_transpose([self.coords_sub(gmul(e, e)) for e in self.E_sub], h)\n        self.NUR = f2_transpose([self.coords_sub(gmul(nu_el, e)) for e in self.E_sub], h)\n        NSR = compose_rows(self.NUR, self.SQR, h)\n        # linear part of the norm N = a0 a1 + a0^2 + nu a1^2 on poly words\n        self.LNROWS = []\n        for r in range(h):\n            m = 0\n            for j in bitlist(self.SQR[r], h):\n                m ^= self.A0ROWS[j]\n            for j in bitlist(NSR[r], h):\n                m ^= self.A1ROWS[j]\n            self.LNROWS.append(m)\n        # GF(8) level: coords3 = low 3 bits of the GF(64) coords of a\n        # GF(8) element; RED3 = v^k for the KT3 reduction\n        self.RED3 = [self.coords3(gpow(v_el, k)) for k in range(5)]\n        self.SQ3R = f2_transpose([self.coords3(gmul(e, e)) for e in self.E3], 3)\n        self.NU3R = f2_transpose([self.coords3(gmul(nup_el, e)) for e in self.E3], 3)\n        NSR3 = compose_rows(self.NU3R, self.SQ3R, 3)\n        # linear part of M = n0 n1 + n0^2 + nu\' n1^2 on GF(64) coords\n        self.LMROWS = [self.SQ3R[r] ^ (NSR3[r] << 3) for r in range(3)]\n        self.FCUBE3 = [self.coords3(gpow(self.elt3(c), 3)) for c in range(8)]\n\n    # -- coordinates --\n    def coords_full(self, v):\n        return rows_apply(self.MINVR, v)\n\n    def coords_sub(self, v):\n        c = self.coords_full(v)\n        assert c < (1 << self.h), "not a subfield element"\n        return c\n\n    def coords3(self, v):\n        c = self.coords_full(v)\n        assert c < 8, "not a GF(8) element"\n        return c\n\n    def elt3(self, c):\n        w = 0\n        for i in bitlist(c, 3):\n            w ^= self.E3[i]\n        return w\n\n    def lin_tables(self):\n        """Return the LIN block of constants.json: the linear maps circuit.py\n        reads.\n        """\n        return {\n            "E12": list(self.E_list),\n            "A0ROWS": list(self.A0ROWS),\n            "A1ROWS": list(self.A1ROWS),\n            "LNROWS": list(self.LNROWS),\n            "SQ3R": list(self.SQ3R),\n            "LMROWS": list(self.LMROWS),\n        }\n\n\n# ---------------------------------------------------------------------------\n# subfield multiplication formulas\n# ---------------------------------------------------------------------------\n\n\ndef search_kt6(fld, tower):\n    """Return the GF(64) multiplication formula in tower coords (E6), 15\n    products: GF(64) = GF(4)[u], 3 terms over GF(4), CRT at the four points\n    + infinity, built directly on the 6-bit coordinate vectors.\n    """\n    f4 = F4(fld)\n    u = tower.u_el\n    gens = [fld.pow(u, i) for i in range(3)]\n    DIG = f4_digits(f4, fld, tower.coords_sub, gens, 3)\n    lams = f4_lams(f4, 3, 0)\n    assert len(lams) == 5\n    E6 = tower.E_sub\n    F = crt_f4_formula(\n        6, DIG, 3, lams, f4.mul, lambda i, j: tower.coords_sub(fld.mul(E6[i], E6[j]))\n    )\n    assert F is not None, "GF(64) CRT formula failed"\n    return [[bitlist(mk, 6), bitlist(mk, 6), bitlist(tm, 6)] for (mk, tm) in F]\n\n\ndef search_kt3(tower):\n    """Return the GF(8) multiplication formula in the v-basis, 6 products:\n    M3TERMS reduced modulo the minimal polynomial of v.\n    """\n    KT3 = []\n    for Am, Bm, Cm in M3TERMS:\n        tm = 0\n        for kk in range(5):\n            if (Cm >> kk) & 1:\n                tm ^= tower.RED3[kk]\n        if tm:\n            KT3.append([bitlist(Am, 3), bitlist(Bm, 3), bitlist(tm, 3)])\n    return KT3\n\n\n# ---------------------------------------------------------------------------\n# quadratic-form formulas: exact solve and exhaustive search (CUBE3Q)\n# ---------------------------------------------------------------------------\n\n\ndef qp_pairs(n):\n    return [(i, j) for i in range(n) for j in range(i + 1, n)]\n\n\ndef _offvec(n, u, w):\n    v = 0\n    for idx, (i, j) in enumerate(qp_pairs(n)):\n        if (((u >> i) & (w >> j)) ^ ((u >> j) & (w >> i))) & 1:\n            v |= 1 << idx\n    return v\n\n\ndef quad_solve(n, prods, fvals):\n    """Given product pairs (u, w),\n    solve for output masks + linear correction so the quadratic formula\n    reproduces fvals on all 2^n elements; returns (tris, corr) or None.\n    """\n    pairs = qp_pairs(n)\n    bas = {}\n    for t_, (u, w) in enumerate(prods):\n        x = _offvec(n, u, w)\n        sel = 1 << t_\n        while x:\n            mb = x.bit_length() - 1\n            if mb in bas:\n                x ^= bas[mb][0]\n                sel ^= bas[mb][1]\n            else:\n                bas[mb] = (x, sel)\n                break\n    tg = [0] * n\n    for idx, (i, j) in enumerate(pairs):\n        cdiff = fvals[(1 << i) | (1 << j)] ^ fvals[1 << i] ^ fvals[1 << j]\n        for m in range(n):\n            if (cdiff >> m) & 1:\n                tg[m] |= 1 << idx\n    outs = []\n    for m in range(n):\n        x = tg[m]\n        sel = 0\n        while x:\n            mb = x.bit_length() - 1\n            if mb not in bas:\n                return None\n            x ^= bas[mb][0]\n            sel ^= bas[mb][1]\n        outs.append(sel)\n    tris = []\n    for t_, (u, w) in enumerate(prods):\n        om = 0\n        for m in range(n):\n            if (outs[m] >> t_) & 1:\n                om |= 1 << m\n        if om:\n            tris.append((u, w, om))\n    corr = [0] * n\n    for i in range(n):\n        cv = fvals[1 << i]\n        for u, w, om in tris:\n            if (u >> i) & 1 and (w >> i) & 1:\n                cv ^= om\n        for m in range(n):\n            if (cv >> m) & 1:\n                corr[m] |= 1 << i\n    for x in range(1 << n):\n        if quad_apply((tris, corr), x, n) != fvals[x]:\n            return None\n    return (tris, corr)\n\n\ndef quad_exhaustive(n, fvals):\n    """EXHAUSTIVE minimum-product quadratic formula for fvals on n bits:\n    candidate products are the unordered pairs u < w of nonzero masks (a\n    pair (u, u) is linear and never needed); subsets are tried in\n    increasing size, lexicographic order, and the first solvable one wins.\n    Feasible for n <= 4; used for GF(8).\n    """\n    from itertools import combinations\n\n    cands = [(u, w) for u in range(1, 1 << n) for w in range(u + 1, 1 << n)]\n    for r in range(len(qp_pairs(n)) + 1):\n        for sub in combinations(cands, r):\n            res = quad_solve(n, list(sub), fvals)\n            if res is not None:\n                tris, corr = res\n                return [[list(t) for t in tris], list(corr)]\n    raise AssertionError("no quadratic formula (impossible)")\n\n\ndef search_cube3q(tower):\n    """Exhaustive, deterministic: x -> x^3 in GF(8), v-basis."""\n    return quad_exhaustive(3, tower.FCUBE3)\n\n\n# ---------------------------------------------------------------------------\n# pairing-form rank decomposition (elimination)\n# ---------------------------------------------------------------------------\n\n\ndef pair_decomp(Brows, n):\n    """B[u] = mask over v of the form on basis pair (u, v).  Returns\n    [(p, qm), ...] with B(a, c) = XOR_k parity(a & p_k) & parity(c & qm_k).\n    """\n    Rw = list(Brows)\n    out = []\n    while True:\n        u0 = None\n        for u in range(n):\n            if Rw[u]:\n                u0 = u\n                break\n        if u0 is None:\n            break\n        qm = Rw[u0]\n        v0 = (qm & -qm).bit_length() - 1\n        p = bitmask(n, lambda u: (Rw[u] >> v0) & 1)\n        for u in bitlist(p, n):\n            Rw[u] ^= qm\n        out.append((p, qm))\n    chk = [0] * n\n    for p, qm in out:\n        for u in bitlist(p, n):\n            chk[u] ^= qm\n    assert chk == list(Brows)\n    return out\n\n\ndef search_pair3(fld, tower):\n    """Rank decomposition of B(r, s) = Tr3(r s^2) on GF(8) v-coords."""\n    gmul = fld.mul\n    E3 = tower.E3\n    B33 = [bitmask(3, lambda j: trace(gmul(E3[i], gmul(E3[j], E3[j])), 3)) for i in range(3)]\n    return [[p, qm] for (p, qm) in pair_decomp(B33, 3)]\n\n\n# ---------------------------------------------------------------------------\n# formula evaluation (search_kt\'s certify_kt, quad_solve\'s quad_apply)\n# ---------------------------------------------------------------------------\n\n\ndef kt_mul_fn(KT):\n    ktm = [\n        (sum(1 << u for u in Ab), sum(1 << u for u in Bb), sum(1 << u for u in Tb))\n        for (Ab, Bb, Tb) in KT\n    ]\n\n    def mul(a, bb):\n        acc = 0\n        for Am, Bm, Tm in ktm:\n            if parity(a & Am) and parity(bb & Bm):\n                acc ^= Tm\n        return acc\n\n    return mul\n\n\ndef certify_kt(KT, fld):\n    """Return True iff KT multiplies exactly in GF(2^b) over the poly basis:\n    checked on the b x b basis pairs, complete for a bilinear form.\n    """\n    kt_mul = kt_mul_fn(KT)\n    return all(\n        kt_mul(1 << i, 1 << j) == fld.mul(1 << i, 1 << j)\n        for i in range(fld.n)\n        for j in range(fld.n)\n    )\n\n\ndef quad_apply(Q, x, n):\n    tris, corr = Q\n    acc = 0\n    for u, w, om in tris:\n        if parity(u & x) and parity(w & x):\n            acc ^= om\n    for m in range(n):\n        if parity(corr[m] & x):\n            acc ^= 1 << m\n    return acc\n\n\ndef run_b12():\n    """All formula searches (every one deterministic); returns (tables,\n    timings) with tables in constants.json shape.\n    """\n    fld = FIELD\n    tower = Tower(fld)\n    tables, tm = {}, {}\n    for name, search in (\n        ("KT12", lambda: search_kt(fld)),\n        ("KT6", lambda: search_kt6(fld, tower)),\n        ("KT3", lambda: search_kt3(tower)),\n        ("CUBE3Q", lambda: search_cube3q(tower)),\n        ("PAIR3", lambda: search_pair3(fld, tower)),\n    ):\n        t0 = time.time()\n        tables[name] = search()\n        tm[name + "_s"] = time.time() - t0\n    tables["LIN"] = tower.lin_tables()\n    return tables, tm\n\n\n# #########################################################################\n# Models\n# #########################################################################\n# Bit-exact classical models of the circuit (Builder-free): the one Python\n# model of its fused-Q BM recursion, and the Psi route (stages 5-7) op for\n# op.  The Psi window search runs them.\n\n# =========================================================================\n# BM recursion (q-form conditional-swap, fused-Q B bank)\n# =========================================================================\n# The B bank rotates at bank-1.  A per-round hook instruments the loop.\n', 'tasks/dqi/bm_forney/gu_linear.py': '"""GF(2) linear-layer synthesis.\n\nSource: conventional Gauss-Jordan row elimination, emitted as the reversed\nelimination schedule — the textbook in-place CNOT/SWAP realization of an\ninvertible F2 matrix, as used across the finite-field literature. ``linear_xor`` is the textbook\nout-of-place form: one CNOT per set matrix entry.\n"""\n\n\ndef linear_in_place(b, qs, rows):\n    """Apply the invertible GF(2) matrix ``rows`` to register ``qs`` in\n    place: qs <- rows * qs. ``rows[i]`` is the packed-int row i (bit j =\n    column j, little-endian to match register order).\n\n    Synthesis is leftmost-pivot Gauss-Jordan elimination recorded and\n    emitted in reverse: 0 Toffoli, only CX and SWAP; no ancillas. Raises\n    (via the identity assert) if the matrix is singular.\n    """\n    n = len(qs)\n    a = list(rows)\n    assert len(a) == n\n    ops = []\n    for col in range(n):\n        pivot = next(r for r in range(col, n) if (a[r] >> col) & 1)\n        if pivot != col:\n            a[col], a[pivot] = a[pivot], a[col]\n            ops.append(("swap", col, pivot))\n        for r in range(n):\n            if r != col and (a[r] >> col) & 1:\n                a[r] ^= a[col]\n                ops.append(("cx", col, r))\n    assert a == [1 << i for i in range(n)], "singular matrix"\n    for kind, i, j in reversed(ops):\n        (b.swap if kind == "swap" else b.cx)(qs[i], qs[j])\n\n\ndef linear_xor(b, src, dst, rows):\n    """XOR the GF(2) matrix-vector product onto a second register:\n    dst ^= rows * src. ``rows[i]`` is the packed-int row i (bit j =\n    column j), for example from :func:`uncompute.classical.f2_rows_of`.\n\n    Emits one CX per set matrix entry in row-major order: for each dst\n    bit i, for each src bit j. 0 Toffoli; no ancillas. ``src`` and\n    ``dst`` must be disjoint.\n    """\n    for i, row in enumerate(rows):\n        for j in range(len(src)):\n            if (row >> j) & 1:\n                b.cx(src[j], dst[i])\n', 'tasks/dqi/bm_forney/gu_port.py': '# SPDX-License-Identifier: Apache-2.0\n# Adapted from andigu/uncompute by Gu et al.; see REPORT.md and ref/gu_README.md.\n#!/usr/bin/env python3\n"""dqi-reed-solomon discovered circuit: fused-Q truncated Berlekamp-Massey\ndecode at (m, n, b) = (4095, 70, 12), then a half-degree Bezout pass that\nturns the per-point Forney division into a precomputed Psi = 1/sigma\'\nmod sigma, so the Chien/Forney sweep costs 23 Toffoli per point (root\ntest 11 + a rank-12 gated pairing 12) instead of 80.  Built natively on\nuncompute.Builder.\n\nThe BM body runs the (2,2,3)-tower field arithmetic under the measured\nwindow schedule and stores no accepted discrepancy: the scale identity\nsigma_0 * DEL == 1 replaces it (README).\nThe Psi route (stages 5-7) holds the Bezout pair in the syndrome TAIL,\nzeroed by extending the in-place omega transform to all 70 rows.\n\nLayout (flat):\n  circuit.py         the tables (field, formulas, schedule; loaded from\n                    constants.json), the circuit layer (term lists,\n                    emitter, Psi-route gadgets, sweep body), and build(),\n                    the stage sequence with its phase marks and cost pins.\n  constants.json    every searched value: the formula tables, the BM\n                    windows (bm_windows), the Psi windows (psi_windows).\n  gen_constants.py  reruns every search and writes constants.json; it\n                    holds the bit-exact classical models the searches run.\n  search.c          the C sampler behind the BM window census.\n\nRUN: uv run python circuit.py -- builds, asserts its cost, and\nstreams to $UNCOMPUTE_OPS_PATH (else ./ops.bin).  Importing is cheap and\nbuilds nothing; build() feeds the occupancy/liveness profilers.\nEvaluate: python tools/eval.py dqi-reed-solomon.\n\nREADME.md in this folder describes the construction.\n"""\n\nimport functools\nimport json\nimport pathlib\nfrom contextlib import contextmanager\n\nimport numpy as np\n\nfrom port_runtime import (Builder, Journal, GF2n, f2_matinv, f2_transpose, parity,\n    declare_registers, linear_in_place, assert_metrics, emit_and_report, load_contract)\nimport os\n\nHERE = pathlib.Path(__file__).resolve().parent\n\n\n# =========================================================================\n# Field and instance\n# =========================================================================\n\nCONTRACT = load_contract(__file__)\nPOLY = CONTRACT["gf_poly"]\nM = CONTRACT["m"]  # code length = nonzero field points\nN = CONTRACT["n_synd"]  # syndrome symbols = BM rounds\nB = CONTRACT["b"]  # bits per symbol\nW = CONTRACT["w_max"]  # unique-decoding radius n/2 = max locator degree\nLAW_CUM = CONTRACT["weight_cum_2p16"]  # the grader\'s 2^16 cumulative weight law\nassert M == (1 << B) - 1 and N == 2 * W\nNH = W // 2 + 1  # words per Euclid half E\', O\' (and per cofactor)\nSQRT_EXP = (M + 1) // 2  # sqrt(x) = x^SQRT_EXP (2048)\n\n# EXP/LOG tables (gamma = 2): EXP[k] = gamma^k for k < 2M, LOG its inverse.\n# ginv is totalized (zero maps to zero), as the tower inversion chain.\nFIELD = GF2n(B, POLY)\nassert FIELD.gen == 2, "gamma = 2 not primitive?!"\nEXP, LOG = FIELD.exp, FIELD.log\ngmul, ginv, gpow, gsqrt = FIELD.mul, FIELD.inv, FIELD.pow, FIELD.sqrt\n# trace(x) is the absolute trace; trace(x, 3) the GF(8) trace of a GF(8) element.\ntrace = FIELD.trace\n\n\ndef bitlist(m, n):\n    return [i for i in range(n) if (m >> i) & 1]\n\n\ndef bitmask(n, pred):\n    return sum(1 << i for i in range(n) if pred(i))\n\n\ndef gf_matinv(Mx):\n    """Inverse of a square matrix over GF(2^B) (Gauss-Jordan)."""\n    n = len(Mx)\n    A = [row[:] + [1 if i == j else 0 for j in range(n)] for i, row in enumerate(Mx)]\n    for col in range(n):\n        piv = next(r for r in range(col, n) if A[r][col])\n        A[col], A[piv] = A[piv], A[col]\n        iv = ginv(A[col][col])\n        A[col] = [gmul(iv, v) for v in A[col]]\n        for r in range(n):\n            if r != col and A[r][col]:\n                f = A[r][col]\n                A[r] = [A[r][c] ^ gmul(f, A[col][c]) for c in range(2 * n)]\n    return [row[n:] for row in A]\n\n\n# =========================================================================\n# A. Fixup vector c (contract.json; task README section 3.2:\n# counter-mode SHA-256 XOF over c_tag, grader-asserted)\n# =========================================================================\n_buf = bytes.fromhex(CONTRACT["c_stream_hex"])\nCBITS = [(_buf[t >> 3] >> (t & 7)) & 1 for t in range(M * B)]\nCMASKJ = [bitmask(B, lambda bp: CBITS[B * j + bp]) for j in range(M)]\n\n# =========================================================================\n# B. Frozen multiplication formulas + tower row masks (constants.json\n# beside circuit.py, written by gen_constants.py).\n# =========================================================================\nCONSTS = json.loads(pathlib.Path(os.environ["DQI_CONSTANTS"]).read_bytes())\nKT = [tuple(map(tuple, t)) for t in CONSTS["KT12"]]  # GF(2^12) mult: 42 products\nKT6 = [tuple(map(tuple, t)) for t in CONSTS["KT6"]]  # GF(64) mult: 15 products\nKT3 = [tuple(map(tuple, t)) for t in CONSTS["KT3"]]  # GF(8) mult: 6 products\n# x^3 in GF(8): 3 products\nCUBE3Q = tuple(map(tuple, CONSTS["CUBE3Q"][0])), tuple(CONSTS["CUBE3Q"][1])\nPAIR3 = [tuple(pq) for pq in CONSTS["PAIR3"]]  # Tr3(r s^2) pairing: rank 3\nLIN = CONSTS["LIN"]  # the tower\'s linear maps\n\n# =========================================================================\n# C. Omega schedule: in-place triangular product from middle-product\n# blocks, each at its field-rank a+b-1 by evaluation-interpolation.\n# Constant GF(2^B) maps are free CX networks, so evaluating both\n# operands at a+b-1 fixed points and interpolating the results back\n# costs only the pointwise quantum-quantum products; the recursion\n# below (balanced binary) is the DP optimum of the alias-free schedule\n# family: 147 products against the dense triangle\'s W(W+1)/2 = 630.\n# Blocks read syn columns strictly below the rows they write, so the\n# hi -> block -> lo emission order stays in-place.\n# =========================================================================\n_OMC = {1: 0}\n_OMS = {1: None}\nfor _n in range(2, W + 1):\n    _best = None\n    _bh = None\n    for _h in range(1, _n // 2 + 1):\n        _v = _OMC[_h] + _OMC[_n - _h]\n        if _best is None or _v < _best:\n            _best, _bh = _v, _h\n    assert _best is not None  # range(1, _n // 2 + 1) is non-empty\n    _OMC[_n] = (_n - 1) + _best\n    _OMS[_n] = _bh\n\nMACS = []  # (r0, a, c0, b): reads syn cols [c0, c0+b), writes [r0, r0+a)\n\n\ndef _build_tri(lo, n):\n    if n == 1:\n        return\n    h = _OMS[n]\n    _build_tri(lo + h, n - h)\n    MACS.append((lo + h, n - h, lo, h))\n    _build_tri(lo, h)\n\n\n# omega70: the same in-place transform extended to all N = 2W rows.  Rows\n# W..N-1 of sigma_m*S mod z^N vanish on every correct decode (the key\n# equation: deg omega < deg sigma <= W), which frees the 35-word syndrome\n# TAIL as workspace for the Bezout pair (stage 5).  Emission\n# order hi -> lo keeps it in place: the tail triangle (rows W..N-1 from\n# rows W..N-2, diagonals 1..W-1), then one off-diagonal middle-product\n# block (rows W..N-1 from rows 0..W-1; its Toeplitz diagonals W+1..N-1\n# are zero -- sigma has W+1 coefficients -- and are clipped from the\n# sigma-side evaluation), then the omega triangle.  The map is unit\n# triangular on all N rows, so its literal reverse restores S on EVERY\n# path, including approximate-BM paths where the tail is not zero.\n_build_tri(W, W)\nMACS.append((W, W, 0, W))\n_build_tri(0, W)\nassert len(MACS) == 2 * (W - 1) + 1\nfor _r0, _a, _c0, _b in MACS:\n    # alias-free in place: every block reads rows strictly below the rows\n    # it writes; the triangles span diagonals 1..W-1, the off-diagonal\n    # block 1..N-1 (clipped to 1..W at emission)\n    assert _c0 + _b == _r0, (_r0, _a, _c0, _b)\n    assert _r0 + _a - 1 - _c0 <= (N - 1 if _b == W else W - 1), (_r0, _a, _c0, _b)\n\n_VINV = {}\n\n\ndef vinv(npts):\n    """(points, inverse Vandermonde) for evaluation at gamma^0..gamma^(npts-1),\n    cached per size.\n    """\n    if npts not in _VINV:\n        pts = [EXP[t] for t in range(npts)]\n        _VINV[npts] = (\n            pts,\n            gf_matinv([[gpow(pts[t], k) for k in range(npts)] for t in range(npts)]),\n        )\n    return _VINV[npts]\n\n\n@functools.cache\ndef cmrows(c):\n    """Row masks of the constant multiplication x -> c*x (a free CX network)."""\n    return f2_transpose([gmul(c, 1 << j) for j in range(B)], B)\n\n\n# =========================================================================\n# D. (2,2,3) tower: GF(2^12) = GF(64)(t) = GF(8)(s)(t), read top-down\n# (gen_constants.py docstring).  A 12-bit tower word is a0 (low 6) + a1 (high 6) t with\n# t^2 + t = nu, conjugate xbar = (a0 + a1) + a1 t; a GF(64) word is\n# n0 (low 3) + n1 (high 3) s, conjugate Nbar = (n0 + n1) + n1 s.  The\n# norm descends both quadratic levels (N = x xbar in GF(64) = a0 a1 +\n# linear, M = N Nbar in GF(8) = n0 n1 + linear); the only nonlinear\n# inverse left is M -> M^3 (a 3-product quadratic form), M^-1 = (M^3)^2\n# a free squaring.  Coordinates: E12[i + 3j + 6k] = v^i s^j t^k; the\n# GF(8) projection of a GF(64) word is a bit slice (no row parities).\n# Every stage maps 0 to 0, so the zflag armor (monicize) carries over.\n# local: gadgets.gf_tower.GF asserts q0 in {4, 8}; this tower is\n# GF(8)(s)(t) and its in-circuit chain runs on A0/A1 row projections of\n# 12-bit poly-basis words, never on tower-packed words.\n# =========================================================================\nE12 = LIN["E12"]  # the 12 tower basis elements (poly ints)\nA0ROWS = LIN["A0ROWS"]  # poly(12) -> a0 (GF(64) coords)\nA1ROWS = LIN["A1ROWS"]  # poly(12) -> a1\nA01ROWS = [A0ROWS[r] ^ A1ROWS[r] for r in range(6)]  # a0 + a1 (xbar\'s constant part)\nLNROWS = LIN["LNROWS"]  # poly(12) -> the linear part of N: a0^2 + nu a1^2\nLMROWS = LIN["LMROWS"]  # GF(64) coords -> the linear part of M: n0^2 + nu\' n1^2\nSQ3R = LIN["SQ3R"]  # squaring on GF(8) coords\nassert (len(E12), len(A0ROWS), len(A1ROWS), len(LNROWS), len(LMROWS), len(SQ3R)) == (\n    12,\n    6,\n    6,\n    6,\n    3,\n    3,\n)\n# the tower elements re-derived from the field (the recipe in the\n# gen_constants.py docstring); E12 must be their products\n_A_el = 2 ^ gpow(2, 64)\n_t_el = gmul(2, ginv(_A_el))\n_u_el = gpow(2, 65)\n_Ap_el = _u_el ^ gpow(_u_el, 8)\n_s_el = gmul(_u_el, ginv(_Ap_el))\n_v_el = gpow(_u_el, 9)\nassert E12 == [\n    gmul(gmul(gpow(_v_el, i), gpow(_s_el, j)), gpow(_t_el, k))\n    for k in range(2)\n    for j in range(2)\n    for i in range(3)\n], "E12 != v^i s^j t^k"\nMINVR = f2_matinv(f2_transpose(E12, B), B)\nassert MINVR[0:6] == A0ROWS and MINVR[6:12] == A1ROWS, "A0/A1 rows != inverse basis matrix"\n\n# =========================================================================\n# E. Trace dual, free-Z interpolation split, GF(8) fixed pairing form\n# =========================================================================\nTRT = [trace(x) for x in range(1 << B)]  # absolute trace table\nTINV = f2_matinv([bitmask(B, lambda c: TRT[gmul(1 << a, 1 << c)]) for a in range(B)], B)\nMU = f2_transpose(TINV, B)\nfor _a in range(B):\n    for _bb in range(B):\n        assert TRT[gmul(1 << _a, MU[_bb])] == (1 if _a == _bb else 0)\n\nLAMJ = [0] * M\nfor _j in range(M):\n    _v = 0\n    for _bpos in bitlist(CMASKJ[_j], B):\n        _v ^= MU[_bpos]\n    LAMJ[_j] = _v\n\n# ---- move N sweep points into free Z gates (Vandermonde interpolation) --\nKILLJ = [j for j in range(M) if LAMJ[j] == 0]\nKILLJ += [j for j in range(M) if LAMJ[j] != 0][: N - len(KILLJ)]\nassert len(KILLJ) == N and len(set(KILLJ)) == N\n_tgt = [gmul(LAMJ[j], ginv(EXP[j])) for j in KILLJ]\n_Ai = gf_matinv([[gpow(EXP[j], i) for i in range(N)] for j in KILLJ])\nUCO = [0] * (N + 1)\nfor _r in range(N):\n    _v = 0\n    for _c in range(N):\n        _v ^= gmul(_Ai[_r][_c], _tgt[_c])\n    UCO[_r + 1] = _v\n\n\ndef U_eval(j):\n    acc = 0\n    for i in range(1, N + 1):\n        if UCO[i]:\n            acc ^= EXP[(LOG[UCO[i]] + i * j) % M]\n    return acc\n\n\nLAMJ2 = [LAMJ[j] ^ U_eval(j) for j in range(M)]\nfor _j in KILLJ:\n    assert LAMJ2[_j] == 0, ("interpolation failed", _j)\nNLIVE = sum(1 for j in range(M) if LAMJ2[j])\nZMASKI = [bitmask(B, lambda bb: TRT[gmul(UCO[i + 1], 1 << bb)]) for i in range(N)]\n\n\ndef _elt3(c):\n    w = 0\n    for i in bitlist(c, 3):\n        w ^= gpow(_v_el, i)\n    return w\n\n\ndef _tr3(x):\n    t = trace(x, 3)\n    assert t in (0, 1)\n    return t\n\n\n# the GF(8) pairing form B(r, s) = Tr3(r * s^2) as PAIR3 spells it: rank 3\nfor _r in range(8):\n    for _s3 in range(8):\n        _acc = 0\n        for _p, _qm in PAIR3:\n            _acc ^= parity(_r & _p) & parity(_s3 & _qm)\n        assert _acc == _tr3(gmul(_elt3(_r), gmul(_elt3(_s3), _elt3(_s3)))), (\n            "pairing form drifted",\n            _r,\n            _s3,\n        )\n\n\ndef pair_rows(lam):\n    """r_a(lam), a < B: parity(r_a & U) = Tr(e_a sqrt(lam U)), lam != 0."""\n    assert lam\n    sq = [gsqrt(gmul(lam, 1 << c)) for c in range(B)]\n    return [bitmask(B, lambda c: TRT[gmul(1 << a_, sq[c])]) for a_ in range(B)]\n\n\n# =========================================================================\n# Constant linear maps of monicize and the Euclid pass\n# =========================================================================\n# MONB[k] = the (a, b) basis pairs whose product has bit k set\nMONB = [[] for _ in range(B)]\nfor _a in range(B):\n    for _b2 in range(B):\n        _vv = gmul(1 << _a, 1 << _b2)\n        for _k2 in range(B):\n            if (_vv >> _k2) & 1:\n                MONB[_k2].append((_a, _b2))\nSQRT_ROWS = f2_transpose([gsqrt(1 << j) for j in range(B)], B)\nSQUARE_ROWS = f2_transpose([gmul(1 << j, 1 << j) for j in range(B)], B)\n\n\n# =========================================================================\n# Truncation schedule\n# =========================================================================\n# Reversal-stack capacity (tagged slots; overflow shots form error\n# class E2, the stack overflow that README.md describes).  Overflow rate\n# by cap under the DQI weight law, measured under the derived windows\n# (403.2M draws): cap 1 -> 1.02e-4 [1.01e-4, 1.03e-4], cap 2 -> 4.1e-7,\n# cap 3 -> 0 seen; max pushes 3.\n# Cap 1 saves STKW = 19 wires at the global peak (both slots materialize\n# by round 4 and stay live through the sweep) and 19 Toffoli per push\n# cascade forward and again in the reverse pass: 2,470 Toffoli and 19\n# qubits against cap 2, a 1.5% lower TQ-cost.  The aggregate sign-error\n# rate is ~1.4e-4 (E1 3.8e-5 + E2 1.02e-4), 7x under the 1e-3 allowance\n# (cap 2: 3.8e-5, 26x).\nSTK_CAP = 1\n\n# Per-round coefficient windows (MEASURED; constants.json bm_windows,\n# derived by gen_constants.py from 403.2M draws under the DQI weight\n# law).  IMAXV bounds the discrepancy convolution, KMSV the\n# flag-controlled slot swaps, KMMV the update macs.  The rule: the\n# smallest per-round windows such that every clipped slot has marginal\n# mass < 4e-5; r=0 and r=N-1 are pinned by the circuit.  Events outside\n# a window form error class E1, the cutoff crossings of README.md.\n_WIN = CONSTS["bm_windows"]\nIMAXV = list(_WIN["IMAXV"])\nKMSV = list(_WIN["KMSV"])\nKMMV = list(_WIN["KMMV"])\nassert len(IMAXV) == len(KMSV) == len(KMMV) == N\nfor _r in range(N):\n    # never wider than the reachable envelope, never narrower than\n    # round 0\'s seed path needs\n    assert IMAXV[_r] <= min(_r, W, (_r + 1) // 2 + 1)\n    assert KMMV[_r] <= KMSV[_r] <= min(_r + 1, W, _r // 2 + 2)\nassert IMAXV[N - 1] == W and KMSV[N - 1] == KMMV[N - 1] == W\nassert KMSV[0] == 1 and KMMV[0] == 0\n\n# Signed d counter: two\'s complement, wide enough for the all-zero\n# syndrome\'s monotone climb (d = N-2 after the last increment) -- DWB-1\n# magnitude bits + the sign bit dW[DWB-1].\nDWB = (N - 2).bit_length() + 1\nassert (1 << (DWB - 1)) - 1 >= N - 2\nSGN = DWB - 1  # the sign bit index (pqc / ctl anchor)\nDMAX = 1 << (DWB - 1)  # two\'s-complement range of dW\n# Stack round tag: TAGB bits hold every pushing round 0..N-1.\nTAGB = (N - 1).bit_length()\nTAGMASK = (1 << TAGB) - 1\nSTKW = B + TAGB  # stack slot: B value lanes + the round tag\nassert DWB == 8  # MBITS = dW[2:7] and search.c hard-code the 8-bit dW\n\n\ndef caps_schedule():\n    """Exact reachable-(d, depth) DP over the d-automaton: d>=0 branches to\n    d+1 (ctl=0, needs D==0) or -d (ctl=1); d<0 forces ctl=0 and a PUSH\n    (pqc = dW[SGN]), d -> d+1.  r=N-1 pushes unconditionally.  CAPS[r] =\n    max reachable stack depth AFTER a push at round r = the emitted\n    cascade extent; CAPS[r]==0 => a push is unreachable => the cascade is\n    elided.\n    """\n    states = {(0, 0)}\n    caps = [0] * N\n    for r in range(N):\n        ns = set()\n        mx = 0\n        for d_, k_ in states:\n            if r == N - 1:\n                ns.add((d_, k_ + 1))\n                mx = max(mx, k_ + 1)\n            elif d_ < 0:\n                ns.add((d_ + 1, k_ + 1))\n                mx = max(mx, k_ + 1)\n            else:\n                ns.add((d_ + 1, k_))\n                ns.add((-d_, k_))\n        caps[r] = mx\n        states = ns\n    return caps\n\n\nCAPS_FULL = caps_schedule()\nassert max(CAPS_FULL) <= W, ("stack cap bound broken", CAPS_FULL)\nassert CAPS_FULL[0] == 0 and CAPS_FULL[1] == 0\n# reachable depth is W-1 before the unconditional r=N-1 push, W after\nassert max(CAPS_FULL[: N - 1]) == W - 1 and CAPS_FULL[N - 1] == W\n# CAP truncation: clamp every emitted cascade extent to STK_CAP.\n# min(c, STK_CAP) == 0 iff c == 0, so the set of rounds carrying a cascade\n# is unchanged; only extents shrink.  Overflow shots (depth would exceed\n# STK_CAP) are the failure class E2 (README.md).  With\n# STK_CAP = 1 every cascade is the single Dw <-> STK[0] Fredkin level.\nCAPS = [min(c, STK_CAP) for c in CAPS_FULL]\nassert max(CAPS) == STK_CAP\n\n# Psi-route cofactor windows (cyclic intervals (lo, n) of the 18-word\n# registers, array-index frame) per CR op of R: the measured union\n# support, recorded only where the op acts (swap: flag = 1; rotation:\n# c = 1; product: (q_lo, q_hi) != 0), over 400k weight-law draws plus the\n# w = 0 / w = 1 anchors (constants.json psi_windows, searched by\n# gen_constants.py).  A support outside a window only mis-sets the\n# Psi value (a sign error on that shot): R^-1 mirrors R op for op, so the\n# registers are restored exactly either way.\nPSI_WIN_IV = {k: {int(j): tuple(v) for j, v in d.items()} for k, d in CONSTS["psi_windows"].items()}\nassert sorted(PSI_WIN_IV) == ["mul", "rz", "swap"] and all(\n    sorted(d) == list(range(NH)) for d in PSI_WIN_IV.values()\n)\nPSI_WIN = {(k, jj): v for k, d in PSI_WIN_IV.items() for jj, v in d.items()}\n\n\n# =========================================================================\n# Terms (Builder-free term-list construction)\n# =========================================================================\n# A term list is the build\'s unit of arithmetic: a list of normalized\n# terms over hard wire ids,\n#\n#   (\'L\', ws, c, tws)             tws ^= parity(ws) ^ c      (free CX/X)\n#   (\'P\', wa, ca, wb, cb, tws)    tws ^= (parity(wa) ^ ca) AND\n#                                        (parity(wb) ^ cb)   (one CCX)\n#\n# built by folding the frozen formula tables (section B) through operand\n# specs (one wire-set per operand bit).  Em emits a term list and\n# erases it.  Everything here is pure construction: no gate is emitted.\n\n\ndef spec_plain(ws):\n    return [frozenset((w,)) for w in ws]\n\n\ndef spec_rows(rows, ws):\n    return [\n        frozenset(ws[j] for j in range(len(ws)) if (rows[r] >> j) & 1) for r in range(len(rows))\n    ]\n\n\ndef fold(spec, bits_):\n    ws = frozenset()\n    for i in bits_:\n        ws = ws ^ spec[i]\n    return tuple(sorted(ws))\n\n\ndef tmask(Tb):\n    """Return a formula product\'s target index set as a B-bit mask."""\n    u = 0\n    for jj in Tb:\n        u ^= 1 << jj\n    return u\n\n\ndef xfold(pairs):\n    """Operand spec bit from (word, row mask) pairs: the XOR (not union)\n    of the selected wires, so repeated wires cancel.\n    """\n    s = set()\n    for word, row in pairs:\n        for j in bitlist(row, B):\n            s ^= {word[j]}\n    return frozenset(s)\n\n\ndef norm_term(wa, ca, wb, cb, tws):\n    if (not wa) and ca == 0:\n        return None\n    if (not wb) and cb == 0:\n        return None\n    if not wa:\n        return ("L", wb, cb, tws)\n    if not wb:\n        return ("L", wa, ca, tws)\n    if wa == wb:\n        if ca == cb:\n            return ("L", wa, ca, tws)\n        return None\n    return ("P", wa, ca, wb, cb, tws)\n\n\ndef bil_terms(rt, table, specA, specB, tgt=None, tws_of=None):\n    """Bilinear terms of one frozen formula table over operand specs:\n    fold each product\'s operand bit-sets through the specs, normalize,\n    append.  Targets come from tgt (indexed by the table\'s target set)\n    or a custom tws_of(Tb).\n    """\n    for Ab, Bb, Tb in table:\n        tws = tws_of(Tb) if tws_of else [tgt[jj] for jj in Tb]  # ty: ignore[not-subscriptable]  # tgt is given when tws_of is not\n        tm = norm_term(fold(specA, Ab), 0, fold(specB, Bb), 0, tws)\n        if tm:\n            rt.append(tm)\n\n\ndef kt6_terms_poly(rt, specA, specB, tgt12, off):\n    """KT6 terms whose GF(64)-coordinate targets land in the 12-bit poly\n    word tgt12 through the tower basis: coord j -> E12[j + off].\n    """\n\n    def tws_of(Tb):\n        pm = 0\n        for u in Tb:\n            pm ^= E12[u + off]\n        return [tgt12[jj] for jj in bitlist(pm, B)]\n\n    bil_terms(rt, KT6, specA, specB, tws_of=tws_of)\n\n\ndef quad_terms(rt, Qf, Xs, tgt, n):\n    tris, corr = Qf\n    sp = spec_plain(Xs)\n    for u, w, om in tris:\n        tm = norm_term(\n            fold(sp, bitlist(u, n)), 0, fold(sp, bitlist(w, n)), 0, [tgt[m] for m in bitlist(om, n)]\n        )\n        if tm:\n            rt.append(tm)\n    for m in range(n):\n        if corr[m]:\n            rt.append(("L", tuple(sorted(Xs[i] for i in bitlist(corr[m], n))), 0, (tgt[m],)))\n\n\ndef lin_rows_terms(rt, rows, src, tgt):\n    """Free linear rows: tgt[r] ^= parity(rows[r] & src)."""\n    for rr in range(len(rows)):\n        if rows[rr]:\n            rt.append(\n                ("L", tuple(sorted(src[jj] for jj in bitlist(rows[rr], len(src)))), 0, (tgt[rr],))\n            )\n\n\ndef prod_rt(tgt, wa, wb):\n    """tgt ^= wa * wb (KT12, 42 products) as a term list."""\n    rt = []\n    bil_terms(rt, KT, spec_plain(wa), spec_plain(wb), tgt)\n    return rt\n\n\ndef one_rt(wa, ca, wb, cb, t):\n    """t ^= (wa ^ ca) AND (wb ^ cb) over wire tuples as a one-term list."""\n    return [norm_term(tuple(sorted(wa)), ca, tuple(sorted(wb)), cb, [t])]\n\n\n# ---- AND ladders ----------------------------------------------------------\ndef ladder_rt(rungs, tree):\n    """AND-ladder term lists over rungs [(wire tuple, polarity), ...]:\n    tree[i-1] <- AND of the first i+1 rungs, each parity tested against\n    its polarity bit (len(rungs) - 1 products).\n    """\n    (w0, c0), (w1, c1) = rungs[0], rungs[1]\n    rts = [one_rt(w0, c0, w1, c1, tree[0])]\n    for i in range(2, len(rungs)):\n        ws, cc = rungs[i]\n        rts.append(one_rt((tree[i - 2],), 0, ws, cc, tree[i - 1]))\n    return rts\n\n\ndef eq_rungs(bits, val):\n    """Rungs whose ladder top is [bits == val] (val = 0: the zero test)."""\n    return [((w,), 1 - ((val >> i) & 1)) for i, w in enumerate(bits)]\n\n\n# ---- the tower inversion chain -------------------------------------------\nCHAIN_WINS = [("nqd", 6), ("m3d", 3), ("s3d", 3), ("yd", 6)]\n\n\ndef norm_chain_terms(src, a0, a1, nq, m3, s3):\n    """Term lists for the norm chain on src: N (KT6 bilinear + linear\n    rows into nq), M (KT3 on the halves of nq + linear rows into m3),\n    M^3 (CUBE3Q into s3).\n    """\n    rtN = []\n    bil_terms(rtN, KT6, a0, a1, nq)\n    lin_rows_terms(rtN, LNROWS, src, nq)\n    rtM = []\n    bil_terms(rtM, KT3, spec_plain(nq[0:3]), spec_plain(nq[3:6]), m3)\n    lin_rows_terms(rtM, LMROWS, nq, m3)\n    rtS = []\n    quad_terms(rtS, CUBE3Q, m3, s3, 3)\n    return rtN, rtM, rtS\n\n\ndef chain_rts(src, wins, out):\n    """Forward tower inversion-chain term lists for out = inv(src): the\n    norm chain, N^-1 = Nbar M^-1 (two KT3 into yd, M^-1 = (M^3)^2 folded\n    into the operand spec as SQ3R rows), then x^-1 = xbar N^-1 (two KT6)\n    recombined into the 12-bit poly-basis out.  wins are the CHAIN_WINS\n    scratch words; rts[:-1] write wins in order, rts[-1] writes out.\n    """\n    nq, m3, s3, yd = wins\n    a0 = spec_rows(A0ROWS, src)\n    a1 = spec_rows(A1ROWS, src)\n    a01 = spec_rows(A01ROWS, src)\n    rtN, rtM, rtS = norm_chain_terms(src, a0, a1, nq, m3, s3)\n    m6 = spec_rows(SQ3R, s3)\n    n0 = spec_plain(nq[0:3])\n    n1 = spec_plain(nq[3:6])\n    n01 = [n0[i] ^ n1[i] for i in range(3)]\n    rtY = []\n    bil_terms(rtY, KT3, n01, m6, yd[0:3])\n    bil_terms(rtY, KT3, n1, m6, yd[3:6])\n    y = spec_plain(yd)\n    rtI = []\n    kt6_terms_poly(rtI, a01, y, out, 0)\n    kt6_terms_poly(rtI, a1, y, out, 6)\n    return [rtN, rtM, rtS, rtY, rtI]\n\n\n# ---- evaluation/interpolation products -----------------------------------\ndef polyprod_rt(Qw, Fw, tgt):\n    """Evaluation/interpolation product: for the polynomials Q(y) =\n    sum_i Qw[i] y^i and F(y) = sum_k Fw[k] y^k (lists of words), XOR\n    coefficient n of Q*F into every word in tgt[n] (a list; empty = drop).\n    Costs len(Qw) + len(Fw) - 1 products (one scalar product when both\n    have one word); operand evaluations and the interpolation rows are\n    free CX networks folded into the operand specs and target sets.\n    """\n    npts = len(Qw) + len(Fw) - 1\n    assert len(tgt) == npts\n    if npts == 1:\n        rt = []\n        for T in tgt[0]:\n            bil_terms(rt, KT, spec_plain(Qw[0]), spec_plain(Fw[0]), T)\n        return rt\n    pts, Iv = vinv(npts)\n    rt = []\n    for t in range(npts):\n        cq = [cmrows(gpow(pts[t], i)) for i in range(len(Qw))]\n        cf = [cmrows(gpow(pts[t], k)) for k in range(len(Fw))]\n        sA = [xfold([(Qw[i], cq[i][bb]) for i in range(len(Qw))]) for bb in range(B)]\n        sB = [xfold([(Fw[k], cf[k][bb]) for k in range(len(Fw))]) for bb in range(B)]\n\n        def tws_of(Tb, t=t):\n            u = tmask(Tb)\n            acc = {}\n            for n in range(npts):\n                if not tgt[n]:\n                    continue\n                v = gmul(Iv[n][t], u)\n                for T in tgt[n]:\n                    for mm in bitlist(v, B):\n                        acc[T[mm]] = acc.get(T[mm], 0) ^ 1\n            return [w for w in sorted(acc) if acc[w]]\n\n        bil_terms(rt, KT, sA, sB, tws_of=tws_of)\n    return [tm for tm in rt if tm[-1]]\n\n\ndef omega_rts(sig, syn):\n    """Return the in-place omega70 block term lists (MACS order) for the\n    monic sigma bank sig (words sig[1..W]) and the syndrome words syn.\n    Each block folds its evaluation maps into the operand parity specs\n    (sigma side over the bank, S side over the rows it reads) and its\n    interpolation rows into the KT target sets, so the a+b-1 pointwise\n    products are the only Toffoli gates.\n    """\n    out = []\n    for r0, a_, c0, b_ in MACS:\n        npts = a_ + b_ - 1\n        pts, Iv = vinv(npts)\n        rt = []\n        for t in range(npts):\n            cmA = [cmrows(Iv[k][t]) for k in range(npts)]\n            cmB = [cmrows(gpow(pts[t], j)) for j in range(b_)]\n            # diagonals beyond W are zero (clipped): sigma has W+1 coefficients\n            sA = [\n                frozenset(\n                    sig[1 + k][j] for k in range(min(npts, W)) for j in bitlist(cmA[k][bb], B)\n                )\n                for bb in range(B)\n            ]\n            sB = [\n                frozenset(\n                    syn[c0 + b_ - 1 - j][jb] for j in range(b_) for jb in bitlist(cmB[j][bb], B)\n                )\n                for bb in range(B)\n            ]\n            wcoef = [gpow(pts[t], i2) for i2 in range(a_)]\n\n            def om_tws(Tb, wcoef=wcoef, r0=r0, a_=a_):\n                u = tmask(Tb)\n                tws = []\n                for i2 in range(a_):\n                    tws.extend(syn[r0 + i2][m] for m in bitlist(gmul(wcoef[i2], u), B))\n                return tws\n\n            bil_terms(rt, KT, sA, sB, tws_of=om_tws)\n        out.append(rt)\n    return out\n\n\n# =========================================================================\n# Emission (term-list emitter, journaled vocabulary, allocators)\n# =========================================================================\n# Em emits the term lists of the Terms section through a live Builder: emit_mac\n# writes a term list forward (one CCX per product; a shared multi-target\n# product fans out through the sT scratch and erases it for free), and\n# emit_free erases a written term list by demolition measurement + phase\n# fixups, 0 Toffoli.  The journaled vocabulary (mac / xor / free / x /\n# cx / cswap / swap / swaps, and the ladder, chain and increment helpers\n# built on them) runs each step now and records its conjugate in a\n# uncompute.bennett.Journal, so a later Journal.reverse() replays the\n# conjugates over the same hard wire ids.\n#\n# Scratch comes from Em\'s current allocator, an Alloc: it round-trips\n# windows through the Builder allocator and logs the window ids a journal\n# touches, so a replay can hold exactly those ids (Alloc.replay).\n\n\ndef word_pairs(xs, ys):\n    """Bitwise wire pairs of two word lists: word by word, bit 0 first."""\n    return [(x[t], y[t]) for x, y in zip(xs, ys) for t in range(B)]\n\n\n# NOJ never records: unjournaled emission through the journaled vocabulary.\nNOJ = Journal()\nNOJ.recording = False\n\n\nclass Em:\n    """Term-list emitter over the held pivots sA, sB and the fan-out\n    scratch sT, with the journaled vocabulary.  The vocabulary records\n    into the current journal J (NOJ outside any using() block), and the\n    scratch helpers draw from the current allocator A.\n    """\n\n    def __init__(self, b, sA, sB, sT, A):\n        self.b = b\n        self.sA = sA\n        self.sB = sB\n        self.sT = sT\n        self.J = NOJ\n        self.A = A\n\n    @contextmanager\n    def using(self, J=None, A=None):\n        """Make J and/or A the current journal / allocator inside the block."""\n        prev = self.J, self.A\n        if J is not None:\n            self.J = J\n        if A is not None:\n            self.A = A\n        yield\n        self.J, self.A = prev\n\n    # ---- the unjournaled core --------------------------------------------\n    def _build(self, ws, cc, s):\n        if len(ws) == 1 and cc == 0:\n            return ws[0], False\n        for w in ws:\n            self.b.cx(w, s)\n        if cc:\n            self.b.x(s)\n        return s, True\n\n    def _unbuild(self, ws, cc, s, built):\n        if not built:\n            return\n        if cc:\n            self.b.x(s)\n        for w in reversed(ws):\n            self.b.cx(w, s)\n\n    def pivot_pair(self, wa, ca, wb, cb, mid):\n        """Emit the shared pivot-operand sandwich: build both pivot operands\n        into sA/sB, run the caller\'s middle 2-qubit op on the built\n        pivots (ua, ub), then unbuild in reverse.  One core for the\n        single-target CCX in emit_mac, the conditioned-CZ repayment in\n        emit_free, and fan_pair\'s full sT window.\n        """\n        ua, ba = self._build(wa, ca, self.sA)\n        ub, bb2 = self._build(wb, cb, self.sB)\n        mid(ua, ub)\n        self._unbuild(wb, cb, self.sB, bb2)\n        self._unbuild(wa, ca, self.sA, ba)\n\n    def fan_pair(self, wa, ca, wb, cb, mid):\n        """Emit the product-pair skeleton over pivot_pair: CCX the pivots\'\n        AND into the sT fan-out scratch, run the caller\'s mid(sT) on it,\n        then measure sT away with its phase repayment.\n        """\n        b = self.b\n\n        def sandwich(ua, ub):\n            b.ccx(ua, ub, self.sT)\n            mid(self.sT)\n            m = b.hmr(self.sT)\n            b.cz(ua, ub, cond=m)\n\n        self.pivot_pair(wa, ca, wb, cb, sandwich)\n\n    def emit_mac(self, rt):\n        b = self.b\n        for t in rt:\n            k = t[0]\n            if k == "L":\n                ws, cc, tws = t[1], t[2], t[3]\n                for tw in tws:\n                    for w in ws:\n                        b.cx(w, tw)\n                    if cc:\n                        b.x(tw)\n            else:\n                wa, ca, wb, cb, tws = t[1], t[2], t[3], t[4], t[5]\n                if len(tws) == 1:\n                    self.pivot_pair(wa, ca, wb, cb, lambda ua, ub, tw=tws[0]: b.ccx(ua, ub, tw))\n                else:\n\n                    def mid(fan, tws=tws):\n                        for tw in tws:\n                            b.cx(fan, tw)\n\n                    self.fan_pair(wa, ca, wb, cb, mid)\n\n    def emit_free(self, rt):\n        b = self.b\n        per = {}\n        order = []\n        for t in rt:\n            for tw in t[-1]:\n                if tw not in per:\n                    per[tw] = []\n                    order.append(tw)\n                per[tw].append(t)\n        for tw in reversed(order):\n            m = b.hmr(tw)\n            for t in per[tw]:\n                k = t[0]\n                if k == "L":\n                    for w in t[1]:\n                        b.z(w, cond=m)\n                    if t[2]:\n                        b.neg(cond=m)\n                else:\n                    self.pivot_pair(t[1], t[2], t[3], t[4], lambda ua, ub: b.cz(ua, ub, cond=m))\n\n    def swap_fn(self, c, pairs):\n        """One closure applying cswap(c, x, y) over pairs in order (c None:\n        plain swaps).\n        """\n        b = self.b\n        if c is None:\n\n            def fn():\n                for x, y in pairs:\n                    b.swap(x, y)\n        else:\n\n            def fn():\n                for x, y in pairs:\n                    b.cswap(c, x, y)\n\n        return fn\n\n    # ---- the journaled vocabulary ----------------------------------------\n    def mac(self, rt):\n        """Write a term list into ZERO targets; the conjugate erases it\n        for free.\n        """\n        self.J.step(lambda: self.emit_mac(rt), lambda: self.emit_free(rt))\n        return rt\n\n    def xor(self, rt):\n        """XOR-accumulate a term list into live targets: self-inverse."""\n        self.J.step(lambda: self.emit_mac(rt))\n        return rt\n\n    def free(self, rt):\n        """Measurement-erase targets that equal the term list (a claim);\n        the conjugate recomputes them.\n        """\n        self.J.step(lambda: self.emit_free(rt), lambda: self.emit_mac(rt))\n\n    def macs(self, rts):\n        for rt in rts:\n            self.mac(rt)\n\n    def frees(self, rts):\n        """Erase term lists in reverse order."""\n        for rt in reversed(rts):\n            self.free(rt)\n\n    def x(self, q):\n        self.J.step(lambda: self.b.x(q))\n\n    def cx(self, c, t):\n        self.J.step(lambda: self.b.cx(c, t))\n\n    def cswap(self, c, x, y):\n        self.J.step(lambda: self.b.cswap(c, x, y))\n\n    def swap(self, x, y):\n        self.J.step(lambda: self.b.swap(x, y))\n\n    def swaps(self, c, pairs):\n        """cswap(c, .) (c None: swap) over pairs as ONE journal step."""\n        self.J.step(self.swap_fn(c, pairs))\n\n    def ladder(self, rungs, tree):\n        """Emit an AND ladder (ladder_rt) and return its term\n        lists; the caller consumes the top and erases with frees().\n        """\n        rts = ladder_rt(rungs, tree)\n        self.macs(rts)\n        return rts\n\n    def inc1(self, x, tree):\n        """x += 1 (mod 2^len(x)) as a constant ripple: len(x) - 2 CCX\n        ladder carries into tree (len(x) - 2 zero wires), each erased by\n        measurement right after its use.\n        """\n        crts = self.ladder([((x[i],), 0) for i in range(len(x) - 1)], tree)\n        for j in range(len(x) - 1, 1, -1):\n            self.cx(tree[j - 2], x[j])\n            self.free(crts[j - 2])\n        self.cx(x[0], x[1])\n        self.x(x[0])\n\n\ndef chain_open(E, src, out):\n    """Open the inversion-chain scratch on E\'s allocator and build the\n    chain term lists for out = inv(src) (chain_rts).\n    """\n    wins = [E.A.open(name, n) for (name, n) in CHAIN_WINS]\n    return wins, chain_rts(src, wins, out)\n\n\ndef chain_close(E, wins):\n    for (name, _n), win in zip(reversed(CHAIN_WINS), reversed(wins)):\n        E.A.close(name, win)\n\n\ndef chain_fwd(E, src, out):\n    wins, rts = chain_open(E, src, out)\n    E.macs(rts)\n    return wins, rts\n\n\ndef chain_undo(E, wins, rts):\n    E.frees(rts)\n    chain_close(E, wins)\n\n\n@contextmanager\ndef chain_live(E, src, out):\n    """out <- inv(src) with the chain intermediates live inside the block;\n    exit erases out and the intermediates.\n    """\n    wins, rts = chain_fwd(E, src, out)\n    yield\n    chain_undo(E, wins, rts)\n\n\n@contextmanager\ndef opened(E, name, n):\n    """Open an n-wire scratch window on E\'s allocator; exit closes it."""\n    qs = E.A.open(name, n)\n    yield qs\n    E.A.close(name, qs)\n\n\n@contextmanager\ndef eq_test(E, bits, val):\n    """Yield the top wire of a [bits == val] ladder; exit\n    erases the ladder.\n    """\n    with opened(E, "tree", len(bits) - 1) as tr:\n        rts = E.ladder(eq_rungs(bits, val), tr)\n        yield tr[len(bits) - 2]\n        E.frees(rts)\n\n\nclass Alloc:\n    """Scratch windows through the Builder allocator: open allocs and\n    claims a window, close releases it and frees it in reverse alloc\n    order.  Journals replay closures over hard wire ids, so open records\n    each window id into the current log set (logging()), and replay()\n    re-materializes exactly a logged set via reclaim before a replay.\n    """\n\n    def __init__(self, b):\n        self.b = b\n        self.log = None  # the set open records window ids into, if any\n        self.round = None  # BM round tag for bank bookkeeping\n        self.bank_log = {}  # round -> bank registers first touched there\n\n    def open(self, name, n):\n        qs = self.b.qubits(n)\n        if self.log is not None:\n            self.log.update(qs)\n        self.b.claim(name, qs)\n        return qs\n\n    def reopen(self, name, qs):\n        """Re-acquire these exact (free) ids as a window."""\n        self.b.reclaim(qs)\n        self.b.claim(name, qs)\n\n    def close(self, name, qs):\n        self.b.release_recycle(name, qs)\n\n    @contextmanager\n    def logging(self, ids):\n        """Record every window id opened inside the block into the set ids."""\n        prev, self.log = self.log, ids\n        yield\n        self.log = prev\n\n    @contextmanager\n    def replay(self, ids, name="jscratch"):\n        """Hold exactly the (free) ids inside the block, for a journal replay."""\n        pool = sorted(ids)\n        if pool:\n            self.reopen(name, pool)\n        yield\n        if pool:\n            self.close(name, pool)\n\n\nclass BankReg:\n    """One lazy bank register: B wires materialized at the first operand\n    touch (allocated, claimed into the bank\'s set, and logged with the\n    touch round).  The fused-Q rotation relabels handles for free, so a\n    physical slot\'s first touch is discovered mechanically instead of\n    derived from the clipped-window schedule.  The mirror replay of the\n    touch round returns the slot to |0> (post-replay state equals the\n    forward state before that round), which is where the reverse pass\n    frees it.\n    """\n\n    __slots__ = ("A", "name", "round", "width", "wires")\n\n    def __init__(self, A, name, width=B):\n        self.A = A\n        self.name = name\n        self.width = width\n        self.wires = None\n        self.round = None\n\n    def mat(self):\n        if self.wires is None:\n            A = self.A\n            self.wires = A.b.qubits(self.width)\n            A.b.claim(self.name, self.wires)\n            self.round = A.round\n            A.bank_log.setdefault(A.round, []).append(self)\n        return self.wires\n\n    def __getitem__(self, k):\n        return self.mat()[k]\n\n    def __iter__(self):\n        return iter(self.mat())\n\n    def __len__(self):\n        return self.width\n\n\n# =========================================================================\n# Psi route (half-degree Euclid F, R, cofactor recurrence)\n# =========================================================================\n# README.md, "Division-free Forney recovery".  The half-degree Euclid\n# pass F compresses the padded locator halves (E\', O\') into quotients;\n# R replays F backwards (restoring E\', O\') while the cofactor recurrence\n# CR builds the Bezout pair (a, b), A E\' + B O\' = z^m, so\n# Psi = B^2 + z A^2 = 1/sigma_m\' at every root.  Every step runs\n# through a uncompute.bennett.Journal over hard wire ids: F records into its\n# journal JF; R is a deferred journal that replays JF\'s block segments\n# backwards (interleaved with the CR steps), R^-1 replays them forwards,\n# and F^-1 replays all of JF backwards.  The allocator logs every scratch\n# id these journals touch (PSI_IDS), and R and R^-1..F^-1 hold exactly that\n# set, so every replay reuses exactly the forward wire ids.  gen_constants.py holds the bit-exact\n# classical model of these gadgets, op for op.\n\n\ndef flag_nz(E, word, q):\n    """q ^= [word != 0]: a zero-test ladder, then NOT top into q (free:\n    CX + X, one journal step).\n    """\n    b = E.b\n    with eq_test(E, word, 0) as top:\n\n        def fn():\n            b.cx(top, q)\n            b.x(q)\n\n        E.J.step(fn)\n\n\n@contextmanager\ndef inv_word(E, src):\n    """Yield out = inv(src) (66 Toffoli), chain intermediates erased for\n    free; exit erases out (36 Toffoli: the intermediates are recomputed).\n    """\n    with opened(E, "inv", B) as out:\n        wins, rts = chain_fwd(E, src, out)\n        chain_undo(E, wins, rts[:-1])\n        yield out\n        # the recompute runs rts built on the chain\'s exact ids\n        for (name, _n), w in zip(CHAIN_WINS, wins):\n            E.A.reopen(name, w)\n        E.macs(rts[:-1])\n        E.free(rts[-1])\n        E.frees(rts[:-1])\n        chain_close(E, wins)\n\n\ndef pad_barrel(E, mbits, ew, ow):\n    """Shift both halves up by m = the 5-bit register mbits (non-cyclic:\n    the barrel of cswap chains, one per set bit of m).\n    """\n    for sft in (16, 8, 4, 2, 1):\n        ctl = mbits[sft.bit_length() - 1]\n        for X in (ew, ow):\n            for k in range(NH - 1 - sft, -1, -1):\n                E.swaps(ctl, word_pairs([X[k]], [X[k + sft]]))\n\n\ndef psi_pre(E, ew, ow, pre):\n    """pre <- [E\'_17 == 0]; cswap the halves on pre (odd L with\n    sigma_{L-1} = 0).  pre stays live until the literal reverse.\n    """\n    with eq_test(E, ew[17], 0) as top:\n        E.cx(top, pre)\n    E.swaps(pre, word_pairs(ew, ow))\n\n\ndef psi_forward(E, f, g, eps, idx):\n    """F on the Euclid bank (f, g: lists of 18 words, top-aligned, f[17]\n    the pivot): round 1, the block-0 zero-pivot repair, then 17 blocks\n    s0..s9.  Mirrors gen_constants.py forward() op for op.  Every free\n    erase claim holds whenever f17 != 0 at the elimination, i.e. on every\n    correct decode (the pre-swap and padding guarantee deg E\' = 17) and on\n    every garbage locator the model has seen.  Returns the final (f, g)\n    word arrays and the per-block records that build_R consumes: the\n    flag / c / q wires and the journal marks where R interleaves the\n    cofactor ops (start; swap, after the flag cswap; q, quotients ready;\n    c, c set; end).\n    """\n    f, g = list(f), list(g)\n    recs = {}\n    # ---- round 1: q1 = g17/f17; g -= q1 f; store q1 in g17\'s word ----\n    mk = {"start": E.J.mark(), "swap": E.J.mark()}  # block 0 swaps on pre, before F\n    with inv_word(E, f[17]) as fi:\n        with opened(E, "q1", B) as q1:\n            E.mac(prod_rt(q1, g[17], fi))\n            E.free(prod_rt(g[17], q1, f[17]))  # claim g17 == q1 f17\n            E.swaps(None, word_pairs([q1], [g[17]]))  # g17\'s word <- q1\n        mk["q"] = E.J.mark()  # q1 lives in g[17]\'s word\n        q1w = g[17]\n        E.xor([op for k in range(17) for op in prod_rt(g[k], q1w, f[k])])\n    g = g[17:] + g[:17]  # rot_up (relabel)\n    # ---- block 0: zero-pivot test + repair (degree defect after round 1)\n    with eq_test(E, g[17], 0) as top:\n        E.cx(top, eps)  # eps <- [g17 == 0]\n    mk["c"] = E.J.mark()\n    for k in range(17, 1, -1):  # rot_up(g, lo=1) on eps\n        E.swaps(eps, word_pairs([g[k]], [g[k - 1]]))\n    mk["end"] = E.J.mark()\n    recs[0] = {"marks": mk, "c": eps, "q": [q1w]}\n    for j in range(1, 18):\n        mk = {"start": E.J.mark()}\n        f[j - 1], g[j - 1] = g[j - 1], f[j - 1]  # boundary swap (relabel)\n        with opened(E, "flag", 1) as (flag,):\n            # s0: set flag to [g17 != 0]\n            flag_nz(E, g[17], flag)\n            # s1: cswap window [j, 17] on flag\n            E.swaps(flag, word_pairs([f[k] for k in range(j, 18)], [g[k] for k in range(j, 18)]))\n            mk["swap"] = E.J.mark()\n            # s2: erase flag against [g17 != 0] (post-swap)\n            flag_nz(E, g[17], flag)\n        # s3: invert f17 into fi; s9 (the with exit) erases it\n        with inv_word(E, f[17]) as fi:\n            # s4: q_hi, t, q_lo\n            with opened(E, "qh", B) as qh, opened(E, "ql", B) as ql:\n                E.mac(prod_rt(qh, g[17], fi))\n                if j <= 16:\n                    with opened(E, "t", B) as t:\n                        trt = E.mac(prod_rt(t, qh, f[16]))\n                        rtl = []\n                        bil_terms(\n                            rtl,\n                            KT,\n                            [frozenset((g[16][i], t[i])) for i in range(B)],\n                            spec_plain(fi),\n                            ql,\n                        )\n                        E.mac(rtl)\n                        E.free(trt)\n                else:\n                    E.mac(prod_rt(ql, g[16], fi))\n                mk["q"] = E.J.mark()\n                # s5: block elimination: target word g[j-1+n] ^= coef_n of\n                # (ql + qh y) * (f[j] + f[j+1] y + ... + f[16] y^(16-j)), n = 0..17-j\n                if j <= 16:\n                    E.xor(\n                        polyprod_rt(\n                            [ql, qh],\n                            [f[k] for k in range(j, 17)],\n                            [[g[j - 1 + n]] for n in range(18 - j)],\n                        ),\n                    )\n                E.free(prod_rt(g[17], qh, f[17]))  # claim g17 == qh f17\n                E.free(prod_rt(g[16], ql, f[17]))  # claim g16\' == ql f17\n                E.swaps(None, word_pairs([qh, ql], [g[17], g[16]]))\n            g = g[16:] + g[:16]  # rot_up twice (relabel)\n            with opened(E, "c", 1) as (c,):\n                # s6: c = [g17 == 0] AND NOT eps\n                with eq_test(E, g[17], 0) as top:\n                    E.mac(one_rt((top,), 0, (eps,), 1, c))\n                mk["c"] = E.J.mark()\n                # s7: controlled rot_up(g, lo=j+1) on c; eps ^= c; idx ^= c*j\n                for k in range(17, j + 1, -1):\n                    E.swaps(c, word_pairs([g[k]], [g[k - 1]]))\n                E.cx(c, eps)\n                for bp in bitlist(j, 5):\n                    E.cx(c, idx[bp])\n                # s8: erase c == eps AND [idx == j]\n                with eq_test(E, idx, j) as top:\n                    E.free(one_rt((top,), 0, (eps,), 0, c))\n        mk["end"] = E.J.mark()\n        recs[j] = {"marks": mk, "flag": flag, "c": c, "q": [ql, qh]}\n    return f, g, recs\n\n\ndef build_R(E, JF, recs, A, Bw, pre, win, eb):\n    """R as a deferred Journal: JF\'s block segments replayed backwards,\n    interleaved with the cofactor recurrence CR on the 18-word cyclic\n    registers A (a) and Bw (b).  win maps (\'mul\'|\'rz\'|\'swap\', j) to the\n    measured union support (lo, n) of the register(s) the op touches\n    (array index frame).  R.forward() runs R; R.reverse() runs R^-1.\n    Returns (R, a, b), the cofactor word lists in their final labels.\n    """\n    a, bb = list(A), list(Bw)\n    R = Journal()\n\n    def seg_rev(x, y):\n        R.defer(lambda: JF.reverse(x, y), lambda: JF.forward(x, y))\n\n    def cr_rz_ctl(c, j):\n        # controlled cyclic rotation up by one (new ws[k] = old ws[k-1])\n        # of the window words: len-1 word cswaps\n        lo, n = win[("rz", j)]\n        if n == 0:\n            return\n        ws = [bb[(lo + i) % NH] for i in range(min(n + 1, NH))]\n        hi = [ws[k] for k in range(len(ws) - 1, 0, -1)]\n        low = [ws[k - 1] for k in range(len(ws) - 1, 0, -1)]\n        R.defer(E.swap_fn(c, word_pairs(hi, low)), E.swap_fn(c, word_pairs(hi[::-1], low[::-1])))\n\n    def cr_mul(qws, j):\n        # b <- z^len(qws) b (relabel); a ^= (sum_i q_i z^(i+1)) b_old,\n        # with qws listed low -> high (ql, qh for a block; [q1] for round 1)\n        b0 = list(bb)\n        lo, n = win[("mul", j)]\n        if n:\n            Fw = [b0[(lo + i) % NH] for i in range(n)]\n            tgt = [[a[(lo + k + 1) % NH]] for k in range(len(qws) + n - 1)]\n            rt = polyprod_rt(list(qws), Fw, tgt)\n            R.defer(lambda: E.emit_mac(rt))\n        return b0[-len(qws) :] + b0[: -len(qws)]\n\n    def cr_swap(c, j):\n        lo, n = win[("swap", j)]\n        ks = [(lo + i) % NH for i in range(n)]\n        R.defer(E.swap_fn(c, word_pairs([a[k] for k in ks], [bb[k] for k in ks])))\n\n    for j in range(17, -1, -1):\n        r = recs[j]\n        mk = r["marks"]\n        seg_rev(mk["c"], mk["end"])\n        cr_rz_ctl(r["c"], j)\n        seg_rev(mk["q"], mk["c"])\n        bb = cr_mul(r["q"], j)\n        seg_rev(mk["swap"], mk["q"])\n        cr_swap(pre if j == 0 else r["flag"], j)\n        seg_rev(mk["start"], mk["swap"])\n    # undo the pre-swap of the halves too: the sweep\'s root test needs E\'\n    # in the E words (psi_pre\'s swap itself stays live until the literal\n    # reverse, which redoes this swap first)\n    ew, ow = eb\n    R.defer(E.swap_fn(pre, word_pairs(ew, ow)))\n    return R, a, bb\n\n\ndef psi_seed(E, h, mbits, A):\n    """A[(m+1) % 18] ^= inv(h), m = the 5-bit register mbits (0..17):\n    one [m == v] ladder (4 Toffoli) and one controlled 12-bit copy per v\n    (12 Toffoli, paid again by the literal reverse).\n    """\n    with inv_word(E, h) as ih:\n        for v in range(NH):\n            with eq_test(E, mbits, v) as top:\n                # XOR-accumulate, recomputed on the reverse: the TAIL word is zero\n                # only on a correct decode, so a measurement erase here would\n                # break syndrome restoration on approximate-BM paths\n                E.xor([norm_term((top,), 0, (ih[t],), 0, [A[(v + 1) % NH][t]]) for t in range(B)])\n\n\n# =========================================================================\n# Sweep (Chien/Forney root test + Psi pairing, one gated repeat body)\n# =========================================================================\n# Per point x = gamma^t (error position j = M - t): with E\'_k, b_k read as\n# if rotated by gamma^(kt) and O\'_k, a_k by gamma^((k+SQRT_EXP)t) (parity sets\n# over the unrotated words; no rotation is emitted),\n#   XOR of the 36 EB words = E\'(x) + sqrt(x) O\'(x) = x^m sqrt(sigma_m(x)),\n#   XOR of the 36 (a, b) words = V = B(x) + sqrt(x) A(x),  Psi(x) = V^2,\n# so flag = [XOR of EB == 0] (11 Toffoli) and the Forney bit\n# Tr(lam omega(x) / sigma_m\'(x)) = Tr(lam omega(x) Psi(x)) = Tr(sqrt(lam U) V)\n# = sum_a parity(r_a(lam) . U) V_a (U = omega(x), loaded into som): 12\n# flag-gated rank-1 terms, one Toffoli each.  23 Toffoli per point.\n#\n# Loop form: the words never move, so every CX of every point acts on\n# fixed wires; only WHICH stored bits a fan-in reads changes with t.  At\n# point t, word k would hold g_k^t times its stored value, so bit bb of the\n# rotated word is the parity of the stored bits selected by row bb of\n# cmrows(g_k^t).  The body emits the union of each fan-in\'s possible bits\n# in the per-point emission order, each CX data-gated on its membership\n# bit at t, so a point\'s executed ops are exactly its per-point spelling\n# (the Em core\'s pivot_pair / fan_pair / emit_free order).  Every other op\n# is gated on the point being live (lam != 0).  A pairing row r_a(lam)\n# with a single bit uses that som wire directly (no sA build), so the\n# pairing CCX/CZ appear once per case (built sA, or som[c]), each gated on\n# its case.\n\nLOG_EB = list(range(NH)) + [(k + SQRT_EXP) % M for k in range(NH)]\nLOG_AB = [(k + SQRT_EXP) % M for k in range(NH)] + list(range(NH))\nCM_ARR = np.array(\n    [[0] * B] + [cmrows(EXP[e]) for e in range(M)], dtype=np.int64\n)  # CM_ARR[1 + e, bb]: cmrows(gamma^e)\n\n\ndef rot_member(logs):\n    """mem[bb, k, j, t]: bit j of word k is in operand group bb at t."""\n    t = np.arange(M, dtype=np.int64)\n    mem = np.zeros((B, len(logs), B, M), dtype=bool)\n    for k, lg in enumerate(logs):\n        rows = CM_ARR[1 + (lg * t) % M]  # [t, bb]\n        for j in range(B):\n            mem[:, k, j, :] = ((rows >> j) & 1).T.astype(bool)\n    return mem\n\n\nLIVE = np.array([bool(LAMJ2[(M - t) % M]) for t in range(M)])\nassert int(LIVE.sum()) == NLIVE\n\n\ndef sweep_plan(b, A, omw, ebw, abw):\n    """Return the body as [(gate vector over t, emitter(s_))] in emission order,\n    s_ the body\'s wire handles; a None vector emits ungated (the tree\n    window open on allocator A).  Mirrors the per-point form: som fan-ins, the root-test\n    ladder (pivot_pair builds, sorted operands), the rank-12 pairing\n    (fan_pair), the ladder erase (emit_free, reversed), the som unwind.\n    """\n    m_om = rot_member(list(range(W))) & LIVE  # som fan-ins\n    m_eb = rot_member(LOG_EB) & LIVE  # root-test groups\n    m_v = rot_member(LOG_AB) & LIVE  # pairing V groups\n    prow = np.zeros((B, M), dtype=np.int64)  # r_a(lam) masks\n    for t in range(M):\n        if LIVE[t]:\n            prow[:, t] = pair_rows(LAMJ2[(M - t) % M])\n    single = np.stack([prow == (1 << c) for c in range(B)], axis=1)  # [a, c, t]\n    built = ~single.any(axis=1) & LIVE  # [a, t]\n    m_sa = np.stack([((prow >> c) & 1).astype(bool) & built for c in range(B)], axis=1)  # [a, c, t]\n    # the per-point spelling the body mirrors: every root-test group is\n    # non-empty (both pivots build), every V group has >= 2 wires (it builds)\n    assert (m_eb.any(axis=(1, 2)) | ~LIVE).all(), "empty root-test group"\n    assert (m_v.sum(axis=(1, 2)) >= 2)[:, LIVE].all(), "degenerate V group"\n    eb_sorted = sorted(\n        (w, k, j) for k, word in enumerate(ebw) for j, w in enumerate(word)\n    )  # terms sort operands\n    assert len({w for w, _, _ in eb_sorted}) == len(eb_sorted)\n\n    plan = []\n\n    def op(vec, fn):\n        plan.append((vec, fn))\n\n    def eb_build(g, s, inverse):\n        cands = [(w, m_eb[g, k, j]) for w, k, j in eb_sorted]\n        if inverse:\n            op(LIVE, lambda s_: b.x(s_[s]))\n            cands.reverse()\n        for w, v in cands:\n            op(v, lambda s_, w=w: b.cx(w, s_[s]))\n        if not inverse:\n            op(LIVE, lambda s_: b.x(s_[s]))\n\n    def ua(i, s_):\n        return s_["sA"] if i == 1 else s_["tr"][i - 2]\n\n    def rung(i, mid):\n        """ladder rung i >= 1: wa = group 0 (i == 1, built in sA) or\n        tree[i-2] (direct), wb = group i (built in sB).\n        """\n        if i == 1:\n            eb_build(0, "sA", False)\n        eb_build(i, "sB", False)\n        op(LIVE, mid)\n        eb_build(i, "sB", True)\n        if i == 1:\n            eb_build(0, "sA", True)\n\n    om = [(bb, k, j) for bb in range(B) for k in range(W) for j in range(B)]\n    for bb, k, j in om:\n        op(m_om[bb, k, j], lambda s_, bb=bb, k=k, j=j: b.cx(omw[k][j], s_["som"][bb]))\n    op(None, lambda s_: s_.__setitem__("tr", A.open("tree", B - 1)))\n    for i in range(1, B):\n        rung(i, lambda s_, i=i: b.ccx(ua(i, s_), s_["sB"], s_["tr"][i - 1]))\n    vw = [(k, j) for k in range(len(abw)) for j in range(B)]\n    for a_ in range(B):\n        cases = [(built[a_], lambda s_: s_["sA"])] + [\n            (single[a_, c] & LIVE, lambda s_, c=c: s_["som"][c]) for c in range(B)\n        ]\n        for c in range(B):\n            op(m_sa[a_, c], lambda s_, c=c: b.cx(s_["som"][c], s_["sA"]))\n        for k, j in vw:\n            op(m_v[a_, k, j], lambda s_, k=k, j=j: b.cx(abw[k][j], s_["sB"]))\n        for v, u in cases:\n            op(v, lambda s_, u=u: b.ccx(u(s_), s_["sB"], s_["sT"]))\n        op(LIVE, lambda s_: b.cz(s_["tr"][B - 2], s_["sT"]))\n        op(LIVE, lambda s_: s_.__setitem__("m", b.hmr(s_["sT"])))\n        for v, u in cases:\n            op(v, lambda s_, u=u: b.cz(u(s_), s_["sB"], cond=s_["m"]))\n        for k, j in reversed(vw):\n            op(m_v[a_, k, j], lambda s_, k=k, j=j: b.cx(abw[k][j], s_["sB"]))\n        for c in reversed(range(B)):\n            op(m_sa[a_, c], lambda s_, c=c: b.cx(s_["som"][c], s_["sA"]))\n    for i in range(B - 1, 0, -1):\n        op(LIVE, lambda s_, i=i: s_.__setitem__("m", b.hmr(s_["tr"][i - 1])))\n        rung(i, lambda s_, i=i: b.cz(ua(i, s_), s_["sB"], cond=s_["m"]))\n    op(LIVE, lambda s_: A.close("tree", s_["tr"]))\n    for bb, k, j in reversed(om):\n        op(m_om[bb, k, j], lambda s_, bb=bb, k=k, j=j: b.cx(omw[k][j], s_["som"][bb]))\n    return plan\n\n\ndef emit_sweep(b, E, A, omw, ebw, abw):\n    """Emit the sweep: one gate-table row of M bits per distinct gate\n    vector (the forward and unwind passes share rows; row r gates\n    iteration t on bit r * M + t), then the repeat(M) body over the held\n    pivots and the som / tree windows of allocator A.\n    """\n    plan = sweep_plan(b, A, omw, ebw, abw)\n    rows = {}\n    for vec, _ in plan:\n        if vec is not None:\n            rows.setdefault(vec.tobytes(), vec)\n    rows.setdefault(LIVE.tobytes(), LIVE)\n    row_of = {key: r for r, key in enumerate(rows)}\n    table = b.declare_table(np.concatenate(list(rows.values())).astype(np.uint8).tolist())\n    with b.repeat(M) as it:\n\n        def gate(vec):\n            return b.gated(table, it + row_of[vec.tobytes()] * M)\n\n        s_ = {"sA": E.sA, "sB": E.sB, "sT": E.sT, "som": A.open("som", B)}\n        for vec, fn in plan:\n            if vec is None:\n                fn(s_)\n            else:\n                with gate(vec):\n                    fn(s_)\n        with gate(LIVE):\n            A.close("som", s_["som"])\n\n\n# =========================================================================\n# BM (stage 2 forward, stage 10 journaled reverse)\n# =========================================================================\ndef flag_test(E, src, gate, ctl):\n    """ctl <- [src != 0] AND parity(gate) (gate=() folds to the bare\n    nonzero test): the tree window, the (B-1)-rung zero-test ladder, the\n    flag AND, and the rung teardown.\n    local: every zero-test tree here has gadgets.mcx\'s n-1 Toffoli cost,\n    but must hold its flag live across later gates and journal each step\n    for the reverse pass; mcx computes and releases inside one call and\n    records nothing.\n    """\n    with eq_test(E, src, 0) as top:\n        E.mac(one_rt((top,), 1, gate, 1, ctl))\n\n\nclass BM:\n    """The q-form conditional-swap BM forward (stage 2) and its journaled\n    reverse (stage 10).  Round 0 and round N-1 are peeled; the general\n    round is a sequence of step methods over explicit round state.  run()\n    makes the BM journal J current, so every step records\n    into it; JROUND holds the\n    journal index at the seed start and at every round start.\n    """\n\n    def __init__(self, E, A, syn, Dw, dW):\n        self.b, self.E, self.A, self.J = E.b, E, A, Journal()\n        self.J.recording = False\n        self.pools = {}  # round -> window ids its journal touches\n        self.syn, self.Dw, self.dW = syn, Dw, dW\n        self.SIG = [BankReg(A, "sig") for _ in range(W + 1)]  # sigma_0..sigma_W\n        self.BB = [BankReg(A, "bb") for _ in range(W + 2)]  # B bank (fused-Q: wrap at W+1)\n        # STK_CAP-slot LIFO push stack, STKW wires per slot: B value lanes + a\n        # TAGB-bit round tag.  The tag records the pushing round and gives\n        # every round\'s control flag an erase anchor at the round tail --\n        # after the d complement, ctl == dW[SGN] AND NOT [top tag == r] -- so\n        # no flag outlives its round.\n        self.STK = [BankReg(A, "stk", STKW) for _ in range(STK_CAP)]\n        assert max(CAPS) <= len(self.STK), ("push cascade would overrun STK", max(CAPS))\n        self.JROUND = [0]\n\n    def round_head(self, r, bb_l):\n        """Open round r (inside its bm/rNN phase block): journal mark, the\n        round\'s cascade stack slots, and the fused-Q bank rotation (a\n        relabel).  Returns the rotated bank.\n        """\n        self.A.round = r\n        self.A.log = self.pools.setdefault(r, set())\n        self.JROUND.append(self.J.mark())\n        # Lazy-materialization invariant: a register touched mid-round may\n        # only take free-list ids that strictly earlier rounds\' windows freed\n        # (those return before the descending replay reaches them).  At the\n        # bank/ctl touch sites this holds because the round\'s own freed ids\n        # are always reconsumed by the windows still open there; the cascade\n        # runs after every window closed, so its stack depths materialize at\n        # the round head instead.  reclaim fails closed on any breach.\n        for s in self.STK[: CAPS[r]]:\n            s.mat()\n        return [bb_l[W + 1]] + bb_l[: W + 1]\n\n    def discrepancy(self, r):\n        """Dw <- sum_{i <= IMAXV[r]} sigma_i S_{r-i} (the truncated window)."""\n        rtD = []\n        for i in range(IMAXV[r] + 1):\n            bil_terms(rtD, KT, spec_plain(self.SIG[i]), spec_plain(self.syn[r - i]), self.Dw)\n        self.E.mac(rtD)\n\n    @contextmanager\n    def quotient(self, ctl, bb_l):\n        """Yield the round quotient m without a stored accepted discrepancy.\n        Invariant (README): sigma_0 * DEL == 1 on every\n        correct-path round, DEL the accepted discrepancy of the q-form\n        recursion, so the recursion\'s m = D\'/DEL\' is\n          ctl=0: m = Delta * sigma_0 = p,\n          ctl=1: m = DEL_old / Delta = 1/(Delta * sigma_0) = inv(p).\n        Exit erases m (a function of p, inv(p), ctl), the chain and inv(p),\n        then p.  Dw = Delta on every branch there; p = Delta * sigma_0_old,\n        and sigma_0_old now sits in bb_l[0] (ctl=1) or SIG[0] (ctl=0, where\n        Delta != 0 forces bb_l[0] == 0); on ctl=1 the update left SIG[0] =\n        1/Delta, so Delta*(bb_l[0] + SIG[0]) = p + ctl:\n          p == Dw * (bb_l[0] ^ SIG[0]) ^ ctl*[1]   (one KT claim).\n        """\n        E = self.E\n        with opened(E, "pq", B) as pv:\n            E.mac(prod_rt(pv, self.Dw, self.SIG[0]))\n            with opened(E, "dinv", B) as dv, chain_live(E, pv, dv), opened(E, "m", B) as mq:\n                # m = p ^ ctl*(p ^ inv(p)): B free copies + B CCX\n                rtM = []\n                for t_ in range(B):\n                    rtM.append(("L", (pv[t_],), 0, (mq[t_],)))\n                    rtM.append(norm_term((ctl,), 0, tuple(sorted((pv[t_], dv[t_]))), 0, (mq[t_],)))\n                E.mac(rtM)\n                yield mq\n                E.free(rtM)\n            rtPe = []\n            bil_terms(\n                rtPe,\n                KT,\n                spec_plain(self.Dw),\n                [frozenset((bb_l[0][j], self.SIG[0][j])) for j in range(B)],\n                pv,\n            )\n            rtPe.append(("L", (ctl,), 0, (pv[0],)))\n            E.free(rtPe)\n\n    def bank_swaps(self, r, ctl, bb_l):\n        """cswap(ctl) of the sigma and B banks over slots 0..KMSV[r]."""\n        for k2 in range(KMSV[r] + 1):\n            if r == 1 and k2 == 1:\n                # provable identity: both branches leave sigma_1 ==\n                # B_1 entering this swap (2-branch enumeration)\n                continue\n            for t_ in range(B):\n                self.E.cswap(ctl, self.SIG[k2][t_], bb_l[k2][t_])\n\n    def update(self, r, k0, mq, bb_l):\n        """sigma_k ^= m * B_k over slots k0..KMMV[r] (one KT12 each)."""\n        for k2 in range(k0, KMMV[r] + 1):\n            self.E.xor(prod_rt(self.SIG[k2], mq, bb_l[k2]))\n\n    def push(self, r, ctl=True):\n        """Emit the fused-Q push cascade on pqc = dW[SGN] (d<0 forces ctl=0);\n        ctl=False spells it with plain swaps (the unconditional r=N-1\n        push, 0 Toffoli, no tag).  Extent CAPS[r] = min(DP extent,\n        STK_CAP): the CLAMPED Fredkin ladder still acts on pairwise-distinct\n        qubits; on a full stack it rotates STK[STK_CAP-1] into Dw (overflow\n        drop, the E2 class); on a not-full stack it is the identical\n        shift as the uncapped ladder.  Inter-slot swaps carry all STKW\n        lanes; the Dw level is value-only.\n        """\n        E, STK = self.E, self.STK\n        capr = CAPS[r]\n        if not capr:\n            return\n        pqc = self.dW[SGN]\n\n        def sw(x, y):\n            if ctl:\n                E.cswap(pqc, x, y)\n            else:\n                E.swap(x, y)\n\n        for i2 in range(capr - 2, -1, -1):\n            for t_ in range(STKW):\n                sw(STK[i2][t_], STK[i2 + 1][t_])\n        for t_ in range(B):\n            sw(self.Dw[t_], STK[0][t_])\n        if not ctl:\n            return\n        # tag the incoming entry with its round: the vacated slot\'s tag lanes\n        # are |0> below overflow, so free CX from the push flag writes r\n        # exactly (overflow shots are the E2 class)\n        for bp in bitlist(r, TAGB):\n            E.cx(pqc, STK[0][B + bp])\n\n    def erase_dw(self, ctl):\n        """Erase Dw.  Post-cascade Dw == ctl * Delta on every branch (d<0:\n        pushed, Dw = 0; ctl=0, d>=0: Delta = 0), and ctl=1 left sigma_0 =\n        1/Delta: Dw == ctl * inv(sigma_0), one chain on SIG[0] and a B-term\n        claim (stack overflow is the E2 class).\n        """\n        E = self.E\n        with opened(E, "wreg", B) as wr, chain_live(E, self.SIG[0], wr):\n            E.free([norm_term((ctl,), 0, (wr[t_],), 0, (self.Dw[t_],)) for t_ in range(B)])\n\n    def close_ctl(self, r, ctl):\n        """d complement if ctl, then erase ctl (the caller closes its\n        window).  Post-complement dW[SGN] = [ctl OR pushed], and a push at\n        this round iff the top tag reads r,\n        so ctl == dW[SGN] AND NOT [top tag == r].  Rounds the DP proves\n        push-free (and r=0) anchor on dW[SGN] alone, for free.  On a stack\n        overflow the rotated-out tag corrupts the compare -- overflow shots\n        are already the E2 class.\n        """\n        E, dW = self.E, self.dW\n        for i in range(DWB):\n            E.cx(ctl, dW[i])\n        if CAPS_FULL[r] == 0:\n            E.free([norm_term((dW[SGN],), 0, (), 1, (ctl,))])\n        else:\n            tag = self.STK[0][B:]  # materialized at round_head\n            with opened(E, "eqt", TAGB - 1) as eqw:\n                flips = bitlist(~r & TAGMASK, TAGB)\n                for bp in flips:\n                    E.x(tag[bp])\n                eqr = E.ladder([((tag[i],), 0) for i in range(TAGB)], eqw)\n                E.free([norm_term((dW[SGN],), 0, (eqw[TAGB - 2],), 1, (ctl,))])\n                E.frees(eqr)\n                for bp in flips:\n                    E.x(tag[bp])\n\n    def inc_d(self):\n        """d += 1: the constant +1 ripple on dW (DWB-2 CCX, MBU-erased\n        journaled carries).\n        local: gadgets.arith.add has no constant-operand form (an all-None\n        operand is a no-op) and no journal hook.\n        """\n        with opened(self.E, "inc", DWB - 2) as ic:\n            self.E.inc1(self.dW, ic)\n\n    def round0(self, bb_l):\n        """r=0 needs no working discrepancy register: D(0)=S_1 lives in\n        syn[0].  ctl = [S_1 != 0] off syn[0], and the round quotient is\n        m = inv(S_1) on both branches (the chain maps 0 to 0).\n        """\n        E, SIG, syn0 = self.E, self.SIG, self.syn[0]\n        with self.b.phase("bm/r00"):\n            bb_l = self.round_head(0, bb_l)\n            with opened(E, "ctl", 1) as (ctl,):\n                flag_test(E, syn0, (), ctl)\n                with opened(E, "m", B) as mq:\n                    wins, rts = chain_fwd(E, syn0, mq)\n                    # SIG[0]=|1>, SIG[1]=|0>, bb_l[0]=|0>, bb_l[1]=|1> are KNOWN basis\n                    # states: the two register cswaps collapse to 4 free CX.\n                    assert KMSV[0] == 1\n                    E.cx(ctl, SIG[0][0])\n                    E.cx(ctl, bb_l[0][0])\n                    E.cx(ctl, SIG[1][0])\n                    E.cx(ctl, bb_l[1][0])\n                    # post-swap bb_l[0] value = ctl, bb_l[1] value = 1^ctl; m nonzero\n                    # only when ctl=1, hence m*bb0 = m (free CX) and m*bb1 = 0 (skip).\n                    for t_ in range(B):\n                        E.cx(mq[t_], SIG[0][t_])\n                    # erase m: post-update m == ctl * sigma_0 on both branches; the\n                    # chain intermediates are functions of syn[0]\n                    E.free([norm_term((ctl,), 0, (SIG[0][t_],), 0, (mq[t_],)) for t_ in range(B)])\n                    chain_undo(E, wins, rts[:-1])\n                self.close_ctl(0, ctl)\n            self.inc_d()\n        return bb_l\n\n    def round(self, r, bb_l):\n        """Emit a general round 0 < r < N-1."""\n        with self.b.phase(f"bm/r{r:02d}"):\n            bb_l = self.round_head(r, bb_l)\n            with opened(self.E, "ctl", 1) as (ctl,):\n                self.discrepancy(r)\n                # ctl is set iff Dw != 0 and d >= 0\n                flag_test(self.E, self.Dw, (self.dW[SGN],), ctl)\n                with self.quotient(ctl, bb_l) as mq:\n                    self.bank_swaps(r, ctl, bb_l)\n                    self.update(r, 0, mq, bb_l)\n                self.push(r)\n                self.erase_dw(ctl)\n                self.close_ctl(r, ctl)\n            if r < N - 2:\n                # +1 always -> d = ctl ? -d : d+1 (at r=N-2 the increment is\n                # dropped: the counter has no later consumer and the complement\n                # alone arms the erase anchor)\n                self.inc_d()\n        return bb_l\n\n    def round_last(self, bb_l):\n        """r=N-1: no ctl.  m = p, computed straight into the m window; the\n        update skips slot 0 (with a length change due (w=W mass) zB occupies\n        slots [N-1-r*, W] and never wraps to 0; otherwise M=0); the push is\n        unconditional (push(ctl=False)), and r=N-1 writes no tag (no later\n        round reads it).\n        """\n        r = N - 1\n        with self.b.phase(f"bm/r{r:02d}"):\n            bb_l = self.round_head(r, bb_l)\n            self.discrepancy(r)\n            with opened(self.E, "m", B) as mq:\n                rtM = self.E.mac(prod_rt(mq, self.Dw, self.SIG[0]))\n                self.update(r, 1, mq, bb_l)\n                # m is a function of Dw and the untouched sigma_0\n                self.E.free(rtM)\n            assert CAPS[r] == STK_CAP\n            self.push(r, ctl=False)\n\n    def run(self):\n        """Stage 2: the seed, round 0, rounds 1..N-2, round N-1; the journal\n        is frozen afterwards (stages 3..9 are mirrored explicitly).\n        """\n        E, J, A = self.E, self.J, self.A\n        J.recording = True\n        A.round = -1  # seed segment: banks touched here free last\n        A.log = self.pools.setdefault(-1, set())\n        with E.using(J=J, A=A):\n            with self.b.phase("bm/seed"):\n                E.x(self.SIG[0][0])\n                E.x(self.BB[0][0])\n            bb_l = self.round0(list(self.BB))\n            for r in range(1, N - 1):\n                bb_l = self.round(r, bb_l)\n            self.round_last(bb_l)\n        self.final_bb_l = [bb_l[W + 1]] + bb_l[:W + 1]\n        J.recording = False\n        self.JROUND.append(J.mark())\n        A.round = None\n        A.log = None\n        # every lazy register must have materialized inside BM: a first touch\n        # in a later stage would have no mirror round to free it at\n        assert all(reg.wires is not None for reg in self.SIG + self.BB + self.STK), (\n            "lazy register untouched during BM"\n        )\n\n    def reverse(self):\n        """Stage 10, the journaled measurement-based reverse pass.  Every BM\n        step went through J with its conjugate (macs record\n        release-or-recompute per the emit-time pebbling choice), so\n        replaying the conjugates in reverse re-materializes each round\'s\n        intermediates at the mirrored points and the replayed Fredkin\n        cascades pop the stack at exactly the right moments.  The journal\n        replays closures over hard wire ids: each round\'s window pool is\n        re-materialized with reclaim right before its replay and freed\n        right after, and every bank slot is freed after the replay of its\n        first-touch round (its state there equals the forward state before\n        that round: |0>).  The availability lines up by time symmetry -- an\n        id a bank slot took at forward round f is released before the\n        replays of rounds < f, exactly when their pools want it back -- and\n        reclaim fails closed on any gap.\n        """\n        b, A, JR = self.b, self.A, self.JROUND\n        freed = 0\n        for k in range(len(JR) - 2, -1, -1):\n            rr = k - 1  # round -1 = the seed\n            with b.phase("reverse/seed" if rr < 0 else f"reverse/r{rr:02d}"):\n                if JR[k + 1] > JR[k]:\n                    with A.replay(self.pools.get(rr, ())):\n                        self.J.reverse(since=JR[k], until=JR[k + 1])\n                for reg in reversed(A.bank_log.get(rr, [])):\n                    A.close(reg.name, reg.wires)\n                    freed += 1\n        assert freed == len(self.SIG) + len(self.BB) + len(self.STK), (\n            "lazy registers leaked past the reverse",\n            freed,\n        )\n\n\n# =========================================================================\n# (Un)monicize (stages 3 and 9) and the in-place omega70 (stages 4 and 8)\n# =========================================================================\n# ARMORED: zflag = [sigma_0 == 0] XORed into W0[0] and SIG[0][0] around the\n# slot loop makes the scaling invertible for ALL data (scale defaults to 1\n# when sigma_0 = 0), so truncation deviations can never make phase garbage.\n\n\ndef w0_pass(E, SIG, mid, W0):\n    """One w0 window: open the chain scratch, run the SIG[0] inversion\n    chain intermediates forward, apply mid to the W0 term list (emit_mac\n    to create W0, emit_free to erase it), then erase the intermediates\n    and close the chain scratch.  The create and destroy windows differ\n    only in mid.\n    """\n    wins, rts = chain_open(E, SIG[0], W0)\n    E.macs(rts[:-1])\n    mid(rts[-1])\n    E.frees(rts[:-1])\n    chain_close(E, wins)\n\n\ndef mon_slot(E, sig_i, scale, eraser, ymon):\n    """sig_i <- scale * sig_i: ymon = scale * sig_i, a demolition erase of\n    sig_i against eraser x ymon through Em\'s measurement-erase core\n    (per-target hmr + conditioned-CZ repayment), then swap ymon into\n    sig_i.  monicize: (scale, eraser) = (W0, SIG[0]); unmonicize: (SIG[0],\n    W0).\n    """\n    E.emit_mac(prod_rt(ymon, scale, sig_i))\n    E.emit_free(\n        [\n            ("P", (eraser[a_],), 0, (ymon[b_],), 0, (sig_i[k2],))\n            for k2 in range(B)\n            for (a_, b_) in MONB[k2]\n        ]\n    )\n    for t_ in range(B):\n        E.b.swap(ymon[t_], sig_i[t_])\n\n\ndef mon_stage(E, SIG, prefix, slot_range, forward):\n    """One whole (un)monicize stage: create W0, arm the zflag armor,\n    run the W scaling slots, disarm, erase W0.  monicize and\n    unmonicize are the same block up to phase prefix, slot order, and\n    the (scale, eraser) pair.\n    """\n    b = E.b\n    # W0 and the zflag ladder (eq_test\'s two halves) outlive the phase that\n    # opens them, so they open and close by hand in their first/last phase.\n    with b.phase(prefix + "/w0+"):\n        W0 = E.A.open("w0", B)\n        w0_pass(E, SIG, E.emit_mac, W0)\n    with b.phase(prefix + "/zflag+"):\n        tr = E.A.open("tree", B - 1)\n        rts = E.ladder(eq_rungs(SIG[0], 0), tr)\n        top = tr[B - 2]\n        b.cx(top, W0[0])\n        b.cx(top, SIG[0][0])\n    scale, eraser = (W0, SIG[0]) if forward else (SIG[0], W0)\n    for i in slot_range:\n        with b.phase(f"{prefix}/s{i:02d}"), opened(E, "ymon", B) as ym:\n            mon_slot(E, SIG[i], scale, eraser, ym)\n    with b.phase(prefix + "/zflag-"):\n        b.cx(top, SIG[0][0])\n        b.cx(top, W0[0])\n        E.frees(rts)\n        E.A.close("tree", tr)\n    with b.phase(prefix + "/w0-"):\n        w0_pass(E, SIG, E.emit_free, W0)\n        E.A.close("w0", W0)\n\n\ndef omega_pass(E, macrt, prefix, order):\n    """Emit the omega70 block term lists in order, one phase block each."""\n    for bi in order:\n        with E.b.phase(f"{prefix}/m{bi:02d}"):\n            E.emit_mac(macrt[bi])\n\n\ndef build():\n    """Build the circuit and assert its pins.  Returns (builder, Toffoli\n    count); the occupancy/liveness profilers consume its wire_log.\n    """\n    # ---- registers, held scratch, allocators --------------------------------\n    b = Builder()\n    with b.phase("init"):  # profile-stage blocks (metadata; emit nothing)\n        syn = list(declare_registers(b, [(f"s{r:02d}", B) for r in range(N)]).values())\n        assert all(list(syn[r]) == list(range(B * r, B * r + B)) for r in range(N)), (\n            "syn ids not contiguous"\n        )  # qubits Br..Br+B-1\n\n        ANC = []\n\n        def An(n):\n            qs = b.qubits(n)\n            ANC.extend(qs)\n            return qs\n\n        Dw = An(B)  # fused-Q working discrepancy register\n        dW = An(DWB)  # two\'s-complement d counter\n        OMW = [syn[i] for i in range(W)]  # omega lives IN PLACE in syn[0..W-1]\n\n        # Ownership wire-sets (claim/release bookkeeping; emits no op).  Every\n        # long-lived wire joins exactly one named set with its full id list, so\n        # the anonymous pool live - sum(owned) is empty from init to cleanup;\n        # tools/occupancy_profile.py validates the discipline fail-closed.\n        b.claim("syn", [w for reg in syn for w in reg])\n        b.claim("dw", Dw)\n        b.claim("dcount", dW)\n\n        # Transient scratch round-trips through the allocator (A.open /\n        # A.close).  Only state that persists between windows (discrepancies,\n        # stack, controls, counters, the Em pivots) stays held; everything that\n        # returns to |0> mid-circuit is freed there, so peak_qubits is the\n        # measured max co-live width and the liveness profile shows the true\n        # windows.  Journal replays (the BM reverse, the Psi route) run\n        # closures over hard wire ids, so A logs the window ids a journal\n        # touches and re-materializes exactly that set before its replay\n        # (A.replay; reclaim fails closed when any id is not free).\n        A = Alloc(b)\n\n        sA = An(1)[0]\n        b.claim("pivA", [sA])\n        sB = An(1)[0]\n        b.claim("pivB", [sB])\n        sT = An(1)[0]\n        b.claim("fanT", [sT])\n        E = Em(b, sA, sB, sT, A)\n        bm = BM(E, A, syn, Dw, dW)\n        SIG = bm.SIG\n\n    # Width ledger: N * B syn + (B + DWB + 3) held (Dw, dW, pivots) + the\n    # lazily-materialized wires ((2W+3) B bank + STK_CAP tagged stack slots)\n    # + the late BM/reverse rounds\' window tiling.  The peak assert below\n    # enforces the total.\n\n    # ---- 1. zlin: free Z part sum_i Tr(u_i S_i) -----------------------------\n    with b.phase("zlin"):\n        for i in range(N):\n            for bb in []:  # variable masks: no fixed syndrome-linear split\n                b.z(syn[i][bb])\n\n    # ---- 2. bm: BM forward (q-form conditional-swap, single-chain m) --------\n    bm.run()\n    BORROW = borrow_template(b, E, A, bm, dW) if os.environ.get("DQI_BORROW") else None\n\n    # ---- 3. monicize: SIG[i] <- inv(sigma_0)*SIG[i] for i=1..W --------------\n    mon_stage(E, SIG, "monicize", range(1, W + 1), True)\n\n    # ---- 4. omega70 = sigma_m*S mod z^N, IN PLACE in syn[0..N-1] ------------\n    # sigma_m is monic: the triangles\' diagonals span sigma_1..sigma_{W-1},\n    # the off-diagonal tail block\'s sigma_1..sigma_W (omega_rts).\n    # Rows W..N-1 (the TAIL) end at 0 on every correct decode and host the\n    # Bezout pair (stage 5).\n    MACRT = omega_rts(SIG, syn)\n    omega_pass(E, MACRT, "omega", range(len(MACRT)))\n\n    # ---- 5. euclid: the half-degree Bezout pass (Psi route) -----------------\n    # sigma_m = E(z)^2 + z O(z)^2 with E_i = sqrt(sigma_2i), O_i =\n    # sqrt(sigma_2i+1): SIG[1..35] are square-rooted IN PLACE (a free linear\n    # map) and the Euclid bank EB is E = [ONE, SIG2, .., SIG34], O = [SIG1, ..,\n    # SIG35] (ONE is a fresh word holding 1 = E_0).  m = floor((35 - L)/2) =\n    # (dW + 2) >> 2 (dW = the BM counter, +2 in place), and the barrel pad\n    # shifts both halves up by m words so deg E\' = 17 on every correct decode.\n    # JSET journals the setup (ONE, the in-place square roots, dW += 2) so\n    # stage 7 undoes it with one reverse.\n    # PSI_IDS logs every scratch id the Psi journals touch; stages R and 7\n    # hold exactly that set around their replays.\n    with b.phase("euclid/setup"):\n        ONE = A.open("one", B)\n        EPS = A.open("zprec", 6)  # eps + 5-bit idx\n        PRE = A.open("pre", 1)[0]\n        PSI_IDS = set()\n        JSET, JPAD, JPRE, JF, JSEED = (Journal(), Journal(), Journal(), Journal(), Journal())\n        with E.using(J=JSET), A.logging(PSI_IDS):\n            E.x(ONE[0])\n            for i in range(1, W + 1):\n                JSET.step(\n                    lambda r=list(SIG[i]): linear_in_place(b, r, SQRT_ROWS),\n                    lambda r=list(SIG[i]): linear_in_place(b, r, SQUARE_ROWS),\n                )\n            # dW += 2: a constant increment on bits 1..7 (ladder carries, MBU-erased)\n            with opened(E, "inc", DWB - 3) as tr:\n                E.inc1(dW[1:DWB], tr)\n        EW = [ONE] + [list(SIG[2 * i]) for i in range(1, NH)]\n        OWD = [list(SIG[2 * i + 1]) for i in range(NH)]\n        MBITS = dW[2:7]  # m = (dW + 2) >> 2\n    with b.phase("euclid/pad"), E.using(J=JPAD):\n        pad_barrel(E, MBITS, EW, OWD)\n    with b.phase("euclid/pre"), E.using(J=JPRE), A.logging(PSI_IDS):\n        psi_pre(E, EW, OWD, PRE)\n    with b.phase("euclid/F"), E.using(J=JF), A.logging(PSI_IDS):\n        F_FIN, _G_FIN, RECS = psi_forward(E, EW, OWD, EPS[0], EPS[1:6])\n\n    # ---- seed + R: the Bezout pair (a, b) grows in the TAIL + one EXT word --\n    # a[(m+1) % 18] <- inv(h) (h = the final F pivot), then R = the journal\n    # reverse of F interleaved with the cofactor recurrence: on exit EB holds\n    # (E\', O\') again and (a, b) = (A, B) with A E\' + B O\' = z^m, deg <= 17.\n    with b.phase("euclid/seed"):\n        TAILW = [list(syn[i]) for i in range(W, N)]\n        AW = TAILW[:NH]\n        with E.using(J=JSEED), A.logging(PSI_IDS):\n            psi_seed(E, F_FIN[NH - 1], MBITS, AW)\n    with b.phase("euclid/R"):\n        with A.replay(PSI_IDS, "psipool"):\n            EXT = A.open("ext", B)  # opened under the replay hold: never a PSI_IDS id\n            BWD = TAILW[NH:] + [EXT]\n            JR, AFIN, BFIN = build_R(E, JF, RECS, AW, BWD, PRE, PSI_WIN, (EW, OWD))\n            JR.forward()\n        A.close("zprec", EPS)\n\n    # ---- 6. sweep: root flag + rank-12 Psi pairing over the M points --------\n    with b.phase("sweep"):\n        emit_sweep(b, E, A, OMW, EW + OWD, AFIN + BFIN)\n\n    # ---- 7. euclid_inv: R^-1, unseed, F^-1, un-pre, un-pad, un-setup --------\n    # The zprec and psipool holds span R..setup, so they open and close by\n    # hand (A.replay\'s two halves) in the first and last phase.\n    PSI_POOL = sorted(PSI_IDS)\n    with b.phase("euclid_inv/R"):\n        A.reopen("zprec", EPS)\n        A.reopen("psipool", PSI_POOL)\n        JR.reverse()\n        A.close("ext", EXT)\n    with b.phase("euclid_inv/seed"):\n        JSEED.reverse()\n    with b.phase("euclid_inv/F"):\n        JF.reverse()\n    with b.phase("euclid_inv/pre"):\n        JPRE.reverse()\n    with b.phase("euclid_inv/pad"):\n        JPAD.reverse()\n    with b.phase("euclid_inv/setup"):\n        JSET.reverse()\n        A.close("psipool", PSI_POOL)\n        A.close("pre", [PRE])\n        A.close("zprec", EPS)\n        A.close("one", ONE)\n\n    # ---- 8. omega_restore: exact reversed block list ------------------------\n    # each block\'s targets are disjoint from its operands, so re-emitting the\n    # same term list un-XORs it; the reversed block order restores S exactly.\n    omega_pass(E, MACRT, "omega_restore", reversed(range(len(MACRT))))\n\n    # ---- 9. unmonicize: SIG[i] <- sigma_0*SIG[i] for i=W..1 -----------------\n    mon_stage(E, SIG, "unmonicize", range(W, 0, -1), False)\n\n    # ---- 10. reverse: the journaled BM reverse pass -------------------------\n    if BORROW is not None:\n        BORROW()\n    bm.reverse()\n\n    # ---- 11. cleanup ---------------------------------------------------------\n    # Release-before-free through the only teardown path: every workspace set\n    # returns its wires (full id lists, reverse claim order) ahead of the\n    # frees; the N * B syndrome wires stay owned to the end as the declared\n    # registers.\n    with b.phase("cleanup"):\n        b.release("fanT", [sT])\n        b.release("pivB", [sB])\n        b.release("pivA", [sA])\n        b.release("dcount", dW)\n        b.release("dw", Dw)\n        b.recycle_all(reversed(ANC))\n\n    # =========================================================================\n    # Metrics gate, pre-emission: the pins under the measured window\n    # schedule and STK_CAP = 1.\n    # =========================================================================\n    TOF = assert_metrics(b, __file__)\n\n    return b, TOF\n\n\nif __name__ == "__main__":\n    emit_and_report(build()[0])\n', 'tasks/dqi/bm_forney/port_runtime.py': '# SPDX-License-Identifier: Apache-2.0\n# Modified port of src/gen_dqi_v6/port_runtime.py from the Apache-2.0\n# notebooks anazemcev/quantum-circuits-source-exact-compilation v4 and\n# kozykappa/quantum-ai-complete-domain-affine-compilation v2 (same embed).\n# Changes: import paths (gu_classical, gu_linear, contract.json) and a\n# hard stop unless kickmix.GF2Field(12) is modulus 4179, primitive element 2.\n# Parents and hashes: tasks/dqi/PROVENANCE.md. No copyright year is added.\n"""Small explicit adapter for Gu et al.\'s published source to official kickmix.\n\nInteger wire handles and LIFO ownership match the published builder. Physical\nkickmix qubits stay allocated while this adapter accounts for their lifetimes;\nevery recycle emits the same reset as the authors\' Builder. Named replay holds\nare checked, so the journal cannot silently alias live scratch.\n"""\nfrom contextlib import contextmanager\nimport json, os, pathlib, time\nimport kickmix as km\nfrom gu_classical import GF2n, f2_matinv, f2_transpose, parity\nfrom gu_linear import linear_in_place\n\n\ndef load_contract(_):\n    """Load the instance and force the shipped competition field.\n\n    Gu et al. use modulus 4331. This competition\'s ``kickmix.GF2Field(12)``\n    uses 4179 (``x^12 + x^6 + x^4 + x + 1``). Formulas are regenerated for\n    that modulus. A mismatch here is a hard stop, not a silent relabel.\n    """\n    contract=json.loads((pathlib.Path(__file__).resolve().parent/\'contract.json\').read_text())\n    field=km.GF2Field(12)\n    if field.modulus!=4179:\n        raise RuntimeError(f\'kickmix GF(2^12) modulus is {field.modulus}, expected 4179\')\n    if field.primitive_element!=2:\n        raise RuntimeError(f\'primitive element is {field.primitive_element}, expected 2\')\n    contract[\'gf_poly\']=field.modulus\n    return contract\n\n\nclass Journal:\n    """Gu et al., uncompute.bennett.Journal; pure conjugate choreography."""\n    def __init__(self): self._steps=[];self.recording=True\n    def mark(self):return len(self._steps)\n    def step(self, fwd, rev=None):\n        fwd()\n        if self.recording:self._steps.append((fwd,rev if rev is not None else fwd))\n    def defer(self, fwd, rev=None):self._steps.append((fwd,rev if rev is not None else fwd))\n    def forward(self,since=0,until=None):\n        for fwd,_ in self._steps[since:until]:fwd()\n    def reverse(self,since=0,until=None):\n        for _,rev in reversed(self._steps[since:until]):rev()\n\n\nclass Builder:\n    def __init__(self):\n        self.k=km.CircuitBuilder()\n        self.physical=[]\n        self.free_qubits=[];self.free_set=set();self.active=set()\n        self.errors=[self.k.create_classical_register(12,name=f\'E{j}\') for j in range(4095)]\n        # Published emitter uses each HMR outcome only for the immediately\n        # following phase repayments, before the next HMR. Reusing one live\n        # classical bit preserves that ordering and avoids dead classical ids.\n        self.measurement=self.k.alloc_bits(1)[0]\n        self.phase_stats=[];self.gate_counts={}\n        self.phase_name=\'init\';self.started=time.monotonic()\n        self.owners={};self.live_profile={};self.current_phase=None\n        self.progress=pathlib.Path(os.environ[\'BRIDGE_OUT\'])/\'build_progress.json\' if \'BRIDGE_OUT\' in os.environ else None\n\n    def qubit(self):\n        if self.free_qubits:\n            q=self.free_qubits.pop();self.free_set.remove(q)\n        else:\n            q=len(self.physical);self.physical.append(self.k.alloc_qubits(1)[0])\n        assert q not in self.active\n        self.active.add(q)\n        self._profile()\n        return q\n    def qubits(self,n):return [self.qubit() for _ in range(n)]\n    def reclaim(self,qs):\n        qs=list(qs);want=set(qs)\n        assert len(want)==len(qs) and want<=self.free_set, (\'reclaim live ids\',want-self.free_set)\n        self.free_qubits[:]=[q for q in self.free_qubits if q not in want]\n        self.free_set-=want;self.active.update(want)\n        self._profile()\n    def recycle(self,q):\n        assert q in self.active,(\'recycle not active\',q)\n        self.k.reset(self.physical[q])\n        self.active.remove(q);self.free_qubits.append(q);self.free_set.add(q)\n        self.owners.pop(q,None)\n    def recycle_all(self,qs):\n        for q in qs:self.recycle(q)\n    def _profile(self,force=False):\n        if self.current_phase is None:return\n        p=self.live_profile.setdefault(self.current_phase,{\'peak_active\':0,\'peak_physical\':0})\n        p[\'peak_physical\']=max(p[\'peak_physical\'],len(self.physical))\n        if len(self.active)>p[\'peak_active\'] or (force and len(self.active)==p[\'peak_active\']):\n            p[\'peak_active\']=len(self.active)\n            counts={}\n            for q in self.active:\n                name=self.owners.get(q,\'unclaimed\')\n                counts[name]=counts.get(name,0)+1\n            p[\'owners_at_peak\']=counts\n    def claim(self,name,qs):\n        for q in qs:self.owners[q]=name\n        self._profile(force=True)\n    def release(self,name,qs):\n        for q in qs:self.owners.pop(q,None)\n    def release_recycle(self,name,qs):self.recycle_all(reversed(qs))\n    def _gate(self,name,*qs,cond=None):\n        if cond is not None:self.k.push_condition(cond)\n        self.gate_counts[name]=self.gate_counts.get(name,0)+1\n        getattr(self.k,name)(*[self.physical[q] for q in qs])\n        if cond is not None:self.k.pop_condition()\n    def x(self,q,cond=None):self._gate(\'x\',q,cond=cond)\n    def z(self,q,cond=None):self._gate(\'z\',q,cond=cond)\n    def cx(self,c,t,cond=None):self._gate(\'cx\',c,t,cond=cond)\n    def cz(self,a,b,cond=None):self._gate(\'cz\',a,b,cond=cond)\n    def ccx(self,a,b,t,cond=None):self._gate(\'ccx\',a,b,t,cond=cond)\n    def ccz(self,a,b,c,cond=None):self._gate(\'ccz\',a,b,c,cond=cond)\n    def swap(self,a,b,cond=None):self._gate(\'swap\',a,b,cond=cond)\n    def cswap(self,c,a,b,cond=None):self._gate(\'cswap\',c,a,b,cond=cond)\n    def neg(self,cond=None):\n        assert cond is not None\n        self.k.neg_if(cond)\n    def hmr(self,q):\n        self.k.hmr(self.physical[q],self.measurement)\n        return self.measurement\n    @contextmanager\n    def phase(self,name):\n        self.k.info_tag(\'@enter:\'+name)\n        self.phase_name=name\n        prev=self.current_phase;self.current_phase=name\n        self._profile()\n        before=dict(self.gate_counts)\n        t=time.monotonic()\n        p={\'phase\':name,\'elapsed\':t-self.started,\'physical_qubits\':len(self.physical),\'active_qubits\':len(self.active)}\n        print(json.dumps(p),flush=True)\n        if self.progress:self.progress.write_text(json.dumps(p,indent=2))\n        yield\n        self.k.info_tag(\'@exit:\'+name)\n        self.phase_stats.append({\'phase\':name,\'seconds\':time.monotonic()-t,\n                                 \'gates\':{k:v-before.get(k,0) for k,v in self.gate_counts.items()}})\n        self.live_profile[name][\'exit_active\']=len(self.active)\n        self.current_phase=prev\n\n\ndef declare_registers(b,decls):\n    regs={}\n    for index,(_,width) in enumerate(decls):\n        assert not b.free_qubits\n        start=len(b.physical)\n        reg=b.k.create_quantum_register(width,name=f\'S{index}\')\n        b.physical.extend(reg)\n        qs=list(range(start,start+width));b.active.update(qs)\n        regs[f\'S{index}\']=qs\n    return regs\n\n\ndef assert_metrics(b,_):\n    return b.gate_counts.get(\'ccx\',0)+b.gate_counts.get(\'cswap\',0)+b.gate_counts.get(\'ccz\',0)\n\n\ndef emit_and_report(b):\n    raise RuntimeError(\'Use run_gu_port.py; no implicit publication or emission destination.\')\n', 'tasks/dqi/bm_forney/round2_gadgets.py': '# SPDX-License-Identifier: Apache-2.0\n"""Round-two reversible scheduling changes; original algebra: Gu et al.\n\nThe quotient swap schedule, early Euclid quotient parking, and early linear\nroot erasure were developed in this workspace (OpenAI Codex, 2026-10-23).\n"""\nfrom contextlib import contextmanager\n\n\ndef install(G,top_first=False,pebble_last=False,norm_pebble=False,compact_from=70):\n    @contextmanager\n    def quotient(self, ctl, bb_l):\n        E=self.E\n        if self.A.round>=compact_from:\n            with compact_quotient(G,self,ctl,bb_l) as mq:yield mq\n            return\n        if pebble_last and self.A.round==68:\n            with pebble_quotient(G,self,ctl,bb_l,full=norm_pebble) as mq:yield mq\n            return\n        with G.opened(E,\'pq\',G.B) as pv:\n            E.mac(G.prod_rt(pv,self.Dw,self.SIG[0]))\n            with G.opened(E,\'dinv\',G.B) as dv, G.chain_live(E,pv,dv):\n                # The quotient occupies p\'s register; swap back before erasing\n                # the inversion scaffold, whose source is the original p.\n                pairs=G.word_pairs([pv],[dv])\n                E.swaps(ctl,pairs)\n                yield pv\n                E.swaps(ctl,pairs)\n            rt=[]\n            G.bil_terms(rt,G.KT,G.spec_plain(self.Dw),\n                [frozenset((bb_l[0][j],self.SIG[0][j])) for j in range(G.B)],pv)\n            rt.append((\'L\',(ctl,),0,(pv[0],)))\n            E.free(rt)\n    G.BM.quotient=quotient\n    if pebble_last or compact_from<70:\n        original_run=G.BM.run\n        def run(self):\n            original_run(self)\n            # An erased scratch window may be reused by a bank later in the\n            # SAME round. Replay already owns that bank and restores it to\n            # zero before the earlier scratch operations execute. Hold only\n            # the other ids; free the bank once at its normal reverse point.\n            for rr,regs in self.A.bank_log.items():\n                for reg in regs:self.pools[rr].difference_update(reg.wires)\n        G.BM.run=run\n\n    # Preserve every journal mark\'s meaning: q_hi is parked in g17 before\n    # q_lo\'s temporary window opens, and the recorded cofactor operand follows\n    # its new location. No polynomial operation is dropped.\n    import inspect\n    if norm_pebble:\n        G.inv_word=lambda E,src:compact_inverse(G,E,src)\n    source=inspect.getsource(G.psi_forward)\n    source=source.replace(\'\'\'            with opened(E, "qh", B) as qh, opened(E, "ql", B) as ql:\n                E.mac(prod_rt(qh, g[17], fi))\'\'\',\'\'\'            with opened(E, "qh", B) as qh:\n                E.mac(prod_rt(qh, g[17], fi))\n                E.free(prod_rt(g[17], qh, f[17]))\n                E.swaps(None, word_pairs([qh], [g[17]]))\n            qh = g[17]\n            with opened(E, "ql", B) as ql:\'\'\')\n    source=source.replace(\'\'\'                E.free(prod_rt(g[17], qh, f[17]))  # claim g17 == qh f17\n\'\'\',\'\')\n    source=source.replace(\'\'\'                E.swaps(None, word_pairs([qh, ql], [g[17], g[16]]))\'\'\',\n                          \'\'\'                E.swaps(None, word_pairs([ql], [g[16]]))\'\'\')\n    if top_first:\n        G.top_poly_rt=lambda qh,ql,fw,tgt,high:top_poly_rt(G,qh,ql,fw,tgt,high)\n        source=source.replace(\'\'\'            qh = g[17]\n            with opened(E, "ql", B) as ql:\n                if j <= 16:\n                    with opened(E, "t", B) as t:\n                        trt = E.mac(prod_rt(t, qh, f[16]))\n                        rtl = []\n                        bil_terms(\n                            rtl,\n                            KT,\n                            [frozenset((g[16][i], t[i])) for i in range(B)],\n                            spec_plain(fi),\n                            ql,\n                        )\n                        E.mac(rtl)\n                        E.free(trt)\n                else:\n                    E.mac(prod_rt(ql, g[16], fi))\'\'\',\'\'\'            qh = g[17]\n            if j <= 16:\n                targets = [[g[j - 1 + n]] for n in range(18 - j)]\n                fw = [f[k] for k in range(j,17)]\n                E.xor(top_poly_rt(qh,None,fw,targets,True))\n            with opened(E, "ql", B) as ql:\n                E.mac(prod_rt(ql, g[16], fi))\'\'\')\n        source=source.replace(\'\'\'                        polyprod_rt(\n                            [ql, qh],\n                            [f[k] for k in range(j, 17)],\n                            [[g[j - 1 + n]] for n in range(18 - j)],\n                        ),\'\'\',\'\'\'                        top_poly_rt(qh,ql,fw,targets,False),\'\'\')\n        assert \'with opened(E, "t", B)\' not in source\n    ns=dict(vars(G));exec(source,ns)\n    G.psi_forward=ns[\'psi_forward\']\n\n\n@contextmanager\ndef compact_quotient(G,bm,ctl,bb_l):\n    E=bm.E\n    with G.opened(E,\'pq\',12) as pv:\n        E.mac(G.prod_rt(pv,bm.Dw,bm.SIG[0]))\n        with G.inv_word(E,pv) as dv:\n            pairs=G.word_pairs([pv],[dv]);E.swaps(ctl,pairs)\n            yield pv\n            E.swaps(ctl,pairs)\n        rt=[]\n        G.bil_terms(rt,G.KT,G.spec_plain(bm.Dw),\n            [frozenset((bb_l[0][j],bm.SIG[0][j])) for j in range(12)],pv)\n        rt.append((\'L\',(ctl,),0,(pv[0],)));E.free(rt)\n\n\n@contextmanager\ndef pebble_quotient(G,bm,ctl,bb_l,full=False):\n    E=bm.E;A=bm.A\n    with G.opened(E,\'pq\',12) as pv:\n        E.mac(G.prod_rt(pv,bm.Dw,bm.SIG[0]))\n        dv=A.open(\'dinv\',12)\n        wins,rts=G.chain_fwd(E,pv,dv)\n        nq,m3,s3,yd=wins\n        E.free(rts[2]);E.free(rts[1])\n        A.close(\'s3d\',s3);A.close(\'m3d\',m3)\n        if full:\n            E.free(rts[0]);A.close(\'nqd\',nq)\n        pairs=G.word_pairs([pv],[dv]);E.swaps(ctl,pairs)\n        yield pv\n        E.swaps(ctl,pairs)\n        # The inverse output needs only p and yd for MBU. Release its twelve\n        # wires BEFORE recomputing the two erased three-bit norm windows.\n        E.free(rts[4]);A.close(\'dinv\',dv)\n        if full:nq=A.open(\'nqd_re\',6)\n        nm=A.open(\'m3d_re\',3);ns=A.open(\'s3d_re\',3)\n        nr=G.chain_rts(pv,[nq,nm,ns,yd],dv)\n        if full:E.mac(nr[0])\n        E.mac(nr[1]);E.mac(nr[2]);E.free(nr[3])\n        E.free(nr[2]);E.free(nr[1]);E.free(nr[0] if full else rts[0])\n        A.close(\'s3d_re\',ns);A.close(\'m3d_re\',nm)\n        A.close(\'yd\',yd);A.close(\'nqd\',nq)\n        rt=[]\n        G.bil_terms(rt,G.KT,G.spec_plain(bm.Dw),\n            [frozenset((bb_l[0][j],bm.SIG[0][j])) for j in range(12)],pv)\n        rt.append((\'L\',(ctl,),0,(pv[0],)));E.free(rt)\n\n\ndef norm_rt(G,src,out):\n    rt=[]\n    G.bil_terms(rt,G.KT6,G.spec_rows(G.A0ROWS,src),G.spec_rows(G.A1ROWS,src),out)\n    G.lin_rows_terms(rt,G.LNROWS,src,out)\n    return rt\n\n\n@contextmanager\ndef compact_inverse(G,E,src):\n    """18-wire inverse schedule using Norm(x^-1) = Norm(x)^-1.\n\n    Forward inverse is still 66 Toffolis. Erasing its output costs 39 rather\n    than36, but no twelve-bit output overlaps an eighteen-bit scaffold.\n    """\n    A=E.A\n    wins=[A.open(name,n) for name,n in G.CHAIN_WINS]\n    nq,m3,s3,yd=wins\n    # The dummy output is only used to construct term lists, never emitted.\n    rts=G.chain_rts(src,wins,src)\n    E.macs(rts[:4])\n    E.free(rts[2]);E.free(rts[1]);E.free(rts[0])\n    A.close(\'s3d\',s3);A.close(\'m3d\',m3);A.close(\'nqd\',nq)\n    out=A.open(\'inv\',12)\n    inv=G.chain_rts(src,wins,out)[4];E.mac(inv)\n    E.free(norm_rt(G,out,yd));A.close(\'yd\',yd)\n    yield out\n    y=A.open(\'yd_un\',6);E.mac(norm_rt(G,out,y))\n    dummy=[out[:6],out[:3],out[:3],y]\n    E.free(G.chain_rts(src,dummy,out)[4]);A.close(\'inv\',out)\n    n=A.open(\'nqd_un\',6);m=A.open(\'m3d_un\',3);s=A.open(\'s3d_un\',3)\n    rt=G.chain_rts(src,[n,m,s,y],src)\n    E.macs(rt[:3]);E.free(rt[3]);E.frees(rt[:3])\n    A.close(\'s3d_un\',s);A.close(\'m3d_un\',m);A.close(\'nqd_un\',n);A.close(\'yd_un\',y)\n\n\n_TOP_V={}\n\ndef certify_norm_identity(G):\n    """Exhaust all4096 field elements over the actual emitted term lists."""\n    src=list(range(12));wins=[list(range(12,18)),list(range(18,21)),list(range(21,24)),list(range(24,30))]\n    out=list(range(30,42));normout=list(range(42,48))\n    rts=G.chain_rts(src,wins,out);nr=norm_rt(G,out,normout)\n    def emit(memory,rt):\n        for term in rt:\n            if term[0]==\'L\':v=term[2]^sum(memory[q] for q in term[1])%2\n            else:v=(term[2]^sum(memory[q] for q in term[1])%2)&(term[4]^sum(memory[q] for q in term[3])%2)\n            for q in term[-1]:memory[q]^=v\n    for v in range(4096):\n        memory=[(v>>q)&1 for q in range(12)]+[0]*36\n        for rt in rts:emit(memory,rt)\n        got=sum(memory[q]<<i for i,q in enumerate(out))\n        assert got==G.ginv(v),(v,got,G.ginv(v))\n        emit(memory,nr)\n        assert [memory[q] for q in wins[-1]]==[memory[q] for q in normout],v\n    return {\'field_elements\':4096,\'inverse_correct\':True,\'norm_inverse_identity\':True,\n            \'method\':\'evaluate exact Gu term lists and norm_rt on every GF4096 element, including zero\'}\n\ndef top_poly_rt(G,qh,ql,fw,tgt,high):\n    """Evaluate a linear-times-polynomial product at finite points + infinity.\n\n    Emit infinity first so its high coefficient changes g16 before q_lo is\n    divided from g16. This removes the redundant twelve-bit top-product copy.\n    The remaining interpolation columns never write the high coefficient.\n    """\n    n=len(fw)+1\n    if n not in _TOP_V:\n        pts=[G.EXP[t] for t in range(n-1)]\n        matrix=[[G.gpow(p,k) for k in range(n)] for p in pts]+[[0]*(n-1)+[1]]\n        _TOP_V[n]=(pts,G.gf_matinv(matrix))\n    pts,iv=_TOP_V[n]\n    rt=[]\n    for t in ([n-1] if high else range(n-1)):\n        if high:\n            sa=G.spec_plain(qh);sb=G.spec_plain(fw[-1])\n        else:\n            sa=[G.xfold([(ql,1<<b),(qh,G.cmrows(pts[t])[b])]) for b in range(G.B)]\n            cf=[G.cmrows(G.gpow(pts[t],k)) for k in range(len(fw))]\n            sb=[G.xfold([(fw[k],cf[k][b]) for k in range(len(fw))]) for b in range(G.B)]\n        def targets(tb,t=t):\n            acc={};u=G.tmask(tb)\n            for k in range(n):\n                v=G.gmul(iv[k][t],u)\n                for word in tgt[k]:\n                    for b in G.bitlist(v,G.B):acc[word[b]]=acc.get(word[b],0)^1\n            return sorted(q for q,odd in acc.items() if odd)\n        G.bil_terms(rt,G.KT,sa,sb,tws_of=targets)\n    return [term for term in rt if term[-1]]\n\n\ndef lean_sweep(b,E,A,omw,ebw,abw,stream=False):\n    import gu_port as G\n    import kickmix as km\n    from port_runtime import linear_in_place\n    raw=b.k;gf=km.GF2Field(12)\n    def arr(qs):return km.array([b.physical[q] for q in qs])\n    def sum_into(words,target):\n        raw.cx(arr([q for word in words for q in word]),arr([q for _ in words for q in target]))\n    words=omw+ebw+abw\n    logs=list(range(35))+G.LOG_EB+G.LOG_AB\n    beta=gf.pow(gf.primitive_element,4094)\n    steps=[gf.pow(beta,e) for e in logs]\n    arrays=[arr(reg) for reg in words]\n    for j in range(4095):\n        val=A.open(\'scan_root\',12);sum_into(ebw,val)\n        tr=A.open(\'tree\',11);rts=E.ladder(G.eq_rungs(val,0),tr);root=tr[-1]\n        linear=[(\'L\',tuple(w[k] for w in ebw),0,(val[k],)) for k in range(12)]\n        E.free(linear);A.close(\'scan_root\',val)\n        om=A.open(\'scan_omega\',1 if stream else 12);ps=A.open(\'scan_psi\',12);loc=A.open(\'scan_located\',1 if stream else 12)\n        sum_into(abw,ps)\n        linear_in_place(b,ps,G.SQUARE_ROWS)\n        if stream:\n            # Bilinearity permits one evaluator bit at a time. The AND\n            # ancilla is repaid before reuse: still twelve Toffolis/point.\n            for k in range(12):\n                raw.cx(arr([w[k] for w in omw]),arr(om*len(omw)))\n                raw.init_and(b.physical[root],arr(om),arr(loc))\n                for h in range(12):\n                    coeff=gf.mul(1<<k,1<<h)\n                    for c in range(12):\n                        if (coeff>>c)&1:\n                            raw.push_condition(b.errors[j][c])\n                            raw.cz(b.physical[loc[0]],b.physical[ps[h]])\n                            raw.pop_condition()\n                raw.del_and(b.physical[root],arr(om),arr(loc))\n                raw.cx(arr([w[k] for w in omw]),arr(om*len(omw)))\n        else:\n            sum_into(omw,om)\n            raw.init_and(b.physical[root],arr(om),arr(loc))\n            raw.gf2_phase_by_product(arr(loc),arr(ps),mask=b.errors[j],field=gf)\n            raw.del_and(b.physical[root],arr(om),arr(loc))\n            sum_into(omw,om)\n        linear_in_place(b,ps,G.SQRT_ROWS)\n        sum_into(abw,ps)\n        A.close(\'scan_located\',loc);A.close(\'scan_psi\',ps);A.close(\'scan_omega\',om)\n        A.reopen(\'scan_root\',val);sum_into(ebw,val)\n        E.frees(rts);A.close(\'tree\',tr)\n        sum_into(ebw,val);A.close(\'scan_root\',val)\n        if j+1<4095:\n            for reg,factor in zip(arrays,steps):\n                if factor!=1:raw.gf2_imul(factor,target=reg,field=gf)\n        if j%512==0:\n            import json,time\n            p={\'phase\':\'lean_sweep\',\'position\':j,\'elapsed\':time.monotonic()-b.started}\n            print(json.dumps(p),flush=True)\n            if b.progress:b.progress.write_text(json.dumps(p))\n    for reg,factor in zip(arrays,steps):\n        if factor!=1:raw.gf2_idiv(gf.pow(factor,4094),target=reg,field=gf)\n\ndef stream_sweep(b,E,A,omw,ebw,abw):\n    return lean_sweep(b,E,A,omw,ebw,abw,stream=True)\n', 'tasks/dqi/bm_forney/symbolic_runtime.py': '"""Exact bounded symbolic partial evaluation for reversible kickmix emission.\n\nEvery form is an XOR of independent formal atoms, including constant atom 0.\nAn AND of forms is interned as a fresh atom; no independence assumption is\nused to infer equalities. Oversized forms are forgotten into unique atoms,\nwhich loses optimization opportunities but cannot create false equalities.\nHMR returns a wire to zero and permits immediate physical reuse. Declared\ninput registers remain pinned. No input examples enter this compiler.\nSPDX-License-Identifier: Apache-2.0\n"""\nimport kickmix as km\nimport port_runtime as R\n\nZERO=frozenset()\nONE=frozenset((0,))\nBase=R.Builder\n\nclass SymbolicBuilder(Base):\n    def __init__(self):\n        super().__init__()\n        self.forms=[];self.pinned=set();self.atom_id=0;self.products={}\n        self.elided={};self.folded={};self.max_native=0\n        self.limit=48\n\n    def atom(self):\n        self.atom_id+=1\n        return frozenset((self.atom_id,))\n\n    def xor(self,a,c):\n        v=a^c\n        return v if len(v)<=self.limit else self.atom()\n\n    def product(self,a,c):\n        if not a or not c:return ZERO\n        if a==ONE:return c\n        if c==ONE or a==c:return a\n        if a^c==ONE:return ZERO\n        # Canonical complemented inputs share the same nonlinear atom.\n        ac=0 in a;cc=0 in c\n        aa=a-ONE;bb=c-ONE\n        key=(aa,bb) if tuple(sorted(aa))<=tuple(sorted(bb)) else (bb,aa)\n        if key not in self.products:self.products[key]=self.atom()\n        v=self.products[key]\n        if ac:v=self.xor(v,bb)\n        if cc:v=self.xor(v,aa)\n        if ac and cc:v=self.xor(v,ONE)\n        return v\n\n    def wire(self,q):\n        if self.physical[q] is None:\n            self.physical[q]=self.k.alloc_qubits(1)[0]\n            if self.forms[q]==ONE:self.k.x(self.physical[q])\n        return self.physical[q]\n\n    def qubit(self):\n        if self.free_qubits:\n            q=self.free_qubits.pop();self.free_set.remove(q)\n            assert self.forms[q]==ZERO\n        else:\n            q=len(self.physical);self.physical.append(None);self.forms.append(ZERO)\n        assert q not in self.active\n        self.active.add(q)\n        return q\n\n    def _constant(self,q):\n        if q in self.pinned or self.forms[q] not in (ZERO,ONE):return\n        if self.physical[q] is not None:\n            if self.forms[q]==ONE:self.k.x(self.physical[q])\n            self.k.free(self.physical[q]);self.physical[q]=None\n\n    def recycle(self,q):\n        assert q in self.active\n        if self.forms[q]:\n            # Retain actual reset semantics on every approximate path.\n            self.k.reset(self.wire(q))\n        if self.physical[q] is not None:self.k.free(self.physical[q])\n        self.physical[q]=None;self.forms[q]=ZERO\n        self.active.remove(q);self.free_qubits.append(q);self.free_set.add(q)\n        self.owners.pop(q,None)\n\n    def _native(self,name,qs,cond):\n        if cond is not None:self.k.push_condition(cond)\n        self.gate_counts[name]=self.gate_counts.get(name,0)+1\n        getattr(self.k,name)(*[self.wire(q) for q in qs])\n        if cond is not None:self.k.pop_condition()\n\n    def _gate(self,name,*qs,cond=None):\n        if getattr(self,\'suspended\',False):\n            self._native(name,qs,cond)\n            if name not in (\'z\',\'cz\',\'ccz\'):\n                changed=qs[-2:] if name in (\'swap\',\'cswap\') else qs[-1:]\n                self.invalidate(changed)\n            return\n        fs=[self.forms[q] for q in qs]\n        def skip():self.elided[name]=self.elided.get(name,0)+1\n        def fold(n,qq):\n            self.folded[name+\'->\'+n]=self.folded.get(name+\'->\'+n,0)+1\n            self._gate(n,*qq,cond=cond)\n        if name in (\'z\',\'cz\',\'ccz\'):\n            if any(f==ZERO for f in fs):skip();return\n            keep=[q for q,f in zip(qs,fs) if f!=ONE]\n            if len(keep)<len(qs):\n                if not keep:\n                    if cond is None:\n                        # Global half-turn retained explicitly using one bit.\n                        q=self.k.alloc_qubits(1)[0];self.k.x(q);self.k.z(q);self.k.x(q);self.k.free(q)\n                    else:self.k.neg_if(cond)\n                else:fold((\'z\',\'cz\',\'ccz\')[len(keep)-1],keep)\n                return\n            self._native(name,qs,cond);return\n        if name in (\'cx\',\'ccx\',\'cswap\'):\n            n=1 if name in (\'cx\',\'cswap\') else 2\n            if any(f==ZERO for f in fs[:n]):skip();return\n            if name==\'cx\' and fs[0]==ONE:fold(\'x\',qs[1:]);return\n            if name==\'ccx\':\n                if fs[0]==ONE:fold(\'cx\',qs[1:]);return\n                if fs[1]==ONE or fs[0]==fs[1]:fold(\'cx\',(qs[0],qs[2]));return\n                if fs[0]^fs[1]==ONE:skip();return\n            if name==\'cswap\':\n                if fs[1]==fs[2]:skip();return\n                if fs[0]==ONE:fold(\'swap\',qs[1:]);return\n        if name==\'swap\' and fs[0]==fs[1]:skip();return\n        if name==\'swap\' and cond is None and not any(q in self.pinned for q in qs):\n            a,c=qs;self.physical[a],self.physical[c]=self.physical[c],self.physical[a]\n            self.forms[a],self.forms[c]=fs[1],fs[0]\n            skip();return\n        self._native(name,qs,cond)\n        if name==\'x\':updates={qs[0]:self.xor(fs[0],ONE)}\n        elif name==\'cx\':updates={qs[1]:self.xor(fs[1],fs[0])}\n        elif name==\'ccx\':updates={qs[2]:self.xor(fs[2],self.product(fs[0],fs[1]))}\n        elif name==\'swap\':updates={qs[0]:fs[1],qs[1]:fs[0]}\n        elif name==\'cswap\':\n            delta=self.product(fs[0],self.xor(fs[1],fs[2]))\n            updates={qs[1]:self.xor(fs[1],delta),qs[2]:self.xor(fs[2],delta)}\n        else:raise ValueError(name)\n        for q,f in updates.items():\n            self.forms[q]=f if cond is None else self.atom()\n            self._constant(q)\n\n    def hmr(self,q):\n        self.k.hmr(self.wire(q),self.measurement)\n        self.forms[q]=ZERO\n        self._constant(q)\n        return self.measurement\n\n    def invalidate(self,qs):\n        # Direct native field operations bypass this adapter. Conservatively\n        # forget their outputs; no value-based optimization may cross them.\n        for q in qs:self.forms[q]=self.atom()\n\ndef declare_registers(b,decls):\n    regs=R.declare_registers(b,decls)\n    assert not b.forms\n    b.forms=[b.atom() for _ in b.physical]\n    b.pinned.update(q for v in regs.values() for q in v)\n    return regs\n', 'tasks/dqi/bm_forney/windows.json': '{\n  "bm_windows": {\n    "IMAXV": [\n      0,\n      1,\n      2,\n      2,\n      3,\n      3,\n      4,\n      4,\n      5,\n      5,\n      6,\n      6,\n      7,\n      7,\n      8,\n      8,\n      9,\n      9,\n      10,\n      10,\n      11,\n      11,\n      12,\n      12,\n      13,\n      13,\n      14,\n      14,\n      15,\n      15,\n      16,\n      16,\n      17,\n      17,\n      18,\n      18,\n      19,\n      19,\n      20,\n      20,\n      21,\n      21,\n      22,\n      22,\n      23,\n      23,\n      24,\n      24,\n      25,\n      25,\n      26,\n      26,\n      27,\n      27,\n      28,\n      28,\n      29,\n      29,\n      30,\n      30,\n      31,\n      31,\n      32,\n      32,\n      33,\n      33,\n      33,\n      34,\n      34,\n      35\n    ],\n    "KMSV": [\n      1,\n      2,\n      2,\n      3,\n      3,\n      4,\n      4,\n      5,\n      5,\n      6,\n      6,\n      7,\n      7,\n      8,\n      8,\n      9,\n      9,\n      10,\n      10,\n      11,\n      11,\n      12,\n      12,\n      13,\n      13,\n      14,\n      14,\n      15,\n      15,\n      16,\n      16,\n      17,\n      17,\n      18,\n      18,\n      19,\n      19,\n      20,\n      20,\n      21,\n      21,\n      22,\n      22,\n      23,\n      23,\n      24,\n      24,\n      25,\n      25,\n      26,\n      26,\n      27,\n      27,\n      28,\n      28,\n      29,\n      29,\n      30,\n      30,\n      31,\n      31,\n      32,\n      32,\n      33,\n      33,\n      33,\n      34,\n      34,\n      35,\n      35\n    ],\n    "KMMV": [\n      0,\n      1,\n      1,\n      2,\n      2,\n      3,\n      3,\n      4,\n      4,\n      5,\n      5,\n      6,\n      6,\n      7,\n      7,\n      8,\n      8,\n      9,\n      9,\n      10,\n      10,\n      11,\n      11,\n      12,\n      12,\n      13,\n      13,\n      14,\n      14,\n      15,\n      15,\n      16,\n      16,\n      17,\n      17,\n      18,\n      18,\n      19,\n      19,\n      20,\n      20,\n      21,\n      21,\n      22,\n      22,\n      23,\n      23,\n      24,\n      24,\n      25,\n      25,\n      26,\n      26,\n      27,\n      27,\n      28,\n      28,\n      29,\n      29,\n      30,\n      30,\n      31,\n      31,\n      32,\n      32,\n      33,\n      33,\n      34,\n      34,\n      35\n    ]\n  },\n  "psi_windows": {\n    "mul": {\n      "17": [\n        0,\n        0\n      ],\n      "16": [\n        1,\n        1\n      ],\n      "15": [\n        2,\n        3\n      ],\n      "14": [\n        3,\n        4\n      ],\n      "13": [\n        4,\n        5\n      ],\n      "12": [\n        5,\n        6\n      ],\n      "11": [\n        6,\n        7\n      ],\n      "10": [\n        7,\n        8\n      ],\n      "9": [\n        8,\n        9\n      ],\n      "8": [\n        9,\n        10\n      ],\n      "7": [\n        10,\n        11\n      ],\n      "6": [\n        11,\n        12\n      ],\n      "5": [\n        12,\n        13\n      ],\n      "4": [\n        13,\n        14\n      ],\n      "3": [\n        14,\n        15\n      ],\n      "2": [\n        15,\n        16\n      ],\n      "1": [\n        16,\n        17\n      ],\n      "0": [\n        0,\n        18\n      ]\n    },\n    "rz": {\n      "17": [\n        0,\n        0\n      ],\n      "16": [\n        0,\n        0\n      ],\n      "15": [\n        2,\n        2\n      ],\n      "14": [\n        3,\n        3\n      ],\n      "13": [\n        4,\n        4\n      ],\n      "12": [\n        5,\n        5\n      ],\n      "11": [\n        6,\n        6\n      ],\n      "10": [\n        7,\n        7\n      ],\n      "9": [\n        8,\n        8\n      ],\n      "8": [\n        9,\n        9\n      ],\n      "7": [\n        10,\n        10\n      ],\n      "6": [\n        11,\n        11\n      ],\n      "5": [\n        12,\n        12\n      ],\n      "4": [\n        13,\n        13\n      ],\n      "3": [\n        14,\n        14\n      ],\n      "2": [\n        15,\n        15\n      ],\n      "1": [\n        16,\n        16\n      ],\n      "0": [\n        17,\n        17\n      ]\n    },\n    "swap": {\n      "17": [\n        1,\n        1\n      ],\n      "16": [\n        2,\n        2\n      ],\n      "15": [\n        3,\n        4\n      ],\n      "14": [\n        4,\n        5\n      ],\n      "13": [\n        5,\n        6\n      ],\n      "12": [\n        6,\n        7\n      ],\n      "11": [\n        7,\n        8\n      ],\n      "10": [\n        8,\n        9\n      ],\n      "9": [\n        9,\n        10\n      ],\n      "8": [\n        10,\n        11\n      ],\n      "7": [\n        11,\n        12\n      ],\n      "6": [\n        12,\n        13\n      ],\n      "5": [\n        13,\n        14\n      ],\n      "4": [\n        14,\n        15\n      ],\n      "3": [\n        15,\n        16\n      ],\n      "2": [\n        16,\n        17\n      ],\n      "1": [\n        0,\n        18\n      ],\n      "0": [\n        0,\n        17\n      ]\n    }\n  }\n}\n', 'tasks/dqi/generate.py': '# SPDX-License-Identifier: Apache-2.0\n# Port driver for the Berlekamp-Massey + Forney decoder. This is a port,\n# not independent clean-room authorship. Parents, used paths, licences\n# and file hashes are in tasks/dqi/PROVENANCE.md and NOTICE.\n# Gu et al., andigu/uncompute, commit 64b5a13aa249f1360846657392e6c66817b26d0f.\n# No .kmb file and no .npz file is an input. No copyright year is added.\n\n"""Berlekamp–Massey + Forney decoder for the competition DQI instance.\n\n``python tasks/dqi/generate.py --out DIR`` writes ``DIR/dqi.kmb`` and nothing\nelse. The instance is fixed: RS(4095, 4025, 71), 70 syndrome registers\n``S0``..``S69``, classical record ``E0``..``E4094``.\n\nEach run regenerates the multiplication formulas and the tower inverse\nchain on the shipped ``GF(2^12)`` (modulus 4179, primitive element 2).\nTruncation windows are the parent\'s measured windows. The fixed SHA-256\nmask is not applied: the sweep kicks the phase from the per-shot record.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib\nimport json\nimport os\nimport sys\nimport tempfile\nimport time\nfrom pathlib import Path\n\n_PORT = Path(__file__).resolve().parent / "bm_forney"\n\n# Replacements that keep the V6 sweep visible to the symbolic builder.\n# The sweep body calls kickmix field ops directly. The symbolic builder\n# must forget those outputs, or a later uncompute will treat them as\n# still-known forms.\n_SWEEP_EDITS = (\n    ("b.physical[q]", "b.wire(q)"),\n    ("b.physical[root]", "b.wire(root)"),\n    ("b.physical[loc[0]]", "b.wire(loc[0])"),\n    ("b.physical[ps[h]]", "b.wire(ps[h])"),\n    (\n        "raw.cx(arr([q for word in words for q in word]),arr([q for _ in words for q in target]))",\n        "raw.cx(arr([q for word in words for q in word]),arr([q for _ in words for q in target]));b.invalidate(target)",\n    ),\n    ("tr=A.open(\'tree\',11);", "b.invalidate(val);tr=A.open(\'tree\',11);"),\n    (\n        "raw.del_and(b.wire(root),arr(om),arr(loc))",\n        "raw.del_and(b.wire(root),arr(om),arr(loc));b.forms[loc[0]]=S.ZERO",\n    ),\n    (\n        "raw.init_and(b.wire(root),arr(om),arr(loc))",\n        "raw.init_and(b.wire(root),arr(om),arr(loc));b.invalidate(loc)",\n    ),\n    (\n        "raw.cx(arr([w[k] for w in omw]),arr(om*len(omw)))",\n        "raw.cx(arr([w[k] for w in omw]),arr(om*len(omw)));b.invalidate(om)",\n    ),\n)\n\n\ndef _port_modules():\n    """Import the port from its directory once. Later calls reuse it."""\n    port = str(_PORT)\n    if port not in sys.path:\n        sys.path.insert(0, port)\n    import gu_formula_derivation as formulas\n\n    return formulas\n\n\ndef regenerate_tables(work: Path) -> dict:\n    """Regenerate field formulas for modulus 4179 and merge the windows.\n\n    The returned object is what ``gu_port`` reads. Multiplication tables\n    and ``LIN`` come from this process. ``bm_windows`` and ``psi_windows``\n    come from the committed parent windows, not from a numeric target file.\n    """\n    formulas = _port_modules()\n    tables, timing = formulas.run_b12()\n    if not formulas.certify_kt(tables["KT12"], formulas.FIELD):\n        raise RuntimeError("KT12 does not multiply GF(2^12) modulus 4179")\n    if formulas.FIELD.gen != 2:\n        raise RuntimeError(f"derived primitive element is {formulas.FIELD.gen}, expected 2")\n    windows = json.loads((_PORT / "windows.json").read_text())\n    constants = dict(windows)\n    constants.update(tables)\n    path = work / "constants.json"\n    path.write_text(json.dumps(constants, sort_keys=True))\n    os.environ["DQI_CONSTANTS"] = str(path)\n    os.environ["DQI_BORROW"] = "affine"\n    return {"timings": timing, "modulus": formulas.CONTRACT["gf_poly"], "primitive": formulas.FIELD.gen}\n\n\ndef _install_sweep(port, symbolic):\n    """Point the sweep at the streamed Chien/Forney body."""\n    source = (_PORT / "round2_gadgets.py").read_text()\n    marker = "def lean_sweep("\n    start = source.index(marker)\n    body = source[start:]\n    for old, new in _SWEEP_EDITS:\n        if old not in body:\n            raise RuntimeError(f"sweep edit missed its target: {old[:60]}")\n        body = body.replace(old, new)\n    namespace = {"S": symbolic}\n    exec(body, namespace)\n\n    def native_sweep(builder, emitter, alloc, omw, ebw, abw):\n        qubits = [q for register in omw + ebw + abw for q in register]\n        builder.suspended = True\n        namespace["stream_sweep"](builder, emitter, alloc, omw, ebw, abw)\n        builder.invalidate(qubits)\n        builder.suspended = False\n\n    port.emit_sweep = native_sweep\n\n\ndef build_decoder(work: Path):\n    """Build the competition decoder. ``work`` holds regenerated constants."""\n    report = regenerate_tables(work)\n    import borrow as borrow_route\n    import gu_port as port\n    import round2_gadgets as gadgets\n    import symbolic_runtime as symbolic\n\n    port = importlib.reload(port)\n    gadgets = importlib.reload(gadgets)\n    port.Builder = symbolic.SymbolicBuilder\n    port.declare_registers = symbolic.declare_registers\n    port.borrow_template = borrow_route.borrow_template\n    gadgets.install(port, top_first=True, pebble_last=True, norm_pebble=True, compact_from=70)\n    _install_sweep(port, symbolic)\n    started = time.monotonic()\n    builder, counted = port.build()\n    builder.k.free(builder.measurement)\n    circuit = builder.k.finish_circuit()\n    report.update(\n        build_seconds=time.monotonic() - started,\n        counted_toffoli_like=int(counted),\n        num_qubits=int(circuit.num_qubits),\n        static_toffoli_like=int(circuit.operation_statistics().num_toffoli_like),\n        num_ops=len(circuit),\n        num_bits=int(circuit.num_bits),\n    )\n    return circuit, report\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--out", type=Path, required=True, help="Directory that receives dqi.kmb.")\n    parser.add_argument("--format", choices=("kmb",), default="kmb")\n    args = parser.parse_args()\n    args.out.mkdir(parents=True, exist_ok=True)\n    output = args.out / "dqi.kmb"\n    with tempfile.TemporaryDirectory(prefix="dqi-bm-") as tmp:\n        circuit, report = build_decoder(Path(tmp))\n        circuit.to_file(output, format=args.format)\n    digest = hashlib.sha256(output.read_bytes()).hexdigest()\n    print(\n        f"Wrote {output}: Q {report[\'num_qubits\']}, "\n        f"static Toffoli-like {report[\'static_toffoli_like\']}, "\n        f"ops {report[\'num_ops\']}, build {report[\'build_seconds\']:.1f}s."\n    )\n    print(f"sha256 {digest}")\n    print("field", report["modulus"], "primitive", report["primitive"])\n\n\nif __name__ == "__main__":\n    main()\n', 'tasks/invsqrt/PROVENANCE.md': '# invsqrt generator provenance\n\nThis generator is a **port**. It is not independent clean-room authorship.\nCoefficient provenance is `regenerated; dataset-adapted calibration; licence unresolved`.\nNothing in this map is a shipment.\nA ledger candidate still needs G0–G5, the full-shot verifier, and a\ncandidate-specific Adjudicator sign-off (D-PRA202-05). Historical P0 rows\nstay unchanged.\n\n## Coefficient fit (dataset-adapted calibration; licence unresolved)\n\n| Field | Value |\n| --- | --- |\n| Module | `tasks/invsqrt/fit_coefficients.py` |\n| Provenance | `regenerated; dataset-adapted calibration; licence unresolved` |\n| Calibration lineage | [kozykappa/quantum-ai-dataset version 1](https://www.kaggle.com/datasets/kozykappa/quantum-ai-dataset?datasetVersionId=21136269), datasetVersionId 21136269; Kaggle licence label "Other (specified in description)" |\n| Calibration source | `code/domain_tables.py`, SHA-256 `59cf4d98899433dcc49d9ce512293be20d33da5334211c20e35323555994a4fb`; `valid_residuals`, `calibrate_segment` and `regenerate_tables_exact` follow lines 8–43 |\n| Dataset table | `code/coefficients.json`, SHA-256 `98bd211f22b158bf05a4d2d934ab966e5e97e54d8307b82e2ca58910c6830763`; locally regenerated output equals 256 of 256 rows |\n| Rights evidence | README SHA-256 `fc19f8538dfbbfe9c8342d6fea10edc2ca178e66cd14eef14c12c3b648cdae70`; `catalog/license_map.csv` SHA-256 `5badf60ae5dc035f890b00e262f8e87345c3f13d323a405e89077b42a4d3fac8`. Neither names the two files; file-specific rights unresolved |\n| Caller | `signed_digit.regenerate_tables_exact` |\n| Target | `2**((e-1)/2) / sqrt(M)` on each segment, then the phase `1/sqrt(x)` |\n| Precision | float64 Lawson (`numpy.linalg.lstsq`, `rcond=None`), then Python ints. Error is in units of `2^-24` |\n| Domain | `x = 1..3133452`; segment `(e, a)` with `e` in {0, 1} and `a` in 0..127; `M` in [1, 2) |\n| Objective | minimax relative error of the quadratic signed-digit Horner value. Lawson: 80 iterations, 3001-point grid, weight `sqrt(M)/scale`. Then an integer radius search |\n| Rounding | Python `round` (half to even) for the K2 and K1 centres. Horner `rd` adds the bit under the kept field for a non-negative value. K0 is an integer argmin |\n| Widths | `f2=8`, `f1=19`, `fyr=27`, `gy=1`, `drop2=(5,)` |\n| Optimiser start | the rounded Lawson coefficient. Not an imported table |\n| Determinism | no random seed. A second in-process fit must match. Drift fails closed on the row digest |\n| Output | 256 `[K2, K1, K0]` triples, row-major `(e, a)`, JSON with separators `(\',\', \':\')` |\n| Row SHA-256 | `a36d1636d31d6b28a3bd5dce475bb3f8ef451b8b50deefc403853f184d4e6dcc`; different serialization, not the dataset file hash |\n\nThe committed recipe object is `RECIPE` in `fit_coefficients.py`.\n\n## Code parents\n\n| Work | Version | Licence | Role |\n| --- | --- | --- | --- |\n| Artem1410 (anazemcev / ViberTeam), "Quantum Circuits \\| Signed-Digit Memory" | v3 = scriptVersionId 361502374 | Apache-2.0, declared by the v3 source itself (cell 5 `SPDX-License-Identifier: Apache-2.0`; cell 20 embeds the Apache-2.0 text and NOTICE). Kaggle\'s version licence field was not retrievable without credentials: open | Immediate parent of `signed_digit.py`. Snapshot SHA-256 `79f6373302cb916a88f65a1b142024d4d2d0b14e41eb58bef3152ebd45ffd0ca`, equal to the public version-pinned download (22 cells; public metadata `currentVersionNumber` 3, `isPrivate` false). Extraction differs only by the added docstring and the removed `cli()` call (PRA-218). URL https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory?scriptVersionId=361502374 |\n| Kozykorpesh (kozykappa), "Quantum AI — Complete-Domain Affine Compilation" | v2 | Apache-2.0 | Method parent of `affine.py` and `peephole.py`. PRA-194 attachment SHA-256 `81befe1e69ee286271fa2eb8acf4bb760a727853bf966945318d918482b1ab24` |\n| Artem1410 (anazemcev / ViberTeam), "Quantum Circuits \\| Source & Exact Compilation" | v4 | Apache-2.0 | Davio orders and the measurement-uncompute rule. PRA-194 attachment SHA-256 `4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb` |\n| Kozykorpesh (kozykappa), "Quantum AI LB score" | V3 run 361677445 | Apache-2.0, as stated by that lineage | Method parent of `schedule90.py`. URL https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445 |\n\n## Credited, not loaded\n\nThese people are credited where their methods are used. The originals are not\ninputs. Sumit v1 versus v2, and Gluzdov v6, stay second-hand until\n[PRA-205](/PRA/issues/PRA-205) pins them.\n\n| Person | What is credited | Version named by the notebooks | Licence on this record |\n| --- | --- | --- | --- |\n| Sumit Chongder | centered signed-digit generator and integer pipeline | run 361357149 **is v1** (public version metadata, PRA-218). The inherited "latest version 2" remark is corrected in `signed_digit.py` by D-PRA220-01 | second-hand; no licence declaration in that snapshot: open. Not loaded |\n| Rudra Bhaskar | derivative split, top-carry reuse, static-high checkpoint | run 361445283 = v7 (public, PRA-218) | retained notice in the Apache-2.0 v3 parent; no publisher licence label retrieved: open. Not loaded |\n| Alex Chilton | shared prefix flags and empty-target shifts | v8 reviewed by PRA-194; the run the parent cites, 361472684, is v3 (public, PRA-218) | **reference-only**. PRA-194 found no notebook-wide licence; PRA-218 found none on v3 either. Not loaded |\n| Kirill Tymchishin | integration of the Rudra/Alex schedule | run 361491814 = v11 (public, PRA-218) | credited by the v3 parent; its cell 1 retains upstream Apache material, no licence field for its own integration: open. Not loaded |\n| Kozykorpesh | widths `8,19,27,1`, Gray schedule, affine rewrite, peephole | V3 run 361677445; affine v2 | Apache-2.0 for the attached affine v2 and the stated V3 lineage |\n| anazemcev / ViberTeam | signed-digit v3 body, v4 Davio orders, measurement uncompute | v3 and v4 | Apache-2.0 |\n| Dmitrii Gluzdov | V6 boundary rule used by the 90-qubit schedule | v6, run 361592338 | second-hand. His notebook is not loaded. `femoco_public_words.npz` is not an input |\n| LightningV08 and Mizero Lucky Gall | slack-maximising rotation drop | Mizero v1 | **reference-only**. PRA-194 found no notebook-wide licence. `calibrate.py` is our scipy linprog search, not a port of that notebook |\n\n## Module map\n\nHashes are SHA-256 of the committed file bytes. "Copied material" is none:\nthese files are ports and new code, not verbatim notebook cells.\n\n| Module | Source | Licence | Copied material | Modifications |\n| --- | --- | --- | --- | --- |\n| `signed_digit.py` | anazemcev Signed-Digit Memory v3; dataset domain-tables calibration | Apache-2.0 for the notebook parent; dataset-adapted calibration licence unresolved | none verbatim | Port of the quadratic signed-digit body, derivative split, borrowed carry, and promised-input rewrite. Coefficient integers come from the fit below. Retained upstream notices stay in the header. SHA-256 `ab08c1f178405cae9872281fbe73ca41c5de160bea4b5ddeece41b55ac708cd5` |\n| `schedule90.py` | Kozykorpesh V3 run 361677445; Davio orders from anazemcev v4 | Apache-2.0 | none verbatim | Port. Gluzdov V6 is the named boundary rule only. SHA-256 `58c88937e18e2c96e5e92f3a2576b3d32661de0d93221d4f288e314bb4c81edb` |\n| `affine.py` | Kozykorpesh affine v2 and anazemcev v4 | Apache-2.0 | none verbatim | Port of the exact-domain Toffoli/CSWAP rewrite. SHA-256 `aa351a287c507c264a8aa7d172e1c7b9bfc1d0dcb78c8fa61b92996947eba290` |\n| `peephole.py` | Kozykorpesh affine v2 involution pass | Apache-2.0 | none verbatim | Port. SHA-256 `d5ed02b0d28392cc637111fc94fb1760c71feda4f8ca7aad6f73944e81fff54c` |\n| `calibrate.py` | our scipy linprog search | Apache-2.0 (this repository) | none | Not a Mizero port. Method credit only. SHA-256 `7fc0c1cac6ff45753c25d98dfa7cd82fb4803332c37fb616af18fabe47cef309` |\n| `fit_coefficients.py` | the mathematical target and dataset-adapted calibration in `RECIPE` | `regenerated; dataset-adapted calibration; licence unresolved` | none | The fit. SHA-256 `35d36f95aa4bb7c63bf8cd24d72df594abdd679a3ed770cb2fcced0200cda839` |\n| `generate.py` | the modules above | port; coefficient provenance: `regenerated; dataset-adapted calibration; licence unresolved` | none | Entry point. Writes only `invsqrt.kmb`. SHA-256 `9ccdffd0cba53a097c78d99d833b9bcc37e9be387f62ed723a8173cd2cdf498c` |\n| `reference.py` | Google\'s `verifiers/invsqrt/nr_reference_circuit.py` (competition package) | Apache-2.0 (Google) | the script, attributed | Port with `--out DIR`; builds the reference circuit shipped as submission-001, SHA-256 `e2bf8eb64350a4460a271fc7f1d591ef125b1e276377e9a0f64b96ab3ca321e1`. G5 generator for `configs/gates/reference.yaml` only (PRA-211). Not a parent of the candidate and not used by `generate.py` |\n| `mirror.py` | the integer model | Apache-2.0 (this repository) | none | G2 certificate. SHA-256 `44f9928fe54f84c45977e9cfe637e7ad2068ab7cda7dd9fb2949a6666cd1513d` |\n| `gates.py` | the ledger hook contract | Apache-2.0 (this repository) | none | G1/G2 hooks. Not a method parent. SHA-256 `3f569d70c97f3c41f7b627050e7a40494d54c546da196788c1bc08f55162890d` |\n\n## Public-source evidence (PRA-218, 2026-10-25)\n\nCaine\'s packet ([PRA-218](/PRA/issues/PRA-218) `invsqrt-licence-evidence`, archive SHA-256\n`38f162467df166c0294e00db8dd1beabfad03c6727cfad2182e229c75ed04fe8`) retrieved every source above\nwithout credentials and pinned it by hash and UTC time. John Wick re-ran its verifier at `3404dff`\n(eight PASS lines) and re-checked the facts below on `main` after PRA-216.\n\n| Item (PRA-179 provenance review) | Closed with public evidence | Still open |\n| --- | --- | --- |\n| Immediate parent and attribution | v3 = 361502374, public, snapshot equal to the committed pin, Apache-2.0 declared in its own source, extraction diff recorded | Kaggle\'s version-specific licence field (public `GetKernelVersion` returns 403) |\n| Transitive code | Sumit 361357149 = v1, Rudra 361445283 = v7, Alex 361472684 = v3, Kirill 361491814 = v11, Kozykorpesh 361456406 = v2 (slug now `quantum-ai-lb-score-6-68108`); all public, cell-level AST comparison recorded | a publisher licence declaration for each of the five versions |\n| Numeric table and calibration dataset | `kozykappa/quantum-ai-dataset`, dataset id 12801707, datasetVersionId 21136269 = version 1, public; licence label **"Other (specified in description)"**; README: "New atlas tooling and derived numerical generation code: Apache-2.0"; `catalog/license_map.csv` does not name `code/coefficients.json` or `code/domain_tables.py` | file-specific rights for both files unresolved; affected candidate held (D-PRA220-01) |\n| Public-source inventory | every listed source is anonymously retrievable today | historical absence of private sharing cannot be proved from present public access |\n\nFinding recorded on PRA-211, now ruled by [D-PRA220-01](../../docs/decisions/D-PRA220-01.md):\nthe locally regenerated fit reproduces the dataset table 256 of 256 rows, and the calibration\nis adapted from its domain-tables module (version and file hashes above). The\n[Adjudicator verdict](/PRA/issues/PRA-220#document-adjudicator-verdict), revision\n`a70476a7-9756-406e-abdb-f7985a26c96e`, supersedes D-PRA200-01\'s `own` and\nlicence-closure conclusion and resolves D-PRA211-04(b) as **unresolved rights; affected candidate held**.\nNo Kaggle upload of a zip whose `invsqrt.kmb` comes from `tasks/invsqrt/generate.py`\'s current coefficient lineage until the Adjudicator accepts file-specific rights evidence for both files ([PRA-222](/PRA/issues/PRA-222)) or clears an independently implemented replacement ([PRA-223](/PRA/issues/PRA-223)). Ledger enforcement is [PRA-224](/PRA/issues/PRA-224); this correction records the hold and does not implement it.\nHistorical decisions and manifests remain unchanged.\n\n## Forbidden inputs (not loaded)\n\nThese names are not opens, warm starts, or test oracles in `tasks/invsqrt` Python:\n\n- `code/coefficients.json` (Kaggle dataset version 21136269)\n- toolbox `fusion/*.npz`\n- `selected/invsqrt.kmb`\n- any `.npz` load\n\nA recorded SHA-256 of a circuit we generated is not a load of that file.\n', 'tasks/invsqrt/__init__.py': '"""Invsqrt generator, integer mirror, and ledger hooks."""\n', 'tasks/invsqrt/affine.py': '"""Exact-domain affine rewrite for the invsqrt circuit.\n\nThis file is a port of the method below. It is not independent authorship.\n\nMethod parents (Apache-2.0), reimplemented here:\n- Kozykorpesh, complete-domain affine compilation v2: linear GF(2) replacement\n  of Toffoli and CSWAP, and reuse of qubit ids whose value is identically 0.\n- anazemcev, source & exact compilation v4: measurement uncompute when the\n  Toffoli result is an affine function of the other wires. The kickback is\n  cancelled by one conditioned CZ. The conditional-Z affine-measurement\n  variant is not used.\n\nEvery accepted replacement is checked against the full 3,133,452-input truth\nvector. A sampled linear system only proposes a combination. Missed rewrites\nare allowed. An invalid replacement is not.\n\nPhase recorded here is the unconditional Z_POW and unconditional CZ/Z\ncontribution. Conditioned diagonals are kickback fixups and are not added.\n"""\n\nfrom __future__ import annotations\n\nimport collections\nimport heapq\nimport re\nfrom decimal import Decimal\nfrom fractions import Fraction\n\nimport numpy as np\n\nMAX_X = 3_133_452\nWORDS = (MAX_X + 63) // 64\n_QUBIT = re.compile(r"(?<!\\w)q(\\d+)(?!\\w)")\n_BIT = re.compile(r"(?<!\\w)b(\\d+)(?!\\w)")\n_METADATA = {\n    "REGISTER",\n    "INFO_TAG",\n    "APPEND_TO_REGISTER",\n    "BIT_STORE0",\n    "BIT_STORE1",\n    "BIT_INVERT",\n    "BIT_XOR",\n}\n_DIAGONAL = {"Z", "CZ", "CCZ", "Z_POW", "NEG"}\n\n\ndef _pack_input_bit(xs: np.ndarray, bit: int) -> np.ndarray:\n    flags = ((xs >> bit) & 1).astype(np.uint8)\n    packed = np.packbits(flags, bitorder="little")\n    buffer = np.zeros(WORDS * 8, dtype=np.uint8)\n    buffer[: len(packed)] = packed\n    return np.ascontiguousarray(buffer.view(np.uint64)[:WORDS])\n\n\nclass CompiledCircuit:\n    """Affine-compiled text plus the phase certificate collected on the way."""\n\n    def __init__(self, text, phase, rotations, peak, rewrites, fresh_bits):\n        self.text = text\n        self.phase = phase\n        self.rotations = rotations\n        self.peak = peak\n        self.rewrites = rewrites\n        self.fresh_bits = fresh_bits\n\n\nclass DomainCompiler:\n    def __init__(self, text: str):\n        self.lines = [line for line in text.splitlines() if line]\n        self.protected = set()\n        for line in self.lines:\n            if line.startswith("APPEND_TO_REGISTER"):\n                self.protected.update(int(match) for match in _QUBIT.findall(line))\n        if self.protected != set(range(22)):\n            raise RuntimeError(f"Expected input qubits 0..21, found {sorted(self.protected)}")\n        seen_bits = [int(match) for line in self.lines for match in _BIT.findall(line)]\n        self.fresh_bit = 1 + max(seen_bits, default=-1)\n        self.zero = np.zeros(WORDS, dtype=np.uint64)\n        self.full = np.full(WORDS, np.uint64(2**64 - 1))\n        if MAX_X % 64:\n            self.full[-1] = np.uint64((1 << (MAX_X % 64)) - 1)\n        self.full.setflags(write=False)\n        self.zero.setflags(write=False)\n        xs = np.arange(1, MAX_X + 1, dtype=np.uint64)\n        self.original = [_pack_input_bit(xs, bit) for bit in range(22)]\n        if int(self.original[0][0] & 1) != 1:\n            raise RuntimeError("Input bit packing is not little-endian on x=1")\n        self.values = {qubit: self.original[qubit] for qubit in range(22)}\n        self.mapping = {qubit: qubit for qubit in range(22)}\n        self.free: list[int] = []\n        self.next_id = 22\n        self.peak = 22\n        self.depth = 0\n        self.stack: list[dict] = []\n        self.out: list[str] = []\n        self.rewrites: collections.Counter = collections.Counter()\n        self.phase = np.zeros(MAX_X, dtype=np.uint64)\n        self.rotations: list[dict] = []\n        self.sample = np.unique(\n            np.r_[\n                np.arange(min(8, WORDS)),\n                np.linspace(0, WORDS - 1, 8, dtype=int),\n            ]\n        )\n        self.false_candidates = 0\n\n    def val(self, qubit: int):\n        return self.values.get(qubit, self.zero)\n\n    def canon(self, array: np.ndarray):\n        if array is self.zero or array is self.full:\n            return array\n        if not array.any():\n            return self.zero\n        if np.array_equal(array, self.full):\n            return self.full\n        array.setflags(write=False)\n        return array\n\n    def equal(self, left, right) -> bool:\n        if left is right:\n            return True\n        return bool(np.array_equal(left, right))\n\n    def xor(self, left, right):\n        if left is self.zero:\n            return right\n        if right is self.zero:\n            return left\n        if left is right:\n            return self.zero\n        return self.canon(np.bitwise_xor(left, right))\n\n    def band(self, left, right):\n        if left is self.zero or right is self.zero:\n            return self.zero\n        if left is self.full:\n            return right\n        if right is self.full or left is right:\n            return left\n        return self.canon(np.bitwise_and(left, right))\n\n    def materialize(self, qubit: int) -> int:\n        if qubit not in self.mapping:\n            if self.free:\n                self.mapping[qubit] = heapq.heappop(self.free)\n            else:\n                self.mapping[qubit] = self.next_id\n                self.next_id += 1\n                self.peak = self.next_id\n        return self.mapping[qubit]\n\n    def release(self):\n        if self.depth:\n            return\n        for qubit in list(self.mapping):\n            if qubit in self.protected:\n                continue\n            if self.val(qubit) is self.zero:\n                heapq.heappush(self.free, self.mapping.pop(qubit))\n                self.values.pop(qubit, None)\n\n    def emit(self, line: str):\n        def replace(match):\n            qubit = int(match.group(1))\n            return f"q{self.materialize(qubit)}"\n\n        self.out.append(_QUBIT.sub(replace, line))\n\n    def literal(self, function, excluded: set[int]):\n        if function is self.zero or function is self.full:\n            return None\n        for qubit, value in self.values.items():\n            if qubit not in excluded and self.equal(function, value):\n                return qubit, False\n        complement = self.xor(function, self.full)\n        for qubit, value in self.values.items():\n            if qubit not in excluded and self.equal(complement, value):\n                return qubit, True\n        return None\n\n    def _signature(self, value, extra: list[int]) -> int:\n        acc = 0\n        for index in self.sample:\n            acc = (acc << 64) | int(value[index])\n        for index in extra:\n            acc = (acc << 64) | int(value[index])\n        return acc\n\n    def linear_terms(self, function, excluded: set[int]):\n        """Propose an affine combination on samples and prove it on every x."""\n        if function is self.zero:\n            return [], False\n        if function is self.full:\n            return [], True\n        variables = [\n            qubit\n            for qubit, value in self.values.items()\n            if qubit not in excluded and value is not self.zero and value is not self.full\n        ]\n        columns = [self.full] + [self.values[qubit] for qubit in variables]\n        extra: list[int] = []\n        for _attempt in range(8):\n            basis = {}\n            for index, value in enumerate(columns):\n                vector = self._signature(value, extra)\n                mask = 1 << index\n                while vector:\n                    pivot = vector.bit_length() - 1\n                    if pivot not in basis:\n                        basis[pivot] = (vector, mask)\n                        break\n                    existing, coeff = basis[pivot]\n                    vector ^= existing\n                    mask ^= coeff\n            vector = self._signature(function, extra)\n            mask = 0\n            while vector:\n                pivot = vector.bit_length() - 1\n                if pivot not in basis:\n                    return None\n                existing, coeff = basis[pivot]\n                vector ^= existing\n                mask ^= coeff\n            chosen = [index for index in range(len(columns)) if (mask >> index) & 1]\n            exact = self.zero\n            for index in chosen:\n                exact = self.xor(exact, columns[index])\n            if self.equal(exact, function):\n                return [variables[index - 1] for index in chosen if index], (0 in chosen)\n            self.false_candidates += 1\n            mismatch = np.flatnonzero(np.not_equal(exact, function))\n            if len(mismatch) == 0 or int(mismatch[0]) in extra:\n                return None\n            extra.append(int(mismatch[0]))\n        return None\n\n    def _clifford(self, target: int, terms, constant: bool) -> list[str]:\n        lines = [f"X q{target}"] if constant else []\n        lines.extend(f"CX q{qubit} q{target}" for qubit in terms)\n        return lines\n\n    def _add_phase(self, bits, units: int):\n        if units == 0 or bits is self.zero:\n            return\n        if bits is self.full:\n            self.phase += np.uint64(units)\n            return\n        flat = np.unpackbits(bits.view(np.uint8), bitorder="little")[:MAX_X]\n        self.phase[flat.astype(bool)] += np.uint64(units)\n\n    def _record_rotation(self, logical: int, theta: Fraction, bits):\n        counted = (theta * 4).denominator != 1\n        units = theta * (1 << 63)\n        if units.denominator != 1:\n            raise RuntimeError(f"Rotation {theta} is not on the 2^-64 lattice")\n        flat = np.ones(MAX_X, dtype=np.uint8) if bits is self.full else (\n            np.unpackbits(bits.view(np.uint8), bitorder="little")[:MAX_X].astype(np.uint8)\n        )\n        self.rotations.append(\n            {\n                "line": len(self.out),\n                "logical": logical,\n                "theta": theta,\n                "counted": counted,\n                "units": int(units),\n                "bits": flat,\n            }\n        )\n\n    def _conditioned(self, parts: list[str]) -> bool:\n        return self.depth > 0 or "if" in parts\n\n    def execute(self, line: str):\n        parts = line.split()\n        op = parts[0]\n        if op in _METADATA:\n            self.emit(line)\n            return\n        if op == "PUSH_CONDITION":\n            self.stack.append(dict(self.values))\n            self.depth += 1\n            self.emit(line)\n            return\n        if op == "POP_CONDITION":\n            before = self.stack.pop()\n            self.depth -= 1\n            if self.depth < 0:\n                raise RuntimeError("Unbalanced POP_CONDITION")\n            keys = set(before) | set(self.values)\n            for qubit in keys:\n                if not self.equal(self.val(qubit), before.get(qubit, self.zero)):\n                    raise RuntimeError(f"Condition join changed q{qubit}")\n            self.emit(line)\n            self.release()\n            return\n        qubits = [int(match) for match in _QUBIT.findall(line)]\n        if op in _DIAGONAL and self._conditioned(parts):\n            if op == "Z_POW":\n                raise RuntimeError(f"Conditioned rotation is outside this compiler: {line}")\n            self.emit(line)\n            return\n        if "if" in parts and op not in _DIAGONAL:\n            raise RuntimeError(f"Inline condition on {op} is unsupported: {line}")\n        if op == "CCX":\n            left, right, target = qubits\n            product = self.band(self.val(left), self.val(right))\n            replacement = None\n            reason = None\n            if product is self.zero:\n                replacement, reason = [], "ccx_zero"\n            elif product is self.full:\n                replacement, reason = [f"X q{target}"], "ccx_one"\n            else:\n                found = self.literal(product, {target})\n                if found is not None:\n                    qubit, inverted = found\n                    replacement = self._clifford(target, [qubit], inverted)\n                    reason = "ccx_literal"\n                else:\n                    terms = self.linear_terms(product, {target})\n                    if terms is not None:\n                        variables, constant = terms\n                        replacement = self._clifford(target, variables, constant)\n                        reason = "ccx_affine"\n            if replacement is None and self.depth == 0:\n                result = self.xor(self.val(target), product)\n                terms = None\n                if result is self.zero:\n                    terms = [], False\n                elif result is self.full:\n                    terms = [], True\n                else:\n                    found = self.literal(result, {target})\n                    if found is not None:\n                        qubit, inverted = found\n                        terms = [qubit], inverted\n                    else:\n                        terms = self.linear_terms(result, {target})\n                if terms is not None:\n                    variables, constant = terms\n                    prepare = self._clifford(target, variables, constant)\n                    bit = self.fresh_bit\n                    self.fresh_bit += 1\n                    replacement = prepare + [\n                        f"HMR q{target} b{bit}",\n                        f"CZ q{left} q{right} if b{bit}",\n                    ] + prepare\n                    reason = "ccx_mbu"\n            if replacement is None:\n                self.emit(line)\n            else:\n                self.rewrites[reason] += 1\n                for emitted in replacement:\n                    self.emit(emitted)\n            self.values[target] = self.xor(self.val(target), product)\n            self.release()\n            return\n        if op == "CSWAP":\n            control, left, right = qubits\n            delta = self.xor(self.val(left), self.val(right))\n            product = self.band(self.val(control), delta)\n            replacement = None\n            reason = None\n            if product is self.zero:\n                replacement, reason = [], "cswap_zero"\n            elif self.equal(product, delta):\n                replacement, reason = [f"SWAP q{left} q{right}"], "cswap_swap"\n            else:\n                found = self.literal(product, {left, right})\n                if found is not None:\n                    qubit, inverted = found\n                    replacement = []\n                    if inverted:\n                        replacement += [f"X q{left}", f"X q{right}"]\n                    replacement += [f"CX q{qubit} q{left}", f"CX q{qubit} q{right}"]\n                    reason = "cswap_literal"\n                else:\n                    terms = self.linear_terms(product, {left, right})\n                    if terms is not None:\n                        variables, constant = terms\n                        replacement = []\n                        if constant:\n                            replacement += [f"X q{left}", f"X q{right}"]\n                        for qubit in variables:\n                            replacement += [f"CX q{qubit} q{left}", f"CX q{qubit} q{right}"]\n                        reason = "cswap_affine"\n            if replacement is None:\n                self.emit(line)\n            else:\n                self.rewrites[reason] += 1\n                for emitted in replacement:\n                    self.emit(emitted)\n            self.values[left] = self.xor(self.val(left), product)\n            self.values[right] = self.xor(self.val(right), product)\n            self.release()\n            return\n        if op == "CX":\n            control, target = qubits\n            control_value = self.val(control)\n            if control_value is self.zero:\n                self.rewrites["cx_zero"] += 1\n            elif control_value is self.full:\n                self.rewrites["cx_one"] += 1\n                self.emit(f"X q{target}")\n            else:\n                self.emit(line)\n            self.values[target] = self.xor(self.val(target), control_value)\n            self.release()\n            return\n        if op == "X":\n            for qubit in qubits:\n                self.values[qubit] = self.xor(self.val(qubit), self.full)\n            self.emit(line)\n            self.release()\n            return\n        if op == "SWAP":\n            left, right = qubits\n            self.values[left], self.values[right] = self.val(right), self.val(left)\n            self.emit(line)\n            self.release()\n            return\n        if op == "R":\n            if all(self.val(qubit) is self.zero for qubit in qubits):\n                self.rewrites["r_zero"] += 1\n            else:\n                self.emit(line)\n            for qubit in qubits:\n                self.values[qubit] = self.zero\n            self.release()\n            return\n        if op == "HMR":\n            self.emit(line)\n            for qubit in qubits[:1]:\n                self.values[qubit] = self.zero\n            self.release()\n            return\n        if op == "Z_POW":\n            theta = Fraction(Decimal(parts[2]))\n            bits = self.val(qubits[0])\n            if bits is self.zero:\n                self.rewrites["zpow_zero"] += 1\n            else:\n                self._record_rotation(qubits[0], theta, bits)\n                self.emit(line)\n                units = theta * (1 << 63)\n                self._add_phase(bits, int(units))\n            self.release()\n            return\n        if op in {"Z", "CZ", "CCZ"}:\n            product = self.full\n            for qubit in qubits:\n                product = self.band(product, self.val(qubit))\n            units = 1 << 63\n            if product is self.zero:\n                self.rewrites[f"{op.lower()}_zero"] += 1\n            else:\n                self.emit(line)\n                self._add_phase(product, units)\n            self.release()\n            return\n        if op == "NEG":\n            self.emit(line)\n            self.phase += np.uint64(1 << 63)\n            self.release()\n            return\n        raise RuntimeError(f"Unsupported operation: {line}")\n\n    def compile(self) -> CompiledCircuit:\n        for number, line in enumerate(self.lines, start=1):\n            self.execute(line)\n            if number % 4000 == 0:\n                print(\n                    f"affine {number}/{len(self.lines)} rewrites {sum(self.rewrites.values())} "\n                    f"peak {self.peak} false {self.false_candidates}",\n                    flush=True,\n                )\n        if self.depth or self.stack:\n            raise RuntimeError("Unbalanced condition stack")\n        for qubit in range(22):\n            if not self.equal(self.val(qubit), self.original[qubit]):\n                raise RuntimeError(f"Input q{qubit} was not restored")\n        for qubit, value in self.values.items():\n            if qubit not in self.protected and value is not self.zero:\n                raise RuntimeError(f"Scratch q{qubit} was not restored")\n        ids = list(self.mapping.values())\n        if len(ids) != len(set(ids)):\n            raise RuntimeError("Physical qubit map is not injective")\n        text = "\\n".join(self.out) + "\\n"\n        return CompiledCircuit(\n            text,\n            self.phase,\n            self.rotations,\n            self.peak,\n            dict(self.rewrites),\n            self.fresh_bit,\n        )\n\n\ndef compile_circuit(circuit) -> CompiledCircuit:\n    """Rewrite `circuit` and return text, phase and rotation activations."""\n    return DomainCompiler(str(circuit)).compile()\n', 'tasks/invsqrt/calibrate.py': '"""Drop one invsqrt rotation when every x keeps a positive margin.\n\nThis file is our scipy linprog calibrator. The slack-maximising drop is\ncredited to LightningV08 and to Mizero Lucky Gall. Mizero v1 has no\nnotebook-wide licence grant and is reference-only: it is not loaded, and it\nis not a shipping parent. This file is not a port of that notebook.\n\nThe linear program maximises the worst slack. Rounding is then checked in\nintegers on all 3,133,452 inputs. No verifier salt, calibration file, or\ncircuit binary is an input.\n\nThe activation bits and the starting phase come from the affine compiler.\nOnly unconditioned, exactly representable Z_POW gates are moved. Clifford\nZ_POW gates stay fixed.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom fractions import Fraction\n\nimport numpy as np\nfrom scipy.optimize import linprog\n\nfrom tasks.invsqrt.peephole import optimize\n\nMAX_X = 3_133_452\nUNIT = 1 << 64\nHALF = UNIT // 2\nREFERENCE_COST = 2_076_774.233715282\n\n\ndef reference_target_tolerance(xs: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    """Official invsqrt target and tolerance for beta = 1, precision 24.\n\n    `r = isqrt(floor(2^130 / x))`, target is `r` rounded at bit 0, tolerance\n    is `min(r >> 25, 2^63)`.\n    """\n    roots = [math.isqrt((1 << 130) // int(x)) for x in xs.tolist()]\n    target = np.array([((root + 1) >> 1) & (UNIT - 1) for root in roots], dtype=np.uint64)\n    tolerance = np.array([min(root >> 25, 1 << 63) for root in roots], dtype=np.uint64)\n    return target, tolerance\n\n\ndef model_phase(cfg, tables) -> np.ndarray:\n    """Integer-model phase in units of 2^-64 turns, for every x."""\n    from tasks.invsqrt.signed_digit import integer_model\n\n    xs = np.arange(1, MAX_X + 1, dtype=np.int64)\n    shift = np.uint64(64 - (cfg.fy + 10))\n    phase = np.empty(MAX_X, dtype=np.uint64)\n    for start in range(0, MAX_X, 131072):\n        block = xs[start : start + 131072]\n        values = integer_model(block, cfg, tables).astype(np.uint64)\n        phase[start : start + len(block)] = values << shift\n    return phase\n\n\ndef assert_phase_matches(compiled, cfg, tables) -> dict:\n    """Require the compiled unconditional phase to equal the integer model."""\n    expected = model_phase(cfg, tables)\n    mismatch = np.flatnonzero(compiled.phase != expected)\n    if len(mismatch):\n        index = int(mismatch[0])\n        raise RuntimeError(\n            "Compiled phase does not match the integer model at "\n            f"x={index + 1}: circuit {int(compiled.phase[index])} "\n            f"model {int(expected[index])} mismatches {len(mismatch)}"\n        )\n    return {"phase_match": True, "inputs": MAX_X}\n\n\ndef _matvec(bits: np.ndarray, columns: list[int], delta: np.ndarray) -> np.ndarray:\n    acc = np.zeros(bits.shape[0], dtype=np.float64)\n    for index, column in enumerate(columns):\n        if delta[index] == 0:\n            continue\n        acc += bits[:, column] * float(delta[index])\n    return acc\n\n\ndef _fit(error, tolerance, bits, free, fixed, bound_units, scale=2.0**28):\n    """Maximise slack with cutting planes. `fixed` is the dropped-rotation shift."""\n    residual = (error.astype(np.float64) + fixed) / scale\n    limit = tolerance.astype(np.float64) / scale\n    width = len(free)\n    active = set(np.linspace(0, MAX_X - 1, 4096, dtype=np.int64).tolist())\n    initial = limit - np.abs(residual)\n    active.update(np.argpartition(initial, 4095)[:4096].tolist())\n    delta = np.zeros(width, dtype=np.float64)\n    margin = 0.0\n    converged = False\n    iterations = 0\n    for iterations in range(1, 61):\n        chosen = np.fromiter(sorted(active), dtype=np.int64)\n        block = bits[chosen][:, free].astype(np.float64)\n        matrix = np.vstack(\n            [np.c_[block, np.ones(len(chosen))], np.c_[-block, np.ones(len(chosen))]]\n        )\n        rhs = np.r_[limit[chosen] - residual[chosen], limit[chosen] + residual[chosen]]\n        bounds = [(-bound_units / scale, bound_units / scale)] * width + [(None, None)]\n        result = linprog(\n            np.r_[np.zeros(width), -1.0],\n            A_ub=matrix,\n            b_ub=rhs,\n            bounds=bounds,\n            method="highs",\n        )\n        if not result.success:\n            raise RuntimeError(result.message)\n        delta, margin = result.x[:width], float(result.x[width])\n        corrected = residual + _matvec(bits, free, delta)\n        slack = limit - np.abs(corrected)\n        actual = float(slack.min())\n        print(\n            f"calibration LP {iterations}: rows {len(chosen)} "\n            f"lp {margin * scale:.3f} exhaustive {actual * scale:.3f}",\n            flush=True,\n        )\n        if actual >= margin - 1e-9 * max(1.0, abs(margin)):\n            converged = True\n            break\n        added = set(np.argpartition(slack, 399)[:400].tolist())\n        if added.issubset(active):\n            break\n        active.update(added)\n    return np.rint(delta * scale).astype(np.int64), {\n        "iterations": iterations,\n        "lp_converged": converged,\n        "active_inputs": len(active),\n    }\n\n\ndef _exact_margin(error, tolerance, bits, deltas) -> tuple[int, int]:\n    shift = np.zeros(MAX_X, dtype=np.int64)\n    for index, delta in enumerate(deltas):\n        if int(delta) == 0:\n            continue\n        shift += bits[:, index].astype(np.int64) * np.int64(delta)\n    largest = max(abs(int(error.min())), abs(int(error.max())))\n    if largest + sum(abs(int(delta)) for delta in deltas) >= HALF:\n        raise RuntimeError("Rounded correction can overflow signed 64-bit phase")\n    corrected = error.astype(np.int64) + shift\n    slack = tolerance.astype(np.int64) - np.abs(corrected)\n    worst = int(np.argmin(slack))\n    return int(slack[worst]), worst\n\n\ndef _angle_key(theta: Fraction) -> float:\n    reduced = theta % 2\n    return abs(float(min(reduced, 2 - reduced)))\n\n\ndef _estimate(qubits, expected_toffoli, t_like, rotations, margin) -> dict:\n    if rotations <= 0 or margin <= 0:\n        return {"estimated_expected_cost": math.inf, "ratio": 0.0, "per_rotation_t": None}\n    epsilon = math.pi * (margin / UNIT) / rotations\n    per_rotation = 0.56 * math.log2(1 / epsilon) + 5.3\n    extra = (t_like + rotations * per_rotation) / 4\n    cost = qubits * (expected_toffoli + extra)\n    return {\n        "estimated_expected_cost": cost,\n        "ratio": REFERENCE_COST / cost,\n        "per_rotation_t": per_rotation,\n        "epsilon": epsilon,\n    }\n\n\ndef _rewrite(lines: list[str], rotations: list[dict], deltas: dict[int, int], dropped: set[int]) -> str:\n    from decimal import Decimal, localcontext\n\n    def exponent_text(theta: Fraction) -> str:\n        with localcontext() as ctx:\n            ctx.prec = 80\n            rendered = format(Decimal(theta.numerator) / Decimal(theta.denominator), "f")\n        if "." not in rendered:\n            rendered += ".0"\n        return rendered\n\n    remove = set()\n    out = list(lines)\n    for index, rotation in enumerate(rotations):\n        if index in dropped:\n            remove.add(rotation["line"])\n            continue\n        delta = int(deltas.get(index, 0))\n        if delta == 0:\n            continue\n        theta = rotation["theta"] + Fraction(delta, HALF)\n        qubit = lines[rotation["line"]].split()[1]\n        out[rotation["line"]] = f"Z_POW {qubit} {exponent_text(theta)}"\n    body = "\\n".join(line for index, line in enumerate(out) if index not in remove and line)\n    return body + "\\n"\n\n\ndef calibrate(compiled, resources: dict) -> dict:\n    """Try the original angles and each single counted drop. Keep the cheapest legal one."""\n    rotations = compiled.rotations\n    counted = [index for index, rotation in enumerate(rotations) if rotation["counted"]]\n    if not counted:\n        raise RuntimeError("The compiled circuit has no counted rotation")\n    bits = np.column_stack([rotation["bits"] for rotation in rotations])\n    xs = np.arange(1, MAX_X + 1, dtype=np.int64)\n    _target, tolerance = reference_target_tolerance(xs)\n    # Reuse the packed target. reference_target_tolerance already walked every x.\n    target = _target\n    error = (compiled.phase - target).view(np.int64)\n    order = sorted(counted, key=lambda index: _angle_key(rotations[index]["theta"]))\n    # The public build drops one rotation. Try the original and the smallest\n    # angle first. Search the other single drops only when that drop has no margin.\n    candidates = [(), (order[0],)]\n    best = None\n    trials = []\n    expanded = False\n    for dropped in candidates:\n        fixed = np.zeros(MAX_X, dtype=np.float64)\n        bound = float(1 << 36)\n        for index in dropped:\n            units = rotations[index]["units"]\n            fixed += bits[:, index] * float(-units)\n            bound = max(bound, 4.0 * abs(units))\n        free = [index for index in counted if index not in dropped]\n        rounded, fit_report = _fit(error, tolerance, bits, free, fixed, bound)\n        deltas = np.zeros(len(rotations), dtype=np.int64)\n        for column, index in enumerate(free):\n            deltas[index] = rounded[column]\n        for index in dropped:\n            deltas[index] = -int(rotations[index]["units"])\n        margin, worst = _exact_margin(error, tolerance, bits, deltas)\n        kept = len(counted) - len(dropped)\n        estimate = _estimate(\n            resources["qubits"],\n            resources["expected_toffoli"],\n            resources["t_like"],\n            kept,\n            margin,\n        )\n        trial = {\n            "dropped": [int(index) for index in dropped],\n            "margin_units": margin,\n            "worst_index": worst,\n            "rotations": kept,\n            "fit": fit_report,\n            **estimate,\n        }\n        trials.append({key: trial[key] for key in ("dropped", "margin_units", "rotations", "ratio")})\n        print(\n            f"calibration drop {list(dropped) or \'none\'}: margin {margin} "\n            f"rotations {kept} ratio {estimate[\'ratio\']:.6f}",\n            flush=True,\n        )\n        if margin <= 0:\n            if dropped == (order[0],) and not expanded:\n                expanded = True\n                candidates.extend((index,) for index in order[1:])\n            continue\n        if best is None or estimate["estimated_expected_cost"] < best["estimate"]["estimated_expected_cost"]:\n            best = {"deltas": deltas, "dropped": set(dropped), "trial": trial, "estimate": estimate}\n    if best is None:\n        raise RuntimeError(f"No calibration kept a positive margin: {trials}")\n    lines = compiled.text.splitlines()\n    rewritten = _rewrite(\n        lines,\n        rotations,\n        {index: int(best["deltas"][index]) for index in range(len(rotations))},\n        best["dropped"],\n    )\n    peepholed, cancelled = optimize(rewritten)\n    return {\n        "text": peepholed,\n        "margin_units": best["trial"]["margin_units"],\n        "worst_x": int(best["trial"]["worst_index"]) + 1,\n        "rotations": best["trial"]["rotations"],\n        "dropped_rotation_indexes": best["trial"]["dropped"],\n        "peephole_cancelled": cancelled,\n        "estimate": best["estimate"],\n        "trials": trials,\n    }\n', 'tasks/invsqrt/fit_coefficients.py': '"""Regenerate the invsqrt QROM table with dataset-adapted calibration.\n\nThe 256 triples are the output of the Lawson weighted minimax followed by the\nexact-domain integer search in `signed_digit.regenerate_tables_exact`. No\nexternal dataset is read, but the calibration lineage has unresolved rights.\nSee RECIPE["calibration_lineage"] for the version-pinned source and hashes.\nThe SHA-256 below is a drift check: if the numeric\nlibraries move a coefficient, generation stops instead of silently changing\nthe scored circuit.\n\n    python tasks/invsqrt/fit_coefficients.py\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport sys\nfrom functools import lru_cache\nfrom pathlib import Path\n\nif __package__ in (None, ""):\n    sys.path.insert(0, str(Path(__file__).resolve().parents[2]))\n\nfrom tasks.invsqrt import signed_digit as sd  # noqa: E402\n\n# Digest of the integer rows this environment derives. Not a dataset pin.\nEXPECTED_ROW_SHA256 = "a36d1636d31d6b28a3bd5dce475bb3f8ef451b8b50deefc403853f184d4e6dcc"\nPROVENANCE = "regenerated; dataset-adapted calibration; licence unresolved"\n\n# Committed fitting recipe. The integer search starts at the rounded Lawson\n# coefficient of the mathematical target. No external table is read, and none\n# is the optimiser start. Historical P0 copies stay in the old records.\nRECIPE = {\n    "provenance": PROVENANCE,\n    "calibration_lineage": {\n        "dataset_url": "https://www.kaggle.com/datasets/kozykappa/quantum-ai-dataset?datasetVersionId=21136269",\n        "dataset_version": 1,\n        "dataset_version_id": 21136269,\n        "domain_tables_module_sha256": "59cf4d98899433dcc49d9ce512293be20d33da5334211c20e35323555994a4fb",\n        "coefficient_table_sha256": "98bd211f22b158bf05a4d2d934ab966e5e97e54d8307b82e2ca58910c6830763",\n    },\n    "target": (\n        "For each segment, y = 2**((e - 1) / 2) / sqrt(M) with M in [1, 2). "\n        "The emitted phase is beta / sqrt(x) at beta = 1."\n    ),\n    "precision": (\n        "IEEE-754 float64 Lawson fit (numpy.linalg.lstsq, rcond=None) on a "\n        "fixed grid. Stored triples are Python ints. Relative error is judged "\n        "in units of 2^-24."\n    ),\n    "domain": (\n        "x = 1..3133452. Segment key (e, a) with e in {0, 1} and a in 0..127. "\n        "The residual set is valid_residuals. M = x_norm / 2^21 in [1, 2)."\n    ),\n    "objective": (\n        "Minimise the maximum relative error of the quadratic signed-digit "\n        "Horner value against the target. Continuous stage: Lawson reweighting, "\n        "80 iterations, 3001-point grid, weight sqrt(M) / scale. Integer stage: "\n        "the same relative error inside a radius of the rounded continuous "\n        "coefficients, then K0 by an integer centre search."\n    ),\n    "rounding": (\n        "K2 and K1 centres use Python round (half to even) of co[2] * 2^f2 and "\n        "-co[1] * 2^f1. Horner shifts use rd: for a non-negative value, add the "\n        "bit immediately below the kept field. K0 is the integer argmin."\n    ),\n    "widths": {"f2": 8, "f1": 19, "fyr": 27, "gy": 1, "drop2": [5]},\n    "search": {\n        "first_c_radius": 1,\n        "first_b_radius": 3,\n        "refine_if_ratio_above": 0.95,\n        "refine_c_radius": 4,\n        "refine_b_radius": 7,\n    },\n    "optimiser_start": (\n        "Rounded Lawson coefficient of the mathematical target. An imported "\n        "coefficient table is not read and is not the start."\n    ),\n    "determinism": (\n        "No random seed. The grid, iteration count and radii are fixed. A "\n        "second in-process fit must return the same integers. Numeric drift "\n        "fails closed: generation raises unless the row SHA-256 equals the "\n        "pinned digest. This is not a saved checkpoint."\n    ),\n    "row_sha256": EXPECTED_ROW_SHA256,\n    "row_encoding": (\n        "JSON array of 256 [K2, K1, K0] triples, row-major (e, a), "\n        "separators (\',\', \':\'), UTF-8, no spaces"\n    ),\n    "excluded_inputs": [\n        "Kaggle coefficient dataset version 21136269",\n        "toolbox fusion archives",\n        "a selected invsqrt circuit binary",\n    ],\n}\n\n\ndef coefficient_rows(tables: dict) -> list[list[int]]:\n    """Row-major (e, a) order used by the 256-entry QROM."""\n    return [[int(v) for v in tables[(e, a)]] for e in range(2) for a in range(128)]\n\n\ndef row_sha256(rows: list[list[int]]) -> str:\n    blob = json.dumps(rows, separators=(",", ":")).encode()\n    return hashlib.sha256(blob).hexdigest()\n\n\n@lru_cache(maxsize=1)\ndef derived_tables():\n    """Fit the selected configuration once per process and check the digest."""\n    cfg = sd.selected_cfg()\n    tables, report = sd.regenerate_tables_exact(cfg)\n    rows = coefficient_rows(tables)\n    digest = row_sha256(rows)\n    if digest != EXPECTED_ROW_SHA256:\n        raise RuntimeError(\n            f"Derived coefficient digest {digest} != {EXPECTED_ROW_SHA256}. "\n            "The numeric fit moved; refuse to build a different circuit."\n        )\n    worst = float(report["worst_ratio"]) / 2**24\n    return cfg, tables, worst, report\n\n\ndef main(argv: list[str] | None = None) -> int:\n    del argv\n    _cfg, _tables, worst, report = derived_tables()\n    payload = {\n        "provenance": PROVENANCE,\n        "source": "tasks/invsqrt/fit_coefficients.py",\n        "method": "Lawson weighted minimax plus exact-domain integer search",\n        "recipe": RECIPE,\n        "row_sha256": EXPECTED_ROW_SHA256,\n        "normalized_worst_relative_error": worst,\n        "worst_ratio": report["worst_ratio"],\n        "worst_key": report["worst_key"],\n        "reachable_normalized_inputs": report["reachable_normalized_inputs"],\n        "rows": 256,\n    }\n    print(json.dumps(payload, indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'tasks/invsqrt/generate.py': '"""Kickmix generator for the invsqrt task.\n\nThis generator is a port. It is not independent clean-room authorship.\nThe coefficient calibration is dataset-adapted; its licence is unresolved.\nThe module map is tasks/invsqrt/PROVENANCE.md.\n\nParents (Apache-2.0), reimplemented in this package:\n- anazemcev, Quantum Circuits | Signed-Digit Memory v3\n  https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory\n  snapshot SHA-256 79f6373302cb916a88f65a1b142024d4d2d0b14e41eb58bef3152ebd45ffd0ca\n- Kozykorpesh, 90-qubit Gray curvature schedule, V3 run 361677445\n  https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445\n- Kozykorpesh, complete-domain affine compilation v2, and anazemcev source &\n  exact compilation v4, for the Toffoli/CSWAP rewrite\n\nAlex Chilton v8 and Mizero Lucky Gall v1 have no notebook-wide licence.\nThey are reference-only and are not loaded.\n\nCoefficients are regenerated by tasks/invsqrt/fit_coefficients.py;\ndataset-adapted calibration rights remain unresolved (D-PRA220-01).\nThe same commit writes one invsqrt.kmb.\n\n    python tasks/invsqrt/generate.py --out DIR\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport io\nimport sys\nfrom functools import lru_cache\nfrom pathlib import Path\n\nif __package__ in (None, ""):\n    sys.path.insert(0, str(Path(__file__).resolve().parents[2]))\n\nimport kickmix as km  # noqa: E402\n\nfrom tasks.invsqrt import affine, calibrate, fit_coefficients, schedule90, signed_digit as sd  # noqa: E402\n\nF2 = 8\nF1 = 19\nFYR = 27\nGY = 1\nDROP2 = (5,)\nQROM_ENTRIES = 256\nWIDTH = 22\nMAX_X = 3_133_452\nBETA = 1\nPARENT_URL = "https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory"\nPARENT_LICENCE = "Apache-2.0"\nPARENT_SNAPSHOT_SHA256 = "79f6373302cb916a88f65a1b142024d4d2d0b14e41eb58bef3152ebd45ffd0ca"\n# Affine method parents. Notebook code only; no notebook binary is loaded.\nAFFINE_URL = "https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation-v2"\nAFFINE_LICENCE = "Apache-2.0"\nSCHEDULE_URL = "https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445"\n# Deterministic bytes of the ported 90-qubit circuit; coefficient rights unresolved.\nCIRCUIT_AUTHORSHIP = "port"\nSELECTED_KMB_SHA256 = "75e8a0e2341bb3e8f8ded1ec1d53e24437410c69e887da9daf7d49bfcc28c2dd"\n\nassert (sd.F2, sd.F1, sd.FYR, sd.GY, sd.DROP2) == (F2, F1, FYR, GY, DROP2)\n\n\ndef _resources(circuit) -> dict:\n    stats = circuit.operation_statistics()\n    static = int(stats.num_toffoli_like)\n    discount = sd.conditional_toffoli_discount(circuit)\n    return {\n        "qubits": int(circuit.num_qubits),\n        "static_toffoli": static,\n        "expected_toffoli": static - discount,\n        "conditional_discount": discount,\n        "t_like": int(stats.num_t_like),\n        "rotations": int(stats.num_z_pow(min_clifford_hierarchy_level=4)),\n    }\n\n\n@lru_cache(maxsize=1)\ndef compilation_report() -> dict:\n    """Fit, schedule, affine-compile, calibrate, and peephole. Cached per process."""\n    cfg, tables, worst = sd.load_or_build_tables()\n    raw, schedule_report = schedule90.build_raw(cfg, tables)\n    compiled = affine.compile_circuit(raw)\n    phase_report = calibrate.assert_phase_matches(compiled, cfg, tables)\n    compiled_circuit = km.Circuit(compiled.text)\n    before = _resources(compiled_circuit)\n    calibrated = calibrate.calibrate(\n        compiled,\n        {\n            "qubits": before["qubits"],\n            "expected_toffoli": before["expected_toffoli"],\n            "t_like": before["t_like"],\n        },\n    )\n    circuit = km.Circuit(calibrated["text"])\n    final = _resources(circuit)\n    estimate = calibrate._estimate(\n        final["qubits"],\n        final["expected_toffoli"],\n        final["t_like"],\n        final["rotations"],\n        calibrated["margin_units"],\n    )\n    if final["rotations"] != calibrated["rotations"]:\n        raise RuntimeError(\n            f"Rotation count {final[\'rotations\']} != calibrated {calibrated[\'rotations\']}"\n        )\n    buffer = io.BytesIO()\n    circuit.to_file(buffer, format="kmb")\n    data = buffer.getvalue()\n    return {\n        "circuit": circuit,\n        "kmb": data,\n        "sha256": hashlib.sha256(data).hexdigest(),\n        "bytes": len(data),\n        "coefficient_provenance": fit_coefficients.PROVENANCE,\n        "circuit_authorship": CIRCUIT_AUTHORSHIP,\n        "table_worst_relative_error": worst,\n        "schedule": schedule_report,\n        "affine_peak_qubits": compiled.peak,\n        "affine_rewrites": compiled.rewrites,\n        "affine_false_candidates": None,\n        "phase": phase_report,\n        "before_calibration": before,\n        "calibration": {\n            "margin_units": calibrated["margin_units"],\n            "worst_x": calibrated["worst_x"],\n            "dropped_rotation_indexes": calibrated["dropped_rotation_indexes"],\n            "peephole_cancelled": calibrated["peephole_cancelled"],\n            "trials": calibrated["trials"],\n        },\n        "final": final,\n        "estimate": estimate,\n    }\n\n\ndef build_invsqrt_circuit():\n    """Build the selected kickmix circuit. Deterministic."""\n    return compilation_report()["circuit"]\n\n\ndef circuit_kmb_bytes() -> bytes:\n    """Canonical .kmb bytes of the selected circuit."""\n    return compilation_report()["kmb"]\n\n\ndef write_invsqrt(out_dir: Path) -> Path:\n    """Write `invsqrt.kmb` into `out_dir` and return its path."""\n    out_dir.mkdir(parents=True, exist_ok=True)\n    target = out_dir / "invsqrt.kmb"\n    target.write_bytes(circuit_kmb_bytes())\n    return target\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--out", type=Path, required=True, help="Directory for invsqrt.kmb.")\n    args = parser.parse_args(argv)\n    target = write_invsqrt(args.out)\n    report = compilation_report()\n    print(f"{target} sha256={report[\'sha256\']} bytes={report[\'bytes\']}")\n    print(\n        f"Q {report[\'final\'][\'qubits\']} expected_toffoli {report[\'final\'][\'expected_toffoli\']} "\n        f"rotations {report[\'final\'][\'rotations\']} ratio {report[\'estimate\'][\'ratio\']:.6f} "\n        f"margin {report[\'calibration\'][\'margin_units\']}"\n    )\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'tasks/invsqrt/peephole.py': '"""Cancel adjacent commuting involutions.\n\nThis file is a port of the method below. It is not independent authorship.\n\nMethod parent (Apache-2.0): the peephole used after rotation calibration in\nKozykorpesh, Quantum AI complete-domain affine compilation v2. Diagonal gates\nand classical conditions are barriers. A match must be the same text within\nthe previous 256 live lines and must not cross a read/write conflict.\n"""\n\nfrom __future__ import annotations\n\nimport collections\nimport re\n\n_QUBIT = re.compile(r"(?<!\\w)q(\\d+)")\n_PHASE = {"Z", "CZ", "CCZ", "Z_POW"}\n_GATES = {"X", "CX", "CCX", "SWAP", "CSWAP"}\n\n\ndef _touch(line: str):\n    """Return (read, write) sets, or None when the line is a barrier."""\n    parts = line.split()\n    if not parts or "if" in parts:\n        return None\n    op = parts[0]\n    if op not in _PHASE | _GATES:\n        return None\n    qubits = {int(match) for match in _QUBIT.findall(line)}\n    if op in _PHASE:\n        return qubits, set()\n    if op == "X":\n        return set(), qubits\n    if op in {"CX", "CCX"}:\n        ordered = [int(match) for match in _QUBIT.findall(line)]\n        return set(ordered[:-1]), {ordered[-1]}\n    if op == "SWAP":\n        return qubits, qubits\n    ordered = [int(match) for match in _QUBIT.findall(line)]\n    return qubits, set(ordered[1:])\n\n\ndef optimize(text: str) -> tuple[str, dict]:\n    """Return peephole text and a count of cancelled gate pairs."""\n    kept: list[str | None] = []\n    counts: collections.Counter = collections.Counter()\n    for line in text.splitlines():\n        if not line:\n            continue\n        described = _touch(line)\n        op = line.split()[0]\n        cancel = False\n        if described is not None and op in _GATES:\n            reads, writes = described\n            for index in range(len(kept) - 1, max(-1, len(kept) - 257), -1):\n                if kept[index] is None:\n                    continue\n                previous = kept[index]\n                if previous == line:\n                    kept[index] = None\n                    counts[op] += 2\n                    cancel = True\n                    break\n                earlier = _touch(previous)\n                if earlier is None:\n                    break\n                earlier_reads, earlier_writes = earlier\n                if writes & earlier_reads or earlier_writes & reads:\n                    break\n        if not cancel:\n            kept.append(line)\n    body = "\\n".join(line for line in kept if line is not None)\n    return body + "\\n", dict(counts)\n', 'tasks/invsqrt/schedule90.py': '"""90-qubit invsqrt schedule: deferred feed-forward and a Gray curvature checkpoint.\n\nMethod parent (Apache-2.0): Kozykorpesh, Quantum AI LB score, V3 run 361677445,\nhttps://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445\nThe positive-Davio reload orders below are the method parameters published in\nanazemcev, Quantum Circuits | Source & Exact Compilation v4 (Apache-2.0).\nThey are not a coefficient dataset. Arithmetic helpers come from our\nsigned-digit generator. The pruned shift is the Dmitrii Gluzdov V6 boundary\nrule as that schedule names it (run 361592338). His notebook is not attached\nand is not loaded. Direct reuse waits on PRA-205. This file is a port.\n\nCoefficients are supplied by the caller. The selected generator regenerates them\nwith dataset-adapted calibration; licence unresolved (D-PRA220-01).\n"""\n\nfrom __future__ import annotations\n\nfrom fractions import Fraction\nfrom functools import lru_cache\n\nimport numpy as np\nimport kickmix as km\n\nfrom tasks.invsqrt import signed_digit as sd\n\n# Published Davio orders for the two curvature bits this schedule reloads\n# (checkpoint at bit 5, curve_count 2). `ands` is the node count we recompute.\nCURVATURE_CHOICES = {\n    "curve5": {"ands": 8, "order": (0, 7, 6, 5, 1, 2, 3, 4), "polarity": 61},\n    "curve6": {"ands": 9, "order": (5, 4, 7, 2, 1, 0, 3, 6), "polarity": 249},\n}\nSELECTED_OPTIONS = {\n    "delay_high": True,\n    "gray_checkpoint": True,\n    "curve_count": 2,\n    "late_coarse": 0,\n}\n\n\ndef mobius_transform(table: list[int]) -> list[int]:\n    """Truth table to algebraic normal form over GF(2), packed in integers."""\n    result = [int(value) for value in table]\n    if not result or len(result) & (len(result) - 1):\n        raise ValueError("truth-table length must be a power of two")\n    width = (len(result) - 1).bit_length()\n    for bit in range(width):\n        step = 1 << bit\n        for start in range(0, len(result), 2 * step):\n            for offset in range(step):\n                result[start + step + offset] ^= result[start + offset]\n    return result\n\n\ndef optimized_shift(builder, flags, word, length, inverse, prune=True):\n    """Shift `word` by the normalization flags. Empty high targets use AND, not CSWAP."""\n    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]\n    occupied = []\n    width = length\n    for stage, amount in stages:\n        occupied.append(min(width, length + 2) if prune and stage == 4 else width)\n        width = min(width + amount, len(word))\n    sequence = list(enumerate(stages))\n    if inverse:\n        sequence = sequence[::-1]\n    for index, (stage, amount) in sequence:\n        indices = range(occupied[index]) if inverse else reversed(range(occupied[index]))\n        boundary = length + sum(step for _, step in stages[:index])\n        for i in indices:\n            if i + amount >= len(word):\n                continue\n            if i + amount >= boundary:\n                if inverse:\n                    builder.cx(word[i + amount], word[i])\n                    builder.del_and(flags[stage], word[i], word[i + amount])\n                else:\n                    builder.init_and(flags[stage], word[i], word[i + amount])\n                    builder.cx(word[i + amount], word[i])\n            else:\n                builder.cswap(flags[stage], word[i], word[i + amount])\n\n\ndef _xor_many_toffoli(builder, left, right, targets):\n    targets = list(targets)\n    if not targets:\n        return\n    first = targets[0]\n    for target in targets[1:]:\n        builder.cx(first, target)\n    builder.ccx(left, right, first)\n    for target in reversed(targets[1:]):\n        builder.cx(first, target)\n\n\ndef init_anf(builder, table, address, target, borrowed=()):\n    """XOR the table into `target`. Leaf products land on the output wires."""\n    address, target = list(address), list(target)\n    width = len(address)\n    coefficient = mobius_transform(table)\n    loan = list(borrowed)[: max(0, width - 2)]\n    extra = list(builder.alloc_qubits(max(0, width - 2) - len(loan)))\n    scratch = loan + extra\n\n    def emit(mask, control):\n        outputs = [target[j] for j in range(len(target)) if (coefficient[mask] >> j) & 1]\n        if not outputs:\n            return\n        if control is None:\n            builder.x(km.array(outputs))\n        else:\n            builder.cx(control, km.array(outputs))\n\n    def walk(mask, last, control):\n        emit(mask, control)\n        for bit in range(last + 1, width):\n            child = mask | (1 << bit)\n            if not mask:\n                walk(child, bit, address[bit])\n            elif bit == width - 1:\n                outputs = [target[j] for j in range(len(target)) if (coefficient[child] >> j) & 1]\n                _xor_many_toffoli(builder, control, address[bit], outputs)\n            else:\n                qubit = scratch[mask.bit_count() - 1]\n                builder.init_and(control, address[bit], qubit)\n                walk(child, bit, qubit)\n                builder.del_and(control, address[bit], qubit)\n\n    walk(0, -1, None)\n    if extra:\n        builder.free(km.array(extra))\n\n\ndef phase_from_records(builder, table, address, records):\n    """Apply the deferred lookup phase once, after the address is free to borrow."""\n    address, records = list(address), list(records)\n    width = len(address)\n    half = width // 2\n    coefficient = mobius_transform(table)\n    left = {1 << j: address[j] for j in range(half)}\n    masks = [mask for mask in range(1, 1 << half) if mask.bit_count() > 1]\n    scratch = list(builder.alloc_qubits(len(masks) + width - half - 1))\n    for mask, qubit in zip(masks, scratch):\n        bit = mask.bit_length() - 1\n        builder.init_and(left[mask ^ (1 << bit)], address[bit], qubit)\n        left[mask] = qubit\n    carry = scratch[len(masks):]\n    parity = builder.alloc_bits(1)[0]\n\n    def emit(high, high_qubit):\n        for low in range(1 << half):\n            value = coefficient[low | (high << half)]\n            if not value:\n                continue\n            builder.bit_store(parity, False)\n            while value:\n                bit = (value & -value).bit_length() - 1\n                builder.cx(records[bit], parity)\n                value &= value - 1\n            if not high and not low:\n                builder.neg_if(parity)\n                continue\n            builder.push_condition(parity)\n            if not low:\n                builder.z(high_qubit)\n            elif not high:\n                builder.z(left[low])\n            else:\n                builder.cz(left[low], high_qubit)\n            builder.pop_condition()\n\n    def walk(mask, last, qubit):\n        emit(mask, qubit)\n        for index in range(last + 1, width - half):\n            child = mask | (1 << index)\n            if not mask:\n                walk(child, index, address[half + index])\n            else:\n                temp = carry[mask.bit_count() - 1]\n                builder.init_and(qubit, address[half + index], temp)\n                walk(child, index, temp)\n                builder.del_and(qubit, address[half + index], temp)\n\n    walk(0, -1, None)\n    for mask in reversed(masks):\n        bit = mask.bit_length() - 1\n        builder.del_and(left[mask ^ (1 << bit)], address[bit], left[mask])\n    builder.free(km.array(scratch))\n    builder.free(parity)\n\n\n@lru_cache(maxsize=None)\ndef _plan_cache(values: tuple, order: tuple, polarity: int):\n    return boolean_plan(values, order, polarity)\n\n\ndef boolean_plan(values, order=(7, 4, 2, 3, 0, 1, 6, 5), polarity=104):\n    """Positive Davio XOR/AND graph. Algebra only; no sampling."""\n    values = list(values)\n    width = len(order)\n    indices = np.arange(1 << width)\n    original = sum((((indices >> j) & 1) << order[j]) for j in range(width)) ^ polarity\n    packed = np.packbits(np.asarray(values, dtype=np.uint8)[original], bitorder="little")\n    truth = int.from_bytes(packed.tobytes(), "little")\n    nodes = []\n    node_ids = {}\n\n    def xor(left, right):\n        return (left[0] ^ right[0], left[1] ^ right[1])\n\n    @lru_cache(maxsize=None)\n    def rec(node, level):\n        ones = (1 << (1 << level)) - 1\n        if node == 0:\n            return (0, frozenset())\n        if node == ones:\n            return (1, frozenset())\n        half = (1 << (1 << (level - 1))) - 1\n        low = node & half\n        high = node >> (1 << (level - 1))\n        left = rec(low, level - 1)\n        right = rec(low ^ high, level - 1)\n        if not right[1]:\n            return xor(left, (0, frozenset((level - 1,)))) if right[0] else left\n        key = (level - 1, right)\n        if key not in node_ids:\n            node_ids[key] = width + len(nodes)\n            nodes.append(key)\n        return xor(left, (0, frozenset((node_ids[key],))))\n\n    result = rec(truth, width)\n    return {"order": tuple(order), "polarity": polarity, "nodes": list(nodes), "result": result}\n\n\ndef restore_boolean(builder, plan, address, target):\n    """Write one addressed bit with the Davio graph, then uncompute the nodes."""\n    address = list(address)\n    width = len(address)\n    flipped = [address[j] for j in range(width) if (plan["polarity"] >> j) & 1]\n    if flipped:\n        builder.x(km.array(flipped))\n    wires = [address[j] for j in plan["order"]]\n    nodes = list(builder.alloc_qubits(len(plan["nodes"])))\n    parity = builder.alloc_qubits(1)[0]\n    wires.extend(nodes)\n\n    def form(expression):\n        if expression[0]:\n            builder.x(parity)\n        for index in sorted(expression[1]):\n            builder.cx(wires[index], parity)\n\n    for index, (variable, right) in enumerate(plan["nodes"]):\n        form(right)\n        builder.init_and(wires[variable], parity, nodes[index])\n        form(right)\n    constant, terms = plan["result"]\n    if constant:\n        builder.x(target)\n    for term in sorted(terms):\n        builder.cx(wires[term], target)\n    for index in reversed(range(len(nodes))):\n        variable, right = plan["nodes"][index]\n        form(right)\n        builder.del_and(wires[variable], parity, nodes[index])\n        form(right)\n    if nodes:\n        builder.free(km.array(nodes))\n    builder.free(parity)\n    if flipped:\n        builder.x(km.array(flipped))\n\n\ndef _choice(name, values):\n    choice = CURVATURE_CHOICES[name]\n    plan = _plan_cache(tuple(int(v) for v in values), tuple(choice["order"]), int(choice["polarity"]))\n    if len(plan["nodes"]) != choice["ands"]:\n        raise RuntimeError(f"{name} Davio node count {len(plan[\'nodes\'])} != {choice[\'ands\']}")\n    return plan\n\n\ndef build_raw(cfg, tables):\n    """Build the selected 90-qubit schedule from `cfg` and fitted `tables`."""\n    if (cfg.f2, cfg.f1, cfg.fyr, cfg.gy, sorted(cfg.drop2)) != (8, 19, 27, 1, [5]):\n        raise ValueError("This schedule is for Cfg(8, 19, 27, 1, drop 5)")\n    options = SELECTED_OPTIONS\n    place = sd.layout(cfg, tables)\n    if (place["w1"], place["LB"], place["S"], place["whi"]) != (18, 21, 20, 8):\n        raise RuntimeError(f"Unexpected layout {place[\'w1\'], place[\'LB\'], place[\'S\'], place[\'whi\']}")\n    split, bounds = sd.prepare_derivative_split(sd, cfg, tables)\n    builder = km.CircuitBuilder()\n    register = builder.create_quantum_register(22, name="x")\n    flags = builder.alloc_qubits(5)\n    sd.normalize(builder, register, flags)\n    builder.x(register[-1])\n    late = options["late_coarse"]\n    c2, residual, coarse, low, high = [\n        builder.alloc_qubits(n) for n in (cfg.w2, 12, 7 - late, 21, 8)\n    ]\n    full_target = km.array(list(low) + list(residual) + list(c2) + list(high) + list(coarse))\n    full_table = [0] * 256\n    rest_table = [0] * 256\n    high_table = [0] * 256\n    low_table = [0] * 256\n    for (parity, address), (k2, _, _) in tables.items():\n        key = (parity, address)\n        index = address + 128 * parity\n        common = place["lo"][key] | (split["residual"][key] << 21) | (k2 << 33)\n        full_table[index] = common | (place["hi"][key] << (33 + cfg.w2)) | (split["coarse"][key] << (41 + cfg.w2))\n        rest_table[index] = split["residual"][key] | (k2 << 12) | (split["coarse"][key] << (12 + cfg.w2))\n        high_table[index] = place["hi"][key]\n        low_table[index] = place["lo"][key]\n    address = km.array(list(register[14:21]) + [flags[0]])\n    delta = register[:14]\n    extra_borrow = []\n\n    def erase(reg, table):\n        bits = builder.alloc_bits(len(reg))\n        builder.hmr(reg, bits)\n        return list(bits), list(table)\n\n    def phase(reg, low_bit):\n        extension = builder.alloc_qubits(9)\n        word = km.array(list(reg) + list(extension) + [register[-1]])\n        optimized_shift(builder, flags, word, len(reg), False, prune=True)\n        for index, qubit in enumerate(word):\n            exponent = low_bit + index - cfg.fy - 10\n            if exponent < 0:\n                builder.z_pow(qubit, Fraction(2) ** (exponent + 1))\n        optimized_shift(builder, flags, word, len(reg), True, prune=True)\n        builder.free(extension)\n\n    def adder(source, target, cin=None, sub=False):\n        sd.folded_gadd(builder, source, target, cin, sub, borrowed=(register[-1], *extra_borrow))\n\n    def inner(inverse):\n        rows = cfg.rrow_list()\n        for index in reversed(rows) if inverse else rows:\n            shift = cfg.sh2[index]\n            if shift >= 0:\n                addend, target, carry = c2[shift:], residual, (c2[shift - 1] if shift else None)\n            else:\n                addend, target, carry = c2, residual[-shift:], None\n            builder.cx(delta[index], target)\n            adder(addend, target, carry, sub=inverse)\n            builder.cx(delta[index], target)\n\n    def outer(derivative, inverse):\n        for index in range(14) if inverse else reversed(range(14)):\n            shift = cfg.sh1[index]\n            carry = derivative[shift - 1] if shift else None\n            builder.cx(delta[index], low)\n            adder(derivative[shift:], low, carry, sub=inverse)\n            builder.cx(delta[index], low)\n\n    init_anf(builder, full_table, address, full_target, borrowed=(register[-1],))\n    if not all(value >> 7 == 1 for value in high_table):\n        raise RuntimeError("Constant-high carry borrowing proof failed")\n    builder.x(high[-1])\n    extra_borrow.append(high[-1])\n    inner(False)\n    extra_borrow.clear()\n    builder.x(high[-1])\n    if cfg.w2 != 7:\n        raise RuntimeError("Curvature erasure is for a 7-bit coefficient")\n    checkpoint_start = 7 - options["curve_count"]\n    for index in range(checkpoint_start, 6):\n        builder.cx(c2[index + 1], c2[index])\n    erased = km.array(list(c2[checkpoint_start:]))\n    checkpoint_table = []\n    for index in range(256):\n        coefficient = tables[(index // 128, index % 128)][0]\n        coefficient ^= coefficient >> 1\n        checkpoint_table.append(coefficient >> checkpoint_start)\n    checkpoint_bits, _checkpoint_values = erase(erased, checkpoint_table)\n    builder.free(erased)\n    phase(high, 20)\n    high_bits, _high_values = erase(high, high_table)\n    builder.free(high)\n    adder(residual[11:], coarse)\n    derivative = residual[:11] + coarse\n    outer(derivative, False)\n    phase(low[cfg.gy:], cfg.gy)\n    outer(derivative, True)\n    low_bits, _low_values = erase(low, low_table)\n    builder.free(low)\n    replacement = builder.alloc_qubits(7 - checkpoint_start)\n    c2 = km.array(list(c2[:checkpoint_start]) + list(replacement))\n    curvature = np.array([tables[(index // 128, index % 128)][0] for index in range(256)], dtype=np.int64)\n    gray = curvature ^ (curvature >> 1)\n    reloads = 0\n    for index in reversed(range(checkpoint_start, 7)):\n        plan = _choice(f"curve{index}", (gray >> index) & 1)\n        restore_boolean(builder, plan, address, c2[index])\n        reloads += len(plan["nodes"])\n        if index < 6:\n            builder.cx(c2[index + 1], c2[index])\n    adder(residual[11:], coarse, sub=True)\n    inner(True)\n    rest_target = km.array(list(residual) + list(c2) + list(coarse))\n    rest_bits, _rest_values = erase(rest_target, rest_table)\n    builder.free(rest_target)\n    for offset, bit in enumerate(checkpoint_bits):\n        index = checkpoint_start + offset\n        builder.cx(bit, rest_bits[12 + index])\n        if index < 6:\n            builder.cx(bit, rest_bits[12 + index + 1])\n    records = low_bits + rest_bits[: 12 + cfg.w2] + high_bits + rest_bits[12 + cfg.w2 :]\n    if len(records) != len(full_target) + late:\n        raise RuntimeError(f"Deferred record width {len(records)} != {len(full_target) + late}")\n    phase_from_records(builder, full_table, address, records)\n    builder.x(register[-1])\n    sd.unnormalize(builder, register, flags)\n    builder.free(flags)\n    circuit = builder.finish_circuit()\n    report = {\n        "variant": "deferred_feedforward_gray_checkpoint",\n        "options": options,\n        "boolean_reload_toffoli": reloads,\n        "qubits": circuit.num_qubits,\n        "static_toffoli": int(circuit.operation_statistics().num_toffoli_like),\n        "split_bounds": bounds,\n        "provenance_method": "Apache-2.0 Kozykorpesh V3 run 361677445, port",\n    }\n    return circuit, report\n', 'tasks/invsqrt/signed_digit.py': '"""Signed-digit invsqrt circuit body (Apache-2.0 derivative).\n\nParent notebook, v3, source snapshot SHA-256\n79f6373302cb916a88f65a1b142024d4d2d0b14e41eb58bef3152ebd45ffd0ca:\nQuantum Circuits | Signed-Digit Memory, Artem1410 (anazemcev),\nhttps://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory\nParent licence: Apache-2.0. Dataset-adapted calibration rights are unresolved\n(D-PRA220-01); the retained upstream SPDX notices do not settle that scope.\n\nThe selected schedule is quadratic signed-digit Horner evaluation, a 256-entry\nQROM, no Newton iteration, a derivative split, one borrowed normalized carry,\nand the promised-input rewrite. Parameters are the constants under this\ndocstring. `tasks/invsqrt/generate.py` is the entry point. It writes only\n`invsqrt.kmb`. Upstream notices in the comments below are retained.\n"""\n\n# SPDX-License-Identifier: Apache-2.0\n# New derivative-split integration; original upstream notices retained below.\n# Sumit Chongder: centered signed-digit generator and integer pipeline,361357149.\n# Rudra Bhaskar: derivative split, top-carry reuse, static-high checkpoint,361445283.\n# https://www.kaggle.com/code/rudrabhaskar/highest-public-score-direct-polynomial-phasing?scriptVersionId=361445283\n# Alex Chilton: shared prefix and empty-target temporary-AND shifts,361472684.\n# Kirill Tymchishin: exact Rudra/Alex integration used as the95Q source baseline,361491814.\n# https://www.kaggle.com/code/kirilltym/quantum-circuits-qrom-tower42?scriptVersionId=361491814\n# Kozykorpesh: explicit 8,19,27,1 coefficient widths. The integers are regenerated\n# with dataset-adapted calibration; licence unresolved (D-PRA220-01).\n# Our changes: generic exact coefficient widths/left shifts, normalized-input\n# borrowing on this derivative schedule, and the proven original-input rewrite.\n# The derivative split and its precision calibrations are credited upstream.\n\n# SPDX-License-Identifier: Apache-2.0\n# Arithmetic research candidate. Retained upstream attribution:\n# Sumit Chongder: readable centered signed-digit generator and integer pipeline.\n# https://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149\n# anazemcev / Artem1410: early high-word phase, folded carry and normalized-input borrowing.\n# https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory?scriptVersionId=361422248\n# Kozykorpesh: exact reachable-domain calibration, Cfg(8,19,27,1).\n# https://www.kaggle.com/code/kozykappa/quantum-ai-compact-arithmetic-banked-qrom?scriptVersionId=361456406\n# Coefficient integers are regenerated by tasks/invsqrt/fit_coefficients.py.\n# Provenance: regenerated; dataset-adapted calibration; licence unresolved.\n# Alex Chilton: shared prefix flags and temporary-AND moves into zero destinations.\n# https://www.kaggle.com/code/alexchilton/where-the-toffolis-go-anatomy-a-cheaper-invsqrt?scriptVersionId=361472684\n# Alex v8 has no notebook-wide licence. It is reference-only and is not loaded.\n# The Apache-2.0 parent of this file is anazemcev Signed-Digit Memory v3, which\n# retains the Alex notice. Sumit run 361357149 is v1 (PRA-218).\n# This revision combines credited methods. No upstream gain is claimed as original.\n# Added own exact rewrite: the original input bound proves x[21]&x[20]==0.\n# Linearize the first conjunction in normalization and the final input phase fixup.\n\n# Attribution: Sumit Chongder, Apache-2.0 as published.\n# https://www.kaggle.com/code/sumitchongder/notebook2f196e646b\n# Sumit run 361357149 is v1 (public version metadata, PRA-218). Kirill\'s v9 cites v1.\n# The Sumit sentences above are the parent notebook\'s notice. This tree does\n# not load that notebook. Whether a later v2 exists stays with PRA-205.\n# This isolated lab regenerates the circuit from readable source; no binaries or\n# remote notebook outputs are inputs. Our added exact integer certificate and\n# reporting helpers and memory-schedule adaptations follow the original generator.\n"""Builds an optimized kickmix circuit for the invsqrt benchmark (Benchmark I).\n\n|x>|0> -> exp(2 pi i / sqrt(x)) |x>|0>,   x in [1, 3133452], 22-bit register \'x\'.\n\nDesign (all arithmetic exactly mirrored by the numpy model below):\n  1. Normalize x by single-bit binary stages (16, 8, 4, 2, 1) into M = x_norm / 2^21 in [1, 2);\n     flags f0..f4 hold the shift count s\' in binary.  e = f0 (parity), s = s\' >> 1.\n     1/sqrt(x) = 2^(s-10) * y,  y = 2^((e-1)/2) / sqrt(M)  in (0.5, 1].\n  2. One 256-entry QROM indexed by (7 mantissa bits, e) XORs quadratic coefficients directly into\n     the working registers (no separate coefficient copies).\n  3. Centered signed-digit Horner: delta_c = sum_i (2 d_i - 1) 2^(i-22) over the 14 low bits.\n     R = K1 - C2*delta_c,  y = K0 - R*delta_c.  Each row is an UNcontrolled Gidney addition\n     wrapped in CNOT flips (Y - A = ~(~Y + A)), with a rounding carry-in.\n  4. The y register is split: rows only touch a 21-bit low register L; the constant high part\n     sits in a separate register and the overflow bit is merged in before phasing.\n  5. Binary de-normalization (shift by s), one Z_POW per bit, then exact mirror uncompute.\n     Lookup uncompute is measurement based (del_lookup); normalization flags are cleared by HMR\n     with classically conditioned phase fixups.\nUsage: python signed_digit_lab.py OUT.kmb [--baseline] (requires kickmix and NumPy)\n"""\nimport sys, math, functools, types\nfrom fractions import Fraction\nimport numpy as np\nimport kickmix as km\n\nW = 22; MB = 21; DB = 14; NSEG = 128\nF2, F1, FYR, GY, DROP2 = 8, 19, 27, 1, (5,)     # precision configuration (verified)\n\n# ----------------------------------------------------------------- numpy model / tables\n@functools.lru_cache(None)\ndef fitc(lo, hi, mid, scale, deg=2, iters=80):\n    f = lambda M: scale / np.sqrt(M)\n    d = np.linspace(lo - mid, hi - mid, 3001)\n    A = np.vander(d, deg + 1, increasing=True)\n    w = np.sqrt(mid + d) / scale\n    wt = np.ones_like(d)\n    for _ in range(iters):                       # Lawson iterations -> weighted minimax\n        sw = w * np.sqrt(wt)\n        coef = np.linalg.lstsq(A * sw[:, None], f(mid + d) * sw, rcond=None)[0]\n        err = np.abs((A @ coef - f(mid + d)) * w)\n        wt = wt * err; wt /= wt.sum()\n    return coef\n\nclass Cfg:\n    def __init__(self, f2, f1, fyr, gy, drop2=()):\n        self.f2, self.f1, self.fyr, self.gy = f2, f1, fyr, gy\n        self.fy = fyr + gy\n        self.w2 = f2 - 1\n        self.sh2 = [f2 + 22 - i - f1 for i in range(DB)]\n        self.sh1 = [f1 + 22 - i - self.fy for i in range(DB)]\n        self.drop2 = set(drop2)\n    def rrow_list(self):\n        return [i for i in reversed(range(DB)) if i not in self.drop2 and self.sh2[i] < self.w2]\n\ndef rd(v, s):\n    if s < 0: return v << (-s)\n    r = v >> s\n    if s >= 1: r = r + ((v >> (s - 1)) & 1)\n    return r\n\ndef pipeline(cfg, K2, K1, d):\n    sig = [2 * ((d >> i) & 1) - 1 for i in range(DB)]\n    R = np.full_like(d, K1)\n    for i in cfg.rrow_list():\n        R = R - sig[i] * rd(K2, cfg.sh2[i])\n    Q = np.zeros_like(d)\n    for i in reversed(range(DB)):\n        Q = Q + sig[i] * rd(R, cfg.sh1[i])\n    return R, Q\n\nMAX_X = 3_133_452\n\ndef valid_residuals(e,a):\n    d=np.arange(1<<DB,dtype=np.int64);xn=(1<<MB)+(a<<DB)+d\n    mask=(d%2==0) if e else ((xn<=MAX_X)|(d%4==0))\n    return d[mask]\n\ndef calibrate_segment(cfg,e,a,c_radius=1,b_radius=3):\n    d=valid_residuals(e,a);lo=1+a*2.**(DB-MB);mid=lo+((1<<DB)-1)/2**(MB+1)\n    scale=2**((e-1)/2);co=fitc(lo,lo+2.**(DB-MB),mid,scale)\n    f=scale/np.sqrt(lo+d*2.**-MB);target=f*2**cfg.fy;w=1/f\n    # Start is the rounded Lawson coefficient, not an imported table.\n    K2c=round(co[2]*2**cfg.f2);K1c=round(-co[1]*2**cfg.f1);best=None;quant=1<<cfg.gy\n    for K2 in range(max(0,K2c-c_radius),min((1<<cfg.w2)-1,K2c+c_radius)+1):\n        for K1 in range(K1c-b_radius,K1c+b_radius+1):\n            R,Q=pipeline(cfg,K2,K1,d)\n            if R.min()<0:continue\n            vals=target+Q;center=.5*(vals.min()+vals.max())+.5*(quant-1)\n            k0s=np.arange(math.floor(center)-quant-1,math.ceil(center)+quant+2,dtype=np.int64)\n            # The stored high anchor and the carry must keep the leading one fixed.\n            k0s=k0s[k0s <= (1<<cfg.fy)-1+int(Q.min())]\n            if len(k0s)==0:continue\n            y=((k0s[:,None]-Q)>>cfg.gy)<<cfg.gy\n            scores=np.max(abs(y-target)*w,axis=1)/2**cfg.fy;i=int(scores.argmin())\n            v=float(scores[i]);K0=int(k0s[i])\n            if best is None or v<best[0]:best=(v,K2,K1,K0)\n    if best is None:raise RuntimeError(\'No legal coefficient triple\')\n    return best\n\ndef regenerate_tables_exact(cfg,refine_threshold=.95):\n    """Fit every segment from the mathematical target.\n\n    The integer search is centred on the rounded Lawson coefficient.\n    No coefficient file is read.\n    """\n    tabs={};ratios=[]\n    for e in(0,1):\n        for a in range(128):\n            best=calibrate_segment(cfg,e,a)\n            if best[0]*2**24>refine_threshold:\n                alt=calibrate_segment(cfg,e,a,c_radius=4,b_radius=7)\n                if alt[0]<best[0]:best=alt\n            tabs[(e,a)]=best[1:];ratios.append(best[0]*2**24)\n    return tabs,dict(worst_ratio=max(ratios),worst_key=[int(np.argmax(ratios))//128,int(np.argmax(ratios))%128],reachable_normalized_inputs=sum(len(valid_residuals(e,a)) for e in(0,1) for a in range(128)))\n\ndef build_tables(cfg, search=2):\n    """Regenerate coefficients with the exact-domain radius policy.\n\n    The radius policy is adapted from the Kozykorpesh dataset domain-tables\n    module (SHA-256 59cf4d98899433dcc49d9ce512293be20d33da5334211c20e35323555994a4fb).\n    Its licence is unresolved; the version-pinned URL is in\n    `fit_coefficients.RECIPE["calibration_lineage"]`. The selected build calls\n    this through `fit_coefficients.derived_tables`.\n    """\n    tables, report = regenerate_tables_exact(cfg)\n    return tables, report[\'worst_ratio\'] / 2**24\n\ndef layout(cfg, tabs, LB=21, S=20):\n    d = np.arange(2**DB, dtype=np.int64)\n    assert max(t[0] for t in tabs.values()) < (1 << cfg.w2)\n    Rmax = 0; per = {}; Ymax = 0\n    for key, (K2, K1, K0) in tabs.items():\n        R, Q = pipeline(cfg, K2, K1, d)\n        assert R.min() >= 0\n        Rmax = max(Rmax, int(R.max()), K1)\n        per[key] = (int(Q.min()), int(Q.max()))\n        Ymax = max(Ymax, K0 - per[key][0])\n    w1 = Rmax.bit_length()\n    lo_tab, hi_tab = {}, {}\n    for key, (K2, K1, K0) in tabs.items():\n        qmin, qmax = per[key]\n        base = K0 % (1 << S); j = 0\n        while base + j * (1 << S) - qmax < 0: j += 1\n        k0lo = base + j * (1 << S)\n        assert k0lo - qmin < (1 << LB), \'layout infeasible\'\n        lo_tab[key] = k0lo; hi_tab[key] = (K0 - k0lo) >> S\n        assert hi_tab[key] >= 0\n    whi = max(max(hi_tab.values()).bit_length(), 1)\n    return dict(w1=w1, LB=LB, S=S, whi=whi, lo=lo_tab, hi=hi_tab)\n\n# ----------------------------------------------------------------- circuit pieces\ndef and_all_into(b, controls, target):\n    k = len(controls)\n    if k == 1: b.cx(controls[0], target); return\n    if k == 2: b.ccx(controls[0], controls[1], target); return\n    anc = b.alloc_qubits(k - 2)\n    b.init_and(controls[0], controls[1], anc[0])\n    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])\n    b.ccx(anc[k - 3], controls[k - 1], target)\n    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])\n    b.del_and(controls[0], controls[1], anc[0])\n    b.free(anc)\n\ndef phase_all(b, controls):\n    k = len(controls)\n    if k == 1: b.z(controls[0]); return\n    if k == 2: b.cz(controls[0], controls[1]); return\n    anc = b.alloc_qubits(k - 2)\n    b.init_and(controls[0], controls[1], anc[0])\n    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])\n    b.cz(anc[k - 3], controls[k - 1])\n    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])\n    b.del_and(controls[0], controls[1], anc[0])\n    b.free(anc)\n\nNSTAGES = [(4, 16), (3, 8), (2, 4), (1, 2), (0, 1)]\n\ndef _baseline_normalize(b, x, f):\n    for (k, d) in NSTAGES:\n        top = x[W - d:W]\n        if d == 1:\n            b.cx(x[W - 1], f[0]); b.x(f[0])\n        else:\n            b.x(top); and_all_into(b, list(top), f[k]); b.x(top)\n        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])\n\n\ndef _combined_normalize(b, x, f):\n    """v3 step 1: f4, f3, f2 (the binary leading-zero count div 4) from one prefix-AND chain on the unshifted x.\n    chain[m] = [x[W-m-2:W] == 0] (top m+2 bits zero); p_j = chain[4j-2] = [lz >= 4j].  f4 = p4, f3 = p2 ^ p4,\n    f2 = p1 ^ p2 ^ p3 ^ p4 ^ p5 (nested prefixes, so XOR = bit 2 of lz).  19 ANDs instead of 15 + 7 + 3."""\n    bits = [x[W - 1 - m] for m in range(20)]                  # x21, x20, ..., x2\n    b.x(km.array(bits))\n    chain = b.alloc_qubits(19)\n    b.init_and(bits[0], bits[1], chain[0])\n    for m in range(1, 19): b.init_and(chain[m - 1], bits[m + 1], chain[m])\n    p = {j: chain[4 * j - 2] for j in range(1, 6)}\n    b.cx(p[4], f[4])\n    b.cx(p[2], f[3]); b.cx(p[4], f[3])\n    for j in range(1, 6): b.cx(p[j], f[2])\n    for m in reversed(range(1, 19)): b.del_and(chain[m - 1], bits[m + 1], chain[m])\n    b.del_and(bits[0], bits[1], chain[0])\n    b.free(chain)\n    b.x(km.array(bits))\n    for (k, d) in NSTAGES:\n        top = x[W - d:W]\n        if d == 1:\n            b.cx(x[W - 1], f[0]); b.x(f[0])\n        elif d == 2:\n            b.x(top); b.ccx(top[0], top[1], f[1]); b.x(top)\n        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])\n\n# MAX_X < 3*2**20 proves the original two top bits are never both one.\n# After inversion, their AND equals XOR with one, on every promised input.\ndef normalize(b, x, f):\n    """v3 step 1: f4, f3, f2 (the binary leading-zero count div 4) from one prefix-AND chain on the unshifted x.\n    chain[m] = [x[W-m-2:W] == 0] (top m+2 bits zero); p_j = chain[4j-2] = [lz >= 4j].  f4 = p4, f3 = p2 ^ p4,\n    f2 = p1 ^ p2 ^ p3 ^ p4 ^ p5 (nested prefixes, so XOR = bit 2 of lz).  19 ANDs instead of 15 + 7 + 3."""\n    bits = [x[W - 1 - m] for m in range(20)]                  # x21, x20, ..., x2\n    b.x(km.array(bits))\n    chain = b.alloc_qubits(19)\n    b.x(chain[0]); b.cx(bits[0],chain[0]); b.cx(bits[1],chain[0])\n    for m in range(1, 19): b.init_and(chain[m - 1], bits[m + 1], chain[m])\n    p = {j: chain[4 * j - 2] for j in range(1, 6)}\n    b.cx(p[4], f[4])\n    b.cx(p[2], f[3]); b.cx(p[4], f[3])\n    for j in range(1, 6): b.cx(p[j], f[2])\n    for m in reversed(range(1, 19)): b.del_and(chain[m - 1], bits[m + 1], chain[m])\n    b.cx(bits[1],chain[0]); b.cx(bits[0],chain[0]); b.x(chain[0])\n    b.free(chain)\n    b.x(km.array(bits))\n    for (k, d) in NSTAGES:\n        top = x[W - d:W]\n        if d == 1:\n            b.cx(x[W - 1], f[0]); b.x(f[0])\n        elif d == 2:\n            b.x(top); b.ccx(top[0], top[1], f[1]); b.x(top)\n        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])\n\ndef _baseline_unnormalize(b, x, f):\n    for (k, d) in reversed(NSTAGES):\n        for i in range(0, W - d): b.cswap(f[k], x[i], x[i + d])\n        top = x[W - d:W]\n        if d == 1:\n            b.x(f[0]); b.cx(x[W - 1], f[0])\n        else:\n            m = b.alloc_bits(1)[0]\n            b.hmr(f[k], m)\n            b.push_condition(m)                     # fixup runs on half the shots on average\n            b.x(top); phase_all(b, list(top)); b.x(top)\n            b.pop_condition()\n            b.free(m)\n\ndef promise_phase_original_top(b,controls):\n    """Phase the original top16-zero predicate using its promised bit relation."""\n    assert len(controls)==16\n    controls=list(reversed(controls))\n    k=len(controls)\n    anc=b.alloc_qubits(k-2)\n    b.x(anc[0]); b.cx(controls[0],anc[0]); b.cx(controls[1],anc[0])\n    for j in range(1,k-2):\n        b.init_and(anc[j-1],controls[j+1],anc[j])\n    b.cz(anc[k-3],controls[k-1])\n    for j in reversed(range(1,k-2)):\n        b.del_and(anc[j-1],controls[j+1],anc[j])\n    b.cx(controls[1],anc[0]); b.cx(controls[0],anc[0]); b.x(anc[0])\n    b.free(anc)\n\ndef unnormalize(b, x, f):\n    for (k, d) in reversed(NSTAGES):\n        for i in range(0, W - d): b.cswap(f[k], x[i], x[i + d])\n        top = x[W - d:W]\n        if d == 1:\n            b.x(f[0]); b.cx(x[W - 1], f[0])\n        else:\n            m = b.alloc_bits(1)[0]\n            b.hmr(f[k], m)\n            b.push_condition(m)                     # fixup runs on half the shots on average\n            b.x(top)\n            if d==16: promise_phase_original_top(b,list(top))\n            else: phase_all(b,list(top))\n            b.x(top)\n            b.pop_condition()\n            b.free(m)\n\ndef gadd(b, a, t, cin=None, sub=False):\n    """t += a + cin (or t -= a + cin), Gidney adder: len(t)-1 Toffolis, uncompute by HMR."""\n    a = list(a); t = list(t); o = len(a); n = len(t)\n    assert o <= n\n    if sub: b.x(km.array(t))\n    c = b.alloc_qubits(n - 1) if n > 1 else km.array([])\n    def C(i): return cin if i == 0 else c[i - 1]\n    for i in range(0, n - 1):\n        if i == 0 and cin is None:\n            if o: b.init_and(a[0], t[0], C(1))\n        elif i < o:\n            b.cx(C(i), a[i]); b.cx(C(i), t[i]); b.init_and(a[i], t[i], C(i + 1)); b.cx(C(i), C(i + 1))\n        else:\n            b.init_and(C(i), t[i], C(i + 1))\n    if n - 1 < o: b.cx(a[n - 1], t[n - 1])\n    if n > 1 or cin is not None: b.cx(C(n - 1), t[n - 1])\n    for i in reversed(range(0, n - 1)):\n        if i == 0 and cin is None:\n            if o: b.del_and(a[0], t[0], C(1)); b.cx(a[0], t[0])\n        elif i < o:\n            b.cx(C(i), C(i + 1)); b.del_and(a[i], t[i], C(i + 1)); b.cx(C(i), a[i]); b.cx(a[i], t[i])\n        else:\n            b.del_and(C(i), t[i], C(i + 1)); b.cx(C(i), t[i])\n    if n > 1: b.free(c)\n    if sub: b.x(km.array(t))\n\ndef _baseline_shift_stages(b, f, reg, L, inverse):\n    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]\n    occ = []; o = L\n    for (k, d) in stages: occ.append(o); o = min(o + d, len(reg))\n    seq = list(enumerate(stages))\n    if not inverse:\n        for idx, (k, d) in seq:\n            for i in reversed(range(0, occ[idx])):\n                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])\n    else:\n        for idx, (k, d) in reversed(seq):\n            for i in range(0, occ[idx]):\n                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])\n\n\ndef shift_stages(b, f, reg, L, inverse):\n    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]\n    occ = []; o = L\n    for (k, d) in stages: occ.append(o); o = min(o + d, len(reg))\n    seq = list(enumerate(stages))\n    if not inverse:\n        for idx, (k, d) in seq:\n            for i in reversed(range(0, occ[idx])):\n                if i + d >= len(reg): continue\n                if i + d >= occ[idx]:                       # v2: target is |0> for every f\n                    b.init_and(f[k], reg[i], reg[i + d]); b.cx(reg[i + d], reg[i])\n                else:\n                    b.cswap(f[k], reg[i], reg[i + d])\n    else:\n        for idx, (k, d) in reversed(seq):\n            for i in range(0, occ[idx]):\n                if i + d >= len(reg): continue\n                if i + d >= occ[idx]:\n                    b.cx(reg[i + d], reg[i]); b.del_and(f[k], reg[i], reg[i + d])\n                else:\n                    b.cswap(f[k], reg[i], reg[i + d])\n\ndef build_circuit(cfg, tabs, lay, adder=None):\n    # The default preserves the public generator exactly. An injected adder\n    # supports isolated, explicitly tested memory experiments.\n    if adder is None: adder=gadd\n    w2, w1, LB, S, whi = cfg.w2, lay[\'w1\'], lay[\'LB\'], lay[\'S\'], lay[\'whi\']\n    gy, fy = cfg.gy, cfg.fy\n    b = km.CircuitBuilder()\n    x = b.create_quantum_register(W, name=\'x\')\n    f = b.alloc_qubits(5)\n    c2 = b.alloc_qubits(w2); R = b.alloc_qubits(w1); Lr = b.alloc_qubits(LB); Yh = b.alloc_qubits(whi)\n    tgt = km.array(list(Lr) + list(R) + list(c2) + list(Yh))\n    table = [0] * 256\n    for (e, a), (K2, K1, K0) in tabs.items():\n        table[a + 128 * e] = lay[\'lo\'][(e, a)] | (K1 << LB) | (K2 << (LB + w1)) | (lay[\'hi\'][(e, a)] << (LB + w1 + w2))\n    addr = km.array(list(x[14:21]) + [f[0]])\n    delta = x[0:DB]\n    def rrows(inverse):\n        seq = cfg.rrow_list()\n        if inverse: seq = list(reversed(seq))\n        for i in seq:\n            sh = cfg.sh2[i]\n            if sh >= 0: off = c2[sh:w2]; t = R[0:w1]; cin = c2[sh - 1] if sh >= 1 else None\n            else: off = c2[0:w2]; t = R[-sh:w1]; cin = None\n            b.cx(delta[i], t); adder(b, off, t, cin, sub=inverse); b.cx(delta[i], t)\n    def yrows(inverse):\n        seq = list(reversed(range(DB)))\n        if inverse: seq.reverse()\n        for i in seq:\n            sh = cfg.sh1[i]; assert 0 <= sh < w1\n            t = Lr[0:LB]; cin = R[sh - 1] if sh >= 1 else None\n            b.cx(delta[i], t); adder(b, R[sh:w1], t, cin, sub=inverse); b.cx(delta[i], t)\n    normalize(b, x, f)\n    b.init_lookup(table=table, address=addr, target=tgt)\n    rrows(False); yrows(False)\n    ext = b.alloc_qubits(10)\n    if LB > S: adder(b, Lr[S:LB], Yh)\n    Cx = km.array(list(Lr[gy:S]) + list(Yh) + list(ext))\n    shift_stages(b, f, Cx, S - gy + whi, False)\n    for j in range(len(Cx)):\n        e = gy + j - fy - 10                       # bit weight 2^e turns\n        if e < 0: b.z_pow(Cx[j], Fraction(2)**(e + 1))\n    shift_stages(b, f, Cx, S - gy + whi, True)\n    if LB > S: adder(b, Lr[S:LB], Yh, sub=True)\n    b.free(ext)\n    yrows(True); rrows(True)\n    b.del_lookup(table=table, address=addr, target=tgt)\n    b.free(tgt)\n    unnormalize(b, x, f)\n    b.free(f)\n    return b.finish_circuit()\n\ndef main(out):\n    cfg = Cfg(F2, F1, FYR, GY, DROP2)\n    tabs, worst = build_tables(cfg)\n    print(f\'model worst relative error / 2^-24 = {worst * 2**24:.3f}\')\n    lay = layout(cfg, tabs)\n    c = build_circuit(cfg, tabs, lay)\n    s = c.operation_statistics()\n    print(\'qubits\', c.num_qubits, \'toffoli(static)\', s.num_toffoli_like,\n          \'rotations\', s.num_z_pow(min_clifford_hierarchy_level=4))\n    c.to_file(out, format=\'kmb\')\n    print(\'wrote\', out)\n\n\ndef selected_cfg():\n    return Cfg(F2,F1,FYR,GY,DROP2)\n\n\ndef config_record(cfg):\n    return dict(f2=cfg.f2,f1=cfg.f1,fyr=cfg.fyr,gy=cfg.gy,drop2=sorted(cfg.drop2))\n\n\n\n\ndef load_or_build_tables(path=None, regenerate=False):\n    """Derive the 256 coefficient triples. No dataset is read.\n\n    `tasks/invsqrt/fit_coefficients.py` is the source. `regenerate=True` fits\n    a second time and requires the same integers. A digest check inside the\n    fitter stops a numeric-library change from silently changing the circuit.\n    """\n    import json\n    from pathlib import Path\n    from tasks.invsqrt.fit_coefficients import PROVENANCE, coefficient_rows, derived_tables\n    cfg, tables, worst, _report = derived_tables()\n    rows = coefficient_rows(tables)\n    if regenerate:\n        again, _again_report = regenerate_tables_exact(cfg)\n        assert coefficient_rows(again) == rows, \'A second fit disagreed with the cached coefficients.\'\n    path = Path(path) if path else None\n    if path and path.exists():\n        saved = json.loads(path.read_text())\n        assert saved[\'config\'] == config_record(cfg)\n        assert saved[\'rows\'] == rows, \'Cached coefficient table differs from the derived model.\'\n        assert saved.get(\'provenance\') == PROVENANCE\n    elif path:\n        path.write_text(json.dumps({\n            \'provenance\': PROVENANCE,\n            \'source\': \'tasks/invsqrt/fit_coefficients.py\',\n            \'config\': config_record(cfg),\n            \'normalized_worst_relative_error\': worst,\n            \'rows\': rows,\n        }, indent=2))\n    return cfg, tables, worst\n\n\ndef integer_model(xs,cfg,tabs):\n    """Exact emitted arithmetic in units 2^-(cfg.fy+10) turns, before mod 1."""\n    xs=np.asarray(xs,dtype=np.int64)\n    if not ((xs>=1)&(xs<=3133452)).all():\n        raise ValueError(\'The supported competition domain is x=1..3133452\')\n    # All inputs fit exactly in float64; frexp gives the exact binary exponent.\n    shifts=21-(np.frexp(xs)[1].astype(np.int64)-1)\n    norm=xs << shifts\n    assert ((norm >> 21)==1).all(), \'Normalized top-bit borrowing requires x[21]=1\'\n    parity=shifts&1\n    index=((norm >> 14)&127)+128*parity\n    table=np.asarray([tabs[(e,a)] for e in range(2) for a in range(128)],dtype=np.int64)\n    K2,K1,K0=(table[index,j] for j in range(3))\n    d=norm&((1 << DB)-1)\n    signs=[2*((d >> i)&1)-1 for i in range(DB)]\n    R=K1.copy()\n    for i in cfg.rrow_list():\n        R-=signs[i]*rd(K2,cfg.sh2[i])\n    Q=np.zeros_like(xs)\n    for i in reversed(range(DB)):\n        Q+=signs[i]*rd(R,cfg.sh1[i])\n    Y=((K0-Q) >> cfg.gy) << cfg.gy\n    if (R<0).any() or (Y<0).any():\n        raise OverflowError(\'An unsigned intermediate is negative\')\n    return Y << (shifts >> 1)\n\n\ndef exhaustive_certificate(cfg,tabs,chunk_size=131072):\n    """Every promised basis input, exact integer target and tolerance.\n\n    This is an arithmetic certificate. Ancillas, restored inputs and phase\n    feed-forward require fresh circuit simulation and the official verifier.\n    """\n    output_fraction=cfg.fy+10\n    worst_ratio,worst_x,min_margin=0.,0,1 << 64\n    for start in range(1,3133453,chunk_size):\n        xs=np.arange(start,min(start+chunk_size,3133453),dtype=np.int64)\n        values=integer_model(xs,cfg,tabs).astype(np.uint64)\n        phase=values << np.uint64(64-output_fraction)\n        roots=[math.isqrt((1 << 130)//int(x)) for x in xs]\n        target=np.array([((r+1) >> 1)&((1 << 64)-1) for r in roots],dtype=np.uint64)\n        tolerance=np.array([min(r >> 25,1 << 63) for r in roots],dtype=np.uint64)\n        error=np.abs((phase-target).view(np.int64)).view(np.uint64)\n        ratio=error/tolerance\n        i=int(np.argmax(ratio))\n        if float(ratio[i])>worst_ratio:\n            worst_ratio,worst_x=float(ratio[i]),int(xs[i])\n        if (error>tolerance).any():\n            i=int(np.flatnonzero(error>tolerance)[0])\n            return {\'pass\':False,\'first_bad_x\':int(xs[i]),\'ratio\':float(ratio[i])}\n        min_margin=min(min_margin,int((tolerance-error).min()))\n    return {\'pass\':True,\'tested_inputs\':3133452,\'worst_ratio\':worst_ratio,\n            \'worst_x\':worst_x,\'min_margin_units\':min_margin,\n            \'phase_fraction_bits\':output_fraction}\n\n\ndef conditional_toffoli_discount(circuit):\n    """Mean discount for this emitted family\'s fair HMR conditional branches."""\n    depth=0\n    total=0\n    discount=0.\n    for line in str(circuit).splitlines():\n        opcode=line.split(\' \',1)[0]\n        if opcode==\'PUSH_CONDITION\':depth+=1\n        elif opcode==\'POP_CONDITION\':depth-=1\n        elif opcode in [\'CCX\',\'CSWAP\']:\n            total+=1\n            if depth:\n                assert depth==1,\'Expected one fair measurement bit in a Toffoli branch.\'\n                discount+=.5\n    assert total==int(circuit.operation_statistics().num_toffoli_like)\n    assert depth==0\n    return discount\n\ndef estimated_resources(circuit,certificate):\n    """Model-based cost; full unchanged official verification remains required."""\n    assert certificate[\'pass\'] and certificate[\'min_margin_units\']>0\n    stats=circuit.operation_statistics()\n    rotations=int(stats.num_z_pow(min_clifford_hierarchy_level=4))\n    epsilon=math.pi*(certificate[\'min_margin_units\']/2**64)/rotations\n    per_rotation=0.56*math.log2(1/epsilon)+5.3\n    static_toffoli=int(stats.num_toffoli_like)\n    # Fair flag measurements activate the emitted normalization phase-fixup\n    # Toffolis on half the shots. Read their actual branch counts so the\n    # original schedules discount 11 and the promise rewrite discounts 10.5.\n    normalization_discount=conditional_toffoli_discount(circuit)\n    expected_toffoli=static_toffoli-normalization_discount\n    t_like=int(stats.num_t_like)\n    extra=(t_like+rotations*per_rotation)/4\n    return {\'qubits\':circuit.num_qubits,\'static_toffoli\':static_toffoli,\n            \'expected_mean_toffoli\':expected_toffoli,\'t_like\':t_like,\n            \'rotations\':rotations,\'rotation_epsilon\':epsilon,\n            \'estimated_expected_cost\':circuit.num_qubits*(expected_toffoli+extra),\n            \'static_cost_upper_estimate\':circuit.num_qubits*(static_toffoli+extra),\n            \'full_official_verification_required\':True}\n\n\n\n\n# ---- Our isolated phase scheduling, folded carry and borrowed workspace ----\ndef folded_gadd(b,a,t,cin=None,sub=False,borrowed=()):\n    a,t=list(a),list(t)\n    o,n=len(a),len(t)\n    assert o<=n and n>=1\n    if sub:\n        for q in t:b.x(q)\n    if n==1:\n        if o:b.cx(a[0],t[0])\n        if cin is not None:b.cx(cin,t[0])\n        if sub:\n            for q in t:b.x(q)\n        return\n    borrowed=list(borrowed)[:max(0,n-2)]\n    count=max(0,n-2-len(borrowed))\n    allocated=b.alloc_qubits(count) if count else []\n    c=borrowed+list(allocated)\n    def C(i):return cin if i==0 else c[i-1]\n    for i in range(n-2):\n        if i==0 and cin is None:\n            if o:b.init_and(a[0],t[0],C(1))\n        elif i<o:\n            b.cx(C(i),a[i]);b.cx(C(i),t[i])\n            b.init_and(a[i],t[i],C(i+1));b.cx(C(i),C(i+1))\n        else:\n            b.init_and(C(i),t[i],C(i+1))\n    i=n-2\n    carry=C(i)\n    if carry is None:\n        if i<o:\n            b.ccx(a[i],t[i],t[-1]);b.cx(a[i],t[i])\n    elif i<o:\n        b.cx(carry,a[i]);b.cx(carry,t[i])\n        b.ccx(a[i],t[i],t[-1]);b.cx(carry,t[-1])\n        b.cx(carry,a[i]);b.cx(a[i],t[i])\n    else:\n        b.ccx(carry,t[i],t[-1]);b.cx(carry,t[i])\n    if n-1<o:b.cx(a[n-1],t[n-1])\n    for i in reversed(range(n-2)):\n        if i==0 and cin is None:\n            if o:b.del_and(a[0],t[0],C(1));b.cx(a[0],t[0])\n        elif i<o:\n            b.cx(C(i),C(i+1));b.del_and(a[i],t[i],C(i+1))\n            b.cx(C(i),a[i]);b.cx(a[i],t[i])\n        else:\n            b.del_and(C(i),t[i],C(i+1));b.cx(C(i),t[i])\n    if count:b.free(allocated)\n    if sub:\n        for q in t:b.x(q)\n\n\ndef truth_table_check(max_width=6,borrowed_count=0):\n    class Bits:\n        def __init__(self):self.v=[]\n        def alloc_qubits(self,n):\n            ids=list(range(len(self.v),len(self.v)+n));self.v.extend([0]*n);return ids\n        def x(self,q):self.v[q]^=1\n        def cx(self,a,t):self.v[t]^=self.v[a]\n        def ccx(self,a,b,t):self.v[t]^=self.v[a]&self.v[b]\n        def init_and(self,a,b,t):\n            assert self.v[t]==0\n            self.v[t]=self.v[a]&self.v[b]\n        def del_and(self,a,b,t):\n            assert self.v[t]==(self.v[a]&self.v[b])\n            self.v[t]=0\n        def free(self,reg):assert not any(self.v[q] for q in reg)\n        def put(self,reg,value):\n            for i,q in enumerate(reg):self.v[q]=(value >> i)&1\n        def read(self,reg):return sum(self.v[q] << i for i,q in enumerate(reg))\n    count=0\n    for n in range(1,max_width+1):\n        for m in range(n+1):\n            for a in range(1 << m):\n                for value in range(1 << n):\n                    for cin_value in (None,0,1):\n                        for sub in (False,True):\n                            b=Bits();areg=b.alloc_qubits(m);target=b.alloc_qubits(n)\n                            cin=b.alloc_qubits(1)[0] if cin_value is not None else None\n                            if cin is not None:b.v[cin]=cin_value\n                            b.put(areg,a);b.put(target,value)\n                            borrowed=b.alloc_qubits(borrowed_count)\n                            folded_gadd(b,areg,target,cin,sub,borrowed=borrowed)\n                            delta=a+(cin_value or 0)\n                            expected=(value-delta if sub else value+delta) % (1 << n)\n                            assert b.read(target)==expected,(n,m,a,value,cin_value,sub)\n                            assert b.read(areg)==a\n                            assert cin is None or b.v[cin]==cin_value\n                            assert not any(b.v[q] for q in borrowed)\n                            count+=1\n    return count\n\n\ndef build_split_circuit(cfg,tabs,lay,adder=None,borrow_top=False):\n    if adder is None:adder=gadd\n    w2,w1,LB,S,whi=cfg.w2,lay[\'w1\'],lay[\'LB\'],lay[\'S\'],lay[\'whi\']\n    b=km.CircuitBuilder()\n    x=b.create_quantum_register(W,name=\'x\')\n    flags=b.alloc_qubits(5)\n    normalize(b,x,flags)\n    # Every promised positive input normalizes to x[21]=1. The arithmetic\n    # address uses x[14:21] and delta x[:14], so this wire is idle in the core.\n    if borrow_top:\n        b.x(x[-1])\n        raw_adder=adder\n        def adder(builder,a,t,cin=None,sub=False):\n            return raw_adder(builder,a,t,cin,sub,borrowed=(x[-1],))\n    c2=b.alloc_qubits(w2)\n    R=b.alloc_qubits(w1)\n    Lr=b.alloc_qubits(LB)\n    Yh=b.alloc_qubits(whi)\n    remaining=km.array(list(Lr)+list(R)+list(c2))\n    target=km.array(list(remaining)+list(Yh))\n    table=[0]*256\n    rest_table=[0]*256\n    high_table=[0]*256\n    for (e,a),(K2,K1,K0) in tabs.items():\n        idx=a+128*e\n        rest_table[idx]=lay[\'lo\'][(e,a)] | (K1 << LB) | (K2 << (LB+w1))\n        high_table[idx]=lay[\'hi\'][(e,a)]\n        table[idx]=rest_table[idx] | (high_table[idx] << (LB+w1+w2))\n    address=km.array(list(x[14:21])+[flags[0]])\n    delta=x[:DB]\n    def rrows(inverse):\n        seq=cfg.rrow_list()\n        if inverse:seq=list(reversed(seq))\n        for i in seq:\n            sh=cfg.sh2[i]\n            if sh>=0:off=c2[sh:w2];t=R[:w1];cin=c2[sh-1] if sh>=1 else None\n            else:off=c2[:w2];t=R[-sh:w1];cin=None\n            b.cx(delta[i],t);adder(b,off,t,cin,sub=inverse);b.cx(delta[i],t)\n    def yrows(inverse):\n        seq=list(reversed(range(DB)))\n        if inverse:seq.reverse()\n        for i in seq:\n            sh=cfg.sh1[i]\n            assert 0<=sh<w1\n            cin=R[sh-1] if sh>=1 else None\n            b.cx(delta[i],Lr);adder(b,R[sh:w1],Lr,cin,sub=inverse);b.cx(delta[i],Lr)\n    def phase(reg,base_bit):\n        ext=b.alloc_qubits(10)\n        word=km.array(list(reg)+list(ext))\n        shift_stages(b,flags,word,len(reg),False)\n        for j in range(len(word)):\n            exponent=base_bit+j-cfg.fy-10\n            if exponent<0:b.z_pow(word[j],Fraction(2)**(exponent+1))\n        shift_stages(b,flags,word,len(reg),True)\n        b.free(ext)\n    b.init_lookup(table=table,address=address,target=target)\n    phase(Yh,S)\n    b.del_lookup(table=high_table,address=address,target=Yh)\n    b.free(Yh)\n    rrows(False)\n    yrows(False)\n    phase(Lr[cfg.gy:LB],cfg.gy)\n    yrows(True)\n    rrows(True)\n    b.del_lookup(table=rest_table,address=address,target=remaining)\n    b.free(remaining)\n    if borrow_top:b.x(x[-1])\n    unnormalize(b,x,flags)\n    b.free(flags)\n    return b.finish_circuit()\n\n\n\ndef build_previous_schedule(cfg,tabs,lay=None):\n    """Regenerate 91Q circuit from the explicit model and credited schedules."""\n    if lay is None: lay=layout(cfg,tabs)\n    return build_split_circuit(cfg,tabs,lay,adder=folded_gadd,borrow_top=True)\n\n\n# The derivative-split version of this API is defined below.\n\n# The derivative-split version of this API is defined below.\n\n\n\n# The derivative-split version of this API is defined below.\n\n\n\n# The derivative-split version of this API is defined below.\n\n\ndef cli():\n    import argparse\n    import json\n    from pathlib import Path\n    parser = argparse.ArgumentParser(description=\'Generate the credited signed-digit circuit and memory-schedule adaptation.\')\n    parser.add_argument(\'output\', nargs=\'?\', default=\'invsqrt.kmb\')\n    parser.add_argument(\'--baseline\', action=\'store_true\')\n    args = parser.parse_args()\n    cfg, tabs, _ = load_or_build_tables()\n    certificate = exhaustive_certificate(cfg, tabs)\n    if not certificate[\'pass\']:\n        raise ArithmeticError(certificate)\n    lay = layout(cfg, tabs)\n    circuit = build_memory_baseline(cfg, tabs, lay) if args.baseline else build_selected(cfg, tabs, lay)\n    circuit.to_file(args.output, format=\'kmb\')\n    report = estimated_resources(circuit, certificate)\n    report[\'arithmetic_certificate\'] = certificate\n    report[\'variant\'] = \'sumit_baseline\' if args.baseline else selected_descriptor()[\'variant\']\n    Path(args.output + \'.resources.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    print(json.dumps(report, indent=2))\n\n\n\n\n\ndef _rudra_gadd_top_reused(b, source, target, cin=None, sub=False):\n    """Gidney addition with n-2 clean carries; the last carry targets the MSB."""\n    import kickmix as km\n    a, t = list(source), list(target)\n    n, m = len(t), len(a)\n    assert n >= 1 and m <= n\n    if sub:\n        b.x(km.array(t))\n    carries = b.alloc_qubits(n - 2) if n > 2 else km.array([])\n    for i in range(n - 1):\n        previous = cin if i == 0 else carries[i - 1]\n        last = i == n - 2\n        out = t[-1] if last else carries[i]\n        if previous is None:\n            if i < m:\n                (b.ccx if last else b.init_and)(a[i], t[i], out)\n        elif i < m:\n            b.cx(previous, a[i]); b.cx(previous, t[i])\n            (b.ccx if last else b.init_and)(a[i], t[i], out)\n            b.cx(previous, out)\n        else:\n            (b.ccx if last else b.init_and)(previous, t[i], out)\n    if n - 1 < m:\n        b.cx(a[n - 1], t[n - 1])\n    if n == 1 and cin is not None:\n        b.cx(cin, t[0])\n    for i in reversed(range(n - 1)):\n        previous = cin if i == 0 else carries[i - 1]\n        last = i == n - 2\n        if previous is None:\n            if i < m:\n                if not last:\n                    b.del_and(a[i], t[i], carries[i])\n                b.cx(a[i], t[i])\n        elif i < m:\n            if not last:\n                b.cx(previous, carries[i]); b.del_and(a[i], t[i], carries[i])\n            b.cx(previous, a[i]); b.cx(a[i], t[i])\n        else:\n            if not last:\n                b.del_and(previous, t[i], carries[i])\n            b.cx(previous, t[i])\n    if n > 2:\n        b.free(carries)\n    if sub:\n        b.x(km.array(t))\n\ndef prepare_derivative_split(reference, cfg, tables):\n    d = np.arange(1 << reference.DB, dtype=np.int64)\n    coarse, residual = {}, {}\n    bounds = {\'normalized_addresses_checked\': 0, \'min_residual\': 1 << 30,\n              \'max_residual\': 0, \'max_table_residual\': 0, \'max_coarse\': 0}\n    for key, (k2, k1, _) in tables.items():\n        derivative, _ = reference.pipeline(cfg, k2, k1, d)\n        base = int(derivative.min()) >> 11\n        low = derivative - (base << 11)\n        coarse[key], residual[key] = base, k1 - (base << 11)\n        assert 0 <= residual[key] < 1 << 12\n        assert (low >= 0).all() and (low < 1 << 12).all()\n        assert ((base + (low >> 11)) < 1 << 7).all()\n        joined = (low & 2047) + ((base + (low >> 11)) << 11)\n        assert np.array_equal(joined, derivative)\n        bounds[\'normalized_addresses_checked\'] += len(d)\n        bounds[\'min_residual\'] = min(bounds[\'min_residual\'], int(low.min()))\n        bounds[\'max_residual\'] = max(bounds[\'max_residual\'], int(low.max()))\n        bounds[\'max_table_residual\'] = max(bounds[\'max_table_residual\'], residual[key])\n        bounds[\'max_coarse\'] = max(bounds[\'max_coarse\'], base)\n    assert cfg.w2 >= 6 and len(cfg.rrow_list()) == 8\n    return {\'coarse\': coarse, \'residual\': residual}, bounds\n\ndef _build_derivative_circuit(reference, cfg, tables, layout, split, borrow_top=False):\n    import kickmix as km\n    assert (layout[\'w1\'], layout[\'LB\'], layout[\'S\'], layout[\'whi\']) == (18, 21, 20, 8)\n    b = km.CircuitBuilder()\n    x = b.create_quantum_register(22, name=\'x\')\n    flags = b.alloc_qubits(5)\n    # These coefficients are still zero during normalization; allocate afterward.\n    reference.normalize(b, x, flags)\n    if borrow_top: b.x(x[-1])\n    def adder(builder,a,t,cin=None,sub=False):\n        if borrow_top: return folded_gadd(builder,a,t,cin,sub,borrowed=(x[-1],))\n        return _rudra_gadd_top_reused(builder,a,t,cin,sub)\n    c2, residual, coarse, low, high = [b.alloc_qubits(n) for n in [cfg.w2, 12, 7, 21, 8]]\n    full_target = km.array(list(low) + list(residual) + list(c2) + list(high) + list(coarse))\n    rest_target = km.array(list(low) + list(residual) + list(c2) + list(coarse))\n    full_table, rest_table, high_table = [0] * 256, [0] * 256, [0] * 256\n    for (parity, address), (k2, _, _) in tables.items():\n        key, index = (parity, address), address + 128 * parity\n        common = layout[\'lo\'][key] | (split[\'residual\'][key] << 21) | (k2 << 33)\n        full_table[index] = common | (layout[\'hi\'][key] << (33+cfg.w2)) | (split[\'coarse\'][key] << (41+cfg.w2))\n        rest_table[index] = common | (split[\'coarse\'][key] << (33+cfg.w2))\n        high_table[index] = layout[\'hi\'][key]\n    address = km.array(list(x[14:21]) + [flags[0]])\n    delta = x[:14]\n\n    def phase_register(register, low_bit):\n        ext = b.alloc_qubits(10)\n        word = km.array(list(register) + list(ext))\n        reference.shift_stages(b, flags, word, len(register), False)\n        for i, qubit in enumerate(word):\n            exponent = low_bit + i - cfg.fy - 10\n            if exponent < 0:\n                b.z_pow(qubit, Fraction(2) ** (exponent + 1))\n        reference.shift_stages(b, flags, word, len(register), True)\n        b.free(ext)\n\n    def inner(inverse):\n        rows = cfg.rrow_list()\n        for i in reversed(rows) if inverse else rows:\n            shift = cfg.sh2[i]\n            # A narrower curvature coefficient can require an exact left shift.\n            # Add to the corresponding residual suffix, leaving lower bits intact.\n            if shift>=0:\n                addend=c2[shift:];target=residual\n                carry=c2[shift-1] if shift else None\n            else:\n                addend=c2;target=residual[-shift:];carry=None\n            b.cx(delta[i],target)\n            adder(b,addend,target,carry,sub=inverse)\n            b.cx(delta[i],target)\n\n    def outer(derivative, inverse):\n        for i in range(14) if inverse else reversed(range(14)):\n            shift = cfg.sh1[i]\n            carry = derivative[shift - 1] if shift else None\n            b.cx(delta[i], low)\n            adder(b, derivative[shift:], low, carry, sub=inverse)\n            b.cx(delta[i], low)\n\n    b.init_lookup(table=full_table, address=address, target=full_target)\n    phase_register(high, 20)\n    # High and the shift workspace have been restored. For every X-measurement\n    # mask m, unlookup contributes (-1)^(m.high(address)); its phase correction\n    # contributes the same sign. Their product is +1, preserving our phase.\n    # Address and its parity flag stay live and unchanged until after erasure.\n    b.del_lookup(table=high_table, address=address, target=high)\n    b.free(high)\n    inner(False)\n    adder(b, residual[11:], coarse)\n    derivative = residual[:11] + coarse\n    outer(derivative, False)\n    # Include bit20 here. There is no output join: floor((L+2^20H)/4)\n    # equals floor(L/4)+2^18H, so the two diagonal phases add exactly.\n    phase_register(low[cfg.gy:], cfg.gy)\n    outer(derivative, True)\n    adder(b, residual[11:], coarse, sub=True)\n    inner(True)\n    b.del_lookup(table=rest_table, address=address, target=rest_target)\n    b.free(rest_target)\n    if borrow_top: b.x(x[-1])\n    reference.unnormalize(b, x, flags)\n    b.free(flags)\n    return b.finish_circuit()\n\n\ndef _derivative_reference(promise):\n    return types.SimpleNamespace(**dict(globals(),\n        normalize=normalize if promise else _combined_normalize,\n        unnormalize=unnormalize if promise else _baseline_unnormalize))\n\n\ndef _build_derivative_variant(cfg,tabs,lay,borrow,promise):\n    if lay is None:lay=layout(cfg,tabs)\n    reference=_derivative_reference(promise)\n    split,bounds=prepare_derivative_split(reference,cfg,tabs)\n    return _build_derivative_circuit(reference,cfg,tabs,lay,split,borrow_top=borrow)\n\n\ndef build_selected(cfg,tabs,lay=None):\n    """Regenerate selected92Q derivative split with borrowed carry and domain rewrite."""\n    return _build_derivative_variant(cfg,tabs,lay,borrow=True,promise=True)\n\n\ndef build_memory_baseline(cfg,tabs,lay=None):\n    """Same Kozykorpesh coefficient model, Rudra/Kirill derivative schedule:93Q."""\n    return _build_derivative_variant(cfg,tabs,lay,borrow=False,promise=False)\n\n\ndef build_borrowed_baseline(cfg,tabs,lay=None):\n    """Same derivative schedule with the normalized input bit borrowed:92Q."""\n    return _build_derivative_variant(cfg,tabs,lay,borrow=True,promise=False)\n\n\ndef build_combined_baseline(cfg,tabs,lay=None):\n    """Compatibility alias for the borrowed-carry comparison."""\n    return build_borrowed_baseline(cfg,tabs,lay)\n\n\ndef comparison_report(cfg,tabs,certificate,lay=None):\n    """One fixed coefficient table: derivative schedule, borrow, domain rewrite."""\n    if lay is None:lay=layout(cfg,tabs)\n    return {\n        \'baseline\':estimated_resources(build_memory_baseline(cfg,tabs,lay),certificate),\n        \'borrowed\':estimated_resources(build_borrowed_baseline(cfg,tabs,lay),certificate),\n        \'selected\':estimated_resources(build_selected(cfg,tabs,lay),certificate),\n        \'selected_descriptor\':selected_descriptor()}\n\n\ndef selected_descriptor():\n    from tasks.invsqrt.fit_coefficients import PROVENANCE\n    return {\n        \'variant\':\'kozy_table_rudra_split_alex_schedule_borrow_and_domain\',\n        \'source_author\':\'Sumit Chongder; Rudra Bhaskar; Alex Chilton; Kozykorpesh; Kirill Tymchishin; anazemcev / Artem1410\',\n        \'source_url\':\'https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory\',\n        \'coefficient_source\':\'Lawson minimax and dataset-adapted exact-domain calibration in tasks/invsqrt/fit_coefficients.py\',\n        \'configuration\':{\'f2\':8,\'f1\':19,\'fyr\':27,\'gy\':1,\'drop2\':[5]},\n        \'license_as_published\':\'Apache-2.0 for the anazemcev v3 parent. Alex v8 and Mizero v1 are reference-only and are not this licence.\',\n        \'circuit_authorship\':\'port\',\n        \'coefficient_provenance\':PROVENANCE,\n        \'baseline_definition\':\'Same fixed Kozykorpesh table with the Rudra derivative split and Alex normalization/shifts, before normalized-input carry borrowing:93Q\',\n        \'baseline_canonical_sha256\':\'9e67c00941ff0623cfaf22f133fe7f70740f8b0a163a4ba64c6fdcd1c9446295\',\n        \'borrowed_canonical_sha256\':\'f3f8457058aecdcf2650e00b411234899cb01933b79a1f6519c0bf28b49591bc\',\n        \'parent_92q_sha256\':\'63f70e623dd995f4f87ca13254f46f991a5ae545d4c20c80e8510f5ff0a49bbd\',\n        \'selected_canonical_sha256\':\'75e8a0e2341bb3e8f8ded1ec1d53e24437410c69e887da9daf7d49bfcc28c2dd\',\n        \'historical_kirill95_sha256\':\'497b161bc468654b5b2570cfaa006ce336dec9b8c89f82f048fcca4e6e8f7919\',\n        \'previous_same_table91_sha256\':\'e3db0cc65b4771b640fbcd73d770d2015c232e9f54bd69479786fb5445ca76b9\',\n        \'measured_counts\':{\'qubits\':90,\'static_toffoli\':1448,\'rotations\':44},\n        \'expected_mean_toffoli\':1437.5,\n        \'estimated_cost\':157751.9485933627,\n        \'changes_in_this_revision\':[\n            \'Regenerate 256 coefficient triples with dataset-adapted calibration; licence unresolved\',\n            \'90-qubit Gray curvature schedule, exact-domain affine rewrite, and one calibrated rotation drop\',\n            \'Borrow normalized x[21] as clean adder carry, restoring it before inverse normalization\',\n            \'Retain our exact original-input top-bit mutual-exclusion rewrite\'],\n        \'upstream_contributions\':\'Rudra derivative split/top-carry/static-high phase checkpoint; Alex prefix flags/empty-target shift; Kirill exact integration; Sumit model; Kozykorpesh precision calibration\',\n        \'full_official_verification\':\'Required separately; actual unchanged full-verifier report identifies the selected hash\'}\n\n\n'}
SOURCE_HASHES = {'LICENSE': '40911d195a0c25e3647660174fbd30f95ad80af621bd1088bbf5649c970d4e58', 'ledger/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ledger/archive.py': '1950107fd862cf70a1ef7fc899737b334928ffb8e03ce0c69bd4c82d428a0f39', 'notices/10-invsqrt.txt': '66b1de01293a73eb12943ec935a6f76f5eb7fa7a2f1284668da6a137963e341f', 'notices/20-dqi.txt': '161966760acb82fc694e644c78b1a1dfdf0d2d9bffde1c4134a3624dc3496fdf', 'notices/30-dfthcsos.txt': '11e5c028e0230f349f283c7f92fdccdc39a4245c7fca1da75d75177c02fae5e7', 'tasks/__init__.py': 'c51df006aff856bcc60c6e5ba7d8d501e41be78e8ad8235ed06534c643bd45cc', 'tasks/dfthcsos/generate.py': '7c133c8e732328869452dbfd83d781147e4169bd33665b7b6c4b5bc33941149a', 'tasks/dqi/PROVENANCE.md': '307771f250d864d2bf707e305ec4b25bd5ccd5bb0e8f67fed39465dbf60b7c2d', 'tasks/dqi/bm_forney/LICENSE': 'cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30', 'tasks/dqi/bm_forney/borrow.py': '8fd2f83cc831d9d60b6a13d75e0a36dd5ed9033dfe9d8b5ec0ee899780cdfe39', 'tasks/dqi/bm_forney/contract.json': 'b5fa02899021142b7f265f1e18f366b0d6f55944023b3cf249f976c655d2097e', 'tasks/dqi/bm_forney/gu_classical.py': 'c05c9e7a84a69e3ad0590b972642dcb30c2c29ca2613a821c82d09d53556ba62', 'tasks/dqi/bm_forney/gu_formula_derivation.py': 'd55e238aa5c7ccb26b0d5cea8fdb911ecbb4ae54782df3e50950140164204dbb', 'tasks/dqi/bm_forney/gu_linear.py': 'd0b89d494ae63ec28e8bac6f33b344c11aaaab378d0cec81683b1a5743ed25d3', 'tasks/dqi/bm_forney/gu_port.py': '93a753e54e0ae28e0ca37a168f0c4e8a740f6e1ab82ff7106b1fdd24bb487250', 'tasks/dqi/bm_forney/port_runtime.py': 'cbfc4562cd110bce7f3d214ec179e02ec00e1165446504be28c0c3e44fbdf293', 'tasks/dqi/bm_forney/round2_gadgets.py': '76dd907a2f19f16c38c49f6a631909cdb732d17fc7809d3797bb7c08e67c5645', 'tasks/dqi/bm_forney/symbolic_runtime.py': 'f25047e0138bfe75cf177afd6d50e7694d047ed8cacf26ac83b7eaa6224058a4', 'tasks/dqi/bm_forney/windows.json': 'a5dff27decbf564d6b810ece4dd345099eac5e4ba3d8260dcbf263f22dc9da99', 'tasks/dqi/generate.py': 'ab6aca7cd25058aab867a1fb529a764136fd13a2b7dce18224bbe0a2122aa08c', 'tasks/invsqrt/PROVENANCE.md': 'e6357ec25e63f3a68db718e82c69e2a3f3b7f4237701476a34c355dfacc6a3fd', 'tasks/invsqrt/__init__.py': '93c5f408a63782c2615a7fd8a69d90deb74e149df9e4b81059b5e38e929ad6ff', 'tasks/invsqrt/affine.py': 'aa351a287c507c264a8aa7d172e1c7b9bfc1d0dcb78c8fa61b92996947eba290', 'tasks/invsqrt/calibrate.py': '7fc0c1cac6ff45753c25d98dfa7cd82fb4803332c37fb616af18fabe47cef309', 'tasks/invsqrt/fit_coefficients.py': '35d36f95aa4bb7c63bf8cd24d72df594abdd679a3ed770cb2fcced0200cda839', 'tasks/invsqrt/generate.py': '9ccdffd0cba53a097c78d99d833b9bcc37e9be387f62ed723a8173cd2cdf498c', 'tasks/invsqrt/peephole.py': 'd5ed02b0d28392cc637111fc94fb1760c71feda4f8ca7aad6f73944e81fff54c', 'tasks/invsqrt/schedule90.py': '58c88937e18e2c96e5e92f3a2576b3d32661de0d93221d4f288e314bb4c81edb', 'tasks/invsqrt/signed_digit.py': 'ab08c1f178405cae9872281fbe73ca41c5de160bea4b5ddeece41b55ac708cd5'}

In [3]:
import importlib.metadata
import json
import os
import platform
import subprocess
import sys
import tempfile
import time

DATA = Path(os.environ.get('VQCC_DATA', '/kaggle/input/google-verifiable-quantum-circuit-compilation')).resolve()
MODE = os.environ.get('VQCC_VERIFY_MODE', 'full')
if MODE not in ('full', 'sample'):
    raise ValueError('VQCC_VERIFY_MODE must be full or sample')
if sys.version_info[:2] != (3, 13) or platform.system() != 'Linux':
    raise RuntimeError('The pinned competition kickmix requires CPython 3.13 on Linux')
for name in ('kickmix', 'verifiers', 'sample_submission'):
    if not (DATA / name).is_dir():
        raise FileNotFoundError(f'VQCC_DATA must contain {name}/')
print('Source commit:', CONFIG['commit'])
print('Manifest:', CONFIG['manifest'], 'SHA-256', CONFIG['manifest_sha256'])
print('Python:', platform.python_version())
print('Packages:', {p: importlib.metadata.version(p) for p in ('numpy', 'scipy')})
print('Verification mode:', MODE)
check_hash(DATA / 'kickmix/kickmix.so', CONFIG['kickmix_sha256'])
for task, relative in {'invsqrt': 'invsqrt/verify_submission_invsqrt.py',
                       'dqi': 'dqi/verify_submission_dqi.py', 'dfthcsos': 'dfthcsos/verify.py'}.items():
    check_hash(DATA / 'verifiers' / relative, CONFIG['verifiers'][task])
check_hash(DATA / 'sample_submission/dfthcsos/submission.npz', CONFIG['files']['dfthcsos/submission.npz'])
# A new directory on each run prevents stale circuits or cached coefficient tables.
WORK = Path(tempfile.mkdtemp(prefix='vqcc-rebuild-', dir=Path.cwd()))
SOURCE = WORK / 'source'
OUTPUT = WORK / 'submission'
OUTPUT.mkdir()
restore_sources(SOURCE, SOURCES, SOURCE_HASHES)
sys.path[:0] = [str(SOURCE), str(DATA / 'kickmix'), str(DATA / 'verifiers')]
import kickmix as km
check_hash(Path(km.__file__), CONFIG['kickmix_sha256'])
ENV = dict(os.environ, VQCC_DATA=str(DATA), PYTHONPATH=os.pathsep.join(
    [str(SOURCE), str(DATA / 'kickmix'), str(DATA / 'verifiers')]))
print('Fresh workspace:', WORK.name)

Source commit: c8bbfbde18bb27f4f89aac90f4b62437877b60f4
Manifest: ledger/manifests/candidates/pra228-dqi-invsqrt-572.json SHA-256 5d76c29f18a43c54c4402385e3c37cd07c18316c8a6e327d4479d12eddd6a8b9
Python: 3.13.13
Packages: {'numpy': '2.5.4', 'scipy': '1.18.1'}
Verification mode: full
kickmix.so: SHA-256 c66b3216c4029e0cdfd09054306aa0feadf53ab2f50c1164e37f752874e5005e


verify_submission_invsqrt.py: SHA-256 ddef520dec6b68d1cf0a6f328ddd36d79eb84570b9192cf3f7acaf073f06733a


verify_submission_dqi.py: SHA-256 f14274bc0943fb9769ddb01131be6063191a15009fe1caf1d4bdd9fd25194e65


verify.py: SHA-256 0775be7a528922367caebe9e4d051d6199811f1aefeaa4b793eaaa4621b8ec9d


submission.npz: SHA-256 14d5cc25eb3bf82a9ccb87c811db28d127bef6d28697dd190dc1e875de3bfe69


LICENSE: SHA-256 40911d195a0c25e3647660174fbd30f95ad80af621bd1088bbf5649c970d4e58
ledger/__init__.py: SHA-256 e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
ledger/archive.py: SHA-256 1950107fd862cf70a1ef7fc899737b334928ffb8e03ce0c69bd4c82d428a0f39
notices/10-invsqrt.txt: SHA-256 66b1de01293a73eb12943ec935a6f76f5eb7fa7a2f1284668da6a137963e341f
notices/20-dqi.txt: SHA-256 161966760acb82fc694e644c78b1a1dfdf0d2d9bffde1c4134a3624dc3496fdf
notices/30-dfthcsos.txt: SHA-256 11e5c028e0230f349f283c7f92fdccdc39a4245c7fca1da75d75177c02fae5e7
tasks/__init__.py: SHA-256 c51df006aff856bcc60c6e5ba7d8d501e41be78e8ad8235ed06534c643bd45cc
tasks/dfthcsos/generate.py: SHA-256 7c133c8e732328869452dbfd83d781147e4169bd33665b7b6c4b5bc33941149a
tasks/dqi/PROVENANCE.md: SHA-256 307771f250d864d2bf707e305ec4b25bd5ccd5bb0e8f67fed39465dbf60b7c2d
tasks/dqi/bm_forney/LICENSE: SHA-256 cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30
tasks/dqi/bm_forney/borrow.py: SHA-256 8fd2f83cc831d

Fresh workspace: vqcc-rebuild-xnhhyepg


## Build all three tasks

The arithmetic and decoder circuits are ported and adapted source; the chemistry circuits
rebuild Google's reference construction. Our orchestration, deterministic packaging and
byte checks are in this repository. The only copied output is Google's declared non-circuit
factorization `dfthcsos/submission.npz`; every `.kmb` is newly generated.
The verifiers also use the reference physics data inside the same competition package.

In [4]:
started = time.monotonic()
subprocess.run([sys.executable, str(SOURCE / 'tasks/invsqrt/generate.py'), '--out', str(OUTPUT)],
               cwd=SOURCE, env=ENV, check=True)
print('invsqrt build seconds:', round(time.monotonic() - started, 2))

affine 4000/27221 rewrites 64 peak 86 false 105


affine 8000/27221 rewrites 114 peak 86 false 105


affine 12000/27221 rewrites 271 peak 90 false 123


affine 16000/27221 rewrites 642 peak 90 false 378


affine 20000/27221 rewrites 887 peak 90 false 383
affine 24000/27221 rewrites 895 peak 90 false 383


...

calibration LP 9: rows 10183 lp 73326606.462 exhaustive -307671729.247


calibration LP 10: rows 10297 lp 73326606.462 exhaustive 73326606.462


calibration drop [18]: margin 73326605 rotations 44 ratio 13.164809


/home/cld-main/work/.paperclip-worktrees/quantum-hackathon/PRA-240-public-notebook/notebooks/vqcc-rebuild-xnhhyepg/submission/invsqrt.kmb sha256=75e8a0e2341bb3e8f8ded1ec1d53e24437410c69e887da9daf7d49bfcc28c2dd bytes=234743
Q 90 expected_toffoli 1437.5 rotations 44 ratio 13.164809 margin 73326605


invsqrt build seconds: 94.11


In [5]:
started = time.monotonic()
subprocess.run([sys.executable, str(SOURCE / 'tasks/dqi/generate.py'), '--out', str(OUTPUT)],
               cwd=SOURCE, env=ENV, check=True)
print('dqi build seconds:', round(time.monotonic() - started, 2))

{"phase": "init", "elapsed": 9.24380001379177e-05, "physical_qubits": 0, "active_qubits": 0}
{"phase": "zlin", "elapsed": 0.00230783800361678, "physical_qubits": 863, "active_qubits": 863}
{"phase": "bm/seed", "elapsed": 0.0024665240052854642, "physical_qubits": 863, "active_qubits": 863}
{"phase": "bm/r00", "elapsed": 0.002762431002338417, "physical_qubits": 887, "active_qubits": 887}
{"phase": "bm/r01", "elapsed": 0.015317217010306194, "physical_qubits": 942, "active_qubits": 911}
{"phase": "bm/r02", "elapsed": 0.13571624300675467, "physical_qubits": 978, "active_qubits": 935}


{"phase": "bm/r03", "elapsed": 0.220081059000222, "physical_qubits": 1009, "active_qubits": 966}
{"phase": "bm/r04", "elapsed": 0.32633432500006165, "physical_qubits": 1033, "active_qubits": 990}


{"phase": "bm/r05", "elapsed": 0.4937996320077218, "physical_qubits": 1045, "active_qubits": 1002}
{"phase": "bm/r06", "elapsed": 0.6690279580070637, "physical_qubits": 1069, "active_qubits": 1026}


{"phase": "bm/r07", "elapsed": 0.8361259110097308, "physical_qubits": 1081, "active_qubits": 1038}
{"phase": "bm/r08", "elapsed": 0.9919513119966723, "physical_qubits": 1105, "active_qubits": 1062}


{"phase": "bm/r09", "elapsed": 1.1309825960051967, "physical_qubits": 1117, "active_qubits": 1074}
{"phase": "bm/r10", "elapsed": 1.278551242998219, "physical_qubits": 1141, "active_qubits": 1098}


...

{"phase": "reverse/r04", "elapsed": 323.78167431100155, "physical_qubits": 1789, "active_qubits": 1002}


{"phase": "reverse/r03", "elapsed": 323.99748183200427, "physical_qubits": 1789, "active_qubits": 990}
{"phase": "reverse/r02", "elapsed": 324.1514718719991, "physical_qubits": 1789, "active_qubits": 966}


{"phase": "reverse/r01", "elapsed": 324.28922553099983, "physical_qubits": 1789, "active_qubits": 935}
{"phase": "reverse/r00", "elapsed": 324.3964223310031, "physical_qubits": 1789, "active_qubits": 911}
{"phase": "reverse/seed", "elapsed": 324.40863108300255, "physical_qubits": 1789, "active_qubits": 887}
{"phase": "cleanup", "elapsed": 324.40897428800236, "physical_qubits": 1789, "active_qubits": 863}


Wrote /home/cld-main/work/.paperclip-worktrees/quantum-hackathon/PRA-240-public-notebook/notebooks/vqcc-rebuild-xnhhyepg/submission/dqi.kmb: Q 1789, static Toffoli-like 445229, ops 93898574, build 325.5s.
sha256 529b075009f7758abeda83a63407f0c1feeb51dd70631ae4335dfd28721b7e08
field 4179 primitive 2


dqi build seconds: 333.07


In [6]:
started = time.monotonic()
subprocess.run([sys.executable, str(SOURCE / 'tasks/dfthcsos/generate.py'), '--out', str(OUTPUT)],
               cwd=SOURCE, env=ENV, check=True)
print('dfthcsos build seconds:', round(time.monotonic() - started, 2))

angle_k_0.kmb: sha256 7bfb3983accc6ff97d6ab41a88deae924f1d1c97cf30e60879d0bff5688a8e25 (match)
angle_k_1.kmb: sha256 a04b6d0d57ce851b123af265066aaab2ab33353ea049d26acd534727466d72b6 (match)
angle_k_2.kmb: sha256 5c1b8987bc3fa1a4b8607e470f8c8f1392c22e7a7dbcb228f0ce9ec986e18058 (match)
angle_k_3.kmb: sha256 16ac67cdcaf3e2c6abdd98c4138ff1ae8a56cfc4ae15622bcdb3d3a3ba1e22cf (match)
angle_k_4.kmb: sha256 d21cdd4c7e94196654aab15ebcabfe10ce8d343151a5e8dc3672e76684b6628d (match)
angle_k_5.kmb: sha256 63b8d4cba36955e6098321f3b2ef3e34401dd276fa629b2e490d043b1743880b (match)
angle_k_6.kmb: sha256 5bf740428e661d035fda860142578f421969e35e0df2e4e024fbf01a7340b378 (match)
angle_k_7.kmb: sha256 606009582111a22bd549b795c8d115ac8bbeb124bc8da24a80380eca6d34163d (match)
angle_k_8.kmb: sha256 57c7cc51f5922fff6e95d76e1f723d4c2d029a241da643b95684d6322157a8fe (match)
angle_k_9.kmb: sha256 22cbe2828988ff954fc07fd971a25e6bc04a02c302bf2cc642e4c3535daaad65 (match)
submission.npz: sha256 14d5cc25eb3bf82a9ccb87c811d

In [7]:
check_files(OUTPUT, CONFIG['files'])
print('All 13 generated/copied file hashes match submission-003.')

dfthcsos/angle_k_0.kmb: angle_k_0.kmb: SHA-256 7bfb3983accc6ff97d6ab41a88deae924f1d1c97cf30e60879d0bff5688a8e25


dfthcsos/angle_k_1.kmb: angle_k_1.kmb: SHA-256 a04b6d0d57ce851b123af265066aaab2ab33353ea049d26acd534727466d72b6


dfthcsos/angle_k_2.kmb: angle_k_2.kmb: SHA-256 5c1b8987bc3fa1a4b8607e470f8c8f1392c22e7a7dbcb228f0ce9ec986e18058


dfthcsos/angle_k_3.kmb: angle_k_3.kmb: SHA-256 16ac67cdcaf3e2c6abdd98c4138ff1ae8a56cfc4ae15622bcdb3d3a3ba1e22cf


dfthcsos/angle_k_4.kmb: angle_k_4.kmb: SHA-256 d21cdd4c7e94196654aab15ebcabfe10ce8d343151a5e8dc3672e76684b6628d


...

dfthcsos/submission.npz: submission.npz: SHA-256 14d5cc25eb3bf82a9ccb87c811db28d127bef6d28697dd190dc1e875de3bfe69


dqi.kmb: 

dqi.kmb: SHA-256 529b075009f7758abeda83a63407f0c1feeb51dd70631ae4335dfd28721b7e08


invsqrt.kmb: invsqrt.kmb: SHA-256 75e8a0e2341bb3e8f8ded1ec1d53e24437410c69e887da9daf7d49bfcc28c2dd


All 13 generated/copied file hashes match submission-003.


## Fresh verifier results

These calls use the competition's unchanged verifiers. Each cell raises on failure.
The run uses the fixed public salt `PRA171_LOCAL_PUBLIC_FIXED_SALT_0001`.
Reduced-shot mode, if selected, is labelled in each result and never presented as a
leaderboard score. No stored verifier receipts are consulted.

In [8]:
import math
from invsqrt import verify_submission_invsqrt as inv
from dqi import verify_submission_dqi as dqi
from dfthcsos import verify as df

SALT = 'PRA171_LOCAL_PUBLIC_FIXED_SALT_0001'
receipts = {}

def record(task, result, started):
    if result.get('status') != 'PASS' or not math.isfinite(result.get('score', float('nan'))) or result['score'] <= 0:
        raise ValueError(f'{task} verifier failed: {result}')
    receipt = {'task': task, 'mode': MODE, 'salt': SALT,
               'seconds': round(time.monotonic() - started, 2), 'result': result}
    receipts[task] = receipt
    print(json.dumps(receipt, indent=2, allow_nan=False))

In [9]:
started = time.monotonic()
spec = inv.ProblemSpecInvSqrt.for_n_p(10, shots_per_x=64 if MODE == 'full' else 1)
record('invsqrt', inv.verify(km.Circuit.from_file(OUTPUT / 'invsqrt.kmb'), spec,
                            salt=SALT, max_workers=4), started)

{
  "task": "invsqrt",
  "mode": "full",
  "salt": "PRA171_LOCAL_PUBLIC_FIXED_SALT_0001",
  "seconds": 233.39,
  "result": {
    "status": "PASS",
    "score": 157751.94999626835,
    "toffoli_equivalent": 1752.7994444029816,
    "toffoli_count": 1437.5000155878406,
    "max_toffoli_count": 1448,
    "t_count": 1.0,
    "rotation_count": 44.0,
    "rotation_eps": 2.8381742236929845e-13,
    "num_qubits": 90,
    "reaction_depth": 200,
    "num_ops": 25510,
    "verified_shots": 200540928,
    "worst_error_over_tolerance": 0.9996227434818049,
    "worst_x": 32,
    "circuit_sha256": "75e8a0e2341bb3e8f8ded1ec1d53e24437410c69e887da9daf7d49bfcc28c2dd"
  }
}


In [10]:
started = time.monotonic()
spec = dqi.ProblemSpecDQI.benchmark_instance(num_shots=102400 if MODE == 'full' else 1536)
record('dqi', dqi.verify(km.Circuit.from_file(OUTPUT / 'dqi.kmb'), spec,
                        salt=SALT, max_workers=4), started)

{
  "task": "dqi",
  "mode": "full",
  "salt": "PRA171_LOCAL_PUBLIC_FIXED_SALT_0001",
  "seconds": 120.38,
  "result": {
    "status": "PASS",
    "score": 1717915181875.0806,
    "m": 4095,
    "n": 70,
    "degree": 12,
    "r": 2016,
    "verified_shots": 102400,
    "value_errors": 0,
    "phase_errors": 3,
    "empirical_error_rate": 2.9296875e-05,
    "effective_error_rate": 0.000983909102617945,
    "log2_delta": -128.0,
    "repetition_factor": 1.2055843238398525,
    "mean_error_weight": 31.046737331746634,
    "toffoli_count": 445229.0,
    "max_toffoli_count": 445229,
    "reaction_depth": 6486,
    "num_ops": 93898574,
    "shot_operation_counts": {
      "NEG": 51173,
      "X": 22372659200,
      "Z": 91663121749,
      "R": 15614464000,
      "HMR": 45198540800,
      "CX": 7837754060800,
      "CZ": 122160561113,
      "SWAP": 201277440000,
      "CSWAP": 6551040000,
      "CCX": 39040409600,
      "INFO_TAG": 174517452800,
      "POP_CONDITION": 422618624000,
      "PU

In [11]:
started = time.monotonic()
record('dfthcsos', df.verify(OUTPUT / 'dfthcsos', salt=SALT,
                             shots_per_address=64 if MODE == 'full' else 1), started)

{
  "task": "dfthcsos",
  "mode": "full",
  "salt": "PRA171_LOCAL_PUBLIC_FIXED_SALT_0001",
  "seconds": 57.33,
  "result": {
    "status": "PASS",
    "score": 3355599588.3155365,
    "lambda_eff": 49.30136925057438,
    "toffoli_count": 93237.0,
    "num_qubits": 730,
    "brot": 12,
    "batch_size": 8,
    "num_circuits": 10,
    "reaction_depth": 110,
    "num_ops": 2060743,
    "verified_shots": 254464,
    "submission_sha256": "276d464ff14da828b6b33e8487d99975494cbdf0c3cb2bd37dd3c3ddc4002695",
    "resources": {
      "brot": 12,
      "keep_bits": 10,
      "angles_toffoli_per_pass": 39740.0,
      "angles_toffoli": 79480.0,
      "angles_qubits": 119,
      "angles_selection_qubits": 12,
      "angles_persistent_qubits": 96,
      "angles_temporary_qubits": 11,
      "outer_state_prep_toffoli": 846,
      "inner_state_prep_toffoli": 9461,
      "givens_rotations_toffoli": 3450,
      "outer_state_prep_persistent_qubits": 46,
      "outer_state_prep_temporary_qubits": 370,
     

In [12]:
from ledger.archive import build_zip

if set(receipts) != {'invsqrt', 'dqi', 'dfthcsos'}:
    raise RuntimeError('All three fresh verifier results are required')
check_files(OUTPUT, CONFIG['files'])
archive = build_zip(OUTPUT, WORK / 'submission.zip')
check_hash(archive, CONFIG['zip_sha256'])
if archive.stat().st_size != CONFIG['zip_bytes']:
    raise ValueError('Archive size differs from the ledger')
# Publish the local output name only after all checks have passed.
import shutil
shutil.copyfile(archive, Path.cwd() / 'submission.zip')
print('submission.zip:', archive.stat().st_size, 'bytes; SHA-256', CONFIG['zip_sha256'])
print('Verification:', MODE, '; Kaggle hidden-score equivalence is not asserted.')

dfthcsos/angle_k_0.kmb: angle_k_0.kmb: SHA-256 7bfb3983accc6ff97d6ab41a88deae924f1d1c97cf30e60879d0bff5688a8e25


dfthcsos/angle_k_1.kmb: angle_k_1.kmb: SHA-256 a04b6d0d57ce851b123af265066aaab2ab33353ea049d26acd534727466d72b6


dfthcsos/angle_k_2.kmb: angle_k_2.kmb: SHA-256 5c1b8987bc3fa1a4b8607e470f8c8f1392c22e7a7dbcb228f0ce9ec986e18058


dfthcsos/angle_k_3.kmb: angle_k_3.kmb: SHA-256 16ac67cdcaf3e2c6abdd98c4138ff1ae8a56cfc4ae15622bcdb3d3a3ba1e22cf


dfthcsos/angle_k_4.kmb: angle_k_4.kmb: SHA-256 d21cdd4c7e94196654aab15ebcabfe10ce8d343151a5e8dc3672e76684b6628d


...

dqi.kmb: 

dqi.kmb: SHA-256 529b075009f7758abeda83a63407f0c1feeb51dd70631ae4335dfd28721b7e08


invsqrt.kmb: invsqrt.kmb: SHA-256 75e8a0e2341bb3e8f8ded1ec1d53e24437410c69e887da9daf7d49bfcc28c2dd


submission.zip: SHA-256 b6709a4692802091f169332630ef54f0c0d9ffe3a6bcd26179e278c2189975ba


submission.zip: 91399451 bytes; SHA-256 b6709a4692802091f169332630ef54f0c0d9ffe3a6bcd26179e278c2189975ba
Verification: full ; Kaggle hidden-score equivalence is not asserted.


## Licence and attribution

The notebook's Praxen-authored code is released under the **Apache License, Version 2.0**.
The full licence is included in the source cell and at [https://github.com/clduab11/quantum-hackathon/blob/c8bbfbde18bb27f4f89aac90f4b62437877b60f4/LICENSE](https://github.com/clduab11/quantum-hackathon/blob/c8bbfbde18bb27f4f89aac90f4b62437877b60f4/LICENSE).
Ported code retains its original notices. This statement does not grant new rights to
third-party material. The invsqrt coefficient table is regenerated with dataset-adapted
calibration; **calibration rights remain unresolved**, as recorded in the repository.
The board authorized this bundle while retaining that record. Publication review must
address that existing limitation; this build does not resolve it.

The following are the repository's complete notices, reproduced verbatim at the pinned
commit. Some describe other experiments or reference implementations: those are identified
in the notices and are not claims that this notebook uses those inputs.

- The invsqrt generator under tasks/invsqrt/ is a port, not independent clean-room authorship. The module map is `tasks/invsqrt/PROVENANCE.md`. The code parent is "Quantum Circuits | Signed-Digit Memory" v3 by Artem1410 (anazemcev / ViberTeam), https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory?scriptVersionId=361502374 (v3), Apache License 2.0 as declared in that version's own source (SPDX header in cell 5, licence text and NOTICE in cell 20; PRA-218). Source snapshot SHA-256 79f6373302cb916a88f65a1b142024d4d2d0b14e41eb58bef3152ebd45ffd0ca, equal to the public version-pinned download. That notebook retains upstream notices for Sumit Chongder, Rudra Bhaskar, Alex Chilton, Kozykorpesh, and Kirill Tymchishin. Those notices are kept in tasks/invsqrt/signed_digit.py. Sumit run 361357149 is v1 (public version metadata, PRA-218); Dmitrii Gluzdov v6 is credited second-hand. Their originals are not loaded. Whether a later Sumit v2 exists upstream stays with PRA-205.
- The 90-qubit Gray curvature schedule in tasks/invsqrt/schedule90.py is a port of Kozykorpesh, "Quantum AI LB score", V3 run 361677445, https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445 , Apache License 2.0, as that method is used by the attached Apache-2.0 notebooks. The pruned shift is the Dmitrii Gluzdov V6 boundary rule that schedule names (run 361592338). His notebook is not an input. The two Davio reload orders are method parameters from anazemcev, "Quantum Circuits | Source & Exact Compilation" v4, Apache License 2.0, attachment SHA-256 4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb.
- The affine Toffoli/CSWAP rewrite in tasks/invsqrt/affine.py is a port of Kozykorpesh, "Quantum AI complete-domain affine compilation" v2, Apache License 2.0, attachment SHA-256 81befe1e69ee286271fa2eb8acf4bb760a727853bf966945318d918482b1ab24, and the measurement-uncompute rule from anazemcev v4. The peephole in tasks/invsqrt/peephole.py is a port of that compilation's involution pass.
- Rotation calibration in tasks/invsqrt/calibrate.py is our scipy linprog search. The slack-maximising drop is credited to LightningV08 and to Mizero Lucky Gall. Mizero v1 has no notebook-wide licence (PRA-194 public-notebooks; PRA-199 item 2) and is reference-only: it is not loaded and it is not a shipping parent. Alex Chilton v8 is the same: credited where the signed-digit parent retains his notice, not loaded, not a licence parent. No calibration file is an input.
- Coefficient triples are produced by tasks/invsqrt/fit_coefficients.py from the mathematical target (Lawson minimax, then an integer search whose start is the rounded Lawson coefficient). Provenance of that table is `regenerated; dataset-adapted calibration; licence unresolved`. The calibration is adapted from the dataset's `code/domain_tables.py` (SHA-256 59cf4d98899433dcc49d9ce512293be20d33da5334211c20e35323555994a4fb); the output equals its `code/coefficients.json` (SHA-256 98bd211f22b158bf05a4d2d934ab966e5e97e54d8307b82e2ca58910c6830763) 256 of 256 rows. Source: kozykappa/quantum-ai-dataset, version 1, datasetVersionId 21136269, https://www.kaggle.com/datasets/kozykappa/quantum-ai-dataset?datasetVersionId=21136269 . Kaggle licence label: "Other (specified in description)". Rights for both files remain unresolved (D-PRA220-01). The Kaggle coefficient dataset version 21136269 and `code/coefficients.json` are not read and are not the optimiser start. Toolbox `fusion/*.npz` and `selected/invsqrt.kmb` are not inputs.
- `tasks/invsqrt/reference.py` is the competition package's reference Newton-Raphson circuit script (`verifiers/invsqrt/nr_reference_circuit.py`), Apache-2.0, Copyright Google Quantum AI authors of that package. Praxen changed the command line (`--out` writes `invsqrt.kmb`). It rebuilds the reference submission (submission-001) for G5 under `configs/gates/reference.yaml` and is not a parent of the Praxen candidate. https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/data

- `tasks/dqi/reference_circuit.py` is the competition package's reference Reed–Solomon decoder (`verifiers/dqi/dqi_reference_circuit.py`), Apache-2.0, Copyright Google Quantum AI authors of that package. https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/data
- `tasks/dqi/generate.py` is a port of a Berlekamp–Massey + Forney decoder, not an independent clean-room work. The module map is `tasks/dqi/PROVENANCE.md`. Transitive code parent: Gu, Kemeny, Yelin, and Lukin, repository `andigu/uncompute`, commit `64b5a13aa249f1360846657392e6c66817b26d0f`, Apache-2.0 (https://github.com/andigu/uncompute). Those four names are the `pyproject.toml` authors. That repository has no NOTICE file and states no copyright year; none is added here. Its `LICENSE` is copied verbatim to `tasks/dqi/bm_forney/LICENSE` (SHA-256 `cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30`). Used upstream paths: `LICENSE`; `uncompute/classical.py`; `uncompute/gadgets/linear.py`; `tasks/dqi-reed-solomon/contract.json` (field modulus changed from 4331 to 4179); `tasks/dqi-reed-solomon/circuits/discovered/constants.json` keys `bm_windows` and `psi_windows` only; `tasks/dqi-reed-solomon/circuits/discovered/circuit.py`; `tasks/dqi-reed-solomon/circuits/discovered/gen_constants.py` (formula section). Not used: `search.c`, `README.md`, `profile.pdf`, `circuit.ops.xz`, `liveness.json`, and the paper text and figures. Bibliographic citation only: Gu, Kemeny, Yelin, and Lukin, arXiv:2610.22367. Immediate code parent of the kickmix schedule: the identical `src/gen_dqi_v6` embed in Artem1410 (anazemcev / ViberTeam), "Quantum Circuits | Source & Exact Compilation" v4, Apache-2.0, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation (attachment SHA-256 `4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb`) and in Kozykorpesh (kozykappa), "Quantum AI — Complete-Domain Affine Compilation" v2, Apache-2.0, https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation (attachment SHA-256 `81befe1e69ee286271fa2eb8acf4bb760a727853bf966945318d918482b1ab24`). That embed attributes the memory schedule to Dmitrii Gluzdov's V6 schedule; his notebook and `femoco_public_words.npz` are not inputs. The shipped borrow route is the notebooks' affine loan. Multiplication tables are regenerated for modulus 4179. Zenodo 10.5281/zenodo.17301475 is not an input. No k256.dev circuit or numeric target is an input.
- anazemcev v4 cell-45: Retained opening notice from cell 45 of "Quantum Circuits | Source & Exact Compilation" v4, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation , attachment SHA-256 4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb: "Quantum Circuits resource labs (2026). Original experiments, explanations and added implementations: anazemcev / ViberTeam. Released under the Apache License, Version 2.0, subject to upstream notices." The directly copied src/gen_dqi_v6 embed supplies tasks/dqi/bm_forney/; tasks/dqi/generate.py adapts its driver and streamed sweep. The existing Gu notice and module map above remain applicable. The following credits are inherited through that embed, not claims that Praxen directly copied those earlier notebooks. Unrelated chemistry, numerical dataset and compiled-input claims in the original NOTICE do not describe inputs to this DQI port.
- Gluzdov field/adapter: Cell-45 attribution (cell 45 of the hash-pinned anazemcev v4 notebook above), retained for tasks/dqi/bm_forney/gu_formula_derivation.py, port_runtime.py and gu_port.py: "Dmitrii Gluzdov: competition-field formula derivation, Kickmix adapter, record-reading phase sweep and readable source-built portfolio. Apache-2.0. Adapted source V3, run 361529794: https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361529794 . The separately scored V1, run 361509836, returned 6.12847 for submission 57567302. V1 and V3 contain identical embedded generator modules. https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361509836 ." Version identity and scores in this passage are the immediate parent's historical attribution, not newly verified measurements. Our field tables are regenerated for modulus 4179; port_runtime.py's import and field checks are modified as recorded in the module map.
- Kozykorpesh symbolic lowering: Inherited through anazemcev v4 cell 45: "Kozykorpesh V3/run361677445, Apache2.0: https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445 ." The applicable DQI credit is "symbolic DQI lowering, and the source context for the portable adapters used in V4." It maps to tasks/dqi/bm_forney/symbolic_runtime.py and its use by tasks/dqi/generate.py. This is the transitive lowering lineage; the identical immediate embed in Kozykorpesh's Complete-Domain Affine Compilation v2 remains credited above. The original passage's Gray-curvature, signed-Givens, difference-bank and scratch-pebbling credits concern other tasks and are not DQI input claims.
- Gluzdov V6: Inherited through anazemcev v4 cell 45: "Dmitrii Gluzdov V6/run361592338, Apache2.0: https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361592338 . Compact inverse-norm schedules, template-word loans and streamed sweep, building on the credited Gu decoder and earlier competition-field port." This schedule lineage maps to tasks/dqi/bm_forney/round2_gadgets.py, borrow.py and gu_port.py, and the streamed-sweep integration in tasks/dqi/generate.py. The directly used source is the immediate notebook embed; Gluzdov's notebook and FeMoCo numerical dataset are not inputs. Our borrow.py is the modified affine-only extract described in the module map.

- PRA-232 module provenance map: `tasks/dfthcsos/qrom.py` is a Praxen reimplementation of truncated algebraic-normal-form (ANF) loading, banked low-address differences, measurement-based ANF erasure and selective ancestor pebbling. Method-only notebook parents (read from the SHA-256-checked source attachments on PRA-194; never executed): kozykappa, "Quantum AI — Complete-Domain Affine Compilation", v2, https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation, Apache-2.0, source SHA-256 81befe1e69ee286271fa2eb8acf4bb760a727853bf966945318d918482b1ab24 (six traversal wires, `gauge_qrom.py`); anazemcev / ViberTeam, "Quantum Circuits — Source & Exact Compilation", v4, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation, Apache-2.0, SHA-256 4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb (`banked_lab.py`, exact ANF phase repair); magmacot, "Quantum Circuits — clean workspace reuse", v2, https://www.kaggle.com/code/magmacot/quantum-circuits-clean-workspace-reuse, Apache-2.0, SHA-256 f3f393b5c4958f7288f69f68cd7762a62bfcc697d94d7d19607e67ff7ed20d9a (pebbling at prefix degree 4). The banked method's upstream attribution includes Kozykorpesh / kozykappa, https://www.kaggle.com/code/kozykappa/quantum-ai-compact-arithmetic-banked-qrom?scriptVersionId=361433617, and Sumit Chongder, https://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149, read through Kirill Tymchishin's QROM + Tower42 v10; retained as credited by these parents, not independently fetched or executed.
- PRA-232 precision experiment: ahmedberatozer, "Quantum AI — DF-THC Group Compression", v1, https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression, Apache-2.0 notebook code, source SHA-256 b221781e9bf8c1c8da8c5251c6556b294ed07def5e57f5a1ea038addbf7a57df, motivates testing 10-bit angles. `qrom.quantize_words` uses nearest-even rescaling of Google's baseline integer words, not that notebook's trained factors or quantization optimizer. No claim is made that its physical result transfers to Google's baseline.
- `tasks/dfthcsos/circuit.py`, `experiment.py` and PRA-232 tests are Praxen orchestration and verification code. The existing `generate.py` supplies Google's attributed batch-XOR schedule and pinned baseline input. `circuit_checks.py` calls Google's unchanged verifier math/resource model; it reads the circuit register width and reports static cost estimates. The only numerical input is Google's Apache-2.0 `sample_submission/dfthcsos/submission.npz`, SHA-256 14d5cc25eb3bf82a9ccb87c811db28d127bef6d28697dd190dc1e875de3bfe69. No `femoco_public_words.npz`, `dfthc_group_32_60_30.npz`, toolbox selected circuits, chemistry directory or k256 artifact was loaded. Signed-gauge factor changes and factor refitting are not implemented here.
- `tasks/dfthcsos/generate.py` ports the package's ordinary-QROM ANGLE builder (`verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py`), Apache-2.0, same authors and URL. The factorization bytes are the package's `sample_submission/dfthcsos/submission.npz` and are not committed.
- PRA-198 uses Google's Apache-2.0 starter kit v4, https://www.kaggle.com/datasets/sohier/verifiable-circuit-compilation-starter-kit/versions/4 (package author Guang Hao Low), as separately declared code and input parents. Original downloaded files and notices are retained unchanged; exact baseline, integral, fixed-shift, encoder and verifier pins are in `ledger/manifests/inputs/pra198-google-v4.json`. Outputs are labelled "own-generated, derived from Google baseline".
- D(w)-penalized structure refit and norm-ranked group pruning: Ahmed Berat Ozer, https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression, v1 described in PRA-194 and v2 source inspected on 2026-10-25, Apache-2.0 (`src/dfthc_train.py`). PRA-198 reimplements these methods with analytic NumPy gradients and SciPy CPU L-BFGS-B. No public factor data, words, derived target, warm start or checkpoint is used.
- Effective-norm / Gram-gauge and banked / six-wire QROM method context: Kozykappa, Complete-Domain Affine Compilation v2, https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation; anazemcev, Source & Exact Compilation v4, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation; Apache-2.0 per PRA-194 / PRA-199. PRA-198 ports no Gram-gauge or banked-QROM code; its cost model is ordinary QROM. The module provenance map is `tasks/dfthcsos/FACTOR_PIPELINE.md`.